# SIH26158 — Single-Pass Drone Video → Georeferenced 3D Model

Converts a single-pass drone video + GPS/flight metadata into a georeferenced,
metric 3D model, with every stage shown live in an in-notebook dashboard.

**Run All** to execute end to end. See `RUN_ON_KAGGLE.md` (in the repo this
notebook was generated from) for upload/dataset/settings instructions.


In [ ]:
# ============================== CONFIG ======================================
# MODE: "QUICK" = first ~90s / ~120 keyframes, target <5 min end-to-end on
#       2x T4 (excluding first-time installs). "FULL" = whole video, target
#       <15 min for a 10-min video.
MODE = "QUICK"

# BACKBONE: "A" (GeoFF3D+SLRF — NOT VIABLE, no published checkpoint exists;
# selecting it raises a clear error rather than pretending to work — see
# PHASE0_NOTES.md), "B" (Pi3X, loaded directly, no SLRF), "C" (MapAnything
# Apache, recommended default — safest install, no torch/CUDA pin).
BACKBONE = "C"

# PRIOR_MODE: "RGB" | "C" (+intrinsics) | "P" (+poses) | "CP" (+both) | "AUTO"
# (resolves to CP when a fine-tuned checkpoint is detected + GPS is present,
# else C; forced to RGB automatically with no GPS regardless of this setting
# — pose/intrinsics priors without real GPS would be fabricated garbage).
PRIOR_MODE = "AUTO"

# Use a fine-tuned UAVFF3D checkpoint (github.com/yanxian-ll/UAVFF3D) if one
# is auto-detected in /kaggle/input (see RUN_ON_KAGGLE.md for how to attach
# it as a dataset). Falls back to the stock pretrained backbone otherwise.
USE_FINETUNED_CHECKPOINT = True

# Off by default per the task spec: a viser server with a public share URL
# for a full-resolution external live-3D view. Never blocks/crashes the
# pipeline if it fails to start (falls back to cloudflared, then just skips).
LIVE_3D_EXTERNAL = False

INPUT_ROOT = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working/outputs"
CACHE_DIR = "/kaggle/working/cache"

print(f"MODE={MODE}  BACKBONE={BACKBONE}  PRIOR_MODE={PRIOR_MODE}  USE_FINETUNED_CHECKPOINT={USE_FINETUNED_CHECKPOINT}")


In [ ]:
# %%writefile (used by every cell below this one) does not create parent
# directories on its own, so create every directory the module-source cells
# write into, up front, before any of them run.
from pathlib import Path

for _d in [
    "sih3d",
    "sih3d/vendor",
    "sih3d/vendor/dinov2",
    "sih3d/vendor/dinov2/hub",
    "sih3d/vendor/dinov2/layers",
    "sih3d/vendor/dinov2/models",
    "sih3d/vendor/dinov2/utils",
    "sih3d/vendor/pi3",
    "sih3d/vendor/pi3/layers",
    "sih3d/vendor/pi3/models",
]:
    Path(_d).mkdir(parents=True, exist_ok=True)
print("sih3d/ package directories ready")


## Module source (`sih3d/`) — generated from the actual source files at build time; collapsed by default, click to expand if you need to inspect it


In [ ]:
%%writefile sih3d/__init__.py
"""sih3d: single-pass drone video -> georeferenced 3D model pipeline (SIH26158)."""

__version__ = "0.1.0"


In [ ]:
%%writefile sih3d/align.py
"""Chunk-to-world alignment: gravity-fixed 4-DoF (GPS) + Sim(3)-between-chunks (no-GPS).

Task requirement (critical): a single-pass flight is a near-straight line, so plain
Sim(3) registration to GPS cannot constrain roll around the flight axis — the GPS
track alone gives no leverage on that rotation. So when GPS is available we NEVER fit
a full Sim(3) transform straight to GPS. Instead:

  1. Fix gravity first (IMU/gimbal attitude when available, else RANSAC on the
     chunk's own points for the dominant ground-plane normal, cross-checked against
     the GPS altitude trend).
  2. With roll/pitch now fixed by gravity, solve only the remaining 4 degrees of
     freedom: scale + yaw (rotation about the now-known vertical axis) + translation.
     This mirrors GeoFF3D's `scale_yaw_translation` alignment mode conceptually, but
     is our own implementation (SLRF's version is architecturally tied to the GeoFF3D
     model we aren't running — see PHASE0_NOTES.md §3).
  3. A lightweight pose-graph refinement then ties adjacent chunks together via their
     overlapping cameras, blended with robust (Huber) GPS unary factors, weighting
     RTK-sourced GPS much higher than consumer GPS when the caller says so.

No-GPS mode: when no telemetry exists at all (backbone auto-selected to C/RGB in
backbone.py), 4-DoF-to-GPS alignment is impossible by definition. We fall back to a
full Sim(3) fit between each chunk and the previous one, using their overlapping
camera centers as point correspondences (our chunking always overlaps consecutive
chunks). This produces a single continuous, metrically-plausible-but-unanchored
model. Every caller of this fallback path MUST propagate `georeferenced=False`
downstream (report.py/export.py/viewer.html all check `ChunkAlignment.georeferenced`).
"""

from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np

from .events import EventBus


@dataclass
class GravityEstimate:
    up_local: np.ndarray  # (3,) unit vector: chunk-local "up", in the backbone's own gauge
    source: str            # "imu" | "gimbal" | "ransac_ground_plane"
    confidence: float = 1.0


@dataclass
class ChunkAlignment:
    scale: float
    R: np.ndarray               # (3,3) rotation, chunk-local -> target frame
    t: np.ndarray                # (3,) translation, chunk-local -> target frame
    mode: str                    # "4dof_gps" | "sim3_chunk_overlap"
    georeferenced: bool
    rmse_m: float | None = None  # residual RMSE of the fit, meters
    gravity_source: str | None = None
    notes: list[str] = field(default_factory=list)

    def apply(self, points: np.ndarray) -> np.ndarray:
        """points: (...,3) chunk-local -> target frame."""
        shape = points.shape
        flat = points.reshape(-1, 3)
        out = (self.scale * (self.R @ flat.T)).T + self.t
        return out.reshape(shape)


# ---------------------------------------------------------------------------
# Gravity estimation
# ---------------------------------------------------------------------------

def estimate_gravity_imu(attitude_deg: tuple[float, float, float] | None) -> GravityEstimate | None:
    """attitude_deg = (yaw, pitch, roll) of the gimbal/IMU for chunk's reference
    camera (camera 0), degrees. Returns world-up expressed in that camera's local
    frame by inverting the known world->camera attitude rotation (ZYX gimbal
    convention: yaw about world z, then pitch about the new y, then roll about the
    new x)."""
    if attitude_deg is None or any(a is None for a in attitude_deg):
        return None
    yaw, pitch, roll = (np.radians(a) for a in attitude_deg)
    cy, sy = np.cos(yaw), np.sin(yaw)
    cp, sp = np.cos(pitch), np.sin(pitch)
    cr, sr = np.cos(roll), np.sin(roll)
    Rz = np.array([[cy, -sy, 0.0], [sy, cy, 0.0], [0.0, 0.0, 1.0]])
    Ry = np.array([[cp, 0.0, sp], [0.0, 1.0, 0.0], [-sp, 0.0, cp]])
    Rx = np.array([[1.0, 0.0, 0.0], [0.0, cr, -sr], [0.0, sr, cr]])
    R_world_to_cam = Rx @ Ry @ Rz
    up_in_cam = R_world_to_cam @ np.array([0.0, 0.0, 1.0])
    norm = np.linalg.norm(up_in_cam)
    if norm < 1e-9:
        return None
    return GravityEstimate(up_local=up_in_cam / norm, source="imu")


def estimate_gravity_ransac(
    points: np.ndarray,
    n_iters: int = 300,
    dist_thresh: float = 0.05,
    min_inlier_frac: float = 0.08,
    rng: np.random.Generator | None = None,
) -> GravityEstimate | None:
    """RANSAC-fit the dominant plane in `points` (N,3), treat its normal as
    chunk-local "up". No sign disambiguation is attempted here (see
    `disambiguate_gravity_sign` — needs the GPS altitude trend, not available
    to this pure-geometry function)."""
    pts = np.asarray(points).reshape(-1, 3)
    pts = pts[np.isfinite(pts).all(axis=1)]
    n = len(pts)
    if n < 50:
        return None
    rng = rng or np.random.default_rng(0)
    best_inliers = -1
    best_normal = None
    for _ in range(n_iters):
        idx = rng.choice(n, size=3, replace=False)
        p0, p1, p2 = pts[idx]
        normal = np.cross(p1 - p0, p2 - p0)
        norm = np.linalg.norm(normal)
        if norm < 1e-9:
            continue
        normal = normal / norm
        d = -np.dot(normal, p0)
        dist = np.abs(pts @ normal + d)
        inliers = int((dist < dist_thresh).sum())
        if inliers > best_inliers:
            best_inliers, best_normal = inliers, normal
    if best_normal is None or best_inliers < min_inlier_frac * n:
        return None
    return GravityEstimate(up_local=best_normal, source="ransac_ground_plane", confidence=best_inliers / n)


def disambiguate_gravity_sign(
    gravity: GravityEstimate, cam_centers_local: np.ndarray, gps_altitude_trend: np.ndarray,
) -> GravityEstimate:
    """Flip `up_local` if needed so it agrees with the GPS altitude trend: cameras
    with a higher projection onto `up_local` must correspond to higher GPS altitude.
    If the correlation is weak/negative even after the best sign choice, halve
    confidence and note it (caller should consider falling back to RANSAC-only /
    no gravity fix at all for this chunk)."""
    if len(cam_centers_local) < 2 or len(gps_altitude_trend) != len(cam_centers_local):
        return gravity
    proj = cam_centers_local @ gravity.up_local
    corr = np.corrcoef(proj, gps_altitude_trend)[0, 1]
    if np.isnan(corr):
        return gravity
    if corr < 0:
        gravity = GravityEstimate(up_local=-gravity.up_local, source=gravity.source, confidence=gravity.confidence)
        corr = -corr
    if corr < 0.3:
        gravity.confidence *= 0.5
    return gravity


def gravity_rotation_matrix(up_local: np.ndarray, world_up: np.ndarray = np.array([0.0, 0.0, 1.0])) -> np.ndarray:
    """Rotation R such that R @ up_local == world_up (Rodrigues' formula)."""
    a = up_local / np.linalg.norm(up_local)
    b = world_up / np.linalg.norm(world_up)
    v = np.cross(a, b)
    s = np.linalg.norm(v)
    c = np.dot(a, b)
    if s < 1e-9:
        return np.eye(3) if c > 0 else _rotation_180_about_any_perpendicular(a)
    vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + vx + vx @ vx * ((1 - c) / (s ** 2))


def _rotation_180_about_any_perpendicular(a: np.ndarray) -> np.ndarray:
    helper = np.array([1.0, 0.0, 0.0]) if abs(a[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
    axis = np.cross(a, helper)
    axis /= np.linalg.norm(axis)
    x, y, z = axis
    return np.array([
        [2 * x * x - 1, 2 * x * y, 2 * x * z],
        [2 * x * y, 2 * y * y - 1, 2 * y * z],
        [2 * x * z, 2 * y * z, 2 * z * z - 1],
    ])


# ---------------------------------------------------------------------------
# 4-DoF (scale + yaw + translation) fit, gravity already fixed
# ---------------------------------------------------------------------------

def solve_4dof(
    src_grav: np.ndarray,   # (M,3) source points, already gravity-rotated (z ~ up)
    dst: np.ndarray,        # (M,3) target points (GPS ENU)
    weights: np.ndarray | None = None,
    huber_delta: float = 2.0,
    n_irls_iters: int = 5,
) -> tuple[float, np.ndarray, np.ndarray, float]:
    """Solve scale s, yaw-only rotation R (about z), translation t minimizing
    weighted robust residuals ||s*R@src_grav[i] + t - dst[i]||. Rotation about z
    doesn't touch the z-component, so we jointly solve the xy similarity (Umeyama
    closed-form, scale + rotation) and z as a separate 1D scale+offset that must
    share the same scale — done via IRLS reweighting the xy Umeyama fit toward
    points whose z also agrees well, then closing z with the shared scale.

    Returns (scale, R (3x3), t (3,), rmse_m).
    """
    src_grav = np.asarray(src_grav, dtype=np.float64)
    dst = np.asarray(dst, dtype=np.float64)
    m = len(src_grav)
    if m < 2:
        raise ValueError("solve_4dof needs at least 2 correspondences")
    w = np.ones(m) if weights is None else np.asarray(weights, dtype=np.float64).copy()
    w = w / w.sum()

    R = np.eye(3)
    scale = 1.0
    t = np.zeros(3)

    for _ in range(n_irls_iters):
        src_c = (src_grav * w[:, None]).sum(axis=0)
        dst_c = (dst * w[:, None]).sum(axis=0)
        src_xy = src_grav[:, :2] - src_c[:2]
        dst_xy = dst[:, :2] - dst_c[:2]

        H = (src_xy * w[:, None]).T @ dst_xy
        U, S, Vt = np.linalg.svd(H)
        d = np.sign(np.linalg.det(Vt.T @ U.T))
        D = np.diag([1.0, d])
        R2 = Vt.T @ D @ U.T  # 2x2 rotation, src_xy -> dst_xy

        var_src_xy = (w * (src_xy ** 2).sum(axis=1)).sum()
        scale = float((S[0] + d * S[1]) / var_src_xy) if var_src_xy > 1e-12 else 1.0
        scale = max(scale, 1e-6)

        R = np.eye(3)
        R[:2, :2] = R2

        t_xy = dst_c[:2] - scale * (R2 @ src_c[:2])
        t_z = dst_c[2] - scale * src_c[2]  # R about z leaves z unchanged
        t = np.array([t_xy[0], t_xy[1], t_z])

        pred = scale * (R @ src_grav.T).T + t
        resid = np.linalg.norm(pred - dst, axis=1)
        w = np.where(resid <= huber_delta, 1.0, huber_delta / np.maximum(resid, 1e-9))
        w = w / w.sum()

    pred = scale * (R @ src_grav.T).T + t
    rmse = float(np.sqrt(np.mean(np.sum((pred - dst) ** 2, axis=1))))
    return scale, R, t, rmse


def align_chunk_4dof(
    cam_centers_local: np.ndarray,
    cam_centers_gps_enu: np.ndarray,
    gravity: GravityEstimate,
    gps_weights: np.ndarray | None = None,
) -> ChunkAlignment:
    R_grav = gravity_rotation_matrix(gravity.up_local)
    src_grav = (R_grav @ np.asarray(cam_centers_local).T).T
    scale, R_yaw, t, rmse = solve_4dof(src_grav, np.asarray(cam_centers_gps_enu), weights=gps_weights)
    R_total = R_yaw @ R_grav
    return ChunkAlignment(
        scale=scale, R=R_total, t=t, mode="4dof_gps", georeferenced=True,
        rmse_m=rmse, gravity_source=gravity.source,
        notes=[f"gravity confidence={gravity.confidence:.2f}"] if gravity.confidence < 0.9 else [],
    )


# ---------------------------------------------------------------------------
# No-GPS fallback: Sim(3) between overlapping chunks
# ---------------------------------------------------------------------------

def solve_sim3(src: np.ndarray, dst: np.ndarray, weights: np.ndarray | None = None) -> tuple[float, np.ndarray, np.ndarray, float]:
    """Full Umeyama Sim(3): scale, 3x3 rotation, translation minimizing
    weighted ||s*R@src[i] + t - dst[i]||^2. Used only when GPS anchoring is
    unavailable (no-GPS mode) — chunk k is registered to chunk k-1 via their
    shared overlapping camera centers, not to any absolute frame."""
    src = np.asarray(src, dtype=np.float64)
    dst = np.asarray(dst, dtype=np.float64)
    m = len(src)
    if m < 3:
        raise ValueError("solve_sim3 needs at least 3 correspondences")
    w = np.ones(m) if weights is None else np.asarray(weights, dtype=np.float64)
    w = w / w.sum()

    src_c = (src * w[:, None]).sum(axis=0)
    dst_c = (dst * w[:, None]).sum(axis=0)
    src0 = src - src_c
    dst0 = dst - dst_c

    H = (src0 * w[:, None]).T @ dst0
    U, S, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    D = np.diag([1.0, 1.0, d])
    R = Vt.T @ D @ U.T

    var_src = (w * (src0 ** 2).sum(axis=1)).sum()
    scale = float((S * np.array([1.0, 1.0, d])).sum() / var_src) if var_src > 1e-12 else 1.0
    scale = max(scale, 1e-6)

    t = dst_c - scale * (R @ src_c)
    pred = scale * (R @ src.T).T + t
    rmse = float(np.sqrt(np.mean(np.sum((pred - dst) ** 2, axis=1))))
    return scale, R, t, rmse


def align_chunk_to_previous(
    overlap_cam_centers_local: np.ndarray,      # this chunk's local coords, for cameras shared with the previous chunk
    overlap_cam_centers_prev_world: np.ndarray,  # those same cameras' already-resolved world coords
) -> ChunkAlignment:
    scale, R, t, rmse = solve_sim3(overlap_cam_centers_local, overlap_cam_centers_prev_world)
    return ChunkAlignment(
        scale=scale, R=R, t=t, mode="sim3_chunk_overlap", georeferenced=False, rmse_m=rmse,
        notes=["no GPS: aligned to previous chunk's overlap only, not to an absolute frame"],
    )


def first_chunk_identity(georeferenced: bool) -> ChunkAlignment:
    """The very first chunk (no previous chunk to register against, and in
    no-GPS mode no GPS to register against either) defines the world frame
    as its own local frame."""
    return ChunkAlignment(
        scale=1.0, R=np.eye(3), t=np.zeros(3), mode="identity_seed",
        georeferenced=georeferenced, rmse_m=0.0,
        notes=[] if georeferenced else ["no GPS: world frame is this chunk's own local frame, not georeferenced"],
    )


# ---------------------------------------------------------------------------
# Lightweight global pose-graph refinement across chunks
# ---------------------------------------------------------------------------

@dataclass
class OverlapConstraint:
    chunk_a: int
    chunk_b: int
    cam_centers_a_local: np.ndarray  # (K,3), chunk_a-local coords of the shared cameras
    cam_centers_b_local: np.ndarray  # (K,3), chunk_b-local coords of the same cameras


@dataclass
class GpsFactor:
    chunk: int
    cam_center_local: np.ndarray  # (3,)
    gps_enu: np.ndarray            # (3,)
    weight: float = 1.0            # higher for RTK/PPK


def refine_pose_graph(
    initial: list[ChunkAlignment],
    overlaps: list[OverlapConstraint],
    gps_factors: list[GpsFactor],
    bus: EventBus,
    huber_delta: float = 2.0,
    max_nfev: int = 200,
) -> list[ChunkAlignment]:
    """Jointly refine each chunk's (scale, yaw, translation) to minimize robust
    residuals from both overlap constraints (chunk-to-chunk agreement on shared
    cameras) and GPS unary factors (chunk-to-world agreement), via
    scipy.optimize.least_squares with a Huber loss. This is a lightweight
    substitute for a full SE3 pose-graph optimizer (g2o/GTSAM are not installed
    and are a real Kaggle install risk we're avoiding) — parameterized as
    (log_scale, yaw, tx, ty, tz) per chunk, i.e. rotation is yaw-only. Chunks
    whose original alignment mode was "sim3_chunk_overlap" (no-GPS) keep a
    yaw-only reparameterization too for simplicity; their absolute rotation is
    already free (no GPS to anchor it), so this only smooths relative
    consistency between chunks, which is the only thing that matters when
    nothing is georeferenced anyway.
    """
    if not initial:
        return initial
    if len(overlaps) == 0 and len(gps_factors) == 0:
        return initial

    from scipy.optimize import least_squares

    n = len(initial)

    def r_of(R: np.ndarray) -> float:
        return float(np.arctan2(R[1, 0], R[0, 0]))

    def R_of(yaw: float) -> np.ndarray:
        c, s = np.cos(yaw), np.sin(yaw)
        return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])

    x0 = np.zeros(5 * n)
    for i, ca in enumerate(initial):
        x0[5 * i + 0] = np.log(max(ca.scale, 1e-6))
        x0[5 * i + 1] = r_of(ca.R)
        x0[5 * i + 2:5 * i + 5] = ca.t

    def unpack(x, i):
        s = np.exp(x[5 * i + 0])
        yaw = x[5 * i + 1]
        t = x[5 * i + 2:5 * i + 5]
        return s, R_of(yaw), t

    def residuals(x):
        res = []
        for oc in overlaps:
            sa, Ra, ta = unpack(x, oc.chunk_a)
            sb, Rb, tb = unpack(x, oc.chunk_b)
            pa = sa * (Ra @ oc.cam_centers_a_local.T).T + ta
            pb = sb * (Rb @ oc.cam_centers_b_local.T).T + tb
            res.append((pa - pb).ravel())
        for gf in gps_factors:
            s, R, t = unpack(x, gf.chunk)
            pred = s * (R @ gf.cam_center_local) + t
            res.append(np.sqrt(gf.weight) * (pred - gf.gps_enu))
        return np.concatenate(res) if res else np.zeros(1)

    result = least_squares(residuals, x0, loss="huber", f_scale=huber_delta, max_nfev=max_nfev)
    bus.log(
        f"Pose graph refinement: {len(overlaps)} overlap constraints, {len(gps_factors)} GPS factors, "
        f"cost {result.cost:.3f}, {'converged' if result.success else 'did not fully converge'}"
    )

    # Recompute each chunk's own RMSE against its GPS factors using the
    # REFINED transform — the pre-refinement rmse_m on `initial` is not a
    # valid "after" value (it was carried straight through here in an
    # earlier version of this function, which is wrong: the whole point of
    # refinement is that per-chunk residuals should change).
    gps_by_chunk: dict[int, list[GpsFactor]] = {}
    for gf in gps_factors:
        gps_by_chunk.setdefault(gf.chunk, []).append(gf)

    refined = []
    for i, ca in enumerate(initial):
        s, R, t = unpack(result.x, i)
        chunk_gps = gps_by_chunk.get(i, [])
        if chunk_gps:
            preds = np.array([s * (R @ gf.cam_center_local) + t for gf in chunk_gps])
            targets = np.array([gf.gps_enu for gf in chunk_gps])
            rmse_after = float(np.sqrt(np.mean(np.sum((preds - targets) ** 2, axis=1))))
        else:
            rmse_after = ca.rmse_m  # no GPS factors for this chunk — nothing to recompute against
        refined.append(ChunkAlignment(
            scale=s, R=R, t=t, mode=ca.mode, georeferenced=ca.georeferenced,
            rmse_m=rmse_after, gravity_source=ca.gravity_source, notes=ca.notes,
        ))
    return refined


In [ ]:
%%writefile sih3d/artifacts.py
"""In-memory artifacts the pipeline accumulates as it runs, beyond what it
publishes as transient events — the per-stage notebook result cells and the
live reconstruction page both read this directly (not by replaying events,
not by re-parsing report.json) since it's already sitting in the same
process. Bounded/capped where it could otherwise grow unboundedly over a
FULL-mode run (e.g. only a few representative chunks' depth/confidence/mask
samples are kept, not every chunk's).
"""

from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np


@dataclass
class KeyframeRecord:
    frame_index: int
    timestamp_s: float
    thumbnail: np.ndarray | None  # small RGB, for the keyframe grid
    sharpness: float
    accepted: bool
    reject_reason: str | None = None


@dataclass
class ChunkAlignmentRecord:
    chunk_idx: int
    mode: str
    rmse_before_m: float | None       # this chunk's own direct-fit residual
    rmse_after_m: float | None = None  # after global pose-graph refinement


@dataclass
class ChunkGeometrySample:
    chunk_idx: int
    rgb: np.ndarray | None            # (H,W,3) uint8, one representative view
    depth: np.ndarray | None          # (H,W) float, camera-space z
    confidence: np.ndarray | None     # (H,W) float in [0,1]
    dynamic_mask: np.ndarray | None   # (H,W) bool, True = excluded (masked)


@dataclass
class RunArtifacts:
    keyframes: list[KeyframeRecord] = field(default_factory=list)
    gps_track_enu: list[tuple[float, float, float]] = field(default_factory=list)
    camera_track_enu: list[tuple[float, float, float]] = field(default_factory=list)
    collinearity_index: float | None = None
    chunk_alignments: list[ChunkAlignmentRecord] = field(default_factory=list)
    geometry_samples: list[ChunkGeometrySample] = field(default_factory=list)  # capped, see pipeline.py
    point_count: int = 0
    point_cloud_preview: np.ndarray | None = None   # (<=50k, 3) downsampled, for quick matplotlib renders
    point_cloud_preview_colors: np.ndarray | None = None
    mesh_n_vertices: int = 0
    mesh_n_faces: int = 0
    mesh_method: str = "none"
    output_paths: dict[str, str] = field(default_factory=dict)  # name -> absolute path, once export.py finishes
    georeferenced: bool = False

    MAX_GEOMETRY_SAMPLES = 6

    def add_geometry_sample(self, sample: ChunkGeometrySample) -> None:
        if len(self.geometry_samples) < self.MAX_GEOMETRY_SAMPLES:
            self.geometry_samples.append(sample)


In [ ]:
%%writefile sih3d/backbone.py
"""Geometry backbone config switch (Options A/B/C) — see PHASE0_NOTES.md.

Verified against actual source (GeoFF3D, MapAnything, UAVFF3D), not guessed:

- **A (GeoFF3D+SLRF)**: not viable. No trained GeoFF3D checkpoint is
  downloadable anywhere; training one is out of budget. `BackboneA.load()`
  raises a clear error rather than pretending to work.
- **B (Pi3X)**: loaded directly via `Pi3X.from_pretrained("yyfz233/Pi3X")`,
  vendored from GeoFF3D (`sih3d/vendor/pi3/`, Apache-2.0) with import paths
  rewritten. We do NOT go through GeoFF3D's SLRF orchestration — SLRF forces
  `load_pretrained_weights=false` and only loads weights via a raw
  `state_dict` load keyed to `Pi3XWrapper`'s `model.*`-prefixed submodule,
  a real footgun (see PHASE0_NOTES.md §6). We do our own chunking, same as C.
- **C (MapAnything Apache)**: `MapAnything.from_pretrained("facebook/map-anything-apache")`
  (pip-installed `mapanything` package, not vendored — Phase 0 confirmed a
  clean install with no torch/CUDA pin). Our own chunking.

Both B and C support loading UAVFF3D fine-tuned checkpoints
(github.com/yanxian-ll/UAVFF3D) auto-detected from /kaggle/input by
io_detect.find_checkpoints(), verified in mapanything/scripts/convert_hf_to_benchmark_checkpoint.py
to be saved as `{"model": <raw state_dict>}` — plain keys, no wrapper
prefix. `strict_load_checkpoint()` never silently drops mismatched keys
(unlike the reference benchmark's own loader, which calls
`load_state_dict(..., strict=False)`); it fails loudly with the exact
missing/unexpected key lists.
"""

from __future__ import annotations

import time
from dataclasses import dataclass
from enum import Enum
from pathlib import Path
from typing import TYPE_CHECKING, Any

import numpy as np

from .events import EventBus

if TYPE_CHECKING:
    from .io_detect import DetectedInputs


class BackboneChoice(str, Enum):
    A = "A"
    B = "B"
    C = "C"


class PriorMode(str, Enum):
    """Matches UAVFF3D's own evaluation-protocol naming (RGB/C/P/CP)."""

    RGB = "RGB"                 # images only
    INTRINSICS = "C"            # + camera intrinsics
    POSES = "P"                 # + camera poses
    INTRINSICS_POSES = "CP"     # + both


@dataclass
class ViewInput:
    image: np.ndarray                       # HWC, uint8 or float32 [0,1]
    intrinsics: np.ndarray | None = None    # 3x3, pixel coords
    camera_pose_c2w: np.ndarray | None = None  # 4x4 OpenCV cam2world, from GPS translation + attitude prior
    frame_index: int = 0
    timestamp_s: float = 0.0


@dataclass
class ChunkResult:
    points_world: np.ndarray        # (N,H,W,3)
    confidence: np.ndarray          # (N,H,W)
    camera_poses_est: np.ndarray    # (N,4,4) cam2world, as predicted by the backbone
    metric_scaling_factor: float | None = None
    backbone_used: str = ""
    prior_mode: str = ""
    checkpoint_source: str = "pretrained"   # "pretrained" | "finetuned:<filename>"
    timing_s: float = 0.0


class BackboneNotAvailableError(RuntimeError):
    """Raised when a config-selected backbone cannot actually run."""


def _to_chw_float(img: np.ndarray):
    import torch

    if img.dtype == np.uint8:
        img = img.astype(np.float32) / 255.0
    t = torch.from_numpy(np.ascontiguousarray(img)).float()
    if t.ndim == 3 and t.shape[-1] == 3:
        t = t.permute(2, 0, 1)
    return t


def strict_load_checkpoint(model, ckpt_path: Path, bus: EventBus, label: str) -> str:
    """Load a checkpoint's state dict with strict key matching.

    Never silently drops mismatched keys the way the reference SLRF/UAVFF3D
    benchmark loaders do (they call `load_state_dict(..., strict=False)` and
    only print the result). Tries the state dict as-is first; if that fails,
    tries mechanically stripping or adding a "model." prefix — the one
    key-naming variant actually observed across GeoFF3D/UAVFF3D checkpoints
    (see PHASE0_NOTES.md §6) — and logs which one worked. Raises
    BackboneNotAvailableError with the missing/unexpected key lists if none
    of these match cleanly.
    """
    import torch

    raw = torch.load(str(ckpt_path), map_location="cpu", weights_only=False)
    state = raw["model"] if isinstance(raw, dict) and "model" in raw else raw
    if not isinstance(state, dict):
        raise BackboneNotAvailableError(f"{label}: checkpoint at {ckpt_path} has no usable state dict")

    own_keys = set(model.state_dict().keys())

    def _matches(candidate: dict) -> bool:
        missing = own_keys - set(candidate.keys())
        unexpected = set(candidate.keys()) - own_keys
        return not missing and not unexpected

    if _matches(state):
        model.load_state_dict(state, strict=True)
        bus.log(f"{label}: checkpoint keys matched exactly ({ckpt_path.name})")
        return "exact"

    stripped = {k[len("model."):]: v for k, v in state.items() if k.startswith("model.")}
    if stripped and _matches(stripped):
        model.load_state_dict(stripped, strict=True)
        bus.log(f"{label}: checkpoint keys matched after stripping 'model.' prefix ({ckpt_path.name})")
        return "stripped_prefix"

    prefixed = {f"model.{k}": v for k, v in state.items()}
    if _matches(prefixed):
        model.load_state_dict(prefixed, strict=True)
        bus.log(f"{label}: checkpoint keys matched after adding 'model.' prefix ({ckpt_path.name})")
        return "added_prefix"

    ckpt_keys = set(state.keys())
    missing = sorted(own_keys - ckpt_keys)
    unexpected = sorted(ckpt_keys - own_keys)
    raise BackboneNotAvailableError(
        f"{label}: checkpoint at {ckpt_path} does not match the model architecture "
        f"(tried as-is and with 'model.' prefix stripped/added). "
        f"{len(missing)} missing keys (e.g. {missing[:5]}), "
        f"{len(unexpected)} unexpected keys (e.g. {unexpected[:5]}). "
        f"Refusing to silently load with strict=False."
    )


class GeometryBackbone:
    name: str = "?"

    def load(self) -> None:
        raise NotImplementedError

    def infer_chunk(self, views: list[ViewInput], prior_mode: PriorMode) -> ChunkResult:
        raise NotImplementedError


class BackboneA(GeometryBackbone):
    """GeoFF3D + SLRF. Not viable — see module docstring and PHASE0_NOTES.md §1/§6."""

    name = "A"

    def load(self) -> None:
        raise BackboneNotAvailableError(
            "Backbone A (GeoFF3D+SLRF) has no publicly downloadable trained checkpoint. "
            "Verified by reading the GeoFF3D source (bash_scripts/run_slrf/geoff3d.sh defaults "
            "to a checkpoint path only produced by running the two-stage training pipeline "
            "yourself — see PHASE0_NOTES.md §1). Training one from scratch is out of budget. "
            "Select BACKBONE='B' or BACKBONE='C' instead."
        )

    def infer_chunk(self, views: list[ViewInput], prior_mode: PriorMode) -> ChunkResult:
        raise BackboneNotAvailableError("Backbone A was never loaded (see load()).")


class BackboneB(GeometryBackbone):
    """Pi3X, loaded directly (no SLRF), our own chunking."""

    name = "B"

    def __init__(self, bus: EventBus, device: str = "cuda:0", checkpoint_path: Path | None = None):
        self.bus = bus
        self.device = device
        self.checkpoint_path = checkpoint_path
        self.model = None
        self.dtype = None
        self.checkpoint_source = "pretrained"

    def load(self) -> None:
        import torch

        from .vendor.pi3.models.pi3x import Pi3X

        self.bus.log("Backbone B: loading Pi3X (yyfz233/Pi3X) directly, no SLRF")
        t0 = time.time()
        self.model = Pi3X.from_pretrained("yyfz233/Pi3X")
        if self.checkpoint_path is not None:
            strict_load_checkpoint(self.model, self.checkpoint_path, self.bus, "Backbone B (Pi3X)")
            self.checkpoint_source = f"finetuned:{self.checkpoint_path.name}"
        self.model.to(self.device)
        self.model.eval()
        if torch.cuda.is_available():
            major, _ = torch.cuda.get_device_capability(self.device)
            self.dtype = torch.bfloat16 if major >= 8 else torch.float16
        else:
            self.dtype = torch.float32
        self.bus.log(
            f"Backbone B: Pi3X loaded in {time.time() - t0:.1f}s "
            f"(checkpoint={self.checkpoint_source}, dtype={self.dtype})"
        )

    def infer_chunk(self, views: list[ViewInput], prior_mode: PriorMode) -> ChunkResult:
        import torch

        assert self.model is not None, "call load() first"
        t0 = time.time()
        device = self.device

        imgs = torch.stack([_to_chw_float(v.image) for v in views], dim=0).unsqueeze(0).to(device)

        use_intrinsics = prior_mode in (PriorMode.INTRINSICS, PriorMode.INTRINSICS_POSES) and all(
            v.intrinsics is not None for v in views
        )
        use_poses = prior_mode in (PriorMode.POSES, PriorMode.INTRINSICS_POSES) and all(
            v.camera_pose_c2w is not None for v in views
        )

        intrinsics = None
        if use_intrinsics:
            intrinsics = torch.stack(
                [torch.from_numpy(v.intrinsics).float() for v in views], dim=0
            ).unsqueeze(0).to(device)

        poses = None
        pose_mask = None
        if use_poses:
            poses = torch.stack(
                [torch.from_numpy(v.camera_pose_c2w).float() for v in views], dim=0
            ).unsqueeze(0).to(device)
            pose_mask = torch.ones(1, len(views), dtype=torch.bool, device=device)

        with torch.autocast(
            device_type="cuda" if str(device).startswith("cuda") else "cpu",
            dtype=self.dtype, enabled=str(device).startswith("cuda"),
        ):
            with torch.no_grad():
                out = self.model(
                    imgs=imgs, intrinsics=intrinsics, poses=poses, pose_mask=pose_mask,
                    with_prior=True, overall_prob=1.0,
                    ray_dirs_prob=1.0 if use_intrinsics else 0.0,
                    cam_prob=1.0 if use_poses else 0.0,
                )
        if str(device).startswith("cuda"):
            torch.cuda.synchronize()

        points = out["points"][0].float().cpu().numpy()
        conf = out["conf"][0, ..., 0].float().cpu().numpy()
        cam_poses = out["camera_poses"][0].float().cpu().numpy()
        metric = float(out["metric"][0].item())

        return ChunkResult(
            points_world=points, confidence=conf, camera_poses_est=cam_poses,
            metric_scaling_factor=metric, backbone_used="B", prior_mode=prior_mode.value,
            checkpoint_source=self.checkpoint_source, timing_s=time.time() - t0,
        )


class BackboneC(GeometryBackbone):
    """MapAnything (Apache), our own chunking."""

    name = "C"

    def __init__(
        self, bus: EventBus, device: str = "cuda:0", checkpoint_path: Path | None = None,
        hf_repo: str = "facebook/map-anything-apache",
    ):
        self.bus = bus
        self.device = device
        self.checkpoint_path = checkpoint_path
        self.hf_repo = hf_repo
        self.model = None
        self.checkpoint_source = "pretrained"

    def load(self) -> None:
        from mapanything.models import MapAnything

        self.bus.log(f"Backbone C: loading MapAnything from {self.hf_repo}")
        t0 = time.time()
        self.model = MapAnything.from_pretrained(self.hf_repo)
        if self.checkpoint_path is not None:
            strict_load_checkpoint(self.model, self.checkpoint_path, self.bus, "Backbone C (MapAnything)")
            self.checkpoint_source = f"finetuned:{self.checkpoint_path.name}"
        self.model.to(self.device)
        self.model.eval()
        self.bus.log(
            f"Backbone C: MapAnything loaded in {time.time() - t0:.1f}s "
            f"(checkpoint={self.checkpoint_source})"
        )

    def infer_chunk(self, views: list[ViewInput], prior_mode: PriorMode) -> ChunkResult:
        import torch

        assert self.model is not None, "call load() first"
        t0 = time.time()
        device = self.device

        use_intrinsics = prior_mode in (PriorMode.INTRINSICS, PriorMode.INTRINSICS_POSES)
        use_poses = prior_mode in (PriorMode.POSES, PriorMode.INTRINSICS_POSES)

        mv_views: list[dict[str, Any]] = []
        for v in views:
            img = _to_chw_float(v.image).unsqueeze(0).to(device)
            entry: dict[str, Any] = {"img": img, "data_norm_type": ["dinov2"]}
            if use_intrinsics and v.intrinsics is not None:
                entry["intrinsics"] = torch.from_numpy(v.intrinsics).float().unsqueeze(0).to(device)
            if use_poses and v.camera_pose_c2w is not None:
                entry["camera_poses"] = torch.from_numpy(v.camera_pose_c2w).float().unsqueeze(0).to(device)
                entry["is_metric_scale"] = True
            mv_views.append(entry)

        with torch.no_grad():
            preds = self.model.infer(
                mv_views, memory_efficient_inference=True, use_amp=True, amp_dtype="fp16",
            )
        if str(device).startswith("cuda"):
            torch.cuda.synchronize()

        points = np.stack([p["pts3d"][0].float().cpu().numpy() for p in preds], axis=0)
        conf = np.stack([p["conf"][0].float().cpu().numpy() for p in preds], axis=0)
        cam_poses = np.stack([p["camera_poses"][0].float().cpu().numpy() for p in preds], axis=0)
        metric = (
            float(preds[0]["metric_scaling_factor"][0].item())
            if "metric_scaling_factor" in preds[0] else None
        )

        return ChunkResult(
            points_world=points, confidence=conf, camera_poses_est=cam_poses,
            metric_scaling_factor=metric, backbone_used="C", prior_mode=prior_mode.value,
            checkpoint_source=self.checkpoint_source, timing_s=time.time() - t0,
        )


_CHECKPOINT_KEY_BY_BACKBONE = {"B": "pi3x", "C": "mapanything"}


def select_backbone(
    choice: str,
    prior_mode: str,
    detected: "DetectedInputs",
    bus: EventBus,
    device: str = "cuda:0",
    use_finetuned: bool = True,
) -> tuple[GeometryBackbone, PriorMode]:
    """Resolve BACKBONE/PRIOR_MODE config, apply no-GPS auto-select, load, return.

    No-GPS mode (task spec): if no telemetry was found (sidecar file or
    embedded stream), force Backbone C with RGB-only priors — MapAnything
    predicts metric scale from images alone; feeding fabricated pose/
    intrinsics priors without real GPS would be worse than none. Chunks are
    aligned to each other only via align.py's Sim(3)-between-chunks
    fallback, and every output gets labeled "APPROXIMATE SCALE — NOT
    GEOREFERENCED" upstream in export.py/report.py/viewer.html.

    prior_mode="AUTO" resolves to CP (intrinsics+poses) when a fine-tuned
    checkpoint is present (per the UAVFF3D authors' own eval protocol, fine-
    tuning targets the CP setting) and telemetry exists, else C
    (intrinsics-only, still useful without pose priors that may hurt dense
    geometry on UAV data per the task's own note — compare via an explicit
    PRIOR_MODE instead of AUTO when benchmarking).
    """
    has_telemetry = detected.telemetry_path is not None or detected.telemetry_kind == "embedded"

    if not has_telemetry:
        if choice != "C" or prior_mode not in ("RGB", "AUTO"):
            bus.log(
                f"No telemetry detected: forcing BACKBONE=C, PRIOR_MODE=RGB (no-GPS mode). "
                f"Outputs will be APPROXIMATE SCALE — NOT GEOREFERENCED.",
                level="warn",
            )
        resolved_choice = "C"
        resolved_prior = PriorMode.RGB
        checkpoint_path = None
    else:
        resolved_choice = choice
        checkpoint_key = _CHECKPOINT_KEY_BY_BACKBONE.get(resolved_choice)
        checkpoint_path = detected.checkpoints.get(checkpoint_key) if (use_finetuned and checkpoint_key) else None
        if prior_mode == "AUTO":
            resolved_prior = PriorMode.INTRINSICS_POSES if checkpoint_path is not None else PriorMode.INTRINSICS
        else:
            resolved_prior = PriorMode(prior_mode)

    if resolved_choice == "A":
        backbone: GeometryBackbone = BackboneA()
    elif resolved_choice == "B":
        backbone = BackboneB(bus, device=device, checkpoint_path=checkpoint_path)
    elif resolved_choice == "C":
        backbone = BackboneC(bus, device=device, checkpoint_path=checkpoint_path)
    else:
        raise ValueError(f"Unknown backbone choice: {resolved_choice!r} (expected A, B, or C)")

    backbone.load()
    return backbone, resolved_prior


In [ ]:
%%writefile sih3d/bootstrap.py
"""Everything that used to be separate notebook cells (environment checks,
dependency installs, cache/input detection) now runs in ONE function, called
from the single RUN cell's own background thread — so the dashboard can
display immediately (before any of this starts) and these steps report
their progress into it instead of raw stdout/pip spam. `bootstrap.run(...)`
blocks until the whole pipeline finishes; that's expected, since it's
already running off the main thread.
"""

from __future__ import annotations

import importlib
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

from .events import EventBus, EventType
from .gpu_monitor import GpuMonitor
from .io_detect import DetectedInputs, detect_all, find_cache_dir
from .pipeline import Pipeline, PipelineConfig
from .report import ReportBuilder
from .telemetry import extract_embedded_telemetry, parse_telemetry


class BootstrapResult:
    def __init__(self):
        self.ok = False
        self.error: str | None = None
        self.pipeline: "Pipeline | None" = None
        self.detected: DetectedInputs | None = None


def _check_internet(timeout_s: float = 5.0) -> bool:
    try:
        urllib.request.urlopen("https://huggingface.co", timeout=timeout_s)
        return True
    except Exception:
        return False


def _check_gpu() -> tuple[int, list[str]]:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=15,
        )
        if out.returncode == 0 and out.stdout.strip():
            return [l.strip() for l in out.stdout.strip().splitlines()].__len__(), [
                l.strip() for l in out.stdout.strip().splitlines()
            ]
    except Exception:
        pass
    return 0, []


def _try_import(name: str) -> bool:
    try:
        importlib.import_module(name)
        return True
    except Exception:
        return False


def _pip_install_silent(
    spec: str, bus: EventBus, label: str | None = None,
    extra_args: list[str] | None = None, find_links: Path | None = None,
) -> None:
    """Captures pip's stdout/stderr (never prints it raw into the notebook
    output — the dashboard's install panel is the only place progress
    shows) and reports one INSTALL_PROGRESS event per package."""
    label = label or spec
    bus.publish(EventType.INSTALL_PROGRESS, package=label, status="installing", elapsed_s=0.0)
    t0 = time.time()
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if find_links:
        cmd += ["--find-links", str(find_links)]
    cmd += (extra_args or []) + [spec]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
        elapsed = time.time() - t0
        if result.returncode != 0:
            bus.publish(EventType.INSTALL_PROGRESS, package=label, status="failed", elapsed_s=elapsed)
            bus.log(f"pip install {label} failed: {result.stderr.strip()[-300:]}", level="warn")
        else:
            bus.publish(EventType.INSTALL_PROGRESS, package=label, status="ok", elapsed_s=elapsed)
    except Exception as e:
        bus.publish(EventType.INSTALL_PROGRESS, package=label, status="failed", elapsed_s=time.time() - t0)
        bus.log(f"pip install {label} raised {type(e).__name__}: {e}", level="warn")


def _mark_present(pkg: str, bus: EventBus) -> None:
    bus.publish(EventType.INSTALL_PROGRESS, package=pkg, status="ok", elapsed_s=0.0)


def run(
    input_root: Path, output_dir: Path, cache_dir: Path,
    mode: str, backbone_choice: str, prior_mode: str, use_finetuned: bool,
    bus: EventBus, report: ReportBuilder, result: BootstrapResult,
) -> None:
    # -- env_check -------------------------------------------------------
    bus.publish(EventType.SETUP_STAGE_START, stage="env_check")
    internet_ok = _check_internet()
    gpu_count, gpu_names = _check_gpu()
    bus.log(f"Internet: {'ON' if internet_ok else 'OFF'}")
    bus.log(f"GPUs detected: {gpu_names or '(none)'}")

    if not internet_ok:
        err = (
            "Internet is OFF for this notebook session. Enable it under "
            "Notebook Settings -> Internet -> On, then Run All again. Model "
            "weights and pip installs need it."
        )
        bus.publish(EventType.SETUP_STAGE_ERROR, stage="env_check", note="Internet is OFF")
        result.error = err
        bus.publish(EventType.PIPELINE_ERROR, error=err, traceback="")
        return

    if gpu_count == 0:
        bus.publish(EventType.SETUP_STAGE_FALLBACK, stage="env_check", note="no GPU detected — will run on CPU (slow)")
    bus.publish(EventType.SETUP_STAGE_DONE, stage="env_check")

    # -- install -----------------------------------------------------------
    bus.publish(EventType.SETUP_STAGE_START, stage="install")
    cache_ds = find_cache_dir(input_root)
    if cache_ds is not None:
        bus.log(f"Found a cache dataset at {cache_ds} — pip will prefer wheels there over downloading")

    simple_deps = [
        ("scipy", "scipy"), ("xatlas", "xatlas"), ("PIL", "pillow"),
        ("laspy", "laspy"), ("rasterio", "rasterio"), ("pyproj", "pyproj"),
        ("trimesh", "trimesh"), ("open3d", "open3d"), ("pymavlink", "pymavlink"),
        ("plotly", "plotly"), ("huggingface_hub", "huggingface_hub"),
        ("safetensors", "safetensors"), ("anywidget", "anywidget"),
    ]
    for mod, pkg in simple_deps:
        if _try_import(mod):
            _mark_present(pkg, bus)
        else:
            _pip_install_silent(pkg, bus, find_links=cache_ds)

    if _try_import("torchcodec"):
        _mark_present("torchcodec", bus)
    else:
        _pip_install_silent("torchcodec", bus, label="torchcodec (optional GPU decode)", find_links=cache_ds)

    if _try_import("ultralytics"):
        _mark_present("ultralytics", bus)
    else:
        _pip_install_silent("ultralytics", bus, label="ultralytics (dynamic-object masking)", find_links=cache_ds)

    if _try_import("mapanything"):
        _mark_present("mapanything", bus)
    else:
        _pip_install_silent(
            "git+https://github.com/facebookresearch/map-anything.git", bus,
            extra_args=["--no-deps"], label="mapanything (Backbone C)", find_links=cache_ds,
        )
        for pkg in ["opencv-python-headless", "hydra-core", "omegaconf", "uniception", "roma"]:
            if not _try_import(pkg.replace("-", "_")):
                _pip_install_silent(pkg, bus, find_links=cache_ds)

    try:
        from .fullscreen_server import ensure_cloudflared

        ensure_cloudflared(bus)
    except Exception as e:
        bus.log(f"cloudflared setup skipped ({e}) — full-screen viewer link will be unavailable", level="warn")

    bus.publish(EventType.SETUP_STAGE_DONE, stage="install")

    # -- detect_inputs -----------------------------------------------------
    bus.publish(EventType.SETUP_STAGE_START, stage="detect_inputs")
    detected = detect_all(input_root)
    result.detected = detected

    if detected.video is None:
        err = f"No video file found under {input_root}. Attach a dataset containing a drone video — see RUN_ON_KAGGLE.md."
        bus.publish(EventType.SETUP_STAGE_ERROR, stage="detect_inputs", note=err)
        result.error = err
        bus.publish(EventType.PIPELINE_ERROR, error=err, traceback="")
        return

    bus.log(
        f"Video: {detected.video.path.name} ({detected.video.width}x{detected.video.height}, "
        f"{detected.video.fps:.1f}fps, {detected.video.duration_s:.0f}s)"
    )
    for w in detected.warnings:
        bus.log(w)

    telemetry = None
    if detected.telemetry_path is not None:
        telemetry = parse_telemetry(detected.telemetry_path, detected.telemetry_kind)
        bus.log(f"Parsed telemetry: {len(telemetry)} samples ({telemetry.kind}); notes: {telemetry.notes}")
    elif detected.telemetry_kind == "embedded":
        telemetry = extract_embedded_telemetry(detected.video.path)
        bus.log(f"Parsed embedded telemetry: {len(telemetry)} samples; notes: {telemetry.notes}")
    if telemetry is None or len(telemetry) == 0:
        bus.log("NO GPS TELEMETRY FOUND. Continuing RGB-only: outputs will be APPROXIMATE SCALE — NOT GEOREFERENCED.", level="warn")

    gpu_monitor = GpuMonitor(bus, interval_s=0.5)

    bus.publish(
        EventType.HEADER_UPDATE,
        video_name=detected.video.path.name,
        resolution=f"{detected.video.width}x{detected.video.height}",
        fps=round(detected.video.fps, 1),
        duration_s=detected.video.duration_s,
        telemetry_type=detected.telemetry_kind or "none",
        telemetry_sample_count=len(telemetry) if telemetry else 0,
        sync_method="timestamp" if (telemetry and telemetry.has_timestamps) else "proportional interpolation",
        mode=mode, gpu_count=gpu_monitor.device_count, gpu_names=gpu_monitor.device_names,
        backbone=backbone_choice, prior_mode=prior_mode,
    )
    bus.publish(EventType.SETUP_STAGE_DONE, stage="detect_inputs")

    # -- full-screen server (best-effort, never blocks the pipeline) -------
    try:
        from .fullscreen_server import start_fullscreen_server

        url = start_fullscreen_server(output_dir, bus)
        bus.publish(EventType.TUNNEL_READY, url=url)
    except Exception as e:
        bus.log(f"Full-screen viewer server unavailable ({e}); inline viewer still works", level="warn")
        bus.publish(EventType.TUNNEL_READY, url=None)

    # -- live reconstruction page (best-effort, never blocks the pipeline) -
    live_page_server = None
    try:
        from .live_page import start_live_page

        live_page_server, live_url = start_live_page(detected.video.path, bus)
        bus.publish(EventType.LIVE_PAGE_READY, url=live_url)
    except Exception as e:
        bus.log(f"Live reconstruction page unavailable ({e}); the dashboard's inline 3D tab is the fallback", level="warn")
        bus.publish(EventType.LIVE_PAGE_READY, url=None)

    # -- pipeline ------------------------------------------------------------
    cfg = PipelineConfig(
        mode=mode, backbone_choice=backbone_choice, prior_mode=prior_mode, use_finetuned=use_finetuned,
        output_dir=output_dir,
        device0="cuda:0" if gpu_monitor.device_count >= 1 else "cpu",
        device1="cuda:1" if gpu_monitor.device_count >= 2 else ("cuda:0" if gpu_monitor.device_count >= 1 else "cpu"),
    )
    pipeline = Pipeline(cfg, detected, telemetry, bus, gpu_monitor, report, live_page=live_page_server)
    result.pipeline = pipeline
    pipeline._run_guarded()  # synchronous — bootstrap.run() already runs off the main thread
    result.ok = bool(pipeline.result and pipeline.result.success)


In [ ]:
%%writefile sih3d/dashboard.py
"""Live ipywidgets dashboard, consuming events drained from the pipeline's
EventBus in the notebook's main thread (the single RUN cell — see
bootstrap.py). Refresh is throttled to <=2Hz via `render_if_due()`;
`on_event()` itself is cheap (just accumulates state) so it never slows
down event draining.

ipywidgets objects can be constructed and updated outside a live Jupyter
kernel (they just have no frontend to render to), so the state-accumulation
logic here is unit-testable headlessly; the actual visual layout can only be
verified in a real notebook.

Deliberately no anywidget and no plotly FigureWidget anywhere in this file.
A first real Kaggle run hit "No version of module anywidget is registered"
— anywidget (like any custom ipywidgets extension, including plotly's
FigureWidget) needs its JS registered with the front end at *kernel start*,
before the notebook even begins executing cells; installing the Python
package at pip-install time, mid-run, is too late; the front end already
decided what it knows. Only ipywidgets' own built-in widgets (Output,
Image, HTML, ...) and plain matplotlib PNGs piped through them are safe to
rely on here — nothing that needs its own front-end extension. The Live 3D
panel is therefore a plain matplotlib top-down + oblique scatter of the
growing point cloud, redrawn on the same throttled cadence as every other
panel (see render_if_due / _render_live3d_panel below), not a live 3D
widget.

The dashboard must render before installs/downloads start (per the task's
"dashboard-first" requirement), so nothing in construction can depend on a
package that isn't preinstalled on Kaggle — ipywidgets/matplotlib/PIL are
safe.
"""

from __future__ import annotations

import io
import time
from collections import deque
from pathlib import Path

import numpy as np

from .events import Event, EventType

STAGE_LABELS = {
    "frame_extraction": "Frame Extraction",
    "camera_trajectory": "Camera Trajectory / Poses",
    "geometric_reconstruction": "Geometric Reconstruction",
    "large_scale_alignment": "Large-Scale Alignment",
    "dense_point_cloud": "Dense 3D Point Cloud",
    "mesh_textured_model": "Mesh / Textured 3D Model",
}
STAGE_ORDER = list(STAGE_LABELS.keys())

SETUP_STAGE_LABELS = {
    "env_check": "Environment Check",
    "install": "Install Dependencies",
    "detect_inputs": "Detect Inputs",
}

_STATUS_COLOR = {
    "pending": "#30363d", "running": "#1f6feb", "done": "#238636",
    "fallback": "#9e6a03", "error": "#da3633",
}


def _encode_png(img_rgb: np.ndarray) -> bytes:
    from PIL import Image

    buf = io.BytesIO()
    Image.fromarray(np.clip(img_rgb, 0, 255).astype(np.uint8)).save(buf, format="PNG")
    return buf.getvalue()


class StageCard:
    def __init__(self, W, key: str, label: str, has_progress: bool = True):
        self.key = key
        self.status = "pending"
        self.elapsed_s = 0.0
        self.fallback_note: str | None = None
        self._start_ts: float | None = None
        self._base_label = label

        self.label = W.HTML(f"<b>{label}</b>")
        self.status_dot = W.HTML(self._dot_html("pending"))
        self.progress = W.FloatProgress(value=0.0, min=0.0, max=1.0, layout=W.Layout(width="90%")) if has_progress else None
        self.elapsed_label = W.Label("")
        children = [W.HBox([self.status_dot, self.label])]
        if self.progress is not None:
            children.append(self.progress)
        children.append(self.elapsed_label)
        self.widget = W.VBox(children, layout=W.Layout(border="1px solid #30363d", padding="6px", width="150px"))

    def _dot_html(self, status: str) -> str:
        color = _STATUS_COLOR.get(status, "#30363d")
        return f'<span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:{color};margin-right:6px;"></span>'

    def start(self, ts: float) -> None:
        self.status, self._start_ts = "running", ts
        self.status_dot.value = self._dot_html("running")

    def done(self, ts: float) -> None:
        self.status = "done"
        self.elapsed_s = ts - self._start_ts if self._start_ts else self.elapsed_s
        self.status_dot.value = self._dot_html("done")
        if self.progress is not None:
            self.progress.value = 1.0
        self.elapsed_label.value = f"{self.elapsed_s:.1f}s"

    def fallback(self, note: str) -> None:
        self.status = "fallback"
        self.fallback_note = note
        self.status_dot.value = self._dot_html("fallback")
        self.label.value = f"<b>{self._base_label}</b> <span style='color:#9e6a03'>[fallback]</span>"

    def error(self, ts: float) -> None:
        self.status = "error"
        self.status_dot.value = self._dot_html("error")

    def set_progress(self, frac: float) -> None:
        if self.progress is not None:
            self.progress.value = max(0.0, min(1.0, frac))

    def tick_elapsed(self, now: float) -> None:
        if self.status == "running" and self._start_ts is not None:
            self.elapsed_label.value = f"{now - self._start_ts:.1f}s"


class Dashboard:
    def __init__(self, header_info: dict, refresh_hz: float = 2.0, output_dir: str | Path | None = None):
        import ipywidgets as W

        self._W = W
        self.refresh_interval = 1.0 / max(refresh_hz, 0.1)
        self._last_render = 0.0
        self._dirty = True
        self.output_dir = Path(output_dir) if output_dir else None

        self.start_ts = time.time()
        self.frames_processed = 0
        self._header_info = dict(header_info)
        self._outputs_tunnel_url: str | None = None
        self._live_page_url: str | None = None

        self._build_header()
        self._build_setup_strip()
        self._build_install_panel()
        self._build_pipeline_strip()
        self._build_gpu_panel(header_info.get("gpu_count", 0))
        self._build_frames_panel()
        self._build_trajectory_panel()
        self._build_geometry_panel()
        self._build_live3d_panel()
        self._build_rasters_panel()
        self._build_log_panel()
        self._build_results_card()

        self.root = W.VBox([
            self.header_box,
            self.setup_box,
            self.pipeline_box,
            W.HBox([self.gpu_box, self.frames_box]),
            W.HBox([self.trajectory_box, self.geometry_box]),
            self.live3d_box,
            self.rasters_box,
            self.log_box,
            self.results_box,
        ])

    def display(self) -> None:
        from IPython.display import display

        display(self.root)

    # -- construction ---------------------------------------------------------

    def _build_header(self) -> None:
        W = self._W
        self.header_html = W.HTML(self._render_header_html())
        self.header_links_html = W.HTML("")
        self.overall_progress = W.FloatProgress(value=0.0, min=0.0, max=1.0, description="Overall:", layout=W.Layout(width="60%"))
        self.elapsed_eta_label = W.Label("elapsed 0s")
        self.header_box = W.VBox([
            self.header_html, self.header_links_html,
            W.HBox([self.overall_progress, self.elapsed_eta_label]),
        ], layout=W.Layout(border="1px solid #30363d", padding="8px", margin="0 0 8px 0"))

    def _render_header_html(self) -> str:
        info = self._header_info
        lines = [
            f"<b>Video:</b> {info.get('video_name', '?')} "
            f"({info.get('resolution', '?')}, {info.get('fps', '?')} fps, {info.get('duration_s', 0):.0f}s)",
            f"<b>Telemetry:</b> {info.get('telemetry_type', 'none')} "
            f"({info.get('telemetry_sample_count', 0)} samples, sync: {info.get('sync_method', '-')})",
            f"<b>Mode:</b> {info.get('mode', '?')} &nbsp; "
            f"<b>GPUs:</b> {', '.join(info.get('gpu_names', [])) or 'none detected'} &nbsp; "
            f"<b>Backbone:</b> {info.get('backbone', '?')} ({info.get('prior_mode', '?')})",
        ]
        return "<br>".join(lines)

    def update_header(self, info: dict) -> None:
        self._header_info.update(info)
        self.header_html.value = self._render_header_html()
        self._dirty = True

    def _update_header_links(self) -> None:
        links = []
        if self._live_page_url:
            links.append(f"<a href='{self._live_page_url}' target='_blank'>Open live reconstruction page</a>")
        if self._outputs_tunnel_url:
            links.append(f"<a href='{self._outputs_tunnel_url}/viewer.html' target='_blank'>Open full-screen viewer</a>")
        self.header_links_html.value = " &nbsp;|&nbsp; ".join(links)

    def _build_setup_strip(self) -> None:
        W = self._W
        self.setup_cards: dict[str, StageCard] = {
            key: StageCard(W, key, label, has_progress=False) for key, label in SETUP_STAGE_LABELS.items()
        }
        self.setup_box = W.HBox(
            [c.widget for c in self.setup_cards.values()],
            layout=W.Layout(overflow_x="auto", margin="0 0 8px 0"),
        )

    def _build_install_panel(self) -> None:
        W = self._W
        self.install_rows: dict[str, dict] = {}
        self.install_html = W.HTML("")
        self.install_box = W.VBox(
            [W.HTML("<b>Installs</b>"), self.install_html],
            layout=W.Layout(border="1px solid #30363d", padding="6px", margin="0 0 8px 0", max_height="140px", overflow_y="auto"),
        )
        self.setup_box.children = list(self.setup_box.children) + [self.install_box]

    def _render_install_panel(self) -> None:
        if not self.install_rows:
            return
        rows = "".join(
            f"<tr><td>{pkg}</td><td>{r['status']}</td><td>{r['elapsed_s']:.1f}s</td></tr>"
            for pkg, r in self.install_rows.items()
        )
        self.install_html.value = f"<table style='font-size:11px'><tr><th>Package</th><th>Status</th><th>Time</th></tr>{rows}</table>"

    def _build_pipeline_strip(self) -> None:
        W = self._W
        self.input_card = W.VBox([
            W.HTML("<b>Drone Video</b>"),
            W.HTML(self._dot_html_static("done")),
        ], layout=W.Layout(border="1px solid #30363d", padding="6px", width="150px"))
        self.stage_cards: dict[str, StageCard] = {key: StageCard(W, key, label) for key, label in STAGE_LABELS.items()}
        self.pipeline_box = W.HBox(
            [self.input_card] + [c.widget for c in self.stage_cards.values()],
            layout=W.Layout(overflow_x="auto", margin="0 0 8px 0"),
        )

    def _dot_html_static(self, status: str) -> str:
        color = _STATUS_COLOR.get(status, "#30363d")
        return f'<span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:{color};"></span>'

    def _build_gpu_panel(self, gpu_count: int) -> None:
        W = self._W
        self.gpu_count = gpu_count
        self.gpu_util_history: list[deque] = [deque(maxlen=120) for _ in range(max(gpu_count, 1))]
        self.gpu_mem_history: list[deque] = [deque(maxlen=120) for _ in range(max(gpu_count, 1))]
        self.gpu_out = W.Output()
        self.gpu_box = W.VBox([W.HTML("<b>GPU</b>"), self.gpu_out], layout=W.Layout(border="1px solid #30363d", padding="6px", width="48%"))

    def _build_frames_panel(self) -> None:
        W = self._W
        self.current_frame_img = W.Image(format="png", layout=W.Layout(width="160px"))
        self.current_frame_label = W.Label("")
        self.filmstrip = W.HBox([], layout=W.Layout(overflow_x="auto"))
        self.filmstrip_items: deque = deque(maxlen=12)
        self.frames_box = W.VBox([
            W.HTML("<b>Frames</b>"), self.current_frame_img, self.current_frame_label, self.filmstrip,
        ], layout=W.Layout(border="1px solid #30363d", padding="6px", width="48%"))

    def _build_trajectory_panel(self) -> None:
        W = self._W
        self.gps_track: list[tuple[float, float]] = []
        self.processed_track: list[tuple[float, float]] = []
        self.camera_track: list[tuple[float, float]] = []
        self.trajectory_out = W.Output()
        self.trajectory_box = W.VBox([W.HTML("<b>Trajectory</b>"), self.trajectory_out], layout=W.Layout(border="1px solid #30363d", padding="6px", width="48%"))

    def _build_geometry_panel(self) -> None:
        W = self._W
        self.geometry_out = W.Output()
        self.geometry_box = W.VBox([W.HTML("<b>Geometry (latest chunk)</b>"), self.geometry_out], layout=W.Layout(border="1px solid #30363d", padding="6px", width="48%"))

    # Cap how many points the Live 3D panel keeps in memory for redraws —
    # unbounded growth over a long FULL-mode run would slow the matplotlib
    # scatter down and bloat notebook memory for no visual benefit past a
    # certain density; beyond the cap, new chunks replace a random sample
    # of existing points so the displayed cloud still reflects the whole
    # scanned area rather than only the most recent chunk.
    _LIVE3D_MAX_POINTS = 200_000

    def _build_live3d_panel(self) -> None:
        W = self._W
        self._live3d_xyz = np.zeros((0, 3), dtype=np.float32)
        self._live3d_color = np.zeros((0, 3), dtype=np.uint8)
        self._live3d_rng = np.random.default_rng(0)
        self._last_live3d_render = 0.0
        self.live3d_out = W.Output()
        self.live3d_box = W.VBox(
            [W.HTML("<b>Live 3D (point cloud, growing per chunk)</b>"), self.live3d_out],
            layout=W.Layout(border="1px solid #30363d", padding="6px", margin="0 0 8px 0"),
        )

    def _build_rasters_panel(self) -> None:
        W = self._W
        self.raster_paths: dict[str, str] = {}
        self.raster_images: dict[str, W.Image] = {}
        self.raster_zoom_sliders: dict[str, W.IntSlider] = {}
        self.rasters_tab = W.Tab()
        self.rasters_box = W.VBox(
            [W.HTML("<b>Rasters</b>"), self.rasters_tab],
            layout=W.Layout(border="1px solid #30363d", padding="6px", margin="0 0 8px 0"),
        )

    def _build_log_panel(self) -> None:
        W = self._W
        self.log_lines: deque = deque(maxlen=30)
        self.log_html = W.HTML("")
        self.log_box = W.VBox([W.HTML("<b>Log</b>"), self.log_html], layout=W.Layout(
            border="1px solid #30363d", padding="6px", max_height="200px", overflow_y="auto", margin="0 0 8px 0",
        ))

    def _build_results_card(self) -> None:
        W = self._W
        self.results_html = W.HTML("<i>Results will appear here once the pipeline finishes.</i>")
        self.package_button = W.Button(description="Package outputs", icon="archive")
        self.package_status = W.Label("")
        self.package_button.on_click(self._on_package_click)
        self.results_box = W.VBox(
            [W.HTML("<b>Results</b>"), self.results_html, W.HBox([self.package_button, self.package_status])],
            layout=W.Layout(border="1px solid #30363d", padding="6px"),
        )

    def _on_package_click(self, _btn) -> None:
        """Downloading stays a separate, deliberate action — never
        triggered automatically by the pipeline itself."""
        if self.output_dir is None or not self.output_dir.exists():
            self.package_status.value = "no output_dir configured"
            return
        import shutil

        self.package_status.value = "zipping..."
        try:
            zip_base = str(self.output_dir.parent / "outputs_package")
            zip_path = shutil.make_archive(zip_base, "zip", root_dir=self.output_dir)
            from IPython.display import FileLink, display

            display(FileLink(zip_path))
            self.package_status.value = f"done: {Path(zip_path).name}"
        except Exception as e:
            self.package_status.value = f"failed: {e}"

    # -- event handling ----------------------------------------------------

    def on_event(self, evt: Event) -> None:
        p = evt.payload
        self._dirty = True

        if evt.type == EventType.SETUP_STAGE_START:
            card = self.setup_cards.get(p.get("stage"))
            if card:
                card.start(evt.ts)
        elif evt.type == EventType.SETUP_STAGE_DONE:
            card = self.setup_cards.get(p.get("stage"))
            if card:
                card.done(evt.ts)
        elif evt.type == EventType.SETUP_STAGE_FALLBACK:
            card = self.setup_cards.get(p.get("stage"))
            if card:
                card.fallback(p.get("note", ""))
        elif evt.type == EventType.SETUP_STAGE_ERROR:
            card = self.setup_cards.get(p.get("stage"))
            if card:
                card.error(evt.ts)

        elif evt.type == EventType.INSTALL_PROGRESS:
            self.install_rows[p.get("package", "?")] = {"status": p.get("status", "?"), "elapsed_s": p.get("elapsed_s", 0.0)}

        elif evt.type == EventType.HEADER_UPDATE:
            self.update_header(p)

        elif evt.type == EventType.TUNNEL_READY:
            self._outputs_tunnel_url = p.get("url")
            self._update_header_links()
        elif evt.type == EventType.LIVE_PAGE_READY:
            self._live_page_url = p.get("url")
            self._update_header_links()

        elif evt.type == EventType.STAGE_START:
            card = self.stage_cards.get(p.get("stage"))
            if card:
                card.start(evt.ts)
        elif evt.type == EventType.STAGE_PROGRESS:
            card = self.stage_cards.get(p.get("stage"))
            if card and "frac" in p:
                card.set_progress(p["frac"])
            self._update_overall_progress()
        elif evt.type == EventType.STAGE_DONE:
            card = self.stage_cards.get(p.get("stage"))
            if card:
                card.done(evt.ts)
            self._update_overall_progress()
        elif evt.type == EventType.STAGE_FALLBACK:
            card = self.stage_cards.get(p.get("stage"))
            if card:
                card.fallback(p.get("note", ""))
        elif evt.type == EventType.STAGE_ERROR:
            card = self.stage_cards.get(p.get("stage"))
            if card:
                card.error(evt.ts)

        elif evt.type == EventType.GPU_SAMPLE:
            idx = p["index"]
            if idx < len(self.gpu_util_history):
                self.gpu_util_history[idx].append(p["util_pct"])
                self.gpu_mem_history[idx].append(p["mem_used_mb"])

        elif evt.type == EventType.FRAME_DECODED:
            self.frames_processed += 1
            frame = p.get("frame")
            if frame is not None:
                self.current_frame_img.value = _encode_png(frame)
            self.current_frame_label.value = f"frame {p.get('frame_index', '?')}  sharpness={p.get('sharpness', 0):.0f}"

        elif evt.type in (EventType.KEYFRAME_ACCEPTED, EventType.KEYFRAME_REJECTED):
            thumb = p.get("thumbnail")
            if thumb is not None:
                accepted = evt.type == EventType.KEYFRAME_ACCEPTED
                self.filmstrip_items.append((thumb, accepted))
                self._rebuild_filmstrip()

        elif evt.type == EventType.TRAJECTORY_POINT:
            kind = p.get("kind", "gps")
            xy = (p["x"], p["y"])
            {"gps": self.gps_track, "processed": self.processed_track, "camera": self.camera_track}.get(kind, self.gps_track).append(xy)

        elif evt.type == EventType.GEOMETRY_CHUNK:
            self._latest_depth = p.get("depth")
            self._latest_confidence = p.get("confidence")

        elif evt.type == EventType.POINTCLOUD_GROWTH:
            xyz = p.get("points")
            if xyz is not None and len(xyz) > 0:
                self._append_live3d_points(xyz, p.get("colors"))

        elif evt.type == EventType.RASTERS_READY:
            for name in ("dsm_path", "orthomosaic_path", "coverage_path"):
                path = p.get(name)
                if path:
                    self.raster_paths[name.replace("_path", "")] = path
            self._rebuild_rasters_tab()

        elif evt.type == EventType.LOG:
            level = p.get("level", "info")
            prefix = {"warn": "⚠", "error": "✗"}.get(level, "")
            self.log_lines.append(f"{prefix} {p.get('message', '')}".strip())

        elif evt.type == EventType.PIPELINE_DONE:
            self._render_results(p)

    def _update_overall_progress(self) -> None:
        n = len(self.stage_cards)
        done = sum(1 for c in self.stage_cards.values() if c.status == "done")
        running_partial = sum(
            (c.progress.value if c.progress is not None else 0.0)
            for c in self.stage_cards.values() if c.status == "running"
        )
        self.overall_progress.value = min(1.0, (done + running_partial) / n)

    def _rebuild_filmstrip(self) -> None:
        W = self._W
        items = []
        for thumb, accepted in self.filmstrip_items:
            img = W.Image(value=_encode_png(thumb), format="png", layout=W.Layout(width="48px", opacity="1.0" if accepted else "0.35"))
            items.append(img)
        self.filmstrip.children = items

    def _append_live3d_points(self, xyz: np.ndarray, colors: np.ndarray | None) -> None:
        n = len(xyz)
        cols = colors.astype(np.uint8) if colors is not None and len(colors) == n else np.full((n, 3), 160, dtype=np.uint8)
        self._live3d_xyz = np.concatenate([self._live3d_xyz, xyz.astype(np.float32)], axis=0)
        self._live3d_color = np.concatenate([self._live3d_color, cols], axis=0)

        total = len(self._live3d_xyz)
        if total > self._LIVE3D_MAX_POINTS:
            idx = self._live3d_rng.choice(total, size=self._LIVE3D_MAX_POINTS, replace=False)
            self._live3d_xyz = self._live3d_xyz[idx]
            self._live3d_color = self._live3d_color[idx]

    def _rebuild_rasters_tab(self) -> None:
        W = self._W
        children = []
        titles = []
        for name, path in self.raster_paths.items():
            try:
                png_bytes = _geotiff_to_png(path)
            except Exception:
                continue
            img = W.Image(value=png_bytes, format="png", layout=W.Layout(width="600px"))
            slider = W.IntSlider(value=600, min=200, max=1600, description="zoom (px)")
            slider.observe(lambda change, im=img: setattr(im.layout, "width", f"{change['new']}px"), names="value")
            children.append(W.VBox([slider, img]))
            titles.append(name)
        self.rasters_tab.children = children
        for i, t in enumerate(titles):
            self.rasters_tab.set_title(i, t)

    # -- rendering -----------------------------------------------------------

    def render_if_due(self, force: bool = False) -> None:
        now = time.time()
        if not force and (not self._dirty or now - self._last_render < self.refresh_interval):
            return
        self._last_render = now
        self._dirty = False

        elapsed = now - self.start_ts
        frac = self.overall_progress.value
        eta = f", ETA {elapsed * (1 - frac) / frac:.0f}s" if frac > 0.02 else ""
        self.elapsed_eta_label.value = f"elapsed {elapsed:.0f}s{eta}"
        for c in self.stage_cards.values():
            c.tick_elapsed(now)
        for c in self.setup_cards.values():
            c.tick_elapsed(now)

        self._render_install_panel()
        self._render_gpu_panel()
        self._render_trajectory_panel()
        self._render_geometry_panel()
        self._render_live3d_panel(now, force)
        self._render_log_panel()

    def _render_gpu_panel(self) -> None:
        try:
            import matplotlib.pyplot as plt
        except Exception:
            return
        with self.gpu_out:
            self.gpu_out.clear_output(wait=True)
            fig, axes = plt.subplots(1, max(self.gpu_count, 1), figsize=(3 * max(self.gpu_count, 1), 1.5))
            axes = np.atleast_1d(axes)
            for i, ax in enumerate(axes):
                hist = list(self.gpu_util_history[i]) if i < len(self.gpu_util_history) else []
                ax.plot(hist, color="#1f6feb")
                ax.set_ylim(0, 100)
                ax.set_title(f"GPU{i} {hist[-1]:.0f}%" if hist else f"GPU{i}", fontsize=8)
                ax.set_xticks([])
            plt.tight_layout()
            plt.show()
            plt.close(fig)

    def _render_trajectory_panel(self) -> None:
        try:
            import matplotlib.pyplot as plt
        except Exception:
            return
        with self.trajectory_out:
            self.trajectory_out.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(4, 3))
            if self.gps_track:
                xs, ys = zip(*self.gps_track)
                ax.plot(xs, ys, "-", color="#8b949e", label="GPS", linewidth=1)
            if self.processed_track:
                xs, ys = zip(*self.processed_track)
                ax.scatter(xs, ys, s=10, color="#1f6feb", label="processed keyframes")
            if self.camera_track:
                xs, ys = zip(*self.camera_track)
                ax.plot(xs, ys, "-", color="#238636", label="estimated camera", linewidth=1)
            ax.set_aspect("equal")
            ax.legend(fontsize=6)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

    def _render_geometry_panel(self) -> None:
        depth = getattr(self, "_latest_depth", None)
        conf = getattr(self, "_latest_confidence", None)
        if depth is None:
            return
        try:
            import matplotlib.pyplot as plt
        except Exception:
            return
        with self.geometry_out:
            self.geometry_out.clear_output(wait=True)
            fig, axes = plt.subplots(1, 2, figsize=(5, 2.2))
            axes[0].imshow(depth, cmap="viridis")
            axes[0].set_title("depth", fontsize=8)
            axes[0].axis("off")
            if conf is not None:
                axes[1].imshow(conf, cmap="magma", vmin=0, vmax=1)
            axes[1].set_title("confidence", fontsize=8)
            axes[1].axis("off")
            plt.tight_layout()
            plt.show()
            plt.close(fig)

    def _render_live3d_panel(self, now: float, force: bool) -> None:
        """A plain matplotlib top-down + oblique scatter of the point
        cloud accumulated so far — see the module docstring for why this
        is static-per-redraw rather than a live 3D widget. Redrawn on a
        coarser cadence than the main throttle (rebuilding a 3D scatter is
        comparatively heavy) — every 3s, or on force (e.g. the final
        render)."""
        if not force and now - self._last_live3d_render < 3.0:
            return
        if len(self._live3d_xyz) == 0:
            return
        self._last_live3d_render = now
        try:
            import matplotlib.pyplot as plt
        except Exception:
            return
        pts = self._live3d_xyz
        colors01 = np.clip(self._live3d_color, 0, 255) / 255.0
        with self.live3d_out:
            self.live3d_out.clear_output(wait=True)
            fig = plt.figure(figsize=(9, 3.6))
            ax_top = fig.add_subplot(121, projection="3d")
            ax_top.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=colors01, s=0.3)
            ax_top.view_init(elev=89, azim=-90)
            ax_top.set_title(f"Top-down ({len(pts):,} pts)", fontsize=8)
            ax_top.set_axis_off()

            ax_obl = fig.add_subplot(122, projection="3d")
            ax_obl.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=colors01, s=0.3)
            ax_obl.view_init(elev=25, azim=-60)
            ax_obl.set_title("Oblique", fontsize=8)
            ax_obl.set_axis_off()

            plt.tight_layout()
            plt.show()
            plt.close(fig)

    def _render_log_panel(self) -> None:
        self.log_html.value = "<pre style='font-size:11px;margin:0;'>" + "\n".join(self.log_lines) + "</pre>"

    def _render_results(self, payload: dict) -> None:
        outputs = payload.get("outputs", [])
        rows = "".join(
            f"<tr><td>{o.get('name')}</td><td>{'OK' if o.get('ok') else 'skipped: ' + str(o.get('skipped_reason'))}</td>"
            f"<td>{(o.get('size_bytes', 0) or 0) / 1e6:.2f} MB</td></tr>" for o in outputs
        )
        viewer_note = ""
        viewer_path = payload.get("viewer_html_path")
        if viewer_path:
            try:
                viewer_src = open(viewer_path, "r", encoding="utf-8").read().replace('"', "&quot;")
                viewer_note = f'<iframe srcdoc="{viewer_src}" style="width:100%;height:500px;border:1px solid #30363d;"></iframe>'
            except Exception:
                viewer_note = f"<p>viewer.html at {viewer_path}</p>"
        self.results_html.value = (
            f"<table><tr><th>File</th><th>Status</th><th>Size</th></tr>{rows}</table>{viewer_note}"
        )

    def snapshot_html(self) -> str:
        """A static HTML render of the dashboard's current state, for
        report.html — ipywidgets themselves don't survive a static export,
        and Kaggle's "Save Version" runs the notebook headless (no one
        watching the live widgets), so report.html is the only place the
        final dashboard state is ever actually seen for that run. Re-uses
        the same matplotlib panels as the live view, saved as embedded
        base64 PNGs instead of rendered into an Output widget."""
        import base64
        import io as _io

        parts = ["<div style='font-family:sans-serif'>"]

        stage_rows = "".join(
            f"<tr><td>{c.key}</td><td>{c.status}</td><td>{c.elapsed_s:.1f}s</td>"
            f"<td>{c.fallback_note or '-'}</td></tr>"
            for c in self.stage_cards.values()
        )
        parts.append(f"<h3>Pipeline stages</h3><table border='1' style='border-collapse:collapse'>"
                     f"<tr><th>Stage</th><th>Status</th><th>Elapsed</th><th>Fallback</th></tr>{stage_rows}</table>")

        try:
            import matplotlib.pyplot as plt

            if any(len(h) for h in self.gpu_util_history):
                fig, axes = plt.subplots(1, max(self.gpu_count, 1), figsize=(3 * max(self.gpu_count, 1), 1.5))
                axes = np.atleast_1d(axes)
                for i, ax in enumerate(axes):
                    hist = list(self.gpu_util_history[i]) if i < len(self.gpu_util_history) else []
                    ax.plot(hist, color="#1f6feb")
                    ax.set_ylim(0, 100)
                    ax.set_title(f"GPU{i}", fontsize=8)
                buf = _io.BytesIO()
                plt.tight_layout()
                fig.savefig(buf, format="png", dpi=100)
                plt.close(fig)
                b64 = base64.b64encode(buf.getvalue()).decode("ascii")
                parts.append(f"<h3>GPU utilization</h3><img src='data:image/png;base64,{b64}'>")
        except Exception:
            pass

        parts.append(f"<h3>Log (last {len(self.log_lines)} lines)</h3><pre style='font-size:11px'>" + "\n".join(self.log_lines) + "</pre>")
        parts.append("</div>")
        return "".join(parts)


def _geotiff_to_png(path: str, max_dim: int = 1600) -> bytes:
    """Reads a GeoTIFF via rasterio and downsamples for a notebook-friendly
    preview (the actual full-resolution file stays on disk untouched)."""
    import rasterio
    from PIL import Image

    with rasterio.open(path) as src:
        data = src.read()  # (bands, H, W)

    if data.shape[0] >= 3:
        arr = np.transpose(data[:3], (1, 2, 0)).astype(np.float32)
    else:
        band = data[0].astype(np.float32)
        arr = np.stack([band, band, band], axis=-1)

    finite = np.isfinite(arr)
    if finite.any():
        lo, hi = np.percentile(arr[finite], [2, 98])
        arr = np.clip((arr - lo) / max(hi - lo, 1e-6), 0, 1)
    else:
        arr = np.zeros_like(arr)
    arr = np.nan_to_num(arr)
    img = Image.fromarray((arr * 255).astype(np.uint8))
    if max(img.size) > max_dim:
        scale = max_dim / max(img.size)
        img = img.resize((int(img.width * scale), int(img.height * scale)))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


In [ ]:
%%writefile sih3d/decode.py
"""GPU-first video decode with a CPU fallback chain.

Priority: torchcodec CUDA decode -> PyNvVideoCodec -> ffmpeg -hwaccel cuda
-> plain CPU decode (OpenCV/PyAV). Every fallback is logged. Decoding is
lazy/on-demand: callers ask for specific frame indices or a time range
(the keyframe selector decides which frames actually get pulled), so we
never decode the whole video into memory up front.

A first Kaggle run (2x T4, 4K source, no telemetry) stalled in frame
extraction for 20+ minutes: CPU pegged at 100%, GPU idle, with the log
claiming "torchcodec on cuda:0". Root cause: `_select_backend()` only
checked that `import torchcodec` succeeded, not that it actually decodes on
the GPU — the PyPI `torchcodec` wheel can be a CPU-only build even when
`torch.cuda.is_available()` is True, and it imports fine either way. Then,
with no fps-based sampling, every raw frame in the QUICK window got decoded
at native 4K one at a time. Fixed by: (1) actually decoding a handful of
frames and checking both the returned tensor's device and achieved fps
before trusting a backend, (2) sampling candidates at a target fps instead
of every raw frame, (3) scaling early (GPU-side via scale_cuda when
hwaccel'd) to a working resolution, and (4) periodic progress logging so a
real stall is visible instead of a silent multi-minute gap in the log.
"""

from __future__ import annotations

import time
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator

import numpy as np

from .events import EventBus, EventType


@dataclass
class DecoderInfo:
    backend: str  # "torchcodec" | "pynvvideocodec" | "ffmpeg_cuda" | "cpu"
    device: str


class FrameDecoder:
    """Yields (frame_index, timestamp_s, HWC uint8 RGB ndarray) tuples.

    Construction probes backends in priority order and logs which one is
    active. `iter_frames(start_s, end_s, target_fps, scale_width)` streams
    frames rather than materializing the whole range.
    """

    def __init__(self, video_path: Path, bus: EventBus, device: str = "cuda:0"):
        self.video_path = Path(video_path)
        self.bus = bus
        self.device = device
        self.info = self._select_backend()
        self.bus.log(f"Video decoder backend: {self.info.backend} on {self.info.device}")

    def _select_backend(self) -> DecoderInfo:
        torchcodec_info = self._verify_torchcodec_gpu()
        if torchcodec_info is not None:
            return torchcodec_info

        try:
            import torch
            if not torch.cuda.is_available():
                raise RuntimeError("no CUDA device")
            import PyNvVideoCodec  # noqa: F401
            return DecoderInfo(backend="pynvvideocodec", device=self.device)
        except Exception as e:
            self.bus.log(f"PyNvVideoCodec unavailable ({e}); trying ffmpeg -hwaccel cuda", level="warn")

        if self._ffmpeg_has_cuda_hwaccel() and self._verify_ffmpeg_cuda():
            return DecoderInfo(backend="ffmpeg_cuda", device=self.device)

        self.bus.log("No GPU decode path available; falling back to CPU decode", level="warn")
        return DecoderInfo(backend="cpu", device="cpu")

    def _verify_ffmpeg_cuda(self, n_probe: int = 30, min_fps_4k: float = 60.0, min_fps_other: float = 24.0) -> bool:
        """`ffmpeg -hwaccels` listing "cuda" only means ffmpeg was compiled
        with CUDA hwaccel support — not that NVDEC actually engages at
        runtime for this codec/driver/container. ffmpeg can silently fall
        back to software decode on a hwaccel init failure rather than
        erroring, which looks identical to a slow-but-working GPU path
        from the caller's side (this is exactly what a real run showed:
        CPU pegged, GPU idle, frame extraction crawling, with the log
        confidently claiming "ffmpeg_cuda"). Same principle as
        _verify_torchcodec_gpu above: actually decode and measure, don't
        trust the capability check alone."""
        try:
            w, h, native_fps = self._probe_dims()
            min_fps = min_fps_4k if max(w, h) >= 3000 else min_fps_other
            probe_end_s = n_probe / max(native_fps, 1.0)

            t0 = time.time()
            count = 0
            for _ in self._iter_ffmpeg(0.0, probe_end_s, stride=1, hwaccel=True, allow_fallback=False):
                count += 1
                if count >= n_probe:
                    break
            elapsed = time.time() - t0
            fps = count / elapsed if elapsed > 0 else 0.0

            if count == 0:
                self.bus.log("ffmpeg -hwaccel cuda produced no frames during verification — using CPU decode instead", level="warn")
                return False
            if fps < min_fps:
                self.bus.log(
                    f"ffmpeg -hwaccel cuda measured only {fps:.1f} fps over {count} frames at {w}x{h} "
                    f"(below the {min_fps:.0f} fps floor — hwaccel likely silently fell back to software) — "
                    f"using CPU decode instead", level="warn",
                )
                return False

            self.bus.log(f"ffmpeg -hwaccel cuda verified: {fps:.1f} fps over {count} real frames at {w}x{h}")
            return True
        except Exception as e:
            self.bus.log(f"ffmpeg -hwaccel cuda verification failed ({e}) — using CPU decode instead", level="warn")
            return False

    def _verify_torchcodec_gpu(self, n_probe: int = 30, min_fps_4k: float = 60.0, min_fps_other: float = 24.0) -> DecoderInfo | None:
        """Decodes a handful of real frames and checks BOTH that they
        actually land on the GPU and that throughput clears a resolution-
        scaled floor — never trust `import torchcodec` succeeding alone
        (see module docstring)."""
        try:
            import torch
            if not torch.cuda.is_available():
                return None
            from torchcodec.decoders import VideoDecoder

            w, h, _ = self._probe_dims()
            min_fps = min_fps_4k if max(w, h) >= 3000 else min_fps_other

            dec = VideoDecoder(str(self.video_path), device=self.device)
            n = min(n_probe, len(dec))
            if n == 0:
                return None

            t0 = time.time()
            on_gpu = True
            for i in range(n):
                frame = dec[i]
                if not str(frame.data.device).startswith("cuda"):
                    on_gpu = False
            elapsed = time.time() - t0
            fps = n / elapsed if elapsed > 0 else 0.0

            if not on_gpu:
                self.bus.log(
                    f"torchcodec requested device={self.device} but decoded frames landed on CPU "
                    f"(the installed wheel is likely a CPU-only build) — trying PyNvVideoCodec", level="warn",
                )
                return None
            if fps < min_fps:
                self.bus.log(
                    f"torchcodec CUDA decode measured {fps:.1f} fps over {n} frames at {w}x{h} "
                    f"(below the {min_fps:.0f} fps floor) — trying PyNvVideoCodec", level="warn",
                )
                return None

            self.bus.log(f"torchcodec CUDA decode verified: {fps:.1f} fps over {n} real frames at {w}x{h}, on {self.device}")
            return DecoderInfo(backend="torchcodec", device=self.device)
        except Exception as e:
            self.bus.log(f"torchcodec CUDA decode unavailable ({e}); trying PyNvVideoCodec", level="warn")
            return None

    def _ffmpeg_has_cuda_hwaccel(self) -> bool:
        import subprocess
        try:
            out = subprocess.run(["ffmpeg", "-hwaccels"], capture_output=True, text=True, timeout=10)
            return "cuda" in out.stdout.lower()
        except Exception:
            return False

    def iter_frames(
        self, start_s: float = 0.0, end_s: float | None = None, stride: int = 1,
        target_fps: float | None = None, scale_width: int | None = None,
    ) -> Iterator[tuple[int, float, np.ndarray]]:
        """`target_fps`, when set, samples candidates at roughly that rate
        (time-based, via ffmpeg's `fps` filter or an equivalent index step)
        instead of every raw frame — this is what actually bounds decode
        work on a long/high-fps source; `stride` is a plain frame-count
        step, used only when `target_fps` is not given. `scale_width`, when
        set and smaller than the source width, scales frames down early
        (GPU-side via scale_cuda on the hwaccel path) to a working
        resolution before they ever hit CPU/pipe I/O or downstream
        sharpness scoring."""
        gen = self._iter_backend(start_s, end_s, stride, target_fps, scale_width)
        yield from self._with_progress_logging(gen, start_s, end_s, target_fps)

    def _iter_backend(self, start_s, end_s, stride, target_fps, scale_width):
        if self.info.backend == "torchcodec":
            yield from self._iter_torchcodec(start_s, end_s, stride, target_fps)
        elif self.info.backend == "pynvvideocodec":
            yield from self._iter_pynvvideocodec(start_s, end_s, stride, target_fps, scale_width)
        elif self.info.backend == "ffmpeg_cuda":
            yield from self._iter_ffmpeg(start_s, end_s, stride, hwaccel=True, target_fps=target_fps, scale_width=scale_width)
        else:
            yield from self._iter_ffmpeg(start_s, end_s, stride, hwaccel=False, target_fps=target_fps, scale_width=scale_width)

    def _with_progress_logging(self, gen, start_s, end_s, target_fps, log_every_s: float = 5.0):
        t0 = time.time()
        last_log = t0
        count = 0
        est_total = None
        if end_s is not None and target_fps:
            est_total = max(1, int((end_s - start_s) * target_fps))
        for item in gen:
            count += 1
            now = time.time()
            if now - last_log >= log_every_s:
                elapsed = now - t0
                fps = count / elapsed if elapsed > 0 else 0.0
                eta = f", ETA {(est_total - count) / fps:.0f}s" if (est_total and fps > 0 and count < est_total) else ""
                total_note = f"/{est_total}" if est_total else ""
                self.bus.log(f"Decode progress: {count}{total_note} frames, {fps:.1f} fps{eta}")
                last_log = now
            yield item

    def _iter_torchcodec(self, start_s, end_s, stride, target_fps):
        from torchcodec.decoders import VideoDecoder
        import torch

        dec = VideoDecoder(str(self.video_path), device=self.device)
        n = len(dec)
        meta = dec.metadata
        fps = meta.average_fps or 30.0
        start_idx = int(start_s * fps)
        end_idx = int(end_s * fps) if end_s is not None else n
        step = max(1, round(fps / target_fps)) if target_fps else stride
        for i in range(start_idx, min(end_idx, n), step):
            frame = dec[i]
            arr = frame.data.permute(1, 2, 0).clamp(0, 255).to("cpu", dtype=torch.uint8).numpy()
            yield i, i / fps, arr

    def _iter_pynvvideocodec(self, start_s, end_s, stride, target_fps, scale_width):
        # PyNvVideoCodec's API varies by version; fall back to ffmpeg CUDA
        # if the demuxer/decoder objects aren't importable/usable at runtime.
        try:
            yield from self._iter_ffmpeg(start_s, end_s, stride, hwaccel=True, target_fps=target_fps, scale_width=scale_width)
        except Exception as e:
            self.bus.log(f"PyNvVideoCodec path failed at runtime ({e}); falling back to CPU decode", level="warn")
            yield from self._iter_ffmpeg(start_s, end_s, stride, hwaccel=False, target_fps=target_fps, scale_width=scale_width)

    def _iter_ffmpeg(self, start_s, end_s, stride, hwaccel: bool, target_fps=None, scale_width=None, allow_fallback: bool = True):
        import subprocess

        w, h, native_fps = self._probe_dims()
        cmd = ["ffmpeg", "-v", "error"]
        if hwaccel:
            # Deliberately NOT `-hwaccel_output_format cuda`: NVDEC still
            # does the actual decode on the GPU, but frames land back in
            # system memory afterward, which lets the (software) `fps`
            # filter run first and cheaply drop most frames BEFORE any of
            # them get uploaded again for scale_cuda below — cheaper than
            # decoding every raw frame at full hw-frame resolution.
            cmd += ["-hwaccel", "cuda"]
        # `-ss` before `-i`: a single fast seek to the QUICK window's start,
        # never a per-frame seek — everything after this is one sequential
        # decode read straight off the pipe.
        cmd += ["-ss", str(start_s), "-i", str(self.video_path)]
        if end_s is not None:
            cmd += ["-t", str(max(0.0, end_s - start_s))]

        out_w, out_h = w, h
        if scale_width and scale_width < w:
            out_w = max(2, int(scale_width) // 2 * 2)
            out_h = max(2, int(round(h * (out_w / w))) // 2 * 2)

        filters = []
        if target_fps:
            filters.append(f"fps={target_fps}")
        elif stride > 1:
            filters.append(f"select='not(mod(n\\,{stride}))'")
        if (out_w, out_h) != (w, h):
            if hwaccel:
                filters += ["hwupload_cuda", f"scale_cuda={out_w}:{out_h}", "hwdownload", "format=nv12"]
            else:
                filters.append(f"scale={out_w}:{out_h}")
        if filters:
            cmd += ["-vf", ",".join(filters)]
        cmd += ["-pix_fmt", "rgb24", "-f", "rawvideo", "-"]

        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        frame_bytes = out_w * out_h * 3
        idx = int(start_s * native_fps)
        idx_step = max(1, round(native_fps / target_fps)) if target_fps else stride
        try:
            while True:
                buf = proc.stdout.read(frame_bytes)
                if len(buf) < frame_bytes:
                    break
                arr = np.frombuffer(buf, dtype=np.uint8).reshape(out_h, out_w, 3)
                yield idx, idx / native_fps, arr
                idx += idx_step
        finally:
            proc.stdout.close()
            proc.wait(timeout=10)
            if proc.returncode not in (0, None) and proc.returncode != 0:
                err = proc.stderr.read().decode("utf-8", "ignore")[-2000:]
                if hwaccel and allow_fallback:
                    self.bus.log(f"ffmpeg CUDA decode failed ({err.strip()[-300:]}); retrying with CPU decode", level="warn")
                    yield from self._iter_ffmpeg(start_s, end_s, stride, hwaccel=False, target_fps=target_fps, scale_width=scale_width)
                elif hwaccel:
                    # allow_fallback=False: used only by _verify_ffmpeg_cuda's
                    # probe, which needs to know hwaccel itself failed rather
                    # than silently receiving CPU-decoded frames it would
                    # mistake for fast GPU decode (confirmed by testing: the
                    # auto-fallback above measured 350+ fps on a failed probe
                    # because those frames were actually from the CPU retry).
                    self.bus.log(f"ffmpeg CUDA decode failed during verification ({err.strip()[-300:]})", level="warn")

    def _probe_dims(self) -> tuple[int, int, float]:
        from .io_detect import probe_video

        vi = probe_video(self.video_path)
        if vi is None:
            raise RuntimeError(f"ffprobe could not read {self.video_path}")
        return vi.width, vi.height, vi.fps or 30.0


In [ ]:
%%writefile sih3d/events.py
"""Thread-safe event bus connecting the pipeline thread to the dashboard.

The pipeline runs in a background thread and calls `bus.publish(...)`.
The dashboard (main thread, ipywidgets) polls `bus.drain()` on a timer.
No locks are held across UI rendering — publish() is O(1) and never blocks
on a full queue (it drops the oldest event instead of blocking the pipeline).
"""

from __future__ import annotations

import queue
import threading
import time
from dataclasses import dataclass, field
from enum import Enum
from typing import Any


class EventType(str, Enum):
    STAGE_START = "stage_start"
    STAGE_PROGRESS = "stage_progress"
    STAGE_DONE = "stage_done"
    STAGE_FALLBACK = "stage_fallback"
    STAGE_ERROR = "stage_error"
    FRAME_DECODED = "frame_decoded"
    KEYFRAME_SELECTED = "keyframe_rejected"
    KEYFRAME_ACCEPTED = "keyframe_accepted"
    KEYFRAME_REJECTED = "keyframe_rejected"
    GPU_SAMPLE = "gpu_sample"
    TRAJECTORY_POINT = "trajectory_point"
    GEOMETRY_CHUNK = "geometry_chunk"
    POINTCLOUD_GROWTH = "pointcloud_growth"
    MESH_PREVIEW = "mesh_preview"
    LOG = "log"
    PIPELINE_DONE = "pipeline_done"
    PIPELINE_ERROR = "pipeline_error"
    HEADER_UPDATE = "header_update"       # bootstrap.py: video/telemetry/GPU info becomes known after detection
    SETUP_STAGE_START = "setup_stage_start"    # bootstrap phases: env_check, install, detect_inputs
    SETUP_STAGE_DONE = "setup_stage_done"
    SETUP_STAGE_FALLBACK = "setup_stage_fallback"
    SETUP_STAGE_ERROR = "setup_stage_error"
    INSTALL_PROGRESS = "install_progress"  # one per package: pending/installing/ok/failed + timing
    TUNNEL_READY = "tunnel_ready"           # fullscreen_server.py: cloudflared URL (or failure) known, for outputs/viewer.html
    LIVE_PAGE_READY = "live_page_ready"     # live_page.py: cloudflared URL (or failure) known, for the live reconstruction page
    RASTERS_READY = "rasters_ready"         # export.py's DSM/orthomosaic/coverage paths, for the Rasters tab


@dataclass
class Event:
    type: EventType
    payload: dict[str, Any] = field(default_factory=dict)
    ts: float = field(default_factory=time.time)


class EventBus:
    """Bounded, drop-oldest, thread-safe pub/sub queue.

    Bounded so a stalled UI consumer can never cause the producer (pipeline
    thread) to block or accumulate unbounded memory.
    """

    def __init__(self, maxsize: int = 4096):
        self._q: "queue.Queue[Event]" = queue.Queue(maxsize=maxsize)
        self._lock = threading.Lock()
        self._log_tail: list[str] = []
        self._log_tail_max = 200

    def publish(self, type: EventType, **payload: Any) -> None:
        evt = Event(type=type, payload=payload)
        if type == EventType.LOG:
            with self._lock:
                self._log_tail.append(str(payload.get("message", "")))
                if len(self._log_tail) > self._log_tail_max:
                    self._log_tail.pop(0)
        try:
            self._q.put_nowait(evt)
        except queue.Full:
            try:
                self._q.get_nowait()
            except queue.Empty:
                pass
            try:
                self._q.put_nowait(evt)
            except queue.Full:
                pass

    def log(self, message: str, level: str = "info") -> None:
        self.publish(EventType.LOG, message=message, level=level)

    def drain(self, max_events: int = 500) -> list[Event]:
        """Non-blocking: pull up to max_events currently queued events."""
        out: list[Event] = []
        for _ in range(max_events):
            try:
                out.append(self._q.get_nowait())
            except queue.Empty:
                break
        return out

    def recent_logs(self) -> list[str]:
        with self._lock:
            return list(self._log_tail)


In [ ]:
%%writefile sih3d/export.py
"""Write every output in outputs/, per the task's format list. Every writer
takes `georeferenced: bool` — when False (no-GPS mode), CRS is skipped on
GeoTIFF/LAS and a local coordinate frame is used instead, matching the
task's no-GPS-mode requirement. Every writer logs and returns a status
instead of raising, except for the point cloud writers (LAS/PLY), which are
never allowed to silently fail — if those break, the whole run has nothing
to show.
"""

from __future__ import annotations

import json
import struct
import subprocess
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np

from .events import EventBus
from .fusion import FusedPointCloud
from .mesh import MeshResult, TextureBakeResult
from .telemetry import enu_to_geodetic


@dataclass
class ExportStatus:
    name: str
    path: Path | None
    ok: bool
    skipped_reason: str | None = None
    size_bytes: int = 0
    timing_s: float = 0.0


# ---------------------------------------------------------------------------
# Point clouds
# ---------------------------------------------------------------------------

def export_pointcloud_ply(cloud: FusedPointCloud, out_path: Path, bus: EventBus) -> ExportStatus:
    """Manual binary_little_endian PLY writer — no dependency beyond numpy,
    so this never fails just because Open3D isn't installed."""
    t0 = time.time()
    n = len(cloud.points)
    header = (
        "ply\nformat binary_little_endian 1.0\n"
        f"element vertex {n}\n"
        "property float x\nproperty float y\nproperty float z\n"
        "property uchar red\nproperty uchar green\nproperty uchar blue\n"
        "property float confidence\nproperty ushort view_count\n"
        "end_header\n"
    ).encode("ascii")

    colors_u8 = np.clip(cloud.colors, 0, 255).astype(np.uint8)
    with open(out_path, "wb") as f:
        f.write(header)
        for i in range(n):
            f.write(struct.pack(
                "<fffBBBfH",
                float(cloud.points[i, 0]), float(cloud.points[i, 1]), float(cloud.points[i, 2]),
                int(colors_u8[i, 0]), int(colors_u8[i, 1]), int(colors_u8[i, 2]),
                float(cloud.confidence[i]), int(min(cloud.view_count[i], 65535)),
            ))

    size = out_path.stat().st_size
    bus.log(f"Wrote {out_path.name}: {n} points ({size / 1e6:.1f} MB)")
    return ExportStatus(name="pointcloud.ply", path=out_path, ok=True, size_bytes=size, timing_s=time.time() - t0)


def export_pointcloud_las(
    cloud: FusedPointCloud, out_path: Path, bus: EventBus, epsg: int | None, georeferenced: bool,
) -> ExportStatus:
    t0 = time.time()
    try:
        import laspy
    except Exception as e:
        bus.log(f"laspy not available ({e}) — pointcloud.las skipped, pointcloud.ply still produced", level="warn")
        return ExportStatus(name="pointcloud.las", path=None, ok=False, skipped_reason=str(e))

    header = laspy.LasHeader(point_format=3, version="1.2")
    header.add_extra_dim(laspy.ExtraBytesParams(name="confidence", type=np.float32))
    header.add_extra_dim(laspy.ExtraBytesParams(name="view_count", type=np.uint16))

    if georeferenced and epsg:
        try:
            import pyproj

            header.add_crs(pyproj.CRS.from_epsg(epsg))
        except Exception as e:
            bus.log(f"Could not attach CRS (EPSG:{epsg}) to LAS header ({e}); writing without CRS", level="warn")

    las = laspy.LasData(header)
    las.x = cloud.points[:, 0]
    las.y = cloud.points[:, 1]
    las.z = cloud.points[:, 2]
    colors_u16 = (np.clip(cloud.colors, 0, 255) * 257).astype(np.uint16)  # 0-255 -> 0-65535 exactly
    las.red = colors_u16[:, 0]
    las.green = colors_u16[:, 1]
    las.blue = colors_u16[:, 2]
    las.confidence = cloud.confidence.astype(np.float32)
    las.view_count = np.clip(cloud.view_count, 0, 65535).astype(np.uint16)

    las.write(str(out_path))
    size = out_path.stat().st_size
    crs_note = f"EPSG:{epsg}" if (georeferenced and epsg) else "no CRS (not georeferenced)"
    bus.log(f"Wrote {out_path.name}: {len(cloud.points)} points, {crs_note} ({size / 1e6:.1f} MB)")
    return ExportStatus(name="pointcloud.las", path=out_path, ok=True, size_bytes=size, timing_s=time.time() - t0)


# ---------------------------------------------------------------------------
# Mesh: OBJ+MTL(+PNG), GLB, FBX (best effort)
# ---------------------------------------------------------------------------

def export_mesh_obj(mesh_result: MeshResult, bake: TextureBakeResult | None, out_dir: Path, bus: EventBus) -> ExportStatus:
    t0 = time.time()
    if mesh_result.mesh is None:
        bus.log("No mesh produced — mesh.obj skipped", level="warn")
        return ExportStatus(name="mesh.obj", path=None, ok=False, skipped_reason="no mesh")

    mesh = mesh_result.mesh
    vertices = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.triangles)
    obj_path = out_dir / "mesh.obj"

    if bake is not None and bake.textured:
        mtl_path = out_dir / "mesh.mtl"
        png_path = out_dir / "mesh_texture.png"
        from PIL import Image

        Image.fromarray(bake.texture_rgb).save(png_path)
        mtl_path.write_text(
            "newmtl material0\nKa 1.0 1.0 1.0\nKd 1.0 1.0 1.0\nKs 0.0 0.0 0.0\n"
            f"map_Kd {png_path.name}\n"
        )
        uv = bake.uv  # (n_faces*3, 2), per-face-corner, matches xatlas `indices` face order
        with open(obj_path, "w") as f:
            f.write(f"mtllib {mtl_path.name}\nusemtl material0\n")
            for v in vertices:
                f.write(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")
            for uv_pair in uv:
                f.write(f"vt {uv_pair[0]:.6f} {1.0 - uv_pair[1]:.6f}\n")
            # Faces reference original mesh vertex indices for position, but
            # sequential 1-based indices for UV (one vt per face-corner).
            for fi, tri in enumerate(triangles):
                a, b, c = tri + 1
                ta, tb, tc = fi * 3 + 1, fi * 3 + 2, fi * 3 + 3
                f.write(f"f {a}/{ta} {b}/{tb} {c}/{tc}\n")
    else:
        colors = np.asarray(mesh.vertex_colors) if mesh.has_vertex_colors() else None
        with open(obj_path, "w") as f:
            for i, v in enumerate(vertices):
                if colors is not None:
                    c = colors[i]
                    f.write(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f} {c[0]:.6f} {c[1]:.6f} {c[2]:.6f}\n")
                else:
                    f.write(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")
            for tri in triangles:
                a, b, c = tri + 1
                f.write(f"f {a} {b} {c}\n")

    size = obj_path.stat().st_size
    bus.log(f"Wrote {obj_path.name}: {len(vertices)} verts, {len(triangles)} faces (textured={bake.textured if bake else False})")
    return ExportStatus(name="mesh.obj", path=obj_path, ok=True, size_bytes=size, timing_s=time.time() - t0)


def export_mesh_glb(mesh_result: MeshResult, bake: TextureBakeResult | None, out_path: Path, bus: EventBus) -> ExportStatus:
    t0 = time.time()
    if mesh_result.mesh is None:
        return ExportStatus(name="mesh.glb", path=None, ok=False, skipped_reason="no mesh")

    try:
        import trimesh
    except Exception as e:
        bus.log(f"trimesh not available ({e}) — mesh.glb skipped", level="warn")
        return ExportStatus(name="mesh.glb", path=None, ok=False, skipped_reason=str(e))

    mesh = mesh_result.mesh
    vertices = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.triangles)

    if bake is not None and bake.textured:
        from PIL import Image

        uv = bake.uv.reshape(-1, 3, 2)
        # trimesh needs one UV per (position) vertex; since xatlas may have
        # duplicated vertices at seams, rebuild a duplicated-vertex mesh so
        # each face-corner UV lines up 1:1 with its own vertex position.
        flat_vertices = vertices[triangles].reshape(-1, 3)
        flat_uv = uv.reshape(-1, 2)
        flat_faces = np.arange(len(flat_vertices)).reshape(-1, 3)
        material = trimesh.visual.material.PBRMaterial(baseColorTexture=Image.fromarray(bake.texture_rgb))
        visual = trimesh.visual.TextureVisuals(uv=flat_uv, material=material)
        tm = trimesh.Trimesh(vertices=flat_vertices, faces=flat_faces, visual=visual, process=False)
    else:
        colors = np.asarray(mesh.vertex_colors) if mesh.has_vertex_colors() else None
        vertex_colors = (np.clip(colors, 0, 1) * 255).astype(np.uint8) if colors is not None else None
        tm = trimesh.Trimesh(vertices=vertices, faces=triangles, vertex_colors=vertex_colors, process=False)

    tm.export(str(out_path))
    size = out_path.stat().st_size
    bus.log(f"Wrote {out_path.name} ({size / 1e6:.1f} MB)")
    return ExportStatus(name="mesh.glb", path=out_path, ok=True, size_bytes=size, timing_s=time.time() - t0)


def export_mesh_fbx(obj_path: Path | None, out_path: Path, bus: EventBus) -> ExportStatus:
    """Best-effort only, per the task spec — assimp via apt/CLI. Logs
    clearly and never raises if assimp isn't installed or the conversion
    fails for any reason."""
    t0 = time.time()
    if obj_path is None or not obj_path.exists():
        return ExportStatus(name="mesh.fbx", path=None, ok=False, skipped_reason="no source mesh.obj")

    try:
        result = subprocess.run(
            ["assimp", "export", str(obj_path), str(out_path)],
            capture_output=True, text=True, timeout=120,
        )
    except FileNotFoundError:
        bus.log("assimp CLI not found — mesh.fbx skipped (best-effort only, install via 'apt-get install assimp-utils')", level="warn")
        return ExportStatus(name="mesh.fbx", path=None, ok=False, skipped_reason="assimp not installed")
    except Exception as e:
        bus.log(f"assimp FBX export raised {type(e).__name__}: {e} — mesh.fbx skipped", level="warn")
        return ExportStatus(name="mesh.fbx", path=None, ok=False, skipped_reason=str(e))

    if result.returncode != 0 or not out_path.exists():
        bus.log(f"assimp FBX export failed (exit {result.returncode}): {result.stderr[-300:]} — mesh.fbx skipped", level="warn")
        return ExportStatus(name="mesh.fbx", path=None, ok=False, skipped_reason=result.stderr[-300:])

    size = out_path.stat().st_size
    bus.log(f"Wrote {out_path.name} via assimp ({size / 1e6:.1f} MB)")
    return ExportStatus(name="mesh.fbx", path=out_path, ok=True, size_bytes=size, timing_s=time.time() - t0)


# ---------------------------------------------------------------------------
# DSM / orthomosaic / coverage rasters
# ---------------------------------------------------------------------------

def _rasterize_grid(
    points_xy: np.ndarray, values: dict[str, np.ndarray], cell_size_m: float, reduce: dict[str, str],
    device: str = "cpu",
) -> tuple[dict[str, np.ndarray], tuple[float, float], int, int]:
    """Rasterizes scattered (x,y)-indexed values onto a regular grid via
    torch.scatter_reduce, as the task spec explicitly asks for. Returns
    (arrays_by_name, (min_x,min_y) grid origin, width, height). Cells with
    no points get NaN (caller decides the nodata value)."""
    import torch

    min_x, min_y = points_xy[:, 0].min(), points_xy[:, 1].min()
    max_x, max_y = points_xy[:, 0].max(), points_xy[:, 1].max()
    width = max(int(np.ceil((max_x - min_x) / cell_size_m)) + 1, 1)
    height = max(int(np.ceil((max_y - min_y) / cell_size_m)) + 1, 1)

    col = ((points_xy[:, 0] - min_x) / cell_size_m).astype(np.int64).clip(0, width - 1)
    row = ((max_y - points_xy[:, 1]) / cell_size_m).astype(np.int64).clip(0, height - 1)  # flip y: raster row 0 = north
    flat_idx = torch.from_numpy(row * width + col).to(device)

    out = {}
    counts = torch.zeros(width * height, device=device)
    counts.scatter_add_(0, flat_idx, torch.ones(len(points_xy), device=device))

    for name, vals in values.items():
        vals_t = torch.from_numpy(np.ascontiguousarray(vals)).float().to(device)
        grid = torch.full((width * height,), float("nan"), device=device)
        mode = reduce.get(name, "mean")
        if mode == "amax":
            grid = torch.full((width * height,), float("-inf"), device=device)
            grid.scatter_reduce_(0, flat_idx, vals_t, reduce="amax", include_self=True)
            grid[counts == 0] = float("nan")
        else:  # mean
            summed = torch.zeros(width * height, device=device)
            summed.scatter_add_(0, flat_idx, vals_t)
            grid = torch.where(counts > 0, summed / counts.clamp_min(1), torch.full_like(summed, float("nan")))
        out[name] = grid.reshape(height, width).cpu().numpy()

    out["_count"] = counts.reshape(height, width).cpu().numpy()
    return out, (float(min_x), float(max_y)), width, height


def _write_geotiff(path: Path, bands: dict[str, np.ndarray], origin_topleft: tuple[float, float], cell_size_m: float, epsg: int | None, bus: EventBus) -> bool:
    try:
        import rasterio
        from rasterio.transform import from_origin
    except Exception as e:
        bus.log(f"rasterio not available ({e}) — {path.name} skipped", level="warn")
        return False

    band_names = list(bands.keys())
    height, width = next(iter(bands.values())).shape
    transform = from_origin(origin_topleft[0], origin_topleft[1], cell_size_m, cell_size_m)
    crs = None
    if epsg:
        try:
            crs = rasterio.crs.CRS.from_epsg(epsg)
        except Exception as e:
            bus.log(f"Could not build CRS EPSG:{epsg} for {path.name} ({e}); writing without CRS", level="warn")

    with rasterio.open(
        path, "w", driver="GTiff", height=height, width=width, count=len(band_names),
        dtype="float32", crs=crs, transform=transform, nodata=np.nan,
    ) as dst:
        for i, name in enumerate(band_names, start=1):
            dst.write(bands[name].astype(np.float32), i)
            dst.set_band_description(i, name)
    return True


def export_dsm_orthomosaic(
    cloud: FusedPointCloud, out_dir: Path, bus: EventBus, epsg: int | None, georeferenced: bool, cell_size_m: float = 0.2,
) -> tuple[ExportStatus, ExportStatus]:
    t0 = time.time()
    if len(cloud.points) < 10:
        skip = ExportStatus(name="dsm.tif/orthomosaic.tif", path=None, ok=False, skipped_reason="too few points")
        return skip, skip

    grids, origin, width, height = _rasterize_grid(
        cloud.points[:, :2],
        {"height": cloud.points[:, 2], "r": cloud.colors[:, 0], "g": cloud.colors[:, 1], "b": cloud.colors[:, 2]},
        cell_size_m, reduce={"height": "amax", "r": "mean", "g": "mean", "b": "mean"},
    )
    dsm_path = out_dir / "dsm.tif"
    ortho_path = out_dir / "orthomosaic.tif"
    epsg_used = epsg if georeferenced else None

    dsm_ok = _write_geotiff(dsm_path, {"height": grids["height"]}, origin, cell_size_m, epsg_used, bus)
    ortho_ok = _write_geotiff(ortho_path, {"r": grids["r"], "g": grids["g"], "b": grids["b"]}, origin, cell_size_m, epsg_used, bus)

    if dsm_ok:
        bus.log(f"Wrote {dsm_path.name}: {width}x{height} cells @ {cell_size_m}m" + ("" if georeferenced else " (no CRS — not georeferenced)"))
    if ortho_ok:
        bus.log(f"Wrote {ortho_path.name}: {width}x{height} cells @ {cell_size_m}m")

    elapsed = time.time() - t0
    dsm_status = ExportStatus(name="dsm.tif", path=dsm_path if dsm_ok else None, ok=dsm_ok, timing_s=elapsed)
    ortho_status = ExportStatus(name="orthomosaic.tif", path=ortho_path if ortho_ok else None, ok=ortho_ok, timing_s=elapsed)
    return dsm_status, ortho_status


def export_coverage(
    cloud: FusedPointCloud, out_dir: Path, bus: EventBus, epsg: int | None, georeferenced: bool, cell_size_m: float = 0.2,
) -> ExportStatus:
    t0 = time.time()
    if len(cloud.points) < 10:
        return ExportStatus(name="coverage.tif", path=None, ok=False, skipped_reason="too few points")

    grids, origin, width, height = _rasterize_grid(
        cloud.points[:, :2],
        {"view_count": cloud.view_count.astype(np.float32), "confidence": cloud.confidence},
        cell_size_m, reduce={"view_count": "mean", "confidence": "mean"},
    )
    path = out_dir / "coverage.tif"
    ok = _write_geotiff(path, {"view_count": grids["view_count"], "confidence": grids["confidence"]}, origin, cell_size_m, epsg if georeferenced else None, bus)
    if ok:
        unobserved_frac = float(np.isnan(grids["view_count"]).mean())
        bus.log(f"Wrote {path.name}: {unobserved_frac * 100:.1f}% of the bounding area unobserved")
    return ExportStatus(name="coverage.tif", path=path if ok else None, ok=ok, timing_s=time.time() - t0)


# ---------------------------------------------------------------------------
# Trajectories
# ---------------------------------------------------------------------------

def export_trajectories(
    gps_track_enu: list[tuple[float, float, float]],
    camera_track_enu: list[tuple[float, float, float]] | None,
    origin_latlonalt: tuple[float, float, float],
    out_dir: Path, bus: EventBus,
) -> tuple[ExportStatus, ExportStatus]:
    t0 = time.time()
    lat0, lon0, alt0 = origin_latlonalt

    def to_lonlat(track):
        return [(lon, lat) for lat, lon, _alt in (enu_to_geodetic(e, n, u, lat0, lon0, alt0) for e, n, u in track)]

    features = []
    if gps_track_enu:
        features.append({
            "type": "Feature", "properties": {"name": "gps_track"},
            "geometry": {"type": "LineString", "coordinates": to_lonlat(gps_track_enu)},
        })
    if camera_track_enu:
        features.append({
            "type": "Feature", "properties": {"name": "estimated_camera_track"},
            "geometry": {"type": "LineString", "coordinates": to_lonlat(camera_track_enu)},
        })

    geojson_path = out_dir / "trajectory.geojson"
    geojson_path.write_text(json.dumps({"type": "FeatureCollection", "features": features}, indent=2))

    kml_placemarks = []
    for feat in features:
        coords_str = " ".join(f"{lon},{lat},0" for lon, lat in feat["geometry"]["coordinates"])
        kml_placemarks.append(
            f"<Placemark><name>{feat['properties']['name']}</name>"
            f"<LineString><coordinates>{coords_str}</coordinates></LineString></Placemark>"
        )
    kml = (
        '<?xml version="1.0" encoding="UTF-8"?>\n'
        '<kml xmlns="http://www.opengis.net/kml/2.2"><Document>\n'
        + "\n".join(kml_placemarks) + "\n</Document></kml>\n"
    )
    kml_path = out_dir / "trajectory.kml"
    kml_path.write_text(kml)

    bus.log(f"Wrote trajectory.geojson/.kml: {len(features)} track(s)")
    elapsed = time.time() - t0
    return (
        ExportStatus(name="trajectory.geojson", path=geojson_path, ok=True, timing_s=elapsed),
        ExportStatus(name="trajectory.kml", path=kml_path, ok=True, timing_s=elapsed),
    )


In [ ]:
%%writefile sih3d/fullscreen_server.py
"""Local HTTP server + cloudflared quick tunnel, serving /kaggle/working/outputs/
so the same self-contained viewer.html (see viewer.py — already embeds its
point cloud/mesh/trajectories inline, no sibling-file fetches needed) can be
opened full-screen and shared with teammates during the session, not just
viewed inline in the notebook.

Every failure here is logged and degrades gracefully — never raises into the
caller, never blocks the pipeline. The "Open full-screen viewer" link in the
dashboard header simply doesn't appear if the tunnel couldn't be established;
the inline (anywidget) viewer keeps working regardless.
"""

from __future__ import annotations

import functools
import http.server
import platform
import re
import shutil
import socket
import stat
import subprocess
import threading
import time
import urllib.request
from pathlib import Path

from .events import EventBus

# Kaggle is Linux — that's the only platform this auto-downloads a binary
# for. On other platforms (e.g. local dev on macOS) it just logs and skips;
# install cloudflared manually there if you want to test this path locally.
_CLOUDFLARED_LINUX_URLS = {
    "x86_64": "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "aarch64": "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-arm64",
}


def ensure_cloudflared(bus: EventBus, dest: Path = Path("/usr/local/bin/cloudflared")) -> Path | None:
    """Best-effort: download the cloudflared binary if it's not already on
    PATH (Kaggle images don't ship it). Never raises."""
    existing = shutil.which("cloudflared")
    if existing:
        return Path(existing)

    if platform.system() != "Linux":
        bus.log(
            f"cloudflared auto-download only supported on Linux (this is {platform.system()}) — "
            f"full-screen viewer link unavailable unless cloudflared is installed manually",
            level="warn",
        )
        return None

    url = _CLOUDFLARED_LINUX_URLS.get(platform.machine())
    if url is None:
        bus.log(f"No known cloudflared build for architecture {platform.machine()} — full-screen viewer link unavailable", level="warn")
        return None

    try:
        dest.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, str(dest))
        dest.chmod(dest.stat().st_mode | stat.S_IEXEC)
        bus.log(f"Downloaded cloudflared to {dest}")
        return dest
    except Exception as e:
        bus.log(f"cloudflared download failed ({e}) — full-screen viewer link unavailable", level="warn")
        return None


def _free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("", 0))
        return s.getsockname()[1]


def _start_http_server(directory: Path, port: int, bus: EventBus) -> None:
    handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=str(directory))
    httpd = http.server.ThreadingHTTPServer(("127.0.0.1", port), handler)
    thread = threading.Thread(target=httpd.serve_forever, daemon=True, name="sih3d-http-server")
    thread.start()
    bus.log(f"Local HTTP server serving {directory} on 127.0.0.1:{port}")


_TRYCLOUDFLARE_RE = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")


def start_cloudflared_tunnel(port: int, bus: EventBus, label: str = "tunnel", timeout_s: float = 20.0) -> str | None:
    """Starts a cloudflared quick tunnel pointing at a local port already
    listening on 127.0.0.1. Shared by start_fullscreen_server() (outputs/)
    and live_page.py (the live reconstruction page) so the cloudflared
    process-management/URL-parsing logic exists exactly once. Returns the
    public https://*.trycloudflare.com URL, or None if anything failed —
    logged, never raised."""
    cloudflared = shutil.which("cloudflared")
    if cloudflared is None and Path("/usr/local/bin/cloudflared").exists():
        cloudflared = "/usr/local/bin/cloudflared"
    if cloudflared is None:
        bus.log(f"cloudflared not available — {label} link unavailable", level="warn")
        return None

    try:
        proc = subprocess.Popen(
            [cloudflared, "tunnel", "--url", f"http://127.0.0.1:{port}"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
    except Exception as e:
        bus.log(f"cloudflared failed to start ({e}) — {label} link unavailable", level="warn")
        return None

    url = None
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        line = proc.stdout.readline()
        if not line:
            if proc.poll() is not None:
                break
            continue
        m = _TRYCLOUDFLARE_RE.search(line)
        if m:
            url = m.group(0)
            break

    if url is None:
        bus.log(f"cloudflared did not produce a tunnel URL within {timeout_s:.0f}s — {label} link unavailable", level="warn")
        try:
            proc.terminate()
        except Exception:
            pass
        return None

    def _drain_stdout() -> None:
        try:
            for _ in proc.stdout:
                pass  # discard further cloudflared chatter — never spam the notebook
        except Exception:
            pass

    threading.Thread(target=_drain_stdout, daemon=True, name=f"sih3d-cloudflared-drain-{label}").start()

    bus.log(f"{label} tunnel ready: {url}")
    return url


def start_fullscreen_server(output_dir: Path, bus: EventBus, timeout_s: float = 20.0) -> str | None:
    """Starts a local HTTP server over `output_dir` and a cloudflared quick
    tunnel pointing at it. Returns the public https://*.trycloudflare.com
    base URL (append "/viewer.html"), or None if anything failed."""
    output_dir.mkdir(parents=True, exist_ok=True)

    try:
        port = _free_port()
        _start_http_server(output_dir, port, bus)
    except Exception as e:
        bus.log(f"Local HTTP server failed to start ({e}) — full-screen viewer unavailable", level="warn")
        return None

    return start_cloudflared_tunnel(port, bus, label="full-screen viewer", timeout_s=timeout_s)


In [ ]:
%%writefile sih3d/fusion.py
"""Multi-chunk point fusion: confidence/view-count-weighted voxel accumulation
(torch, GPU-resident) + an optional TSDF volume for meshing.

Two fusion products, both built incrementally as chunks arrive (matching the
dashboard's "point cloud growing chunk by chunk" panel):

1. `VoxelPointFusion` — always available, pure torch. Hashes points into a
   voxel grid and keeps a running confidence-weighted average position/color
   per voxel plus a view count, via `scatter_add`/`scatter_reduce` (the
   primitive the task spec explicitly calls out for GPU-resident grid work).
   This alone produces the "dense 3D point cloud" pipeline stage output.

2. `Open3DTsdfFusion` — attempted for mesh.py's marching-cubes path (denser,
   smoother surfaces than meshing straight off a point cloud). Tries Open3D's
   tensor CUDA integration first, falls back to legacy CPU TSDF, and returns
   `None` (logged) if Open3D isn't usable at all — mesh.py then meshes
   directly from the point cloud (Poisson/ball-pivoting) instead.
"""

from __future__ import annotations

import time
from dataclasses import dataclass

import numpy as np

from .events import EventBus, EventType


@dataclass
class FusedPointCloud:
    points: np.ndarray        # (N,3) world frame, meters
    colors: np.ndarray        # (N,3) uint8-range floats, RGB
    view_count: np.ndarray    # (N,) int32
    confidence: np.ndarray    # (N,) float32, max confidence observed


class VoxelPointFusion:
    """Confidence/view-count-weighted voxel-grid point accumulation.

    Voxel keys are bit-packed 63-bit integers from (x,y,z) grid coordinates
    (21 bits/axis, offset to stay non-negative — comfortably covers a
    +/-1e5 m scene at a 0.1 m voxel size). The key->row mapping is a plain
    Python dict on CPU (negligible cost relative to the point data itself,
    and far simpler/more correct than an unbounded GPU hash table); the
    actual weighted-sum accumulators live on `device` as growing tensors.
    """

    _AXIS_BITS = 21
    _OFFSET = 1 << 20

    def __init__(self, voxel_size_m: float, device: str = "cuda:0"):
        self.voxel_size = voxel_size_m
        self.device = device
        self._key_to_row: dict[int, int] = {}

        import torch

        self._torch = torch
        self._weight_sum = torch.zeros(0, device=device)
        self._pos_sum = torch.zeros(0, 3, device=device)
        self._color_sum = torch.zeros(0, 3, device=device)
        self._view_count = torch.zeros(0, dtype=torch.int32, device=device)
        self._max_conf = torch.zeros(0, device=device)

    def __len__(self) -> int:
        return self._weight_sum.shape[0]

    def _voxel_keys(self, points):
        torch = self._torch
        coords = torch.floor(points / self.voxel_size).to(torch.int64) + self._OFFSET
        coords = coords.clamp(0, (1 << self._AXIS_BITS) - 1)
        return (coords[:, 0] << (2 * self._AXIS_BITS)) | (coords[:, 1] << self._AXIS_BITS) | coords[:, 2]

    def add_chunk(
        self,
        points_world: np.ndarray,
        colors_rgb: np.ndarray,
        confidence: np.ndarray,
        dynamic_mask: np.ndarray | None = None,
        min_confidence: float = 0.1,
    ) -> int:
        """Shapes: points_world/colors_rgb (...,3), confidence/dynamic_mask (...)
        with matching leading dims (e.g. (N,H,W,{3,1})). Returns the number of
        newly-created voxels (for progress reporting)."""
        torch = self._torch
        pts = np.asarray(points_world).reshape(-1, 3)
        cols = np.asarray(colors_rgb).reshape(-1, 3)
        conf = np.asarray(confidence).reshape(-1)

        keep = np.isfinite(pts).all(axis=1) & np.isfinite(conf) & (conf >= min_confidence)
        if dynamic_mask is not None:
            keep &= ~np.asarray(dynamic_mask).reshape(-1)
        if not keep.any():
            return 0
        pts, cols, conf = pts[keep], cols[keep], conf[keep]

        pts_t = torch.from_numpy(np.ascontiguousarray(pts)).float().to(self.device)
        cols_t = torch.from_numpy(np.ascontiguousarray(cols)).float().to(self.device)
        conf_t = torch.from_numpy(np.ascontiguousarray(conf)).float().to(self.device)

        keys = self._voxel_keys(pts_t)
        uniq_keys, inverse = torch.unique(keys, return_inverse=True)
        n_uniq = uniq_keys.shape[0]

        local_weight = torch.zeros(n_uniq, device=self.device).scatter_add_(0, inverse, conf_t)
        local_pos = torch.zeros(n_uniq, 3, device=self.device).scatter_add_(
            0, inverse.unsqueeze(1).expand(-1, 3), pts_t * conf_t.unsqueeze(1)
        )
        local_color = torch.zeros(n_uniq, 3, device=self.device).scatter_add_(
            0, inverse.unsqueeze(1).expand(-1, 3), cols_t * conf_t.unsqueeze(1)
        )
        local_count = torch.zeros(n_uniq, device=self.device).scatter_add_(0, inverse, torch.ones_like(conf_t))
        local_max_conf = torch.zeros(n_uniq, device=self.device).scatter_reduce(
            0, inverse, conf_t, reduce="amax", include_self=False
        )

        uniq_keys_cpu = uniq_keys.cpu().numpy()
        new_keys, existing_rows, existing_local_idx, new_local_idx = [], [], [], []
        for i, k in enumerate(uniq_keys_cpu):
            row = self._key_to_row.get(int(k))
            if row is None:
                new_keys.append(int(k))
                new_local_idx.append(i)
            else:
                existing_rows.append(row)
                existing_local_idx.append(i)

        if existing_rows:
            rows_t = torch.tensor(existing_rows, device=self.device, dtype=torch.long)
            idx_t = torch.tensor(existing_local_idx, device=self.device, dtype=torch.long)
            self._weight_sum.index_add_(0, rows_t, local_weight[idx_t])
            self._pos_sum.index_add_(0, rows_t, local_pos[idx_t])
            self._color_sum.index_add_(0, rows_t, local_color[idx_t])
            self._view_count.index_add_(0, rows_t, local_count[idx_t].to(torch.int32))
            self._max_conf[rows_t] = torch.maximum(self._max_conf[rows_t], local_max_conf[idx_t])

        n_new = len(new_keys)
        if n_new:
            base = self._weight_sum.shape[0]
            for offset, k in enumerate(new_keys):
                self._key_to_row[k] = base + offset
            idx_t = torch.tensor(new_local_idx, device=self.device, dtype=torch.long)
            self._weight_sum = torch.cat([self._weight_sum, local_weight[idx_t]])
            self._pos_sum = torch.cat([self._pos_sum, local_pos[idx_t]])
            self._color_sum = torch.cat([self._color_sum, local_color[idx_t]])
            self._view_count = torch.cat([self._view_count, local_count[idx_t].to(torch.int32)])
            self._max_conf = torch.cat([self._max_conf, local_max_conf[idx_t]])

        return n_new

    def extract_points(self, min_view_count: int = 1) -> FusedPointCloud:
        """`min_view_count` > 1 doubles as the "multi-view consistency" filter
        the task asks for — a voxel only ever observed once is far more
        likely a floater/artifact than real geometry."""
        keep = (self._view_count >= min_view_count) & (self._weight_sum > 1e-9)
        w = self._weight_sum[keep].clamp_min(1e-9)
        pos = (self._pos_sum[keep] / w.unsqueeze(1)).cpu().numpy()
        color = (self._color_sum[keep] / w.unsqueeze(1)).cpu().numpy()
        view_count = self._view_count[keep].cpu().numpy()
        confidence = self._max_conf[keep].cpu().numpy()
        return FusedPointCloud(points=pos, colors=np.clip(color, 0, 255), view_count=view_count, confidence=confidence)


def remove_statistical_outliers(cloud: FusedPointCloud, bus: EventBus, nb_neighbors: int = 16, std_ratio: float = 2.0) -> FusedPointCloud:
    """Optional extra cleanup pass via Open3D, if available. Never required —
    VoxelPointFusion's confidence/view-count filtering already does most of
    the work; this just catches remaining isolated floaters."""
    try:
        import open3d as o3d
    except Exception as e:
        bus.log(f"Open3D unavailable for outlier removal ({e}); skipping this cleanup pass", level="warn")
        return cloud

    if len(cloud.points) < nb_neighbors + 1:
        return cloud

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(cloud.points)
    _, inlier_idx = pcd.remove_statistical_outlier(nb_neighbors=nb_neighbors, std_ratio=std_ratio)
    inlier_idx = np.asarray(inlier_idx)
    removed = len(cloud.points) - len(inlier_idx)
    if removed:
        bus.log(f"Statistical outlier removal: dropped {removed}/{len(cloud.points)} points")
    return FusedPointCloud(
        points=cloud.points[inlier_idx], colors=cloud.colors[inlier_idx],
        view_count=cloud.view_count[inlier_idx], confidence=cloud.confidence[inlier_idx],
    )


class Open3DTsdfFusion:
    """TSDF volume integration for mesh.py's marching-cubes path. Tries
    Open3D's tensor CUDA VoxelBlockGrid first, falls back to the legacy CPU
    ScalableTSDFVolume, and is simply unavailable (mesh.py falls back to
    point-based meshing) if Open3D can't be imported at all.

    Depth per chunk is derived from the backbone's own per-pixel 3D points
    (already in world frame after align.py) by re-projecting into camera
    space via the estimated camera pose — we don't need a separate depth
    sensor/estimator since the geometry backbone already gives us dense
    per-pixel points directly.
    """

    def __init__(self, bus: EventBus, voxel_size_m: float, sdf_trunc_m: float | None = None):
        self.bus = bus
        self.voxel_size = voxel_size_m
        self.sdf_trunc = sdf_trunc_m or voxel_size_m * 4
        self.backend = "none"
        self._volume = None
        self._device_tsdf = None
        self._init()

    def _init(self) -> None:
        try:
            import open3d as o3d
        except Exception as e:
            self.bus.log(f"Open3D not available ({e}) — TSDF meshing disabled, mesh.py will use point-based meshing", level="warn")
            return

        try:
            if hasattr(o3d, "core") and o3d.core.cuda.is_available():
                self._device_tsdf = o3d.core.Device("CUDA:0")
                self._volume = o3d.t.geometry.VoxelBlockGrid(
                    attr_names=("tsdf", "weight", "color"),
                    attr_dtypes=(o3d.core.float32, o3d.core.float32, o3d.core.float32),
                    attr_channels=((1), (1), (3)),
                    voxel_size=self.voxel_size, block_resolution=16, block_count=50000,
                    device=self._device_tsdf,
                )
                self.backend = "open3d_tensor_cuda"
                self.bus.log("TSDF fusion: using Open3D tensor CUDA VoxelBlockGrid")
                return
        except Exception as e:
            self.bus.log(f"Open3D tensor CUDA TSDF unavailable ({e}); falling back to legacy CPU TSDF", level="warn")

        try:
            self._volume = o3d.pipelines.integration.ScalableTSDFVolume(
                voxel_length=self.voxel_size, sdf_trunc=self.sdf_trunc,
                color_type=o3d.pipelines.integration.TSDFVolumeColorType.RGB8,
            )
            self.backend = "open3d_legacy_cpu"
            self.bus.log("TSDF fusion: using Open3D legacy CPU ScalableTSDFVolume (slower — CUDA tensor path unavailable)")
        except Exception as e:
            self.bus.log(f"Open3D legacy TSDF also unavailable ({e}) — TSDF meshing disabled", level="warn")
            self.backend = "none"

    @property
    def available(self) -> bool:
        return self.backend != "none"

    def integrate_chunk(
        self, points_cam: np.ndarray, colors_rgb: np.ndarray, intrinsics: np.ndarray,
        camera_pose_c2w: np.ndarray, image_shape: tuple[int, int],
    ) -> None:
        """points_cam: (H,W,3) points in *camera* space (z=depth along optical
        axis) for one view; colors_rgb: (H,W,3) uint8. Skips silently (logged
        once by _init) if no TSDF backend is available."""
        if not self.available:
            return
        import open3d as o3d

        h, w = image_shape
        depth = points_cam[..., 2].astype(np.float32)
        depth[depth <= 0] = 0.0

        if self.backend == "open3d_legacy_cpu":
            depth_img = o3d.geometry.Image(depth)
            color_img = o3d.geometry.Image(np.ascontiguousarray(colors_rgb.astype(np.uint8)))
            intr = o3d.camera.PinholeCameraIntrinsic(
                w, h, float(intrinsics[0, 0]), float(intrinsics[1, 1]), float(intrinsics[0, 2]), float(intrinsics[1, 2])
            )
            rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
                color_img, depth_img, depth_scale=1.0, depth_trunc=self.sdf_trunc * 50, convert_rgb_to_intensity=False
            )
            extrinsic = np.linalg.inv(camera_pose_c2w)  # world-to-camera, what Open3D's legacy API expects
            self._volume.integrate(rgbd, intr, extrinsic)
        # Tensor-CUDA integration path intentionally omitted here: its exact
        # VoxelBlockGrid.integrate() call signature needs verification
        # against the installed Open3D version on the actual Kaggle image
        # (API has changed across Open3D releases) — flagged in
        # PHASE0_NOTES.md as a must-verify-on-Kaggle item rather than
        # guessed at here.

    def extract_mesh(self):
        """Returns an open3d.geometry.TriangleMesh via marching cubes, or
        None if unavailable/empty."""
        if not self.available or self.backend != "open3d_legacy_cpu":
            return None
        mesh = self._volume.extract_triangle_mesh()
        if len(mesh.vertices) == 0:
            return None
        mesh.compute_vertex_normals()
        return mesh


In [ ]:
%%writefile sih3d/gpu_monitor.py
"""Background GPU utilization/memory sampler (pynvml), 0.5s cadence.

Publishes GPU_SAMPLE events to the shared bus and keeps an in-memory
per-GPU history so report.py can compute per-stage average utilization
(stage boundaries are correlated by timestamp against STAGE_START/DONE
events already on the bus's log, tracked separately in report.py).
"""

from __future__ import annotations

import threading
import time
from dataclasses import dataclass, field

from .events import EventBus, EventType

try:
    import pynvml

    _NVML_OK = True
except Exception:
    pynvml = None
    _NVML_OK = False


@dataclass
class GpuSample:
    ts: float
    index: int
    name: str
    util_pct: float
    mem_used_mb: float
    mem_total_mb: float


@dataclass
class GpuHistory:
    samples: list[GpuSample] = field(default_factory=list)

    def append(self, s: GpuSample, cap: int = 20000) -> None:
        self.samples.append(s)
        if len(self.samples) > cap:
            del self.samples[: len(self.samples) - cap]

    def average_util(self, index: int, t0: float, t1: float) -> float | None:
        vals = [s.util_pct for s in self.samples if s.index == index and t0 <= s.ts <= t1]
        return sum(vals) / len(vals) if vals else None


class GpuMonitor:
    """Runs a daemon thread sampling every `interval_s` seconds.

    Falls back to a no-GPU / no-pynvml stub that still publishes zeroed
    samples at the same cadence (logged once), so the dashboard's GPU
    panel never has to special-case "no GPU" — it just shows flat lines.
    """

    def __init__(self, bus: EventBus, interval_s: float = 0.5):
        self.bus = bus
        self.interval_s = interval_s
        self.history = GpuHistory()
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None
        self._device_count = 0
        self._handles: list = []
        self._names: list[str] = []
        self._init_nvml()

    def _init_nvml(self) -> None:
        if not _NVML_OK:
            self.bus.log("pynvml not available — GPU panel will show no data", level="warn")
            return
        try:
            pynvml.nvmlInit()
            self._device_count = pynvml.nvmlDeviceGetCount()
            for i in range(self._device_count):
                h = pynvml.nvmlDeviceGetHandleByIndex(i)
                self._handles.append(h)
                name = pynvml.nvmlDeviceGetName(h)
                if isinstance(name, bytes):
                    name = name.decode("utf-8", "ignore")
                self._names.append(name)
        except Exception as e:
            self.bus.log(f"pynvml init failed ({e}) — GPU panel will show no data", level="warn")
            self._device_count = 0

    @property
    def device_count(self) -> int:
        return self._device_count

    @property
    def device_names(self) -> list[str]:
        return list(self._names)

    def start(self) -> None:
        if self._device_count == 0:
            return
        self._stop.clear()
        self._thread = threading.Thread(target=self._run, name="gpu-monitor", daemon=True)
        self._thread.start()

    def stop(self) -> None:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=2.0)

    def _run(self) -> None:
        while not self._stop.is_set():
            t0 = time.time()
            for i, h in enumerate(self._handles):
                try:
                    util = pynvml.nvmlDeviceGetUtilizationRates(h)
                    mem = pynvml.nvmlDeviceGetMemoryInfo(h)
                    sample = GpuSample(
                        ts=t0,
                        index=i,
                        name=self._names[i],
                        util_pct=float(util.gpu),
                        mem_used_mb=mem.used / (1024 * 1024),
                        mem_total_mb=mem.total / (1024 * 1024),
                    )
                except Exception:
                    sample = GpuSample(ts=t0, index=i, name=self._names[i], util_pct=0.0, mem_used_mb=0.0, mem_total_mb=0.0)
                self.history.append(sample)
                self.bus.publish(
                    EventType.GPU_SAMPLE,
                    index=sample.index,
                    name=sample.name,
                    util_pct=sample.util_pct,
                    mem_used_mb=sample.mem_used_mb,
                    mem_total_mb=sample.mem_total_mb,
                    ts=sample.ts,
                )
            elapsed = time.time() - t0
            self._stop.wait(max(0.0, self.interval_s - elapsed))


In [ ]:
%%writefile sih3d/io_detect.py
"""Zero-manual-path input detection.

Recursively scans an input root (Kaggle: /kaggle/input/) for:
  - drone video(s): picks the longest by ffprobe duration, lists the rest
  - telemetry matched to the chosen video (basename first, then folder)
  - camera intrinsics (json/yaml)
  - a cache dataset (previously published wheels/checkpoints)

Never raises on missing optional inputs — callers decide what to do with
`None` telemetry (RGB-only relative-scale fallback is handled in align.py).
"""

from __future__ import annotations

import json
import subprocess
from dataclasses import dataclass, field
from pathlib import Path

VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi", ".m4v", ".ts"}
TELEMETRY_EXTS = {".srt", ".csv", ".gpx", ".kml", ".json", ".tlog", ".bin"}
INTRINSICS_HINT_EXTS = {".json", ".yaml", ".yml"}
INTRINSICS_KEYS = {"fx", "fy", "cx", "cy", "camera_matrix", "K", "intrinsics", "intrinsic_matrix"}
CACHE_DIR_HINTS = {"cache", "checkpoints", "wheels", "sih3d_cache", "sih3d-cache"}
CHECKPOINT_EXTS = {".pth", ".pt", ".safetensors"}
# Name fragments identifying which backbone a fine-tuned checkpoint belongs to
# (UAVFF3D-style filenames, e.g. "mapa_finetuning_.../checkpoint-best.pth",
# "pi3x_finetuning_.../checkpoint-best.pth"). Order matters: more specific
# fragments first so "pi3x" doesn't also match a stray "pi3" substring check.
CHECKPOINT_NAME_HINTS: dict[str, list[str]] = {
    "mapanything": ["mapanything", "map_anything", "map-anything", "mapa"],
    "pi3x": ["pi3x"],
    "pi3": ["pi3"],
    "vggt": ["vggt"],
}


@dataclass
class VideoInfo:
    path: Path
    duration_s: float
    width: int
    height: int
    fps: float
    codec: str = ""


@dataclass
class DetectedInputs:
    video: VideoInfo | None = None
    other_videos: list[VideoInfo] = field(default_factory=list)
    telemetry_path: Path | None = None
    telemetry_kind: str | None = None  # "srt" | "csv" | "gpx" | "kml" | "json" | "mavlink" | "embedded"
    intrinsics_path: Path | None = None
    intrinsics: dict | None = None
    cache_dir: Path | None = None
    checkpoints: dict[str, Path] = field(default_factory=dict)  # backbone key -> checkpoint path
    warnings: list[str] = field(default_factory=list)


def _run_ffprobe(path: Path) -> dict | None:
    try:
        out = subprocess.run(
            [
                "ffprobe", "-v", "error", "-print_format", "json",
                "-show_format", "-show_streams", str(path),
            ],
            capture_output=True, text=True, timeout=30, check=True,
        )
        return json.loads(out.stdout)
    except Exception:
        return None


def probe_video(path: Path) -> VideoInfo | None:
    info = _run_ffprobe(path)
    if info is None:
        return None
    vstream = next((s for s in info.get("streams", []) if s.get("codec_type") == "video"), None)
    if vstream is None:
        return None
    fmt = info.get("format", {})
    duration = float(fmt.get("duration") or vstream.get("duration") or 0.0)
    num, den = (vstream.get("r_frame_rate", "0/1").split("/") + ["1"])[:2]
    fps = float(num) / float(den) if float(den or 1) else 0.0
    return VideoInfo(
        path=path,
        duration_s=duration,
        width=int(vstream.get("width", 0)),
        height=int(vstream.get("height", 0)),
        fps=fps,
        codec=str(vstream.get("codec_name", "")),
    )


def _iter_files(root: Path):
    if not root.exists():
        return
    for p in root.rglob("*"):
        if p.is_file():
            yield p


def find_videos(root: Path) -> list[VideoInfo]:
    videos = []
    for p in _iter_files(root):
        if p.suffix.lower() in VIDEO_EXTS:
            vi = probe_video(p)
            if vi is not None:
                videos.append(vi)
    videos.sort(key=lambda v: v.duration_s, reverse=True)
    return videos


def _basename_stem(p: Path) -> str:
    return p.stem.lower()


def find_telemetry(root: Path, video: VideoInfo) -> tuple[Path | None, str | None]:
    """Match telemetry to the chosen video: basename first, then folder."""
    candidates = [p for p in _iter_files(root) if p.suffix.lower() in TELEMETRY_EXTS]
    if not candidates:
        return None, None

    video_stem = _basename_stem(video.path)
    same_name = [p for p in candidates if _basename_stem(p) == video_stem]
    if same_name:
        chosen = same_name[0]
        return chosen, _kind_from_suffix(chosen)

    same_folder = [p for p in candidates if p.parent == video.path.parent]
    if same_folder:
        # Prefer SRT (richest per-frame gimbal/GPS) if present in-folder.
        srt = [p for p in same_folder if p.suffix.lower() == ".srt"]
        chosen = srt[0] if srt else same_folder[0]
        return chosen, _kind_from_suffix(chosen)

    return None, None


def _kind_from_suffix(p: Path) -> str:
    suf = p.suffix.lower()
    return {
        ".srt": "srt", ".csv": "csv", ".gpx": "gpx", ".kml": "kml",
        ".json": "json", ".tlog": "mavlink", ".bin": "mavlink",
    }.get(suf, suf.lstrip("."))


def find_intrinsics(root: Path) -> tuple[Path | None, dict | None]:
    for p in _iter_files(root):
        if p.suffix.lower() not in INTRINSICS_HINT_EXTS:
            continue
        try:
            if p.suffix.lower() == ".json":
                data = json.loads(p.read_text())
            else:
                import yaml  # deferred import; only needed if a yaml file is present

                data = yaml.safe_load(p.read_text())
        except Exception:
            continue
        if not isinstance(data, dict):
            continue
        flat_keys = set(_flatten_keys(data))
        if flat_keys & INTRINSICS_KEYS:
            return p, data
    return None, None


def _flatten_keys(d: dict, depth: int = 0):
    if depth > 3:
        return
    for k, v in d.items():
        yield k
        if isinstance(v, dict):
            yield from _flatten_keys(v, depth + 1)


def find_cache_dir(input_root: Path) -> Path | None:
    if not input_root.exists():
        return None
    for child in input_root.iterdir():
        if not child.is_dir():
            continue
        name = child.name.lower()
        if any(hint in name for hint in CACHE_DIR_HINTS):
            return child
        # Heuristic: a dataset containing .whl files or a checkpoints/ subdir
        try:
            if any(f.suffix == ".whl" for f in child.rglob("*.whl")):
                return child
            if (child / "checkpoints").exists():
                return child
        except Exception:
            continue
    return None


def find_checkpoints(input_root: Path) -> dict[str, Path]:
    """Find fine-tuned backbone checkpoints anywhere under /kaggle/input.

    Matches by filename fragment against CHECKPOINT_NAME_HINTS (e.g. a
    published UAVFF3D-checkpoints dataset containing
    ".../mapa_finetuning_.../checkpoint-best.pth"). Does not open/validate
    the file — that's backbone.py's job (strict key-checked load, fail
    loudly on mismatch). If multiple candidates match the same backbone,
    prefers a path containing "best", then the most recently modified.
    """
    found: dict[str, list[Path]] = {}
    for p in _iter_files(input_root):
        if p.suffix.lower() not in CHECKPOINT_EXTS:
            continue
        haystack = str(p).lower()
        for backbone_key, hints in CHECKPOINT_NAME_HINTS.items():
            if backbone_key in found:
                continue  # already resolved this backbone key at a higher-priority hint
            if any(hint in haystack for hint in hints):
                found.setdefault(backbone_key, []).append(p)

    resolved: dict[str, Path] = {}
    for backbone_key, candidates in found.items():
        candidates.sort(key=lambda c: (0 if "best" in c.name.lower() else 1, -c.stat().st_mtime))
        resolved[backbone_key] = candidates[0]
    return resolved


def detect_all(input_root: Path = Path("/kaggle/input")) -> DetectedInputs:
    result = DetectedInputs()

    videos = find_videos(input_root)
    if not videos:
        result.warnings.append(f"No video files found under {input_root}")
        return result

    result.video = videos[0]
    result.other_videos = videos[1:]
    if result.other_videos:
        names = ", ".join(v.path.name for v in result.other_videos)
        result.warnings.append(f"Multiple videos found; using longest ({result.video.path.name}). Ignored: {names}")

    tel_path, tel_kind = find_telemetry(input_root, result.video)
    if tel_path is None:
        result.warnings.append("No telemetry file found — will check for embedded subtitle/data stream, then fall back to RGB-only")
    result.telemetry_path = tel_path
    result.telemetry_kind = tel_kind

    intr_path, intr_data = find_intrinsics(input_root)
    result.intrinsics_path = intr_path
    result.intrinsics = intr_data
    if intr_path is None:
        result.warnings.append("No intrinsics file found — will use metadata or model-estimated intrinsics")

    result.cache_dir = find_cache_dir(input_root)

    result.checkpoints = find_checkpoints(input_root)
    if result.checkpoints:
        found_str = ", ".join(f"{k}={v.name}" for k, v in result.checkpoints.items())
        result.warnings.append(f"Fine-tuned checkpoint(s) detected: {found_str}")

    return result


In [ ]:
%%writefile sih3d/keyframes.py
"""Keyframe selection: sharpness (Laplacian variance, torch conv, GPU) + GPS-distance
spacing for target overlap. Entirely in torch so it can run on the same GPU stream as
decoding without a CPU round-trip per frame.
"""

from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np

from .events import EventBus

_LAPLACIAN_KERNEL = [[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]]


@dataclass
class KeyframeCandidate:
    frame_index: int
    timestamp_s: float
    sharpness: float
    accepted: bool
    reject_reason: str | None = None
    gps_enu: tuple[float, float, float] | None = None


@dataclass
class KeyframeSelection:
    accepted: list[KeyframeCandidate] = field(default_factory=list)
    rejected: list[KeyframeCandidate] = field(default_factory=list)


def _laplacian_variance_batch(frames: "torch.Tensor", downscale: int = 4) -> "torch.Tensor":
    """frames: (N,H,W,3) uint8 or float on GPU/CPU. Returns (N,) sharpness scores.

    Downscales first (sharpness selection doesn't need full resolution and this
    keeps the conv cheap), converts to grayscale, and runs a fixed 3x3 Laplacian
    kernel via torch conv2d. Score = variance of the Laplacian response, the
    standard "blur detection" metric.
    """
    import torch
    import torch.nn.functional as F

    if frames.dtype == torch.uint8:
        frames = frames.float()
    n, h, w, _ = frames.shape
    gray = (0.299 * frames[..., 0] + 0.587 * frames[..., 1] + 0.114 * frames[..., 2])  # (N,H,W)
    gray = gray.unsqueeze(1)  # (N,1,H,W)
    if downscale > 1:
        gray = F.avg_pool2d(gray, kernel_size=downscale, stride=downscale)
    kernel = torch.tensor(_LAPLACIAN_KERNEL, dtype=gray.dtype, device=gray.device).view(1, 1, 3, 3)
    # Replicate-pad (not conv2d's default zero-pad) so a real image doesn't
    # get a fake high-frequency edge at the border from an artificial jump
    # to 0 — that would otherwise dominate the variance on a small
    # downsampled grid and inflate scores for flat/dark-bordered frames.
    gray_padded = F.pad(gray, (1, 1, 1, 1), mode="replicate")
    lap = F.conv2d(gray_padded, kernel, padding=0)
    return lap.var(dim=(1, 2, 3))


def compute_sharpness(frame_hwc: np.ndarray, device: str = "cpu") -> float:
    """Single-frame convenience wrapper (used by the dashboard's "current frame"
    panel, which shows one frame's score as it's decoded)."""
    import torch

    t = torch.from_numpy(frame_hwc).unsqueeze(0).to(device)
    return float(_laplacian_variance_batch(t).item())


def compute_sharpness_batch(frames_nhwc: np.ndarray, device: str = "cpu", batch_size: int = 32) -> np.ndarray:
    """Batches the GPU upload/cast instead of putting the whole candidate
    set on the device as one tensor — a real Kaggle run OOM'd here on a 4K
    video: casting a few hundred full-resolution frames to float32 on the
    same device the geometry backbone's weights already occupy needed
    several more GB than was left (each frame at scale_width=1920 is
    already ~6MB as uint8, ~24MB once cast to float32; hundreds of those
    at once adds up fast regardless of what else is resident on the GPU).
    Batching bounds peak memory to one batch's size no matter how many
    candidates there are in total."""
    import torch

    n = len(frames_nhwc)
    if n == 0:
        return np.zeros((0,), dtype=np.float32)
    out = np.empty((n,), dtype=np.float32)
    for i in range(0, n, batch_size):
        t = torch.from_numpy(frames_nhwc[i:i + batch_size]).to(device)
        out[i:i + batch_size] = _laplacian_variance_batch(t).cpu().numpy()
        del t
    return out


def _haversine_like_enu_distance(a: tuple[float, float, float], b: tuple[float, float, float]) -> float:
    return float(np.linalg.norm(np.array(a) - np.array(b)))


def select_keyframes(
    frame_indices: list[int],
    frame_timestamps: list[float],
    frames_hwc: list[np.ndarray],
    gps_enu_per_frame: list[tuple[float, float, float]] | None,
    bus: EventBus,
    sharpness_percentile_floor: float = 15.0,
    min_gps_spacing_m: float = 2.0,
    max_no_gps_frame_stride: int = 5,
    device: str = "cpu",
) -> KeyframeSelection:
    """Two-stage selection:

    1. Sharpness gate: reject frames scoring below `sharpness_percentile_floor`
       of the batch's own sharpness distribution (adapts to the video's overall
       focus quality instead of a fixed threshold).
    2. Spacing gate: among sharpness-surviving frames, greedily keep a frame
       only if it's at least `min_gps_spacing_m` from the last *kept* frame
       (by GPS ENU distance, for target view overlap), or — with no GPS —
       every `max_no_gps_frame_stride`'th surviving frame.

    Every rejection is recorded with a reason so the dashboard's filmstrip can
    grey out rejected frames instead of silently dropping them.
    """
    n = len(frame_indices)
    if n == 0:
        return KeyframeSelection()

    sharpness = compute_sharpness_batch(np.stack(frames_hwc, axis=0), device=device)
    floor = float(np.percentile(sharpness, sharpness_percentile_floor))

    selection = KeyframeSelection()
    last_kept_gps = None
    kept_since_gps_none = 0

    for i in range(n):
        gps = gps_enu_per_frame[i] if gps_enu_per_frame is not None else None
        cand = KeyframeCandidate(
            frame_index=frame_indices[i], timestamp_s=frame_timestamps[i],
            sharpness=float(sharpness[i]), accepted=False, gps_enu=gps,
        )

        if sharpness[i] < floor:
            cand.reject_reason = f"sharpness {sharpness[i]:.1f} below floor {floor:.1f}"
            selection.rejected.append(cand)
            continue

        if gps is not None:
            if last_kept_gps is not None and _haversine_like_enu_distance(gps, last_kept_gps) < min_gps_spacing_m:
                cand.reject_reason = f"GPS spacing < {min_gps_spacing_m}m from last keyframe"
                selection.rejected.append(cand)
                continue
            last_kept_gps = gps
        else:
            kept_since_gps_none += 1
            if kept_since_gps_none % max_no_gps_frame_stride != 1 and n > max_no_gps_frame_stride:
                cand.reject_reason = "frame-stride spacing (no GPS available)"
                selection.rejected.append(cand)
                continue

        cand.accepted = True
        selection.accepted.append(cand)

    bus.log(
        f"Keyframe selection: {len(selection.accepted)} accepted, {len(selection.rejected)} rejected "
        f"(sharpness floor={floor:.1f}, from {n} candidates)"
    )
    return selection


In [ ]:
%%writefile sih3d/live_page.py
"""A small FastAPI server, on its own port (exposed via its own cloudflared
tunnel — see fullscreen_server.start_cloudflared_tunnel, shared code), for
a dedicated live-reconstruction page separate from the in-notebook
dashboard: input video on the left, a live three.js 3D view on the right.

Streams over WebSocket as binary frames (JSON header message immediately
followed by a binary payload message — not base64, not JSON-encoded
numbers), pushed PER KEYFRAME rather than per backbone-inference chunk, so
the client can reveal points frame by frame in sync with the video even
though a whole chunk's geometry becomes available at once computationally.

Every message is kept in an in-memory history list and broadcast to
connected clients. A client that connects late, or reconnects after a
hiccup, gets the full history replayed first, then switches to live tail —
so "replay mode" isn't a special server feature, it's just the client
re-walking the same history array it would build up live, which is also
what makes replay work fully even if live streaming hiccupped.
"""

from __future__ import annotations

import asyncio
import socket
import threading
import time
from pathlib import Path

import numpy as np
# Imported at MODULE level deliberately, not inside _build_app(): this file
# has `from __future__ import annotations`, so every type annotation
# (including `websocket: WebSocket` below) is a lazily-evaluated string.
# FastAPI resolves those strings via typing.get_type_hints(), which looks
# the name up in the function's __globals__ — the ws_endpoint closure's
# enclosing MODULE globals, not _build_app()'s local scope. A local import
# of WebSocket here left it out of those globals entirely, so FastAPI
# silently failed to recognize the parameter as an injected WebSocket
# connection and instead treated "websocket" as a required query string
# parameter — every connection closed with code 1008 ("Field required"),
# which uvicorn's websocket layer in turn reports to the client as a
# generic HTTP 403. Confirmed via TestClient (no network involved) after
# an otherwise-identical inline reproduction without `from __future__
# import annotations` worked fine — isolating the cause to exactly this.
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import FileResponse, HTMLResponse

from .events import EventBus
from .fullscreen_server import start_cloudflared_tunnel


class LivePageServer:
    def __init__(self, video_path: Path, bus: EventBus):
        self.video_path = video_path
        self.bus = bus
        self.history: list[tuple[dict, bytes]] = []
        self._clients: set = set()
        self._loop: "asyncio.AbstractEventLoop | None" = None
        self.port: int | None = None
        self._app = self._build_app()

    # -- server -------------------------------------------------------------

    def _build_app(self):
        app = FastAPI()

        @app.get("/")
        async def index():
            return HTMLResponse(_HTML_TEMPLATE)

        @app.get("/video")
        async def video():
            # Starlette's FileResponse supports Range requests natively,
            # which is what lets the client scrub the video during replay.
            return FileResponse(str(self.video_path))

        @app.websocket("/ws")
        async def ws_endpoint(websocket: WebSocket):
            await websocket.accept()
            self._clients.add(websocket)
            try:
                for header, payload in list(self.history):
                    await websocket.send_json(header)
                    if payload:
                        await websocket.send_bytes(payload)
                while True:
                    await websocket.receive_text()  # keep-alive ping; content ignored
            except WebSocketDisconnect:
                pass
            except Exception:
                pass
            finally:
                self._clients.discard(websocket)

        return app

    def start(self, port: int | None = None) -> int:
        import uvicorn

        if port is None:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.bind(("", 0))
                port = s.getsockname()[1]
        self.port = port

        def _run() -> None:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            self._loop = loop
            config = uvicorn.Config(self._app, host="0.0.0.0", port=port, log_level="warning")
            server = uvicorn.Server(config)
            loop.run_until_complete(server.serve())

        threading.Thread(target=_run, daemon=True, name="sih3d-live-page").start()

        deadline = time.time() + 5.0
        while self._loop is None and time.time() < deadline:
            time.sleep(0.02)

        self.bus.log(f"Live reconstruction page server listening on 127.0.0.1:{port}")
        return port

    def _broadcast(self, header: dict, payload: bytes) -> None:
        self.history.append((header, payload))
        if self._loop is None:
            return

        async def _send() -> None:
            dead = []
            for ws in list(self._clients):
                try:
                    await ws.send_json(header)
                    if payload:
                        await ws.send_bytes(payload)
                except Exception:
                    dead.append(ws)
            for ws in dead:
                self._clients.discard(ws)

        try:
            asyncio.run_coroutine_threadsafe(_send(), self._loop)
        except Exception:
            pass

    # -- push API, called from pipeline.py -----------------------------------

    def push_points(
        self, points: np.ndarray, colors: np.ndarray, confidence: np.ndarray, masked: np.ndarray,
        frame_index: int, timestamp_s: float,
    ) -> None:
        n = len(points)
        header = {"kind": "points", "n": n, "frame_index": frame_index, "timestamp_s": timestamp_s, "seq": len(self.history)}
        payload = (
            points.astype(np.float32).tobytes()
            + np.clip(colors, 0, 255).astype(np.uint8).tobytes()
            + confidence.astype(np.float32).tobytes()
            + masked.astype(np.uint8).tobytes()
        )
        self._broadcast(header, payload)

    def push_camera_pose(
        self, frame_index: int, timestamp_s: float,
        position: np.ndarray, forward: np.ndarray, up: np.ndarray,
    ) -> None:
        header = {
            "kind": "camera_pose", "frame_index": frame_index, "timestamp_s": timestamp_s,
            "position": [float(v) for v in position], "forward": [float(v) for v in forward], "up": [float(v) for v in up],
        }
        self._broadcast(header, b"")

    def push_mesh(
        self, vertices: np.ndarray, indices: np.ndarray,
        colors: np.ndarray | None = None, uv: np.ndarray | None = None,
        texture_png: bytes | None = None, stage: str = "coarse",
    ) -> None:
        parts = [
            vertices.astype(np.float32).tobytes(),
            indices.astype(np.uint32).tobytes(),
            np.clip(colors, 0, 255).astype(np.uint8).tobytes() if colors is not None else b"",
            uv.astype(np.float32).tobytes() if uv is not None else b"",
            texture_png if texture_png is not None else b"",
        ]
        header = {
            "kind": "mesh", "stage": stage, "lengths": [len(p) for p in parts],
            "has_colors": colors is not None, "has_uv": uv is not None, "has_texture": texture_png is not None,
        }
        self._broadcast(header, b"".join(parts))


def start_live_page(video_path: Path, bus: EventBus) -> tuple[LivePageServer | None, str | None]:
    """Starts the server + its own cloudflared tunnel. Never raises —
    returns (None, None) on any failure, logged; the caller (bootstrap.py)
    just doesn't show the "Open live reconstruction page" link and the
    dashboard's inline 3D tab remains the fallback."""
    try:
        server = LivePageServer(video_path, bus)
        port = server.start()
    except Exception as e:
        bus.log(f"Live reconstruction page server failed to start ({e}) — falling back to the dashboard's inline 3D tab", level="warn")
        return None, None

    url = start_cloudflared_tunnel(port, bus, label="live reconstruction page")
    return server, url


_HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>SIH26158 Live Reconstruction</title>
<style>
  html, body { margin:0; height:100%; background:#0b0f14; color:#e6edf3; font-family:-apple-system,sans-serif; overflow:hidden; }
  #layout { display:flex; height:100vh; }
  #left { width:38%; display:flex; flex-direction:column; border-right:1px solid #30363d; }
  #right { flex:1; position:relative; }
  video { width:100%; background:#000; }
  #controls { padding:8px; font-size:12px; display:flex; flex-direction:column; gap:6px; overflow-y:auto; }
  #controls label { display:flex; align-items:center; gap:6px; }
  #timeline { width:100%; }
  button, select { font-size:11px; padding:3px 6px; }
  #hud { position:absolute; top:8px; left:8px; background:rgba(13,17,23,.85); border:1px solid #30363d;
         border-radius:6px; padding:6px 10px; font-size:12px; z-index:5; }
  #status { position:absolute; bottom:8px; left:8px; font-size:11px; color:#8b949e; z-index:5; }
</style>
</head>
<body>
<div id="layout">
  <div id="left">
    <video id="vid" src="/video" muted playsinline preload="auto"></video>
    <div id="controls">
      <div><b>SIH26158 — Live Reconstruction</b></div>
      <label><input type="checkbox" id="chkAutoFollow" checked> Auto-follow camera</label>
      <label><input type="checkbox" id="chkMask"> Show masked-out objects (red)</label>
      <label><input type="checkbox" id="chkConf"> Confidence coloring</label>
      <label><input type="checkbox" id="chkTraj" checked> Trajectory</label>
      <label><input type="checkbox" id="chkMesh" checked> Textured mesh (once available)</label>
      <div>
        <label><input type="radio" name="mode" value="live" id="modeLive" checked> Live</label>
        <label><input type="radio" name="mode" value="replay" id="modeReplay"> Replay</label>
      </div>
      <input type="range" id="timeline" min="0" max="0" value="0" step="1" disabled>
      <div id="timelineLabel">frame -</div>
    </div>
  </div>
  <div id="right">
    <div id="hud">Points: 0 shown / 0 total</div>
    <div id="status">connecting...</div>
  </div>
</div>

<script type="module">
import * as THREE from "https://esm.sh/three@0.160.0";
import { OrbitControls } from "https://esm.sh/three@0.160.0/examples/jsm/controls/OrbitControls.js";

const vid = document.getElementById("vid");
// `preload="auto"` alone isn't reliably honored by every browser in every
// context (confirmed directly: readyState stayed 0/HAVE_NOTHING on first
// load here until an explicit .load() was called, after which it went
// straight to 4/HAVE_ENOUGH_DATA) — force it deterministically.
vid.load();
const hud = document.getElementById("hud");
const statusEl = document.getElementById("status");
const timeline = document.getElementById("timeline");
const timelineLabel = document.getElementById("timelineLabel");
const chkAutoFollow = document.getElementById("chkAutoFollow");
const chkMask = document.getElementById("chkMask");
const chkConf = document.getElementById("chkConf");
const chkTraj = document.getElementById("chkTraj");
const chkMesh = document.getElementById("chkMesh");
const modeLive = document.getElementById("modeLive");
const modeReplay = document.getElementById("modeReplay");

const rightEl = document.getElementById("right");
const scene = new THREE.Scene();
scene.background = new THREE.Color(0x0b0f14);
const camera = new THREE.PerspectiveCamera(60, 1, 0.01, 1e6);
camera.position.set(20, 20, 20);
const renderer = new THREE.WebGLRenderer({ antialias: true });
rightEl.appendChild(renderer.domElement);
scene.add(new THREE.AmbientLight(0xffffff, 0.8));
const dirLight = new THREE.DirectionalLight(0xffffff, 0.6);
dirLight.position.set(1, 2, 1);
scene.add(dirLight);
scene.add(new THREE.AxesHelper(5));

const orbit = new OrbitControls(camera, renderer.domElement);
orbit.enableDamping = true;

function resize() {
  const w = rightEl.clientWidth, h = rightEl.clientHeight;
  camera.aspect = w / h;
  camera.updateProjectionMatrix();
  renderer.setSize(w, h);
}
new ResizeObserver(resize).observe(rightEl);
resize();

// -- point history: every "points" message ever received, in arrival order.
// Both LIVE and REPLAY draw from this same array — replay just re-walks it
// up to a scrubbed index, so it works even if the live WS connection
// hiccupped (the server replays full history to a (re)connecting client).
const pointHistory = []; // {n, frame_index, timestamp_s, xyz, rgb, conf, masked}
let totalPointsSeen = 0;
let revealIndex = 0; // how many history entries are currently shown

const posArr = [];
const colArr = [];
const pointsGeo = new THREE.BufferGeometry();
const pointsMat = new THREE.PointsMaterial({ size: 0.03, vertexColors: true });
const pointsObj = new THREE.Points(pointsGeo, pointsMat);
scene.add(pointsObj);

function rebuildPointsUpTo(idx) {
  let positions = [], colors = [];
  for (let i = 0; i < idx; i++) {
    const e = pointHistory[i];
    for (let j = 0; j < e.n; j++) {
      positions.push(e.xyz[j*3], e.xyz[j*3+1], e.xyz[j*3+2]);
      if (chkMask.checked && e.masked[j]) {
        colors.push(1, 0, 0);
      } else if (chkConf.checked) {
        const v = e.conf[j];
        colors.push(Math.min(1, v*2), Math.min(1, 2-v*2), Math.max(0, 1-v*2));
      } else {
        colors.push(e.rgb[j*3]/255, e.rgb[j*3+1]/255, e.rgb[j*3+2]/255);
      }
    }
  }
  pointsGeo.setAttribute("position", new THREE.Float32BufferAttribute(positions, 3));
  pointsGeo.setAttribute("color", new THREE.Float32BufferAttribute(colors, 3));
  pointsGeo.computeBoundingSphere();
  hud.textContent = `Points: ${(positions.length/3).toLocaleString()} shown / ${totalPointsSeen.toLocaleString()} total`;
}

// -- camera frustum + trajectory ------------------------------------------
const frustum = new THREE.CameraHelper(new THREE.PerspectiveCamera(50, 1.3, 0.1, 2));
scene.add(frustum);
let trajGroup = new THREE.Group();
scene.add(trajGroup);
const trajPoints = [];

function updateFrustum(pos, forward, up) {
  frustum.camera.position.set(pos[0], pos[1], pos[2]);
  const target = [pos[0]+forward[0], pos[1]+forward[1], pos[2]+forward[2]];
  frustum.camera.up.set(up[0], up[1], up[2]);
  frustum.camera.lookAt(target[0], target[1], target[2]);
  frustum.camera.updateProjectionMatrix();
  frustum.update();

  trajPoints.push(new THREE.Vector3(pos[0], pos[1], pos[2]));
  trajGroup.clear();
  if (trajPoints.length >= 2) {
    const geo = new THREE.BufferGeometry().setFromPoints(trajPoints);
    trajGroup.add(new THREE.Line(geo, new THREE.LineBasicMaterial({ color: 0x2ea043 })));
  }

  if (chkAutoFollow.checked) {
    orbit.target.set(pos[0], pos[1], pos[2]);
    camera.position.set(pos[0] - forward[0]*10, pos[1] - forward[1]*10, pos[2] - forward[2]*10 + 5);
  }
}

// -- mesh -------------------------------------------------------------------
let meshObj = null;
function applyMesh(header, buf) {
  const [vLen, iLen, cLen, uvLen, texLen] = header.lengths;
  let off = 0;
  const vertices = new Float32Array(buf, off, vLen/4); off += vLen;
  const indices = new Uint32Array(buf, off, iLen/4); off += iLen;
  const colorsBuf = header.has_colors ? new Uint8Array(buf, off, cLen) : null; off += cLen;
  const uvBuf = header.has_uv ? new Float32Array(buf, off, uvLen/4) : null; off += uvLen;
  const texBuf = header.has_texture ? new Uint8Array(buf, off, texLen) : null;

  const geo = new THREE.BufferGeometry();
  geo.setAttribute("position", new THREE.BufferAttribute(vertices, 3));
  geo.setIndex(new THREE.BufferAttribute(indices, 1));
  let material;
  if (header.has_uv && header.has_texture) {
    geo.setAttribute("uv", new THREE.BufferAttribute(uvBuf, 2));
    const blob = new Blob([texBuf], { type: "image/png" });
    const url = URL.createObjectURL(blob);
    const tex = new THREE.TextureLoader().load(url, () => URL.revokeObjectURL(url));
    material = new THREE.MeshStandardMaterial({ map: tex });
  } else if (header.has_colors) {
    const colorsF32 = new Float32Array(colorsBuf.length);
    for (let i = 0; i < colorsBuf.length; i++) colorsF32[i] = colorsBuf[i] / 255;
    geo.setAttribute("color", new THREE.BufferAttribute(colorsF32, 3));
    material = new THREE.MeshStandardMaterial({ vertexColors: true });
  } else {
    material = new THREE.MeshStandardMaterial({ color: 0x888888 });
  }
  geo.computeVertexNormals();
  if (meshObj) scene.remove(meshObj);
  meshObj = new THREE.Mesh(geo, material);
  meshObj.visible = chkMesh.checked;
  scene.add(meshObj);
}

// -- WebSocket: JSON header message immediately followed by a binary message
let pendingHeader = null;
function connect() {
  const proto = location.protocol === "https:" ? "wss:" : "ws:";
  const ws = new WebSocket(`${proto}//${location.host}/ws`);
  ws.binaryType = "arraybuffer";
  ws.onopen = () => { statusEl.textContent = "connected"; };
  ws.onclose = () => { statusEl.textContent = "disconnected — retrying in 2s"; setTimeout(connect, 2000); };
  ws.onerror = () => { statusEl.textContent = "connection error"; };
  ws.onmessage = (event) => {
    if (typeof event.data === "string") {
      const header = JSON.parse(event.data);
      if (header.kind === "camera_pose") {
        if (modeLive.checked) updateFrustum(header.position, header.forward, header.up);
        return; // no binary payload for camera_pose
      }
      pendingHeader = header;
    } else {
      const header = pendingHeader;
      pendingHeader = null;
      if (!header) return;
      if (header.kind === "points") {
        const buf = event.data;
        let off = 0;
        const xyz = new Float32Array(buf, off, header.n*3); off += header.n*3*4;
        const rgb = new Uint8Array(buf, off, header.n*3); off += header.n*3;
        const conf = new Float32Array(buf, off, header.n); off += header.n*4;
        const masked = new Uint8Array(buf, off, header.n);
        pointHistory.push({ n: header.n, frame_index: header.frame_index, timestamp_s: header.timestamp_s, xyz, rgb, conf, masked });
        totalPointsSeen += header.n;
        timeline.max = pointHistory.length - 1;
        if (modeLive.checked) {
          revealIndex = pointHistory.length;
          rebuildPointsUpTo(revealIndex);
          if (chkAutoFollow.checked || vid.paused) vid.currentTime = header.timestamp_s;
        }
      } else if (header.kind === "mesh") {
        if (chkMesh.checked || true) applyMesh(header, event.data);
      }
    }
  };
}
connect();

// -- toggles ------------------------------------------------------------
chkMask.onchange = chkConf.onchange = () => rebuildPointsUpTo(revealIndex);
chkTraj.onchange = () => { trajGroup.visible = chkTraj.checked; };
chkMesh.onchange = () => { if (meshObj) meshObj.visible = chkMesh.checked; };

// -- replay mode ----------------------------------------------------------
modeReplay.onchange = () => { timeline.disabled = !modeReplay.checked; vid.pause(); };
modeLive.onchange = () => { timeline.disabled = true; };
timeline.oninput = () => {
  const idx = parseInt(timeline.value, 10);
  revealIndex = idx + 1;
  rebuildPointsUpTo(revealIndex);
  if (pointHistory[idx]) {
    vid.currentTime = pointHistory[idx].timestamp_s;
    timelineLabel.textContent = `frame ${pointHistory[idx].frame_index} (t=${pointHistory[idx].timestamp_s.toFixed(2)}s)`;
  }
};

// -- render loop -----------------------------------------------------------
function animate() {
  requestAnimationFrame(animate);
  orbit.update();
  renderer.render(scene, camera);
}
animate();
</script>
</body>
</html>
"""


In [ ]:
%%writefile sih3d/masks.py
"""Dynamic-object masking: a small fp16 segmentation model (YOLO-seg) flags
vehicles/people/animals so fusion.py can exclude those pixels from the point
cloud (moving objects corrupt multi-view fusion — they aren't in the same
place across views the way static scene geometry is).

Designed to run on GPU1 while GPU0 runs the geometry backbone on the next
chunk (see the pipeline's two-GPU producer/consumer wiring) — this module
only does the segmentation itself; the threading/queueing lives in the
pipeline orchestration, not here.
"""

from __future__ import annotations

import time
from dataclasses import dataclass

import numpy as np

from .events import EventBus

# COCO class indices YOLO-seg models are typically trained on, restricted to
# things that actually move in drone footage.
_DYNAMIC_CLASS_NAMES = {
    "person", "bicycle", "car", "motorcycle", "bus", "truck", "train", "boat",
    "bird", "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe",
}


@dataclass
class MaskResult:
    mask: np.ndarray            # (H,W) bool, True = dynamic/exclude-from-fusion
    detections: int
    backend: str
    timing_s: float


class DynamicObjectMasker:
    """Lazily loads a YOLO-seg model on first use (so importing this module
    never requires ultralytics/torch until masking is actually enabled).
    Falls back to an all-static (empty) mask, logged once, if the model
    can't be loaded — dynamic-object exclusion is a quality improvement,
    not a hard requirement, so this must never crash the pipeline."""

    def __init__(self, bus: EventBus, device: str = "cuda:1", model_name: str = "yolo11n-seg.pt", confidence: float = 0.35):
        self.bus = bus
        self.device = device
        self.model_name = model_name
        self.confidence = confidence
        self.model = None
        self.backend = "none"
        self._load_attempted = False

    def _ensure_loaded(self) -> None:
        if self._load_attempted:
            return
        self._load_attempted = True
        try:
            from ultralytics import YOLO

            t0 = time.time()
            self.model = YOLO(self.model_name)
            self.model.to(self.device)
            self.backend = "yolo-seg"
            self.bus.log(f"Dynamic-object masker: loaded {self.model_name} on {self.device} in {time.time() - t0:.1f}s")
        except Exception as e:
            self.backend = "none"
            self.bus.log(
                f"Dynamic-object masker unavailable ({type(e).__name__}: {e}) — "
                f"proceeding with no dynamic-object exclusion (all-static mask)",
                level="warn",
            )

    def mask_frame(self, frame_hwc: np.ndarray) -> MaskResult:
        self._ensure_loaded()
        t0 = time.time()
        h, w = frame_hwc.shape[:2]

        if self.model is None:
            return MaskResult(mask=np.zeros((h, w), dtype=bool), detections=0, backend="none", timing_s=time.time() - t0)

        try:
            results = self.model.predict(frame_hwc, verbose=False, conf=self.confidence, half=True, device=self.device)
        except Exception as e:
            self.bus.log(f"Dynamic-object mask inference failed on one frame ({e}); treating as all-static", level="warn")
            return MaskResult(mask=np.zeros((h, w), dtype=bool), detections=0, backend="error_fallback", timing_s=time.time() - t0)

        mask = np.zeros((h, w), dtype=bool)
        n_det = 0
        for r in results:
            if r.masks is None:
                continue
            names = r.names
            for seg_mask, cls_idx in zip(r.masks.data, r.boxes.cls):
                cls_name = names.get(int(cls_idx), "") if isinstance(names, dict) else str(names[int(cls_idx)])
                if cls_name not in _DYNAMIC_CLASS_NAMES:
                    continue
                m = seg_mask.detach().float().cpu().numpy()
                if m.shape != (h, w):
                    import cv2

                    m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
                mask |= m > 0.5
                n_det += 1

        return MaskResult(mask=mask, detections=n_det, backend=self.backend, timing_s=time.time() - t0)

    def mask_batch(self, frames_hwc: list[np.ndarray]) -> list[MaskResult]:
        return [self.mask_frame(f) for f in frames_hwc]


In [ ]:
%%writefile sih3d/mesh.py
"""Mesh generation: vertex-colored mesh first (always), then best-effort
texture baking from full-resolution keyframes.

Vertex-colored mesh: from the TSDF volume via marching cubes if
`fusion.Open3DTsdfFusion` produced one (denser/smoother), else directly from
the fused point cloud via Open3D Poisson surface reconstruction. If Open3D
itself isn't available at all, meshing is skipped with a clearly logged
reason — the pipeline still ships the point cloud (LAS/PLY), matching the
robustness rule "log clearly if unavailable; never crash".

Texture baking: xatlas UV-unwraps the mesh, then for each face we pick the
"best view" among the full-resolution keyframes (most fronto-parallel,
i.e. maximizing -dot(face_normal, view_direction), among cameras where the
face's centroid actually projects inside the image) and fill that face's
triangle in the atlas with the color sampled at its centroid's projection.
This is a flat-per-face bake (constant color per triangle, not full per-texel
projective sampling) — deliberately simpler than a full rasterizer, and
still literally matches the task's own phrasing ("best-view per face").
Skipped (logged) if it would blow the time budget or if xatlas isn't
installed.
"""

from __future__ import annotations

import time
from dataclasses import dataclass, field

import numpy as np

from .events import EventBus
from .fusion import FusedPointCloud, Open3DTsdfFusion


@dataclass
class MeshResult:
    mesh: "object | None"       # open3d.geometry.TriangleMesh, vertex-colored
    method: str                 # "tsdf_marching_cubes" | "poisson" | "none"
    n_vertices: int = 0
    n_faces: int = 0


@dataclass
class TextureBakeResult:
    textured: bool
    texture_rgb: np.ndarray | None = None   # (H,W,3) uint8
    uv: np.ndarray | None = None             # (n_faces*3, 2) float32, per-face-corner UVs
    skipped_reason: str | None = None
    timing_s: float = 0.0


def build_vertex_colored_mesh(
    cloud: FusedPointCloud, tsdf: Open3DTsdfFusion | None, bus: EventBus,
    poisson_depth: int = 9,
) -> MeshResult:
    if tsdf is not None and tsdf.available:
        mesh = tsdf.extract_mesh()
        if mesh is not None:
            bus.log(f"Mesh: extracted via TSDF marching cubes ({len(mesh.vertices)} vertices)")
            return MeshResult(mesh=mesh, method="tsdf_marching_cubes", n_vertices=len(mesh.vertices), n_faces=len(mesh.triangles))
        bus.log("TSDF available but produced an empty mesh; falling back to point-based Poisson meshing", level="warn")

    try:
        import open3d as o3d
    except Exception as e:
        bus.log(f"Open3D not available ({e}) — mesh generation skipped, point cloud outputs are unaffected", level="warn")
        return MeshResult(mesh=None, method="none")

    if len(cloud.points) < 10:
        bus.log("Too few fused points for meshing — mesh generation skipped", level="warn")
        return MeshResult(mesh=None, method="none")

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(cloud.points)
    colors01 = np.clip(cloud.colors, 0, 255) / 255.0
    pcd.colors = o3d.utility.Vector3dVector(colors01)
    pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.5, max_nn=30))
    pcd.orient_normals_consistent_tangent_plane(30)
    normals = np.asarray(pcd.normals)

    # Open3D's native Poisson/ball-pivoting solvers can hard-abort the whole
    # process on degenerate/pathological point distributions (observed
    # directly: a synthetic test point cloud crashed the interpreter with
    # "libc++abi: terminating" from deep inside PoissonRecon's C++ — no
    # Python try/except can catch a native abort). Both are run in an
    # isolated subprocess so a crash there kills only that subprocess; the
    # pipeline sees it as an ordinary failure and falls back normally.
    method = "poisson"
    poisson_out = _run_isolated_meshing(_poisson_worker, (cloud.points, colors01, normals, poisson_depth), bus, "Poisson")
    if poisson_out is not None:
        vertices, triangles, vcolors = poisson_out
    else:
        bus.log("Poisson reconstruction failed/crashed; trying ball-pivoting as a second fallback", level="warn")
        distances = pcd.compute_nearest_neighbor_distance()
        avg_dist = float(np.mean(distances)) if len(distances) else 0.05
        radii = [avg_dist * r for r in (1.5, 2.0, 3.0)]
        bp_out = _run_isolated_meshing(_ball_pivot_worker, (cloud.points, colors01, normals, radii), bus, "ball-pivoting")
        if bp_out is not None:
            vertices, triangles, vcolors = bp_out
            method = "ball_pivoting"
        else:
            bus.log(
                "Ball-pivoting also failed/crashed; trying 2.5D Delaunay triangulation as a third fallback "
                "(a ground-projected heightfield mesh — often the more robust choice for open, nadir-view "
                "aerial point clouds anyway, since Poisson/ball-pivoting assume more closed/volumetric input)",
                level="warn",
            )
            dt_out = _run_isolated_meshing(_delaunay_2p5d_worker, (cloud.points, colors01), bus, "Delaunay 2.5D", timeout_s=30.0)
            if dt_out is None:
                bus.log("2.5D Delaunay also failed — mesh generation skipped", level="warn")
                return MeshResult(mesh=None, method="none")
            vertices, triangles, vcolors = dt_out
            method = "delaunay_2p5d"

    if len(vertices) == 0:
        bus.log("Meshing produced zero vertices — mesh generation skipped", level="warn")
        return MeshResult(mesh=None, method="none")

    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices = o3d.utility.Vector3dVector(vertices)
    mesh.triangles = o3d.utility.Vector3iVector(triangles)
    if vcolors is not None:
        mesh.vertex_colors = o3d.utility.Vector3dVector(vcolors)
    mesh.compute_vertex_normals()
    bus.log(f"Mesh: built via {method} from point cloud ({len(mesh.vertices)} vertices, {len(mesh.triangles)} faces)")
    return MeshResult(mesh=mesh, method=method, n_vertices=len(mesh.vertices), n_faces=len(mesh.triangles))


def _poisson_worker(points, colors, normals, depth, result_queue) -> None:
    import numpy as _np
    import open3d as _o3d

    try:
        pcd = _o3d.geometry.PointCloud()
        pcd.points = _o3d.utility.Vector3dVector(points)
        pcd.colors = _o3d.utility.Vector3dVector(colors)
        pcd.normals = _o3d.utility.Vector3dVector(normals)
        mesh, densities = _o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=depth)
        densities = _np.asarray(densities)
        # Trim low-density (extrapolated/hallucinated) vertices at the
        # reconstruction's outer fringe — Poisson fills holes by design,
        # which without trimming produces a bloated blob past the actual
        # observed surface.
        keep = densities >= _np.quantile(densities, 0.02)
        mesh.remove_vertices_by_mask(~keep)
        mesh.remove_degenerate_triangles()
        mesh.remove_unreferenced_vertices()
        result_queue.put((
            _np.asarray(mesh.vertices), _np.asarray(mesh.triangles),
            _np.asarray(mesh.vertex_colors) if mesh.has_vertex_colors() else None,
        ))
    except Exception as e:
        result_queue.put(RuntimeError(f"{type(e).__name__}: {e}"))


def _ball_pivot_worker(points, colors, normals, radii, result_queue) -> None:
    import numpy as _np
    import open3d as _o3d

    try:
        pcd = _o3d.geometry.PointCloud()
        pcd.points = _o3d.utility.Vector3dVector(points)
        pcd.colors = _o3d.utility.Vector3dVector(colors)
        pcd.normals = _o3d.utility.Vector3dVector(normals)
        mesh = _o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(pcd, _o3d.utility.DoubleVector(radii))
        result_queue.put((
            _np.asarray(mesh.vertices), _np.asarray(mesh.triangles),
            _np.asarray(mesh.vertex_colors) if mesh.has_vertex_colors() else None,
        ))
    except Exception as e:
        result_queue.put(RuntimeError(f"{type(e).__name__}: {e}"))


def _delaunay_2p5d_worker(points, colors, result_queue) -> None:
    """Ground-projected 2.5D triangulation: Delaunay on the XY projection,
    Z carried through as height. A much more mature, robust algorithm than
    Poisson/ball-pivoting for this specific shape of input (scipy's Qhull
    binding essentially never crashes or hangs on a well-formed 2D point
    set), and arguably the more *appropriate* one for open, single-pass
    nadir aerial capture in the first place — it's the same footprint
    concept as the DSM raster (export.py's export_dsm_orthomosaic), just
    kept as an actual triangle mesh instead of a rasterized grid."""
    import numpy as _np
    from scipy.spatial import Delaunay as _Delaunay

    try:
        xy = points[:, :2]
        tri = _Delaunay(xy)
        simplices = tri.simplices

        p0, p1, p2 = points[simplices[:, 0]], points[simplices[:, 1]], points[simplices[:, 2]]
        edge_lens = _np.stack([
            _np.linalg.norm(p0[:, :2] - p1[:, :2], axis=1),
            _np.linalg.norm(p1[:, :2] - p2[:, :2], axis=1),
            _np.linalg.norm(p2[:, :2] - p0[:, :2], axis=1),
        ], axis=1)
        max_edge = edge_lens.max(axis=1)
        # Delaunay triangulates the full convex hull, including spurious
        # triangles that bridge across real gaps in an open point cloud
        # (unobserved areas). Drop the longest outlier edges.
        thresh = max(float(_np.percentile(max_edge, 95)) * 2.0, 1e-6)
        simplices = simplices[max_edge < thresh]

        result_queue.put((points, simplices, colors))
    except Exception as e:
        result_queue.put(RuntimeError(f"{type(e).__name__}: {e}"))


def _run_isolated_meshing(worker_fn, args: tuple, bus: EventBus, label: str, timeout_s: float = 45.0):
    """Runs `worker_fn(*args, result_queue)` in a subprocess. Returns
    (vertices, triangles, colors) on success, None on any failure —
    including a native crash (nonzero/None exit code) or a timeout, both
    logged the same way as an ordinary caught exception.

    Drains the queue BEFORE calling proc.join() — calling join() first is a
    classic multiprocessing deadlock (documented in Python's own docs, and
    hit directly here): a mesh result is a few MB, comfortably over the OS
    pipe buffer, so Queue.put() in the child blocks on its background
    feeder thread until the parent reads, and the child can't fully exit
    (so join() never returns) until put() finishes. Poisson's occasional
    fast "crashed" report earlier masked this — it usually aborted before
    ever reaching put() — but ball-pivoting and even a plain scipy Delaunay
    call (independently confirmed to run in ~0.2s standalone) both hung
    until forcibly killed, because their results *did* reach put().
    """
    import multiprocessing as mp

    # "fork" copies the already-fully-loaded parent process (torch/open3d/
    # scipy already imported) via copy-on-write, essentially instant.
    # "spawn" re-imports everything from scratch in the child — observed
    # locally to cost 15-20s+ per attempt just on startup. Kaggle (Linux)
    # supports fork; only platforms without it (Windows) fall back to spawn.
    ctx = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else mp.get_context("spawn")
    result_queue: "mp.Queue" = ctx.Queue()
    proc = ctx.Process(target=worker_fn, args=(*args, result_queue))
    proc.start()

    deadline = time.time() + timeout_s
    result = _SENTINEL_NO_RESULT = object()
    while time.time() < deadline:
        try:
            result = result_queue.get(timeout=0.1)
            break
        except Exception:
            pass
        if not proc.is_alive():
            break

    if result is _SENTINEL_NO_RESULT:
        try:
            result = result_queue.get_nowait()
        except Exception:
            result = _SENTINEL_NO_RESULT

    if proc.is_alive():
        proc.terminate()
    proc.join(5)

    if result is not _SENTINEL_NO_RESULT:
        if isinstance(result, Exception):
            bus.log(f"{label} failed: {result}", level="warn")
            return None
        return result

    if proc.exitcode not in (0, None) and proc.exitcode != -15:  # -15 = our own terminate()
        bus.log(f"{label} subprocess crashed (exit code {proc.exitcode}, likely a native library abort) — isolated, pipeline continues", level="warn")
    else:
        bus.log(f"{label} timed out after {timeout_s:.0f}s — treating as failed", level="warn")
    return None


@dataclass
class KeyframeForBaking:
    image_rgb: np.ndarray        # (H,W,3) uint8, full resolution
    camera_pose_c2w: np.ndarray  # (4,4)
    intrinsics: np.ndarray       # (3,3)


def _project_points(points_world: np.ndarray, camera_pose_c2w: np.ndarray, intrinsics: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Returns (pixel_xy (N,2), depth (N,)) — depth <= 0 means behind the camera."""
    w2c = np.linalg.inv(camera_pose_c2w)
    pts_h = np.concatenate([points_world, np.ones((len(points_world), 1))], axis=1)
    pts_cam = (w2c @ pts_h.T).T[:, :3]
    depth = pts_cam[:, 2]
    with np.errstate(divide="ignore", invalid="ignore"):
        pix = (intrinsics @ pts_cam.T).T
        pix_xy = pix[:, :2] / pix[:, 2:3]
    return pix_xy, depth


def bake_texture(
    mesh, keyframes: list[KeyframeForBaking], bus: EventBus,
    time_budget_s: float = 60.0, atlas_resolution: int = 2048,
) -> TextureBakeResult:
    t0 = time.time()
    try:
        import xatlas
    except Exception as e:
        return TextureBakeResult(textured=False, skipped_reason=f"xatlas not installed ({e})")

    if mesh is None or len(mesh.triangles) == 0:
        return TextureBakeResult(textured=False, skipped_reason="no mesh to texture")
    if not keyframes:
        return TextureBakeResult(textured=False, skipped_reason="no full-resolution keyframes available for baking")

    try:
        from PIL import Image, ImageDraw
    except Exception as e:
        return TextureBakeResult(textured=False, skipped_reason=f"PIL not available ({e})")

    vertices = np.asarray(mesh.vertices)
    faces = np.asarray(mesh.triangles)
    normals = np.asarray(mesh.triangle_normals) if mesh.has_triangle_normals() else None
    if normals is None:
        mesh.compute_triangle_normals()
        normals = np.asarray(mesh.triangle_normals)

    bus.log(f"Texture baking: UV-unwrapping {len(faces)} faces via xatlas...")
    vmapping, indices, uvs = xatlas.parametrize(vertices, faces)
    # xatlas may duplicate/reorder vertices at UV seams; `vmapping` maps each
    # new (post-unwrap) vertex back to its original vertex index.
    unwrapped_positions = vertices[vmapping]
    face_centroids_world = unwrapped_positions[indices].mean(axis=1)  # (n_faces, 3), matches `indices`' face order
    face_normals = normals  # original per-triangle normals, same face order as `faces`/`indices` (xatlas preserves face order)

    atlas = Image.new("RGB", (atlas_resolution, atlas_resolution), (128, 128, 128))
    draw = ImageDraw.Draw(atlas)

    n_faces = len(indices)
    baked = 0
    skipped_time_budget = False

    for fi in range(n_faces):
        if time.time() - t0 > time_budget_s:
            skipped_time_budget = True
            break

        centroid = face_centroids_world[fi]
        normal = face_normals[fi] if fi < len(face_normals) else np.array([0.0, 0.0, 1.0])

        best_score = -1.0
        best_color = None
        for kf in keyframes:
            view_dir = centroid - kf.camera_pose_c2w[:3, 3]
            dist = np.linalg.norm(view_dir)
            if dist < 1e-6:
                continue
            view_dir = view_dir / dist
            facing = -float(np.dot(normal, view_dir))
            if facing <= 0.05:  # back-facing or near-grazing: unusable
                continue

            pix, depth = _project_points(centroid[None, :], kf.camera_pose_c2w, kf.intrinsics)
            if depth[0] <= 0:
                continue
            x, y = pix[0]
            h, w = kf.image_rgb.shape[:2]
            if not (0 <= x < w and 0 <= y < h):
                continue

            score = facing / max(dist, 1e-3)  # prefer fronto-parallel and close
            if score > best_score:
                best_score = score
                best_color = kf.image_rgb[int(y), int(x)]

        if best_color is None:
            continue

        uv_tri = uvs[indices[fi]] * atlas_resolution
        poly = [(float(uv_tri[k, 0]), float(atlas_resolution - uv_tri[k, 1])) for k in range(3)]
        draw.polygon(poly, fill=tuple(int(c) for c in best_color))
        baked += 1

    elapsed = time.time() - t0
    if skipped_time_budget:
        bus.log(f"Texture baking hit the time budget ({time_budget_s:.0f}s) after {baked}/{n_faces} faces — using partial bake", level="warn")
    else:
        bus.log(f"Texture baking: {baked}/{n_faces} faces colored in {elapsed:.1f}s")

    if baked == 0:
        return TextureBakeResult(textured=False, skipped_reason="no face could be matched to any keyframe view", timing_s=elapsed)

    return TextureBakeResult(
        textured=True, texture_rgb=np.array(atlas), uv=uvs[indices].reshape(-1, 2).astype(np.float32),
        timing_s=elapsed,
    )


In [ ]:
%%writefile sih3d/pipeline.py
"""Pipeline orchestration (v1): correctness and a working QUICK run first,
deep GPU overlap later.

v1 design, deliberately simple:
  - One background pipeline thread does: input scan -> frame extraction +
    keyframe selection -> per-keyframe pose/intrinsics priors -> a chunk loop
    that runs the geometry backbone (GPU0) and immediately gravity-fixed-
    aligns each chunk.
  - When 2 GPUs are detected, a second worker thread (GPU1) consumes raw
    chunk results from a bounded queue and does dynamic-object masking +
    voxel fusion, so GPU0 can start the next chunk's backbone inference
    without waiting on masking/fusion. On one GPU (or CPU), masking+fusion
    just run inline in the same thread right after the backbone call.
  - v2 (not implemented here) would add CUDA-stream overlap for
    decode/copy/compute and let GPU1 also prefetch the *next* chunk's
    frames while GPU0 is still busy — see decode.py's docstring. Adding
    that before v1 produces a single correct end-to-end QUICK run on real
    Kaggle hardware would be optimizing something we haven't confirmed
    works yet.

Every stage is wrapped so a failure inside it degrades (logs a fallback,
returns something the rest of the pipeline can keep going with) rather than
killing the run. Only the one-time setup step (selecting/loading the
backbone) is allowed to be fatal — there's nothing useful to do without a
model — and even then the run still writes whatever partial report/outputs
exist before re-raising, and the exception+traceback are captured on
`PipelineResult` for the notebook's debug cell.

v1 simplifications, called out explicitly (not silently cut corners):
  - Each chunk's point cloud is fused into `VoxelPointFusion` using that
    chunk's own direct alignment (gravity-fixed 4-DoF vs GPS, or Sim(3) vs
    the previous chunk with no GPS) as soon as it's computed. The end-of-run
    pose-graph refinement (align.refine_pose_graph) is used to report a more
    accurate global alignment RMSE and to write a consistent
    trajectory.geojson/.kml, but does NOT retroactively re-fuse
    already-accumulated points with the refined transforms (that would need
    keeping every chunk's raw pre-alignment points in memory, which we
    deliberately don't for a first working run). A v2 could re-fuse.
  - No TSDF integration in v1 (mesh.py's Open3DTsdfFusion path is left
    unused here) — meshing goes straight from the fused point cloud via
    Poisson, which is already a real, working fallback path with its own
    tests. TSDF's tensor-CUDA call site was intentionally left unverified
    against Kaggle's actual Open3D version (see PHASE0_NOTES.md); wiring it
    in before that's confirmed would just be guessing.
  - Full-resolution keyframe images (needed for texture baking) are kept
    in memory rather than re-decoded on demand, bounded by QUICK mode's own
    keyframe cap. FULL mode with many more/larger keyframes may need this
    revisited during FULL-mode performance tuning (a later, explicit step).
"""

from __future__ import annotations

import queue
import threading
import time
import traceback
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np

from . import align, export, masks
from . import backbone as backbone_mod
from . import mesh as mesh_mod
from .artifacts import ChunkAlignmentRecord, ChunkGeometrySample, KeyframeRecord, RunArtifacts
from .backbone import ChunkResult, PriorMode, ViewInput
from .decode import FrameDecoder
from .events import Event, EventBus, EventType
from .fusion import FusedPointCloud, VoxelPointFusion, remove_statistical_outliers
from .gpu_monitor import GpuMonitor
from .io_detect import DetectedInputs
from .keyframes import compute_sharpness, select_keyframes
from .mesh import KeyframeForBaking
from .report import ReportBuilder
from .telemetry import (
    TelemetryTrack, collinearity_index as compute_collinearity, sync_to_frame_times, track_to_enu,
)


@dataclass
class PipelineConfig:
    mode: str = "QUICK"
    backbone_choice: str = "C"
    prior_mode: str = "AUTO"
    use_finetuned: bool = True
    chunk_size: int = 30
    chunk_overlap: int = 6
    quick_seconds: float = 90.0
    quick_max_keyframes: int = 120
    backbone_input_size: int = 518
    output_dir: Path = Path("/kaggle/working/outputs")
    device0: str = "cuda:0"
    device1: str = "cuda:1"
    texture_time_budget_s: float = 60.0
    dsm_cell_size_m: float = 0.2
    min_confidence: float = 0.1
    min_view_count: int = 1
    voxel_size_m: float = 0.05
    keyframe_min_gps_spacing_m: float = 2.0
    keyframe_max_no_gps_stride: int = 5
    keyframe_sharpness_percentile_floor: float = 15.0
    keyframe_sample_fps: float = 4.0
    decode_working_width: int = 1920
    watchdog_stall_s: float = 60.0


@dataclass
class PreparedKeyframe:
    frame_index: int
    timestamp_s: float
    image_full: np.ndarray
    gps_enu: tuple[float, float, float] | None
    intrinsics: np.ndarray
    camera_pose_c2w_prior: np.ndarray | None


@dataclass
class PipelineResult:
    success: bool
    error: str | None = None
    traceback: str | None = None
    outputs: list = field(default_factory=list)


def _default_intrinsics(w: int, h: int) -> np.ndarray:
    """Rough-FOV fallback when no calibration/metadata is available: assumes
    a typical consumer/drone-camera field of view (roughly 70-80 deg
    horizontal). This is explicitly a fallback, logged by the caller — real
    intrinsics from metadata or a detected calibration file always win."""
    f = 0.9 * max(w, h)
    return np.array([[f, 0, w / 2.0], [0, f, h / 2.0], [0, 0, 1.0]])


def _attitude_to_cam2world(position_enu: np.ndarray, attitude_deg: tuple[float, float, float]) -> np.ndarray:
    """Builds a full 4x4 cam2world pose from a GPS ENU position + gimbal/IMU
    attitude, reusing align.py's world->cam rotation construction (same ZYX
    gimbal convention) since it's already implemented and tested there."""
    from .align import estimate_gravity_imu

    g = estimate_gravity_imu(attitude_deg)
    yaw, pitch, roll = (np.radians(a) for a in attitude_deg)
    cy, sy = np.cos(yaw), np.sin(yaw)
    cp, sp = np.cos(pitch), np.sin(pitch)
    cr, sr = np.cos(roll), np.sin(roll)
    Rz = np.array([[cy, -sy, 0.0], [sy, cy, 0.0], [0.0, 0.0, 1.0]])
    Ry = np.array([[cp, 0.0, sp], [0.0, 1.0, 0.0], [-sp, 0.0, cp]])
    Rx = np.array([[1.0, 0.0, 0.0], [0.0, cr, -sr], [0.0, sr, cr]])
    R_world_to_cam = Rx @ Ry @ Rz
    R_cam_to_world = R_world_to_cam.T
    pose = np.eye(4)
    pose[:3, :3] = R_cam_to_world
    pose[:3, 3] = position_enu
    return pose


def _resize_for_backbone(img: np.ndarray, size: int) -> np.ndarray:
    from PIL import Image

    pil = Image.fromarray(img)
    pil = pil.resize((size, size), Image.BILINEAR)
    return np.array(pil)


def _make_thumbnail(img: np.ndarray, max_width: int = 120) -> np.ndarray:
    """Small RGB copy for the notebook's keyframe grid — keeping full-res
    frames around for every keyframe (not just accepted ones, since
    rejected ones are shown greyed out too) would be a real memory cost at
    FULL-mode scale."""
    from PIL import Image

    h, w = img.shape[:2]
    if w <= max_width:
        return img.copy()
    new_h = max(1, int(h * max_width / w))
    return np.array(Image.fromarray(img).resize((max_width, new_h), Image.BILINEAR))


class Pipeline:
    def __init__(
        self, config: PipelineConfig, detected: DetectedInputs, telemetry: TelemetryTrack | None,
        bus: EventBus, gpu_monitor: GpuMonitor, report: ReportBuilder,
        backbone_override: tuple | None = None, live_page=None,
    ):
        """`backbone_override`: (GeometryBackbone, PriorMode), pre-built and
        already loaded. Testing/dependency-injection hook only — the real
        notebook cells never pass this, so production behavior (backbone
        chosen via backbone.select_backbone from BACKBONE/PRIOR_MODE config)
        is unaffected. Used by the local dry-run harness to exercise the
        entire orchestration loop on CPU with a synthetic backbone, without
        downloading real multi-GB models.

        `live_page`: an already-started live_page.LivePageServer, or None if
        the tunnel/server failed to come up (bootstrap.py logs that and the
        pipeline just runs without pushing to it — the dashboard's inline 3D
        tab remains the fallback view). Kept as a plain optional handle
        rather than a required dependency so the local dry-run harness and
        any future non-notebook caller don't need to stand up a real server."""
        self.config = config
        self.detected = detected
        self.telemetry = telemetry
        self.bus = bus
        self.gpu_monitor = gpu_monitor
        self.report = report
        self._backbone_override = backbone_override
        self.live_page = live_page

        self.stop_event = threading.Event()
        self._thread: threading.Thread | None = None
        self._gpu1_thread: threading.Thread | None = None
        self._chunk_queue: "queue.Queue" = queue.Queue(maxsize=3)
        self._SENTINEL = object()

        # Watchdog: touched by every _emit()/_log() call, polled by a
        # background thread — if a stage sits without producing a single
        # event for watchdog_stall_s, that's exactly the kind of silent
        # stall a first real Kaggle run hit (20+ min stuck in frame
        # extraction with no indication anything was wrong).
        self._last_progress_ts = time.time()
        self._current_stage: str | None = None
        self._watchdog_stop = threading.Event()
        self._watchdog_thread: threading.Thread | None = None
        self._watchdog_last_warn_ts: float = 0.0

        self.two_gpu = gpu_monitor.device_count >= 2
        self.fusion_acc = VoxelPointFusion(config.voxel_size_m, device=config.device0)
        self.masker: masks.DynamicObjectMasker | None = None

        self.chunk_alignments: list[align.ChunkAlignment] = []
        self.overlap_constraints: list[align.OverlapConstraint] = []
        self.gps_factors: list[align.GpsFactor] = []
        self._prev_chunk_cam_local: dict[int, np.ndarray] = {}  # keyframe_index -> local cam center, from the previous chunk only

        self.result: PipelineResult | None = None
        self._export_statuses: list = []
        self.georeferenced: bool = self.detected.telemetry_path is not None or self.detected.telemetry_kind == "embedded"
        self.epsg: int | None = None
        self.artifacts = RunArtifacts()
        self._chunk_alignment_records: dict[int, ChunkAlignmentRecord] = {}

    # -- lifecycle ------------------------------------------------------------

    def start(self) -> None:
        self._thread = threading.Thread(target=self._run_guarded, name="sih3d-pipeline", daemon=True)
        self._thread.start()

    def stop(self, timeout: float = 60.0) -> None:
        self.stop_event.set()
        if self._thread:
            self._thread.join(timeout=timeout)
        if self._gpu1_thread:
            self._gpu1_thread.join(timeout=timeout)

    def is_alive(self) -> bool:
        return self._thread is not None and self._thread.is_alive()

    def _run_guarded(self) -> None:
        try:
            self._run()
            self.result = PipelineResult(success=True, outputs=self._export_statuses)
        except Exception as e:
            tb = traceback.format_exc()
            self._log(f"PIPELINE FATAL ERROR: {type(e).__name__}: {e}", level="error")
            self._emit(EventType.PIPELINE_ERROR, error=str(e), traceback=tb)
            self.result = PipelineResult(success=False, error=str(e), traceback=tb, outputs=self._export_statuses)
        finally:
            self.stop_event.set()
            self._watchdog_stop.set()
            try:
                self.gpu_monitor.stop()
            except Exception:
                pass
            self.report.finish()
            viewer_path = self.config.output_dir / "viewer.html"
            self._emit(
                EventType.PIPELINE_DONE,
                outputs=[o.__dict__ for o in self._export_statuses],
                viewer_html_path=str(viewer_path) if viewer_path.exists() else None,
            )

    def _emit(self, event_type: EventType, **payload) -> None:
        """Publishes to the bus (for the dashboard/any external consumer)
        AND feeds ReportBuilder directly and synchronously. report-building
        is cheap bookkeeping, not rendering, so there's no reason report.json
        should depend on an external consumer having drained the bus in
        time — that's exactly the race this fixes (caught by the local dry
        run: report.json showed every stage as "pending" because nothing
        was draining the bus into `report` at all yet)."""
        ts = time.time()
        self._last_progress_ts = ts
        if event_type == EventType.STAGE_START:
            self._current_stage = payload.get("stage")
        elif event_type in (EventType.STAGE_DONE, EventType.STAGE_ERROR):
            self._current_stage = None
        self.bus.publish(event_type, **payload)
        self.report.on_event(Event(type=event_type, payload=payload, ts=ts))

    def _log(self, message: str, level: str = "info") -> None:
        self._last_progress_ts = time.time()
        self.bus.log(message, level=level)
        self.report.on_event(Event(type=EventType.LOG, payload={"message": message, "level": level}, ts=time.time()))

    def _watchdog_loop(self) -> None:
        """Polls for silent stalls (see class docstring / decode.py's own
        docstring for the real Kaggle run this is a response to). Only
        warns while a stage is actually in progress (STAGE_START seen,
        STAGE_DONE/ERROR not yet) — a long gap between stages (e.g.
        waiting on a subprocess mesh export) isn't a stall by itself as
        long as SOME stage is marked active; every meaningful unit of work
        inside a stage goes through _emit()/_log(), which is what resets
        the clock this checks."""
        while not self._watchdog_stop.wait(10.0):
            if self._current_stage is None:
                continue
            now = time.time()
            stalled_s = now - self._last_progress_ts
            if stalled_s < self.config.watchdog_stall_s:
                continue
            if now - self._watchdog_last_warn_ts < 30.0:
                continue  # already warned recently about this same stall
            self._watchdog_last_warn_ts = now
            self.bus.log(
                f"WATCHDOG: no progress for {stalled_s:.0f}s in stage '{self._current_stage}' — "
                f"this may be a stall, not necessarily an error", level="warn",
            )

    def _fallback(self, stage: str, exc: Exception) -> None:
        note = f"{type(exc).__name__}: {exc}"
        self._log(f"Stage '{stage}': {note} — continuing with degraded output", level="warn")
        self._emit(EventType.STAGE_FALLBACK, stage=stage, note=note)

    def _push_live(self, fn, *args, **kwargs) -> None:
        """Best-effort call into self.live_page. Disables further pushes
        (rather than logging per-keyframe) on the first failure — a broken
        live-page connection would otherwise spam one STAGE_FALLBACK per
        frame; the dashboard's inline 3D tab keeps working regardless."""
        if self.live_page is None:
            return
        try:
            fn(*args, **kwargs)
        except Exception as e:
            self._log(f"Live reconstruction page push failed ({type(e).__name__}: {e}) — disabling further pushes; inline 3D tab still works", level="warn")
            self.live_page = None

    # -- main run ---------------------------------------------------------

    def _run(self) -> None:
        cfg = self.config
        cfg.output_dir.mkdir(parents=True, exist_ok=True)
        self.gpu_monitor.start()
        self._last_progress_ts = time.time()
        self._watchdog_thread = threading.Thread(target=self._watchdog_loop, daemon=True, name="sih3d-watchdog")
        self._watchdog_thread.start()

        video = self.detected.video
        if video is None:
            raise RuntimeError("no video detected — nothing to process")

        # georeferenced only depends on what was detected, not on the
        # backbone — resolved here so frame_extraction/camera_trajectory
        # below can use it without the backbone needing to exist yet.
        self.georeferenced = self.detected.telemetry_path is not None or self.detected.telemetry_kind == "embedded"
        if not self.georeferenced:
            self._log("NO TELEMETRY: outputs will be APPROXIMATE SCALE — NOT GEOREFERENCED", level="warn")

        if self.two_gpu:
            self.masker = masks.DynamicObjectMasker(self.bus, device=cfg.device1)

        # -- Stage: frame_extraction -------------------------------------
        # Backbone selection/loading is deliberately NOT done before this
        # stage (it used to be) — a first real Kaggle run OOM'd here: with
        # MapAnything already resident on device0 (~13.7 of a 14.6 GiB T4),
        # frame extraction's own sharpness-scoring batch (all candidate
        # frames as one tensor, also on device0) had nowhere near enough
        # headroom left. Frame extraction and camera_trajectory use
        # neither the backbone nor its device, so loading it only right
        # before the chunk loop that actually needs it — after chunking,
        # below — means it competes for GPU0 memory with nothing but
        # itself.
        self._emit(EventType.STAGE_START, stage="frame_extraction")
        try:
            keyframes = self._extract_keyframes(video)
        except Exception as e:
            self._fallback("frame_extraction", e)
            keyframes = []
        self._emit(EventType.STAGE_DONE, stage="frame_extraction")
        if not keyframes:
            raise RuntimeError("frame extraction produced zero usable keyframes")

        # -- Stage: camera_trajectory --------------------------------------
        self._emit(EventType.STAGE_START, stage="camera_trajectory")
        try:
            self._attach_priors(keyframes)
        except Exception as e:
            self._fallback("camera_trajectory", e)
        for kf in keyframes:
            if kf.gps_enu is not None:
                self._emit(EventType.TRAJECTORY_POINT, kind="gps", x=kf.gps_enu[0], y=kf.gps_enu[1])
        self._emit(EventType.STAGE_DONE, stage="camera_trajectory")

        collinearity = None
        if self.georeferenced:
            enu_pts = [kf.gps_enu for kf in keyframes if kf.gps_enu is not None]
            if len(enu_pts) >= 3:
                collinearity = compute_collinearity(enu_pts)
                self._log(f"Flight-track collinearity index: {collinearity:.3f} (near 0 = straight single pass)")
        self.artifacts.collinearity_index = collinearity
        self.artifacts.georeferenced = self.georeferenced

        # -- Chunking --------------------------------------------------------
        chunks = self._make_chunks(keyframes)
        self._log(f"Chunked {len(keyframes)} keyframes into {len(chunks)} chunk(s) (size={cfg.chunk_size}, overlap={cfg.chunk_overlap})")

        # Backbone selection is the one allowed-fatal setup step, and the
        # only thing here allowed to put multi-GB of weights on device0 —
        # deliberately done only now, right before the chunk loop that's
        # the first thing to actually need it (see the frame_extraction
        # comment above for why: it used to run before frame extraction
        # and OOM'd there on a real 4K Kaggle run).
        if self._backbone_override is not None:
            bb, prior_mode = self._backbone_override
        else:
            bb, prior_mode = backbone_mod.select_backbone(
                cfg.backbone_choice, cfg.prior_mode, self.detected, self.bus,
                device=cfg.device0, use_finetuned=cfg.use_finetuned,
            )
        self.report.set_run_config(
            mode=cfg.mode, backbone=bb.name, prior_mode=prior_mode.value,
            checkpoint_source=getattr(bb, "checkpoint_source", "pretrained"),
            gpus=self.gpu_monitor.device_names,
        )

        self._gpu1_thread = None
        if self.two_gpu:
            self._gpu1_thread = threading.Thread(target=self._gpu1_worker, name="sih3d-gpu1-worker", daemon=True)
            self._gpu1_thread.start()

        self._emit(EventType.STAGE_START, stage="geometric_reconstruction")
        self._emit(EventType.STAGE_START, stage="large_scale_alignment")
        self._emit(EventType.STAGE_START, stage="dense_point_cloud")

        for chunk_idx, chunk_kfs in enumerate(chunks):
            if self.stop_event.is_set():
                break
            chunk_result = self._process_chunk(bb, prior_mode, chunk_kfs, chunk_idx)
            frac = (chunk_idx + 1) / len(chunks)
            self._emit(EventType.STAGE_PROGRESS, stage="geometric_reconstruction", frac=frac)
            self._emit(EventType.STAGE_PROGRESS, stage="large_scale_alignment", frac=frac)

            if chunk_result is None:
                continue

            if self.two_gpu:
                try:
                    self._chunk_queue.put((chunk_idx, chunk_kfs, chunk_result), timeout=30)
                except queue.Full:
                    self._fallback("dense_point_cloud", RuntimeError("GPU1 worker queue full/stalled; dropping chunk"))
            else:
                self._mask_and_fuse_chunk(chunk_idx, chunk_kfs, chunk_result)
                self._emit(EventType.STAGE_PROGRESS, stage="dense_point_cloud", frac=frac)

        if self.two_gpu:
            self._chunk_queue.put(self._SENTINEL)
            if self._gpu1_thread:
                self._gpu1_thread.join(timeout=300)

        self._emit(EventType.STAGE_DONE, stage="geometric_reconstruction")
        self._emit(EventType.STAGE_DONE, stage="large_scale_alignment")
        self._emit(EventType.STAGE_DONE, stage="dense_point_cloud")

        alignment_rmse = None
        try:
            alignment_rmse = self._refine_global_alignment()
        except Exception as e:
            self._fallback("large_scale_alignment", e)

        # -- Stage: mesh_textured_model ---------------------------------
        self._emit(EventType.STAGE_START, stage="mesh_textured_model")
        cloud = self.fusion_acc.extract_points(min_view_count=cfg.min_view_count)
        try:
            cloud = remove_statistical_outliers(cloud, self.bus)
        except Exception as e:
            self._fallback("mesh_textured_model", e)

        self.artifacts.point_count = len(cloud.points)
        if len(cloud.points) > 0:
            preview_n = min(len(cloud.points), 50_000)
            preview_idx = np.random.default_rng(0).choice(len(cloud.points), size=preview_n, replace=False)
            self.artifacts.point_cloud_preview = cloud.points[preview_idx]
            self.artifacts.point_cloud_preview_colors = cloud.colors[preview_idx]

        mesh_result = mesh_mod.MeshResult(mesh=None, method="none")
        bake_result = None
        try:
            mesh_result = mesh_mod.build_vertex_colored_mesh(cloud, tsdf=None, bus=self.bus)
            if mesh_result.mesh is not None:
                self._emit(EventType.MESH_PREVIEW, mesh=mesh_result, bake=None, stage="coarse")
        except Exception as e:
            self._fallback("mesh_textured_model", e)

        try:
            if mesh_result.mesh is not None:
                kf_for_bake = [
                    KeyframeForBaking(image_rgb=kf.image_full, camera_pose_c2w=self._resolved_pose(kf, chunk_idx=None), intrinsics=kf.intrinsics)
                    for kf in keyframes if getattr(kf, "_resolved_world_pose", None) is not None
                ]
                bake_result = mesh_mod.bake_texture(mesh_result.mesh, kf_for_bake, self.bus, time_budget_s=cfg.texture_time_budget_s)
                if bake_result is not None and bake_result.textured:
                    self._emit(EventType.MESH_PREVIEW, mesh=mesh_result, bake=bake_result, stage="textured")
        except Exception as e:
            self._fallback("mesh_textured_model", e)
        self._emit(EventType.STAGE_DONE, stage="mesh_textured_model")

        self.artifacts.mesh_n_vertices = mesh_result.n_vertices
        self.artifacts.mesh_n_faces = mesh_result.n_faces
        self.artifacts.mesh_method = mesh_result.method

        self.report.set_geometry_summary(
            georeferenced=self.georeferenced, collinearity_index=collinearity, alignment_rmse_m=alignment_rmse,
            point_count=len(cloud.points), mesh_faces=mesh_result.n_faces,
        )

        # -- Export ------------------------------------------------------
        self._export_all(cloud, mesh_result, bake_result, keyframes)

    # -- frame extraction / keyframes --------------------------------------

    def _extract_keyframes(self, video) -> list[PreparedKeyframe]:
        cfg = self.config
        decoder = FrameDecoder(video.path, self.bus, device=cfg.device0)

        end_s = cfg.quick_seconds if cfg.mode == "QUICK" else None
        raw_indices, raw_ts, raw_frames = [], [], []
        for idx, t, frame in decoder.iter_frames(
            start_s=0.0, end_s=end_s,
            target_fps=cfg.keyframe_sample_fps, scale_width=cfg.decode_working_width,
        ):
            if self.stop_event.is_set():
                break
            raw_indices.append(idx)
            raw_ts.append(t)
            raw_frames.append(frame)
            # A real per-frame score, not a placeholder — the frames panel
            # used to always show "sharpness=0" here because the actual
            # accept/reject scoring happens afterward in one GPU batch over
            # ALL collected frames (select_keyframes below), not per frame
            # as each is decoded. This single-frame call is cheap (frames
            # are already scale_width-limited) and purely for the live
            # display; select_keyframes' batch scores are still what
            # decides acceptance.
            live_sharpness = compute_sharpness(frame, device=cfg.device0 if "cuda" in cfg.device0 else "cpu")
            self._emit(EventType.FRAME_DECODED, frame=frame, frame_index=idx, sharpness=live_sharpness)
            if cfg.mode == "QUICK" and len(raw_frames) >= cfg.quick_max_keyframes * 4:
                break  # decode a bounded multiple of the target count; selection will thin it out

        if not raw_frames:
            return []

        gps_per_frame = None
        if self.telemetry is not None and self.telemetry.samples:
            synced = sync_to_frame_times(self.telemetry, raw_ts)
            enu_pts, origin, zone_epsg = track_to_enu(self.telemetry)
            self.epsg = zone_epsg[1]
            origin_lat, origin_lon, origin_alt = origin
            gps_per_frame = []
            for s in synced:
                if s is None:
                    gps_per_frame.append(None)
                    continue
                from .telemetry import geodetic_to_ecef, ecef_to_enu

                alt = s.alt if s.alt is not None else 0.0
                x, y, z = geodetic_to_ecef(s.lat, s.lon, alt)
                gps_per_frame.append(ecef_to_enu(x, y, z, origin_lat, origin_lon, origin_alt))
            self._telemetry_synced = synced

        selection = select_keyframes(
            raw_indices, raw_ts, raw_frames, gps_per_frame, self.bus,
            sharpness_percentile_floor=cfg.keyframe_sharpness_percentile_floor,
            min_gps_spacing_m=cfg.keyframe_min_gps_spacing_m,
            max_no_gps_frame_stride=cfg.keyframe_max_no_gps_stride,
            device=cfg.device0 if "cuda" in cfg.device0 else "cpu",
        )
        for c in selection.rejected:
            img = raw_frames[raw_indices.index(c.frame_index)] if c.frame_index in raw_indices else None
            self._emit(EventType.KEYFRAME_REJECTED, thumbnail=img)
            self.artifacts.keyframes.append(KeyframeRecord(
                frame_index=c.frame_index, timestamp_s=c.timestamp_s,
                thumbnail=_make_thumbnail(img) if img is not None else None,
                sharpness=c.sharpness, accepted=False, reject_reason=c.reject_reason,
            ))

        accepted = selection.accepted
        if cfg.mode == "QUICK" and len(accepted) > cfg.quick_max_keyframes:
            step = len(accepted) / cfg.quick_max_keyframes
            accepted = [accepted[int(i * step)] for i in range(cfg.quick_max_keyframes)]

        prepared = []
        for cand in accepted:
            local_i = raw_indices.index(cand.frame_index)
            img = raw_frames[local_i]
            h, w = img.shape[:2]
            intrinsics = self._resolve_intrinsics(w, h)
            prepared.append(PreparedKeyframe(
                frame_index=cand.frame_index, timestamp_s=cand.timestamp_s, image_full=img,
                gps_enu=cand.gps_enu, intrinsics=intrinsics, camera_pose_c2w_prior=None,
            ))
            self._emit(EventType.KEYFRAME_ACCEPTED, thumbnail=img)
            self.artifacts.keyframes.append(KeyframeRecord(
                frame_index=cand.frame_index, timestamp_s=cand.timestamp_s,
                thumbnail=_make_thumbnail(img), sharpness=cand.sharpness, accepted=True,
            ))
            if cand.gps_enu is not None:
                self.artifacts.gps_track_enu.append(cand.gps_enu)

        return prepared

    def _resolve_intrinsics(self, w: int, h: int) -> np.ndarray:
        info = self.detected.intrinsics
        if info:
            flat = {}

            def collect(d, depth=0):
                if depth > 3:
                    return
                for k, v in d.items():
                    flat[k] = v
                    if isinstance(v, dict):
                        collect(v, depth + 1)

            collect(info)
            if all(k in flat for k in ("fx", "fy", "cx", "cy")):
                return np.array([[flat["fx"], 0, flat["cx"]], [0, flat["fy"], flat["cy"]], [0, 0, 1.0]])
            for key in ("camera_matrix", "K", "intrinsic_matrix"):
                if key in flat:
                    mat = np.array(flat[key], dtype=float)
                    if mat.shape == (3, 3):
                        return mat
        return _default_intrinsics(w, h)

    def _attach_priors(self, keyframes: list[PreparedKeyframe]) -> None:
        if self.telemetry is None or not self.telemetry.samples:
            return
        synced = getattr(self, "_telemetry_synced", None)
        for kf in keyframes:
            if kf.gps_enu is None:
                continue
            sample_idx = None
            # `synced` is aligned to the same frame timestamps used during
            # extraction; find the matching sample by nearest timestamp
            # (cheap linear scan — keyframe counts are small by design).
            if synced:
                best = min(range(len(synced)), key=lambda i: abs((synced[i].t if synced[i] else 1e18) - kf.timestamp_s), default=None)
                if best is not None and synced[best] is not None:
                    s = synced[best]
                    attitude = (s.gimbal_yaw or s.yaw, s.gimbal_pitch or s.pitch, s.gimbal_roll or s.roll)
                    if all(a is not None for a in attitude):
                        kf.camera_pose_c2w_prior = _attitude_to_cam2world(np.array(kf.gps_enu), attitude)
                        continue
            # No attitude available: translation-only prior (identity rotation).
            pose = np.eye(4)
            pose[:3, 3] = kf.gps_enu
            kf.camera_pose_c2w_prior = pose

    # -- chunking --------------------------------------------------------

    def _make_chunks(self, keyframes: list[PreparedKeyframe]) -> list[list[PreparedKeyframe]]:
        cfg = self.config
        step = max(cfg.chunk_size - cfg.chunk_overlap, 1)
        chunks = []
        i = 0
        n = len(keyframes)
        while i < n:
            chunk = keyframes[i:i + cfg.chunk_size]
            if chunk:
                chunks.append(chunk)
            if i + cfg.chunk_size >= n:
                break
            i += step
        return chunks or [keyframes]

    # -- per-chunk processing ----------------------------------------------

    def _process_chunk(self, bb, prior_mode: PriorMode, chunk_kfs: list[PreparedKeyframe], chunk_idx: int) -> ChunkResult | None:
        cfg = self.config
        views = [
            ViewInput(
                image=_resize_for_backbone(kf.image_full, cfg.backbone_input_size),
                intrinsics=self._scale_intrinsics(kf.intrinsics, kf.image_full.shape, cfg.backbone_input_size),
                camera_pose_c2w=kf.camera_pose_c2w_prior,
                frame_index=kf.frame_index, timestamp_s=kf.timestamp_s,
            )
            for kf in chunk_kfs
        ]

        chunk_size = len(views)
        while True:
            try:
                result = bb.infer_chunk(views, prior_mode)
                break
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and chunk_size > 2:
                    chunk_size = max(chunk_size // 2, 2)
                    self._log(f"OOM on chunk {chunk_idx} — halving to {chunk_size} views and retrying", level="warn")
                    self._emit(EventType.STAGE_FALLBACK, stage="geometric_reconstruction", note=f"OOM, halved chunk to {chunk_size}")
                    views = views[:chunk_size]
                    try:
                        import torch

                        torch.cuda.empty_cache()
                    except Exception:
                        pass
                    continue
                self._fallback("geometric_reconstruction", e)
                return None
            except Exception as e:
                self._fallback("geometric_reconstruction", e)
                return None

        chunk_kfs = chunk_kfs[:chunk_size]
        try:
            self._align_chunk(chunk_kfs, result, chunk_idx)
        except Exception as e:
            self._fallback("large_scale_alignment", e)

        depth_preview = result.points_world[0, ..., 2] if len(result.points_world) else None
        conf_preview = result.confidence[0] if len(result.confidence) else None
        self._emit(EventType.GEOMETRY_CHUNK, depth=depth_preview, confidence=conf_preview)

        return result

    def _scale_intrinsics(self, K: np.ndarray, orig_shape: tuple, target_size: int) -> np.ndarray:
        h, w = orig_shape[:2]
        sx, sy = target_size / w, target_size / h
        K2 = K.copy()
        K2[0, 0] *= sx
        K2[0, 2] *= sx
        K2[1, 1] *= sy
        K2[1, 2] *= sy
        return K2

    def _align_chunk(self, chunk_kfs: list[PreparedKeyframe], result: ChunkResult, chunk_idx: int) -> None:
        cam_local = result.camera_poses_est[:, :3, 3]

        if self.georeferenced:
            gps_pts = np.array([kf.gps_enu if kf.gps_enu is not None else [np.nan] * 3 for kf in chunk_kfs])
            valid = ~np.isnan(gps_pts).any(axis=1)
            if valid.sum() >= 2:
                attitudes = [
                    getattr(kf, "_attitude", None) for kf in chunk_kfs
                ]
                gravity = align.estimate_gravity_ransac(result.points_world[valid].reshape(-1, 3)) or align.GravityEstimate(
                    up_local=np.array([0.0, 0.0, 1.0]), source="assumed_identity", confidence=0.2,
                )
                if valid.sum() >= 2:
                    alt_trend = gps_pts[valid][:, 2]
                    gravity = align.disambiguate_gravity_sign(gravity, cam_local[valid], alt_trend)
                alignment = align.align_chunk_4dof(cam_local[valid], gps_pts[valid], gravity)
            else:
                alignment = align.first_chunk_identity(georeferenced=True)
        else:
            overlap_keys = [kf.frame_index for kf in chunk_kfs if kf.frame_index in self._prev_chunk_cam_local]
            if chunk_idx == 0 or not overlap_keys:
                alignment = align.first_chunk_identity(georeferenced=False)
            else:
                idx_map = {kf.frame_index: i for i, kf in enumerate(chunk_kfs)}
                src = np.array([cam_local[idx_map[k]] for k in overlap_keys])
                dst = np.array([self._prev_chunk_cam_local[k] for k in overlap_keys])
                if len(src) >= 3:
                    alignment = align.align_chunk_to_previous(src, dst)
                else:
                    alignment = self.chunk_alignments[-1] if self.chunk_alignments else align.first_chunk_identity(False)

        self.chunk_alignments.append(alignment)
        record = ChunkAlignmentRecord(chunk_idx=chunk_idx, mode=alignment.mode, rmse_before_m=alignment.rmse_m)
        self._chunk_alignment_records[chunk_idx] = record
        self.artifacts.chunk_alignments.append(record)

        world_cam = alignment.apply(cam_local)
        for i, (kf, wc) in enumerate(zip(chunk_kfs, world_cam)):
            kf._resolved_world_pose = wc  # noqa: SLF001 — internal bookkeeping between pipeline stages
            self._emit(EventType.TRAJECTORY_POINT, kind="camera", x=wc[0], y=wc[1])
            self.artifacts.camera_track_enu.append(tuple(wc))
            if self.georeferenced and kf.gps_enu is not None:
                self.gps_factors.append(align.GpsFactor(chunk=chunk_idx, cam_center_local=cam_local[len(self.gps_factors) % len(cam_local)], gps_enu=np.array(kf.gps_enu)))
            if self.live_page is not None:
                # Forward/up are only used by the client to orient the
                # camera-frustum helper, not for any geometry math here — a
                # coarse estimate (next waypoint direction, world +Z up) is
                # enough for that purpose.
                forward = world_cam[i + 1] - wc if i + 1 < len(world_cam) else (wc - world_cam[i - 1] if i > 0 else np.array([1.0, 0.0, 0.0]))
                norm = np.linalg.norm(forward)
                forward = forward / norm if norm > 1e-9 else np.array([1.0, 0.0, 0.0])
                self._push_live(
                    self.live_page.push_camera_pose,
                    frame_index=kf.frame_index, timestamp_s=kf.timestamp_s,
                    position=wc, forward=forward, up=np.array([0.0, 0.0, 1.0]),
                )

        overlap_keys = [kf.frame_index for kf in chunk_kfs if kf.frame_index in self._prev_chunk_cam_local]
        if overlap_keys:
            idx_map = {kf.frame_index: i for i, kf in enumerate(chunk_kfs)}
            self.overlap_constraints.append(align.OverlapConstraint(
                chunk_a=chunk_idx - 1, chunk_b=chunk_idx,
                cam_centers_a_local=np.array([self._prev_chunk_cam_local[k] for k in overlap_keys]),
                cam_centers_b_local=np.array([cam_local[idx_map[k]] for k in overlap_keys]),
            ))

        overlap_start = max(len(chunk_kfs) - self.config.chunk_overlap, 0)
        self._prev_chunk_cam_local = {kf.frame_index: cam_local[i] for i, kf in enumerate(chunk_kfs[overlap_start:], start=overlap_start)}

        self._last_alignment = alignment

    def _resolved_pose(self, kf: PreparedKeyframe, chunk_idx) -> np.ndarray:
        wc = getattr(kf, "_resolved_world_pose", None)
        pose = np.eye(4)
        if wc is not None:
            pose[:3, 3] = wc
        return pose

    # -- masking + fusion (inline or GPU1 worker) ---------------------------

    def _mask_and_fuse_chunk(self, chunk_idx: int, chunk_kfs: list[PreparedKeyframe], result: ChunkResult) -> None:
        alignment = self.chunk_alignments[chunk_idx] if chunk_idx < len(self.chunk_alignments) else None
        if alignment is None:
            return

        points_world = alignment.apply(result.points_world)
        colors = np.stack([_resize_for_backbone(kf.image_full, points_world.shape[1]) for kf in chunk_kfs], axis=0)

        dynamic_mask = None
        if self.masker is not None:
            try:
                masks_list = [self.masker.mask_frame(colors[i]).mask for i in range(len(chunk_kfs))]
                dynamic_mask = np.stack(masks_list, axis=0)
            except Exception as e:
                self._fallback("dense_point_cloud", e)

        self.artifacts.add_geometry_sample(ChunkGeometrySample(
            chunk_idx=chunk_idx, rgb=colors[0].copy(),
            depth=result.points_world[0, ..., 2].copy(), confidence=result.confidence[0].copy(),
            dynamic_mask=dynamic_mask[0].copy() if dynamic_mask is not None else None,
        ))

        n_new = self.fusion_acc.add_chunk(
            points_world, colors, result.confidence, dynamic_mask=dynamic_mask,
            min_confidence=self.config.min_confidence,
        )
        # Full-resolution chunk, not downsampled — the inline viewer (see
        # dashboard.py's Live 3D panel) needs real data to stream, and it
        # does its own LOD above 2M accumulated points. A too-small preview
        # here would defeat the point of a "full-quality" live view.
        flat_pts = points_world.reshape(-1, 3)
        flat_cols = colors.reshape(-1, 3)
        flat_conf = result.confidence.reshape(-1)
        keep = np.isfinite(flat_pts).all(axis=1)
        if dynamic_mask is not None:
            keep &= ~dynamic_mask.reshape(-1)
        self._emit(EventType.POINTCLOUD_GROWTH, points=flat_pts[keep], colors=flat_cols[keep], confidence=flat_conf[keep])
        self.artifacts.point_count = len(self.fusion_acc)
        _ = n_new

        # Live reconstruction page: pushed PER FRAME (not the whole chunk in
        # one message) so the client can reveal points frame by frame in
        # sync with the left-hand video instead of a single chunk-sized pop.
        # `masked` marks points the dynamic-object masker dropped from the
        # fused cloud above — the client shows those in red when its
        # "show masked-out objects" toggle is on rather than hiding them,
        # so still emit them here (only NaNs are actually dropped).
        if self.live_page is not None:
            for i, kf in enumerate(chunk_kfs):
                pts_i = points_world[i].reshape(-1, 3)
                keep_i = np.isfinite(pts_i).all(axis=1)
                masked_i = (
                    dynamic_mask[i].reshape(-1)[keep_i].astype(np.uint8)
                    if dynamic_mask is not None
                    else np.zeros(int(keep_i.sum()), dtype=np.uint8)
                )
                self._push_live(
                    self.live_page.push_points,
                    pts_i[keep_i], colors[i].reshape(-1, 3)[keep_i],
                    result.confidence[i].reshape(-1)[keep_i], masked_i,
                    frame_index=kf.frame_index, timestamp_s=kf.timestamp_s,
                )

    def _gpu1_worker(self) -> None:
        while True:
            item = self._chunk_queue.get()
            if item is self._SENTINEL:
                break
            chunk_idx, chunk_kfs, result = item
            try:
                self._mask_and_fuse_chunk(chunk_idx, chunk_kfs, result)
            except Exception as e:
                self._fallback("dense_point_cloud", e)
            frac = None
            self._emit(EventType.STAGE_PROGRESS, stage="dense_point_cloud", frac=1.0)

    # -- global alignment refinement -----------------------------------------

    def _refine_global_alignment(self) -> float | None:
        if not self.chunk_alignments:
            return None
        if not self.overlap_constraints and not self.gps_factors:
            rmses = [a.rmse_m for a in self.chunk_alignments if a.rmse_m is not None]
            return float(np.mean(rmses)) if rmses else None

        refined = align.refine_pose_graph(self.chunk_alignments, self.overlap_constraints, self.gps_factors, self.bus)
        self.chunk_alignments = refined
        for i, ca in enumerate(refined):
            record = self._chunk_alignment_records.get(i)
            if record is not None:
                record.rmse_after_m = ca.rmse_m
        rmses = [a.rmse_m for a in refined if a.rmse_m is not None]
        return float(np.mean(rmses)) if rmses else None

    # -- export --------------------------------------------------------------

    def _export_all(self, cloud: FusedPointCloud, mesh_result, bake_result, keyframes: list[PreparedKeyframe]) -> None:
        cfg = self.config
        out = cfg.output_dir

        def rec(status):
            self._export_statuses.append(status)
            self.report.add_output(status)
            if status.ok and status.path is not None:
                self.artifacts.output_paths[status.name] = str(status.path)

        rec(export.export_pointcloud_ply(cloud, out / "pointcloud.ply", self.bus))
        rec(export.export_pointcloud_las(cloud, out / "pointcloud.las", self.bus, self.epsg, self.georeferenced))

        obj_status = export.export_mesh_obj(mesh_result, bake_result, out, self.bus)
        rec(obj_status)
        glb_status = export.export_mesh_glb(mesh_result, bake_result, out / "mesh.glb", self.bus)
        rec(glb_status)
        rec(export.export_mesh_fbx(obj_status.path, out / "mesh.fbx", self.bus))

        dsm_status, ortho_status = export.export_dsm_orthomosaic(cloud, out, self.bus, self.epsg, self.georeferenced, cfg.dsm_cell_size_m)
        rec(dsm_status)
        rec(ortho_status)
        coverage_status = export.export_coverage(cloud, out, self.bus, self.epsg, self.georeferenced, cfg.dsm_cell_size_m)
        rec(coverage_status)
        self._emit(
            EventType.RASTERS_READY,
            dsm_path=str(dsm_status.path) if dsm_status.ok else None,
            orthomosaic_path=str(ortho_status.path) if ortho_status.ok else None,
            coverage_path=str(coverage_status.path) if coverage_status.ok else None,
        )

        gps_track = [kf.gps_enu for kf in keyframes if kf.gps_enu is not None]
        cam_track = [getattr(kf, "_resolved_world_pose", None) for kf in keyframes]
        cam_track = [c for c in cam_track if c is not None]
        if gps_track and self.telemetry is not None:
            origin = track_to_enu(self.telemetry)[1]
            geo_status, kml_status = export.export_trajectories(gps_track, cam_track, origin, out, self.bus)
            rec(geo_status)
            rec(kml_status)

        try:
            from .viewer import write_viewer_html

            viewer_status = write_viewer_html(
                out / "viewer.html", self.bus,
                cloud=cloud, mesh_glb_path=glb_status.path,
                gps_track_enu=gps_track, camera_track_enu=cam_track,
                georeferenced=self.georeferenced,
            )
            rec(viewer_status)
        except Exception as e:
            self._fallback("mesh_textured_model", e)

        self.report.write_json(out / "report.json")
        self.report.write_html(out / "report.html")


In [ ]:
%%writefile sih3d/report.py
"""Builds report.json / report.html from the pipeline's event stream.

Deliberately decoupled from EventBus.drain() itself (a queue.Queue can only
have one destructive consumer) — the orchestration loop drains the bus once
per tick and fans each event out to both `dashboard.on_event()` and
`ReportBuilder.on_event()`. This module only accumulates state from events
plus a handful of explicit setters for values that aren't naturally
event-shaped (point/face counts, alignment RMSE, etc.).
"""

from __future__ import annotations

import json
import time
from dataclasses import dataclass, field
from pathlib import Path

from .events import Event, EventType

STAGE_ORDER = [
    "frame_extraction",
    "camera_trajectory",
    "geometric_reconstruction",
    "large_scale_alignment",
    "dense_point_cloud",
    "mesh_textured_model",
]

_FALLBACK_KEYWORDS = ("fallback", "falling back", "unavailable", "skip")


@dataclass
class StageInfo:
    name: str
    status: str = "pending"
    start_ts: float | None = None
    end_ts: float | None = None
    fallback_notes: list[str] = field(default_factory=list)


class ReportBuilder:
    def __init__(self):
        self.stages: dict[str, StageInfo] = {name: StageInfo(name=name) for name in STAGE_ORDER}
        self.warnings: list[str] = []
        self.fallbacks: list[str] = []
        self.backbone_used: str | None = None
        self.prior_mode: str | None = None
        self.checkpoint_source: str | None = None
        self.mode: str | None = None
        self.gpus: list[str] = []
        self.georeferenced: bool | None = None
        self.collinearity_index: float | None = None
        self.alignment_rmse_m: float | None = None
        self.keyframe_count = 0
        self.rejected_keyframe_count = 0
        self.point_count = 0
        self.mesh_faces = 0
        self.outputs: list[dict] = []
        self.started_at = time.time()
        self.finished_at: float | None = None
        self._gpu_samples: list[tuple[float, int, float]] = []

    # -- event-driven updates -------------------------------------------------

    def on_event(self, evt: Event) -> None:
        p = evt.payload
        if evt.type == EventType.STAGE_START:
            st = self.stages.get(p.get("stage"))
            if st:
                st.status, st.start_ts = "running", evt.ts
        elif evt.type == EventType.STAGE_DONE:
            st = self.stages.get(p.get("stage"))
            if st:
                st.status, st.end_ts = "done", evt.ts
        elif evt.type == EventType.STAGE_FALLBACK:
            st = self.stages.get(p.get("stage"))
            note = p.get("note", "")
            if st:
                st.status = "fallback" if st.status == "running" else st.status
                st.fallback_notes.append(note)
            self.fallbacks.append(f"[{p.get('stage')}] {note}" if p.get("stage") else note)
        elif evt.type == EventType.STAGE_ERROR:
            st = self.stages.get(p.get("stage"))
            if st:
                st.status, st.end_ts = "error", evt.ts
        elif evt.type == EventType.GPU_SAMPLE:
            self._gpu_samples.append((p["ts"], p["index"], p["util_pct"]))
        elif evt.type == EventType.KEYFRAME_ACCEPTED:
            self.keyframe_count += 1
        elif evt.type == EventType.KEYFRAME_REJECTED:
            self.rejected_keyframe_count += 1
        elif evt.type == EventType.LOG:
            level, msg = p.get("level", "info"), p.get("message", "")
            if level == "warn":
                self.warnings.append(msg)
                if any(k in msg.lower() for k in _FALLBACK_KEYWORDS):
                    self.fallbacks.append(msg)

    # -- explicit setters for non-event-shaped values --------------------------

    def set_run_config(self, mode: str, backbone: str, prior_mode: str, checkpoint_source: str, gpus: list[str]) -> None:
        self.mode, self.backbone_used, self.prior_mode = mode, backbone, prior_mode
        self.checkpoint_source, self.gpus = checkpoint_source, gpus

    def set_geometry_summary(
        self, georeferenced: bool, collinearity_index: float | None, alignment_rmse_m: float | None,
        point_count: int, mesh_faces: int,
    ) -> None:
        self.georeferenced = georeferenced
        self.collinearity_index = collinearity_index
        self.alignment_rmse_m = alignment_rmse_m
        self.point_count = point_count
        self.mesh_faces = mesh_faces

    def add_output(self, status) -> None:
        """`status`: export.ExportStatus (kept as a plain dict here to avoid
        a circular import — export.py already knows its own shape)."""
        self.outputs.append({
            "name": status.name, "ok": status.ok, "skipped_reason": status.skipped_reason,
            "size_bytes": status.size_bytes, "path": str(status.path) if status.path else None,
        })

    def finish(self) -> None:
        self.finished_at = time.time()

    # -- derived values ---------------------------------------------------------

    def stage_avg_gpu_util(self, name: str) -> dict[int, float]:
        st = self.stages.get(name)
        if st is None or st.start_ts is None:
            return {}
        end = st.end_ts or time.time()
        by_gpu: dict[int, list[float]] = {}
        for ts, idx, util in self._gpu_samples:
            if st.start_ts <= ts <= end:
                by_gpu.setdefault(idx, []).append(util)
        return {idx: sum(v) / len(v) for idx, v in by_gpu.items()}

    def to_dict(self) -> dict:
        stages_out = []
        for name, st in self.stages.items():
            elapsed = (st.end_ts - st.start_ts) if (st.start_ts and st.end_ts) else None
            stages_out.append({
                "name": name, "status": st.status, "elapsed_s": elapsed,
                "avg_gpu_util_pct": self.stage_avg_gpu_util(name),
                "fallback_notes": st.fallback_notes,
            })
        total_elapsed = (self.finished_at - self.started_at) if self.finished_at else None
        return {
            "mode": self.mode, "backbone": self.backbone_used, "prior_mode": self.prior_mode,
            "checkpoint_source": self.checkpoint_source, "gpus": self.gpus,
            "georeferenced": self.georeferenced, "collinearity_index": self.collinearity_index,
            "alignment_rmse_m": self.alignment_rmse_m,
            "keyframe_count": self.keyframe_count, "rejected_keyframe_count": self.rejected_keyframe_count,
            "point_count": self.point_count, "mesh_faces": self.mesh_faces,
            "stages": stages_out, "fallbacks_triggered": self.fallbacks, "warnings": self.warnings,
            "outputs": self.outputs, "started_at": self.started_at, "finished_at": self.finished_at,
            "total_elapsed_s": total_elapsed,
        }

    def low_util_stages(self, threshold: float = 50.0) -> list[tuple[str, float]]:
        out = []
        for name in self.stages:
            util = self.stage_avg_gpu_util(name)
            if util:
                worst = min(util.values())
                if worst < threshold:
                    out.append((name, worst))
        return out

    # -- writers -----------------------------------------------------------

    def write_json(self, path: Path) -> None:
        path.write_text(json.dumps(self.to_dict(), indent=2, default=str))

    def write_html(self, path: Path, dashboard_snapshot_html: str = "") -> None:
        d = self.to_dict()

        def fmt_s(v):
            return f"{v:.1f}s" if v is not None else "-"

        rows = "".join(
            f"<tr><td>{s['name']}</td><td>{s['status']}</td><td>{fmt_s(s['elapsed_s'])}</td>"
            f"<td>{', '.join(f'GPU{k}: {v:.0f}%' for k, v in s['avg_gpu_util_pct'].items()) or '-'}</td>"
            f"<td>{'; '.join(s['fallback_notes']) or '-'}</td></tr>"
            for s in d["stages"]
        )
        low_util = self.low_util_stages()
        low_util_html = "".join(f"<li>Stage '{name}' averaged {util:.0f}% GPU utilization (below 50%)</li>" for name, util in low_util)
        outputs_rows = "".join(
            f"<tr><td>{o['name']}</td><td>{'OK' if o['ok'] else 'SKIPPED: ' + str(o.get('skipped_reason', ''))}</td>"
            f"<td>{(o.get('size_bytes', 0) or 0) / 1e6:.2f} MB</td></tr>"
            for o in d["outputs"]
        )
        fallback_items = "".join(f"<li class='warn'>{f}</li>" for f in d["fallbacks_triggered"]) or "<li>none</li>"

        html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>SIH26158 Pipeline Report</title>
<style>
body {{ font-family: -apple-system, Segoe UI, sans-serif; margin: 2rem; background:#0b0f14; color:#e6edf3; }}
table {{ border-collapse: collapse; width: 100%; margin-bottom: 2rem; }}
th, td {{ border: 1px solid #30363d; padding: 6px 10px; text-align: left; font-size: 0.9rem; }}
th {{ background: #161b22; }}
h1, h2 {{ font-weight: 600; }}
.badge {{ display:inline-block; padding:2px 8px; border-radius:4px; background:#1f6feb; margin-right:6px; }}
.badge.warn {{ background:#9e6a03; }}
.warn {{ color: #d29922; }}
</style></head>
<body>
<h1>SIH26158 — Drone Video to Georeferenced 3D Model</h1>
<p>
<span class="badge">Mode: {d['mode']}</span>
<span class="badge">Backbone: {d['backbone']} ({d['prior_mode']})</span>
<span class="badge {'warn' if not d['georeferenced'] else ''}">{'Georeferenced' if d['georeferenced'] else 'NOT GEOREFERENCED'}</span>
<span class="badge">Total time: {fmt_s(d['total_elapsed_s'])}</span>
</p>
<h2>Stages</h2>
<table><tr><th>Stage</th><th>Status</th><th>Elapsed</th><th>Avg GPU Util</th><th>Fallback</th></tr>{rows}</table>
<h2>Geometry</h2>
<ul>
<li>Keyframes: {d['keyframe_count']} accepted, {d['rejected_keyframe_count']} rejected</li>
<li>Points: {d['point_count']:,}</li>
<li>Mesh faces: {d['mesh_faces']:,}</li>
<li>Collinearity index: {d['collinearity_index']}</li>
<li>Alignment RMSE vs GPS: {d['alignment_rmse_m']} m</li>
</ul>
<h2>Outputs</h2>
<table><tr><th>File</th><th>Status</th><th>Size</th></tr>{outputs_rows}</table>
<h2>Fallbacks triggered ({len(d['fallbacks_triggered'])})</h2>
<ul>{fallback_items}</ul>
{"<h2>GPU Utilization Warnings</h2><ul>" + low_util_html + "</ul>" if low_util_html else ""}
<h2>Final Dashboard Snapshot</h2>
{dashboard_snapshot_html}
</body></html>"""
        path.write_text(html)


In [ ]:
%%writefile sih3d/stage_sync.py
"""Per-stage completion signals for the notebook's per-stage result cells.

Fed from the SAME bus-drain loop that feeds dashboard.on_event()/
report.on_event() — a bus.drain() queue can only have one destructive
consumer (see events.py), so this is a third fan-out target in that same
loop, not an independent reader. Each result cell calls wait_for(stage)
before rendering, which blocks (showing a live "waiting" line, the cell's
own responsibility) until that stage's STAGE_DONE/STAGE_ERROR fires, or
the whole pipeline ends early (e.g. a fatal setup error before that stage
was ever reached) — either way the wait returns rather than hanging
forever.
"""

from __future__ import annotations

import threading

from .events import Event, EventType
from .report import STAGE_ORDER


class StageSync:
    def __init__(self, stage_names: list[str] = STAGE_ORDER):
        self.done_events = {name: threading.Event() for name in stage_names}
        self.error_events = {name: threading.Event() for name in stage_names}
        self.pipeline_done = threading.Event()
        self.pipeline_error: str | None = None

    def on_event(self, evt: Event) -> None:
        if evt.type == EventType.STAGE_DONE:
            ev = self.done_events.get(evt.payload.get("stage"))
            if ev:
                ev.set()
        elif evt.type == EventType.STAGE_ERROR:
            name = evt.payload.get("stage")
            if name in self.error_events:
                self.error_events[name].set()
                self.done_events[name].set()
        elif evt.type in (EventType.PIPELINE_DONE, EventType.PIPELINE_ERROR):
            if evt.type == EventType.PIPELINE_ERROR:
                self.pipeline_error = evt.payload.get("error")
            self.pipeline_done.set()
            # Unblock any stage that never even started (a fatal setup
            # error before it was reached) so its result cell doesn't hang.
            for ev in self.done_events.values():
                ev.set()

    def wait_for(self, stage: str, timeout: float | None = None) -> bool:
        ev = self.done_events.get(stage)
        if ev is None:
            return True
        return ev.wait(timeout)

    def failed(self, stage: str) -> bool:
        ev = self.error_events.get(stage)
        return bool(ev and ev.is_set())


In [ ]:
%%writefile sih3d/stage_views.py
"""Pre-rendered (matplotlib, static) result views for the notebook's
per-stage cells. Every function here renders directly into the calling
cell's output via plt.show() — Jupyter captures that as a static image, so
the notebook file itself stays light (no live widgets re-embedded per
cell, no growing JSON state). Each function reads straight from
RunArtifacts (already populated by pipeline.py as it runs) and, where
useful, ReportBuilder for timing/GPU data.

Every function degrades gracefully (prints a plain-text note) if the data
it needs isn't there yet — a stage cell run against a run that hit an
early fallback shouldn't raise, it should say what's missing.
"""

from __future__ import annotations

import numpy as np

from .artifacts import RunArtifacts
from .report import ReportBuilder


def _require_matplotlib():
    import matplotlib.pyplot as plt

    return plt


def render_frame_extraction(artifacts: RunArtifacts) -> None:
    plt = _require_matplotlib()
    if not artifacts.keyframes:
        print("No keyframe data captured.")
        return

    accepted = [k for k in artifacts.keyframes if k.accepted]
    rejected = [k for k in artifacts.keyframes if not k.accepted]
    print(f"Keyframes: {len(accepted)} accepted, {len(rejected)} rejected (of {len(artifacts.keyframes)} candidates)")

    ordered = sorted(artifacts.keyframes, key=lambda k: k.frame_index)
    n = len(ordered)
    cols = min(12, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.1, rows * 1.1))
    axes = np.atleast_2d(axes)
    for i, kf in enumerate(ordered):
        ax = axes[i // cols, i % cols]
        if kf.thumbnail is not None:
            ax.imshow(kf.thumbnail, alpha=1.0 if kf.accepted else 0.35)
        ax.set_title(f"{kf.sharpness:.0f}", fontsize=6, color="#2ea043" if kf.accepted else "#d29922")
        ax.axis("off")
    for i in range(n, rows * cols):
        axes[i // cols, i % cols].axis("off")
    fig.suptitle("Keyframe grid (rejected greyed out, title = sharpness score)", fontsize=9)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    fig2, ax2 = plt.subplots(figsize=(8, 2.5))
    xs_a = [k.frame_index for k in accepted]
    ys_a = [k.sharpness for k in accepted]
    xs_r = [k.frame_index for k in rejected]
    ys_r = [k.sharpness for k in rejected]
    ax2.scatter(xs_a, ys_a, s=10, color="#2ea043", label="accepted")
    ax2.scatter(xs_r, ys_r, s=10, color="#d29922", label="rejected")
    ax2.set_xlabel("frame index")
    ax2.set_ylabel("sharpness")
    ax2.legend(fontsize=8)
    ax2.set_title("Sharpness by frame")
    plt.tight_layout()
    plt.show()
    plt.close(fig2)


def render_poses(artifacts: RunArtifacts) -> None:
    plt = _require_matplotlib()
    if not artifacts.gps_track_enu and not artifacts.camera_track_enu:
        print("No trajectory data captured (no GPS and no resolved camera poses).")
        return

    fig = plt.figure(figsize=(7, 5))
    ax = fig.add_subplot(111, projection="3d")
    if artifacts.gps_track_enu:
        g = np.array(artifacts.gps_track_enu)
        ax.plot(g[:, 0], g[:, 1], g[:, 2], color="#8b949e", label="GPS", linewidth=1.5)
    if artifacts.camera_track_enu:
        c = np.array(artifacts.camera_track_enu)
        ax.plot(c[:, 0], c[:, 1], c[:, 2], color="#2ea043", label="estimated camera", linewidth=1.5)
    ax.set_xlabel("E (m)")
    ax.set_ylabel("N (m)")
    ax.set_zlabel("U (m)")
    ax.legend(fontsize=8)
    title = "Camera trajectory: GPS vs. estimated"
    if artifacts.collinearity_index is not None:
        title += f"  |  collinearity index: {artifacts.collinearity_index:.4f} (near 0 = straight single pass)"
    ax.set_title(title, fontsize=9)
    plt.tight_layout()
    plt.show()
    plt.close(fig)


def render_geometric_reconstruction(artifacts: RunArtifacts) -> None:
    plt = _require_matplotlib()
    if not artifacts.geometry_samples:
        print("No geometry samples captured.")
        return

    n = len(artifacts.geometry_samples)
    fig, axes = plt.subplots(n, 4, figsize=(12, 3 * n), squeeze=False)
    for row, sample in enumerate(artifacts.geometry_samples):
        ax_rgb, ax_depth, ax_conf, ax_mask = axes[row]
        if sample.rgb is not None:
            ax_rgb.imshow(sample.rgb)
        ax_rgb.set_title(f"chunk {sample.chunk_idx}: RGB", fontsize=8)
        ax_rgb.axis("off")

        if sample.depth is not None:
            ax_depth.imshow(sample.depth, cmap="viridis")
        ax_depth.set_title("depth", fontsize=8)
        ax_depth.axis("off")

        if sample.confidence is not None:
            ax_conf.imshow(sample.confidence, cmap="magma", vmin=0, vmax=1)
        ax_conf.set_title("confidence", fontsize=8)
        ax_conf.axis("off")

        if sample.rgb is not None:
            overlay = sample.rgb.copy().astype(np.float32)
            if sample.dynamic_mask is not None and sample.dynamic_mask.any():
                red = np.zeros_like(overlay)
                red[..., 0] = 255
                m = sample.dynamic_mask[..., None].astype(np.float32)
                overlay = overlay * (1 - 0.5 * m) + red * (0.5 * m)
            ax_mask.imshow(overlay.astype(np.uint8))
        mask_note = "dynamic mask (red)" if (sample.dynamic_mask is not None and sample.dynamic_mask.any()) else "no dynamic objects masked"
        ax_mask.set_title(mask_note, fontsize=8)
        ax_mask.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)


def render_large_scale_alignment(artifacts: RunArtifacts) -> None:
    plt = _require_matplotlib()
    if not artifacts.chunk_alignments:
        print("No per-chunk alignment data captured (no-GPS mode, or alignment fell back before any chunk completed).")
        return

    records = sorted(artifacts.chunk_alignments, key=lambda r: r.chunk_idx)
    idx = [r.chunk_idx for r in records]
    before = [r.rmse_before_m if r.rmse_before_m is not None else np.nan for r in records]
    after = [r.rmse_after_m if r.rmse_after_m is not None else np.nan for r in records]

    fig, ax = plt.subplots(figsize=(8, 3))
    width = 0.35
    x = np.arange(len(idx))
    ax.bar(x - width / 2, before, width, label="before refinement", color="#9e6a03")
    ax.bar(x + width / 2, after, width, label="after refinement", color="#2ea043")
    ax.set_xticks(x)
    ax.set_xticklabels([str(i) for i in idx])
    ax.set_xlabel("chunk index")
    ax.set_ylabel("RMSE vs GPS (m)")
    ax.set_title(f"Per-chunk alignment residual, before/after pose-graph refinement ({records[0].mode})")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    plt.close(fig)


def render_dense_point_cloud(artifacts: RunArtifacts) -> None:
    plt = _require_matplotlib()
    print(f"Point count: {artifacts.point_count:,}")
    if artifacts.point_cloud_preview is None or len(artifacts.point_cloud_preview) == 0:
        print("No point cloud preview captured.")
        return

    pts = artifacts.point_cloud_preview
    colors = artifacts.point_cloud_preview_colors
    colors01 = np.clip(colors, 0, 255) / 255.0 if colors is not None else None

    fig = plt.figure(figsize=(11, 5))
    ax_top = fig.add_subplot(121, projection="3d")
    ax_top.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=colors01, s=0.5)
    ax_top.view_init(elev=89, azim=-90)
    ax_top.set_title("Top-down", fontsize=9)
    ax_top.set_axis_off()

    ax_obl = fig.add_subplot(122, projection="3d")
    ax_obl.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=colors01, s=0.5)
    ax_obl.view_init(elev=25, azim=-60)
    ax_obl.set_title("Oblique", fontsize=9)
    ax_obl.set_axis_off()

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    coverage_path = artifacts.output_paths.get("coverage.tif")
    if coverage_path:
        try:
            from .dashboard import _geotiff_to_png
            from PIL import Image
            import io

            png = _geotiff_to_png(coverage_path)
            img = Image.open(io.BytesIO(png))
            fig2, ax2 = plt.subplots(figsize=(5, 5))
            ax2.imshow(img)
            ax2.set_title("Coverage map (view count / confidence)", fontsize=9)
            ax2.axis("off")
            plt.tight_layout()
            plt.show()
            plt.close(fig2)
        except Exception as e:
            print(f"Coverage map preview unavailable: {e}")
    else:
        print("coverage.tif not yet written.")


def render_mesh_textured_model(artifacts: RunArtifacts) -> None:
    plt = _require_matplotlib()
    print(f"Mesh: {artifacts.mesh_n_vertices:,} vertices, {artifacts.mesh_n_faces:,} faces (method: {artifacts.mesh_method})")

    mesh_path = artifacts.output_paths.get("mesh.glb") or artifacts.output_paths.get("mesh.obj")
    rendered = False
    if mesh_path:
        try:
            rendered = _render_mesh_offscreen(mesh_path, plt)
        except Exception as e:
            print(f"Offscreen mesh render unavailable ({e}); showing a flat wireframe fallback instead.")
    if not rendered and mesh_path:
        try:
            _render_mesh_wireframe(mesh_path, plt)
        except Exception as e:
            print(f"Mesh preview unavailable: {e}")
    elif not mesh_path:
        print("No mesh file available yet.")

    for name, title in [("dsm.tif", "DSM"), ("orthomosaic.tif", "Orthomosaic")]:
        path = artifacts.output_paths.get(name)
        if not path:
            print(f"{title}: not yet written.")
            continue
        try:
            from .dashboard import _geotiff_to_png
            from PIL import Image
            import io

            png = _geotiff_to_png(path, max_dim=500)
            img = Image.open(io.BytesIO(png))
            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow(img)
            ax.set_title(title, fontsize=9)
            ax.axis("off")
            plt.tight_layout()
            plt.show()
            plt.close(fig)
        except Exception as e:
            print(f"{title} preview unavailable: {e}")


def _render_mesh_offscreen(mesh_path: str, plt) -> bool:
    """Tries a real textured/shaded render via Open3D's offscreen
    renderer, top-down + oblique (matching the point-cloud preview's two
    views). Needs a working (EGL/OSMesa) headless GL context, which isn't
    guaranteed on Kaggle — returns False (never raises past this function)
    so the caller falls back to a flat wireframe."""
    import open3d as o3d
    import open3d.visualization.rendering as rendering

    mesh = o3d.io.read_triangle_mesh(mesh_path, enable_post_processing=True)
    if len(mesh.vertices) == 0:
        return False
    mesh.compute_vertex_normals()

    center = mesh.get_center()
    extent = np.asarray(mesh.get_max_bound()) - np.asarray(mesh.get_min_bound())
    radius = float(np.linalg.norm(extent)) or 10.0

    renderer = rendering.OffscreenRenderer(640, 480)
    mat = rendering.MaterialRecord()
    mat.shader = "defaultLit"
    renderer.scene.add_geometry("mesh", mesh, mat)
    renderer.scene.set_background([0.05, 0.06, 0.08, 1.0])

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    views = [("Top-down", center + [0, 0, radius], [0, 1, 0]), ("Oblique", center + [radius, radius, radius], [0, 0, 1])]
    for ax, (title, eye, up) in zip(axes, views):
        renderer.scene.camera.look_at(center, eye, up)
        img = renderer.render_to_image()
        ax.imshow(np.asarray(img))
        ax.set_title(f"Textured mesh — {title}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    return True


def _render_mesh_wireframe(mesh_path: str, plt) -> None:
    """Flat-shaded matplotlib fallback, top-down + oblique: no texture,
    but shows real geometry/vertex colors, and never needs a GL context."""
    import trimesh
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection

    m = trimesh.load(mesh_path, process=False)
    if hasattr(m, "geometry"):  # a Scene (e.g. GLB), take the first mesh
        m = next(iter(m.geometry.values()))
    verts = np.asarray(m.vertices)
    faces = np.asarray(m.faces)
    if len(faces) > 20000:
        idx = np.random.default_rng(0).choice(len(faces), size=20000, replace=False)
        faces = faces[idx]

    face_colors = None
    if hasattr(m.visual, "vertex_colors") and m.visual.vertex_colors is not None:
        vc = np.asarray(m.visual.vertex_colors)[:, :3] / 255.0
        face_colors = vc[faces].mean(axis=1)

    fig = plt.figure(figsize=(11, 5))
    views = [("Top-down", 89, -90), ("Oblique", 25, -60)]
    for i, (title, elev, azim) in enumerate(views, start=1):
        ax = fig.add_subplot(1, 2, i, projection="3d")
        poly = Poly3DCollection(verts[faces], facecolor=face_colors if face_colors is not None else "#888888", linewidths=0)
        ax.add_collection3d(poly)
        ax.set_xlim(verts[:, 0].min(), verts[:, 0].max())
        ax.set_ylim(verts[:, 1].min(), verts[:, 1].max())
        ax.set_zlim(verts[:, 2].min(), verts[:, 2].max())
        ax.view_init(elev=elev, azim=azim)
        ax.set_title(f"Mesh (flat-shaded fallback) — {title}", fontsize=9)
        ax.set_axis_off()
    plt.tight_layout()
    plt.show()
    plt.close(fig)


def render_final_summary(report: ReportBuilder) -> None:
    plt = _require_matplotlib()
    d = report.to_dict()

    rows = "".join(
        f"<tr><td>{s['name']}</td><td>{s['status']}</td>"
        f"<td>{s['elapsed_s']:.1f}s</td>" if s["elapsed_s"] is not None else "<td>-</td>"
        f"<td>{', '.join(f'GPU{k}: {v:.0f}%' for k, v in s['avg_gpu_util_pct'].items()) or '-'}</td></tr>"
        for s in d["stages"]
    )
    from IPython.display import HTML, display

    display(HTML(f"<table><tr><th>Stage</th><th>Status</th><th>Elapsed</th><th>Avg GPU Util</th></tr>{rows}</table>"))
    print(f"\nTotal elapsed: {d['total_elapsed_s']:.1f}s" if d["total_elapsed_s"] else "")
    print(f"Georeferenced: {d['georeferenced']}")
    print(f"Alignment RMSE vs GPS: {d['alignment_rmse_m']} m")
    print(f"Collinearity index: {d['collinearity_index']}")
    if d["fallbacks_triggered"]:
        print(f"\n{len(d['fallbacks_triggered'])} fallback(s) triggered:")
        for f in d["fallbacks_triggered"]:
            print(" -", f)


In [ ]:
%%writefile sih3d/telemetry.py
"""Telemetry parsing + GPS/attitude sync to frames + ENU conversion.

Supports: DJI .SRT (both the bracket format and the older GPS(lon,lat,alt)
format), telemetry embedded as a subtitle/data stream inside the video
(extracted via ffmpeg), .csv (auto column matching), .gpx, .kml, .json,
and MAVLink .tlog/.bin (via pymavlink).

Never raises on a missing/unparseable file — returns an empty TelemetryTrack
and lets the caller (io_detect / the pipeline) decide to fall back to
RGB-only reconstruction.
"""

from __future__ import annotations

import csv
import json
import math
import re
import subprocess
import tempfile
from dataclasses import dataclass, field
from pathlib import Path
from xml.etree import ElementTree as ET


@dataclass
class TelemetrySample:
    t: float  # seconds, relative to track start (or absolute unix if known)
    lat: float
    lon: float
    alt: float | None = None  # meters, absolute or relative — see `alt_is_relative`
    alt_is_relative: bool = False
    yaw: float | None = None  # degrees
    pitch: float | None = None
    roll: float | None = None
    gimbal_yaw: float | None = None
    gimbal_pitch: float | None = None
    gimbal_roll: float | None = None
    source_index: int = 0


@dataclass
class TelemetryTrack:
    samples: list[TelemetrySample] = field(default_factory=list)
    kind: str = "none"
    has_timestamps: bool = True
    notes: list[str] = field(default_factory=list)

    def __len__(self) -> int:
        return len(self.samples)

    def __bool__(self) -> bool:
        return len(self.samples) > 0


# ---------------------------------------------------------------------------
# DJI SRT
# ---------------------------------------------------------------------------

_SRT_TIME_RE = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})[,.](\d{1,3})\s*-->\s*(\d{2}):(\d{2}):(\d{2})[,.](\d{1,3})"
)
# Newer bracket format, e.g.:
# [latitude: 22.5726] [longitude: 88.3639] [rel_alt: 12.3 abs_alt: 45.6]
# optionally with [gb_yaw: 1.2] [gb_pitch: -30.0] [gb_roll: 0.0]
_BRACKET_LAT = re.compile(r"\[latitude\s*:\s*(-?\d+\.?\d*)\]")
_BRACKET_LON = re.compile(r"\[longitude\s*:\s*(-?\d+\.?\d*)\]")
_BRACKET_RELALT = re.compile(r"rel_alt\s*:\s*(-?\d+\.?\d*)")
_BRACKET_ABSALT = re.compile(r"abs_alt\s*:\s*(-?\d+\.?\d*)")
_BRACKET_GB_YAW = re.compile(r"gb_yaw\s*:\s*(-?\d+\.?\d*)")
_BRACKET_GB_PITCH = re.compile(r"gb_pitch\s*:\s*(-?\d+\.?\d*)")
_BRACKET_GB_ROLL = re.compile(r"gb_roll\s*:\s*(-?\d+\.?\d*)")
# Older format: GPS(lon,lat,alt)
_OLD_GPS = re.compile(r"GPS\s*\(\s*(-?\d+\.?\d*)\s*,\s*(-?\d+\.?\d*)\s*,\s*(-?\d+\.?\d*)\s*\)")


def _srt_time_to_seconds(h, m, s, ms) -> float:
    ms = ms.ljust(3, "0")[:3]
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000.0


def parse_srt(path: Path) -> TelemetryTrack:
    text = path.read_text(errors="ignore")
    blocks = re.split(r"\n\s*\n", text.strip())
    samples: list[TelemetrySample] = []
    fmt_seen = None

    for idx, block in enumerate(blocks):
        tmatch = _SRT_TIME_RE.search(block)
        t = None
        if tmatch:
            h1, m1, s1, ms1, *_ = tmatch.groups()
            t = _srt_time_to_seconds(h1, m1, s1, ms1)

        lat_m = _BRACKET_LAT.search(block)
        lon_m = _BRACKET_LON.search(block)
        if lat_m and lon_m:
            fmt_seen = "bracket"
            lat = float(lat_m.group(1))
            lon = float(lon_m.group(1))
            rel_m = _BRACKET_RELALT.search(block)
            abs_m = _BRACKET_ABSALT.search(block)
            alt = None
            alt_is_relative = False
            if abs_m:
                alt = float(abs_m.group(1))
            elif rel_m:
                alt = float(rel_m.group(1))
                alt_is_relative = True
            gy = _BRACKET_GB_YAW.search(block)
            gp = _BRACKET_GB_PITCH.search(block)
            gr = _BRACKET_GB_ROLL.search(block)
            samples.append(TelemetrySample(
                t=t if t is not None else float(idx),
                lat=lat, lon=lon, alt=alt, alt_is_relative=alt_is_relative,
                gimbal_yaw=float(gy.group(1)) if gy else None,
                gimbal_pitch=float(gp.group(1)) if gp else None,
                gimbal_roll=float(gr.group(1)) if gr else None,
                source_index=idx,
            ))
            continue

        old_m = _OLD_GPS.search(block)
        if old_m:
            fmt_seen = "gps_tuple"
            lon, lat, alt = (float(x) for x in old_m.groups())
            samples.append(TelemetrySample(
                t=t if t is not None else float(idx), lat=lat, lon=lon, alt=alt,
                source_index=idx,
            ))

    track = TelemetryTrack(samples=samples, kind="srt")
    if fmt_seen:
        track.notes.append(f"DJI SRT format detected: {fmt_seen}")
    if not samples:
        track.notes.append("SRT parsed but no GPS entries matched either known format")
    return track


# ---------------------------------------------------------------------------
# Embedded subtitle/data stream extraction (ffmpeg)
# ---------------------------------------------------------------------------

def extract_embedded_telemetry(video_path: Path) -> TelemetryTrack:
    """Probe for a subtitle/data stream inside the video and extract it.

    DJI drones sometimes mux the same SRT-style telemetry as a subtitle
    stream instead of shipping a sidecar .srt file.
    """
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-print_format", "json", "-show_streams", str(video_path)],
        capture_output=True, text=True, timeout=30,
    )
    try:
        info = json.loads(probe.stdout)
    except Exception:
        return TelemetryTrack(kind="embedded", notes=["ffprobe failed on video for embedded telemetry check"])

    sub_streams = [
        s for s in info.get("streams", [])
        if s.get("codec_type") in ("subtitle", "data")
    ]
    if not sub_streams:
        return TelemetryTrack(kind="embedded", notes=["no subtitle/data stream found in video"])

    stream_index = sub_streams[0]["index"]
    with tempfile.NamedTemporaryFile(suffix=".srt", delete=False) as tmp:
        tmp_path = Path(tmp.name)
    try:
        res = subprocess.run(
            ["ffmpeg", "-y", "-i", str(video_path), "-map", f"0:{stream_index}", str(tmp_path)],
            capture_output=True, text=True, timeout=120,
        )
        if res.returncode != 0 or not tmp_path.exists() or tmp_path.stat().st_size == 0:
            return TelemetryTrack(kind="embedded", notes=["ffmpeg extraction of embedded stream produced no usable data"])
        track = parse_srt(tmp_path)
        track.kind = "embedded"
        track.notes.append(f"extracted from embedded stream index {stream_index}")
        return track
    finally:
        tmp_path.unlink(missing_ok=True)


# ---------------------------------------------------------------------------
# CSV (auto column matching)
# ---------------------------------------------------------------------------

_COL_ALIASES = {
    "lat": {"lat", "latitude"},
    "lon": {"lon", "lng", "longitude"},
    "alt": {"alt", "altitude", "rel_alt", "relative_altitude", "abs_alt"},
    "time": {"time", "timestamp", "t", "time_s", "datetime"},
    "yaw": {"yaw", "heading"},
    "pitch": {"pitch"},
    "roll": {"roll"},
    "gimbal_yaw": {"gimbal_yaw", "gb_yaw"},
    "gimbal_pitch": {"gimbal_pitch", "gb_pitch"},
    "gimbal_roll": {"gimbal_roll", "gb_roll"},
}


def _match_columns(fieldnames: list[str]) -> dict[str, str]:
    lower = {f.lower().strip(): f for f in fieldnames}
    matched: dict[str, str] = {}
    for canon, aliases in _COL_ALIASES.items():
        for alias in aliases:
            if alias in lower:
                matched[canon] = lower[alias]
                break
    return matched


def _parse_time_value(v: str, idx: int) -> float:
    v = v.strip()
    try:
        return float(v)
    except ValueError:
        pass
    for fmt_try in ("iso",):
        try:
            from datetime import datetime

            return datetime.fromisoformat(v.replace("Z", "+00:00")).timestamp()
        except Exception:
            continue
    return float(idx)


def parse_csv(path: Path) -> TelemetryTrack:
    with open(path, newline="", errors="ignore") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None:
            return TelemetryTrack(kind="csv", notes=["CSV has no header row"])
        cols = _match_columns(reader.fieldnames)
        if "lat" not in cols or "lon" not in cols:
            return TelemetryTrack(kind="csv", notes=[f"CSV missing lat/lon columns; found headers: {reader.fieldnames}"])

        samples = []
        has_time = "time" in cols
        for idx, row in enumerate(reader):
            try:
                lat = float(row[cols["lat"]])
                lon = float(row[cols["lon"]])
            except (ValueError, KeyError):
                continue
            alt = None
            if "alt" in cols:
                try:
                    alt = float(row[cols["alt"]])
                except (ValueError, KeyError):
                    alt = None
            t = _parse_time_value(row[cols["time"]], idx) if has_time else float(idx)

            def _f(key):
                if key not in cols:
                    return None
                try:
                    return float(row[cols[key]])
                except (ValueError, KeyError):
                    return None

            samples.append(TelemetrySample(
                t=t, lat=lat, lon=lon, alt=alt,
                alt_is_relative="rel_alt" in cols.get("alt", ""),
                yaw=_f("yaw"), pitch=_f("pitch"), roll=_f("roll"),
                gimbal_yaw=_f("gimbal_yaw"), gimbal_pitch=_f("gimbal_pitch"), gimbal_roll=_f("gimbal_roll"),
                source_index=idx,
            ))
        track = TelemetryTrack(samples=samples, kind="csv", has_timestamps=has_time)
        if not has_time:
            track.notes.append("CSV had no time column; using row index, will interpolate proportionally over duration")
        return track


# ---------------------------------------------------------------------------
# GPX
# ---------------------------------------------------------------------------

def parse_gpx(path: Path) -> TelemetryTrack:
    try:
        tree = ET.parse(path)
    except Exception as e:
        return TelemetryTrack(kind="gpx", notes=[f"GPX parse failed: {e}"])
    root = tree.getroot()
    ns_match = re.match(r"\{(.*)\}", root.tag)
    ns = {"g": ns_match.group(1)} if ns_match else {}

    def tag(name):
        return f"g:{name}" if ns else name

    samples = []
    pts = root.findall(f".//{tag('trkpt')}", ns) or root.findall(f".//{tag('wpt')}", ns)
    for idx, pt in enumerate(pts):
        try:
            lat = float(pt.attrib["lat"])
            lon = float(pt.attrib["lon"])
        except (KeyError, ValueError):
            continue
        ele_el = pt.find(tag("ele"), ns)
        alt = float(ele_el.text) if ele_el is not None and ele_el.text else None
        time_el = pt.find(tag("time"), ns)
        t = None
        if time_el is not None and time_el.text:
            try:
                from datetime import datetime

                t = datetime.fromisoformat(time_el.text.replace("Z", "+00:00")).timestamp()
            except Exception:
                t = None
        samples.append(TelemetrySample(t=t if t is not None else float(idx), lat=lat, lon=lon, alt=alt, source_index=idx))

    track = TelemetryTrack(samples=samples, kind="gpx", has_timestamps=all(s.t is not None for s in samples))
    return track


# ---------------------------------------------------------------------------
# KML
# ---------------------------------------------------------------------------

def parse_kml(path: Path) -> TelemetryTrack:
    try:
        tree = ET.parse(path)
    except Exception as e:
        return TelemetryTrack(kind="kml", notes=[f"KML parse failed: {e}"])
    root = tree.getroot()
    ns_match = re.match(r"\{(.*)\}", root.tag)
    ns = {"k": ns_match.group(1)} if ns_match else {}

    def tag(name):
        return f"k:{name}" if ns else name

    samples = []
    idx = 0
    for coord_el in root.findall(f".//{tag('coordinates')}", ns):
        if not coord_el.text:
            continue
        for tup in coord_el.text.strip().split():
            parts = tup.split(",")
            if len(parts) < 2:
                continue
            lon, lat = float(parts[0]), float(parts[1])
            alt = float(parts[2]) if len(parts) > 2 else None
            samples.append(TelemetrySample(t=float(idx), lat=lat, lon=lon, alt=alt, source_index=idx))
            idx += 1

    track = TelemetryTrack(samples=samples, kind="kml", has_timestamps=False)
    if samples:
        track.notes.append("KML has no per-point timestamps; using sequence index, will interpolate proportionally")
    return track


# ---------------------------------------------------------------------------
# JSON (generic list-of-records)
# ---------------------------------------------------------------------------

def parse_json_telemetry(path: Path) -> TelemetryTrack:
    try:
        data = json.loads(path.read_text())
    except Exception as e:
        return TelemetryTrack(kind="json", notes=[f"JSON parse failed: {e}"])

    records = data
    if isinstance(data, dict):
        for key in ("samples", "points", "telemetry", "track", "data"):
            if key in data and isinstance(data[key], list):
                records = data[key]
                break
        else:
            return TelemetryTrack(kind="json", notes=["JSON is an object but no list of records found under common keys"])
    if not isinstance(records, list):
        return TelemetryTrack(kind="json", notes=["JSON telemetry is not a list of records"])

    aliases = {k: v for k, v in _COL_ALIASES.items()}
    samples = []
    has_time = True
    for idx, rec in enumerate(records):
        if not isinstance(rec, dict):
            continue
        lower_rec = {k.lower(): v for k, v in rec.items()}

        def find(canon):
            for alias in aliases[canon]:
                if alias in lower_rec:
                    return lower_rec[alias]
            return None

        lat, lon = find("lat"), find("lon")
        if lat is None or lon is None:
            continue
        t_raw = find("time")
        if t_raw is None:
            has_time = False
            t = float(idx)
        else:
            t = t_raw if isinstance(t_raw, (int, float)) else _parse_time_value(str(t_raw), idx)
        samples.append(TelemetrySample(
            t=float(t), lat=float(lat), lon=float(lon),
            alt=float(find("alt")) if find("alt") is not None else None,
            yaw=float(find("yaw")) if find("yaw") is not None else None,
            pitch=float(find("pitch")) if find("pitch") is not None else None,
            roll=float(find("roll")) if find("roll") is not None else None,
            source_index=idx,
        ))
    return TelemetryTrack(samples=samples, kind="json", has_timestamps=has_time)


# ---------------------------------------------------------------------------
# MAVLink (.tlog / .bin) via pymavlink
# ---------------------------------------------------------------------------

def parse_mavlink(path: Path) -> TelemetryTrack:
    try:
        from pymavlink import mavutil
    except ImportError:
        return TelemetryTrack(kind="mavlink", notes=["pymavlink not installed — cannot parse MAVLink log"])

    try:
        conn = mavutil.mavlink_connection(str(path))
    except Exception as e:
        return TelemetryTrack(kind="mavlink", notes=[f"failed to open MAVLink log: {e}"])

    samples = []
    idx = 0
    attitude_cache = {"yaw": None, "pitch": None, "roll": None}
    while True:
        try:
            msg = conn.recv_match(blocking=False)
        except Exception:
            break
        if msg is None:
            break
        mtype = msg.get_type()
        if mtype == "ATTITUDE":
            attitude_cache["yaw"] = math.degrees(msg.yaw)
            attitude_cache["pitch"] = math.degrees(msg.pitch)
            attitude_cache["roll"] = math.degrees(msg.roll)
        elif mtype in ("GLOBAL_POSITION_INT", "GPS_RAW_INT"):
            lat = msg.lat / 1e7
            lon = msg.lon / 1e7
            alt = getattr(msg, "relative_alt", None)
            alt = (alt / 1000.0) if alt is not None else (getattr(msg, "alt", 0) / 1000.0)
            t = getattr(msg, "time_boot_ms", idx * 100) / 1000.0
            samples.append(TelemetrySample(
                t=t, lat=lat, lon=lon, alt=alt, alt_is_relative=hasattr(msg, "relative_alt"),
                yaw=attitude_cache["yaw"], pitch=attitude_cache["pitch"], roll=attitude_cache["roll"],
                source_index=idx,
            ))
            idx += 1

    return TelemetryTrack(samples=samples, kind="mavlink")


# ---------------------------------------------------------------------------
# Dispatch
# ---------------------------------------------------------------------------

_PARSERS = {
    "srt": parse_srt,
    "csv": parse_csv,
    "gpx": parse_gpx,
    "kml": parse_kml,
    "json": parse_json_telemetry,
    "mavlink": parse_mavlink,
}


def parse_telemetry(path: Path, kind: str) -> TelemetryTrack:
    parser = _PARSERS.get(kind)
    if parser is None:
        return TelemetryTrack(kind=kind, notes=[f"no parser for telemetry kind '{kind}'"])
    try:
        return parser(path)
    except Exception as e:
        return TelemetryTrack(kind=kind, notes=[f"parser raised {type(e).__name__}: {e}"])


# ---------------------------------------------------------------------------
# GPS -> local ENU (meters) + UTM zone/EPSG
# ---------------------------------------------------------------------------

WGS84_A = 6378137.0
WGS84_F = 1 / 298.257223563


def utm_zone_epsg(lat: float, lon: float) -> tuple[int, int]:
    zone = int(math.floor((lon + 180) / 6) + 1)
    epsg = (32600 if lat >= 0 else 32700) + zone
    return zone, epsg


def geodetic_to_ecef(lat_deg: float, lon_deg: float, alt_m: float) -> tuple[float, float, float]:
    lat, lon = math.radians(lat_deg), math.radians(lon_deg)
    e2 = WGS84_F * (2 - WGS84_F)
    n = WGS84_A / math.sqrt(1 - e2 * math.sin(lat) ** 2)
    x = (n + alt_m) * math.cos(lat) * math.cos(lon)
    y = (n + alt_m) * math.cos(lat) * math.sin(lon)
    z = (n * (1 - e2) + alt_m) * math.sin(lat)
    return x, y, z


def ecef_to_enu(x, y, z, lat0_deg, lon0_deg, alt0_m) -> tuple[float, float, float]:
    x0, y0, z0 = geodetic_to_ecef(lat0_deg, lon0_deg, alt0_m)
    dx, dy, dz = x - x0, y - y0, z - z0
    lat0, lon0 = math.radians(lat0_deg), math.radians(lon0_deg)
    sl, cl = math.sin(lat0), math.cos(lat0)
    so, co = math.sin(lon0), math.cos(lon0)
    e = -so * dx + co * dy
    n = -sl * co * dx - sl * so * dy + cl * dz
    u = cl * co * dx + cl * so * dy + sl * dz
    return e, n, u


def enu_to_ecef(e: float, n: float, u: float, lat0_deg: float, lon0_deg: float, alt0_m: float) -> tuple[float, float, float]:
    """Inverse of ecef_to_enu — used to convert an aligned world-frame
    (post-align.py) point or camera track back to ECEF/geodetic for
    trajectory.geojson/.kml export, which need lon/lat."""
    lat0, lon0 = math.radians(lat0_deg), math.radians(lon0_deg)
    sl, cl = math.sin(lat0), math.cos(lat0)
    so, co = math.sin(lon0), math.cos(lon0)
    # Transpose of the rotation used in ecef_to_enu (it's orthonormal).
    dx = -so * e - sl * co * n + cl * co * u
    dy = co * e - sl * so * n + cl * so * u
    dz = cl * n + sl * u
    x0, y0, z0 = geodetic_to_ecef(lat0_deg, lon0_deg, alt0_m)
    return x0 + dx, y0 + dy, z0 + dz


def ecef_to_geodetic(x: float, y: float, z: float, n_iters: int = 5) -> tuple[float, float, float]:
    """Iterative ECEF -> geodetic (lat_deg, lon_deg, alt_m). A handful of
    Newton iterations on the WGS84 ellipsoid converges to sub-millimeter
    accuracy, which is more than enough given our GPS input is already
    consumer-grade at best."""
    lon = math.atan2(y, x)
    e2 = WGS84_F * (2 - WGS84_F)
    p = math.hypot(x, y)
    lat = math.atan2(z, p * (1 - e2))
    alt = 0.0
    for _ in range(n_iters):
        sin_lat = math.sin(lat)
        n = WGS84_A / math.sqrt(1 - e2 * sin_lat ** 2)
        alt = p / math.cos(lat) - n
        lat = math.atan2(z, p * (1 - e2 * n / (n + alt)))
    return math.degrees(lat), math.degrees(lon), alt


def enu_to_geodetic(e: float, n: float, u: float, lat0_deg: float, lon0_deg: float, alt0_m: float) -> tuple[float, float, float]:
    x, y, z = enu_to_ecef(e, n, u, lat0_deg, lon0_deg, alt0_m)
    return ecef_to_geodetic(x, y, z)


def track_to_enu(track: TelemetryTrack) -> tuple[list[tuple[float, float, float]], tuple[float, float, float], tuple[int, int]]:
    """Returns (enu_points, origin(lat,lon,alt), (utm_zone, epsg))."""
    if not track:
        return [], (0.0, 0.0, 0.0), (0, 0)
    origin = track.samples[0]
    alt0 = origin.alt if origin.alt is not None else 0.0
    origin_tup = (origin.lat, origin.lon, alt0)
    enu = []
    for s in track.samples:
        alt = s.alt if s.alt is not None else 0.0
        x, y, z = geodetic_to_ecef(s.lat, s.lon, alt)
        enu.append(ecef_to_enu(x, y, z, *origin_tup))
    zone_epsg = utm_zone_epsg(origin.lat, origin.lon)
    return enu, origin_tup, zone_epsg


# ---------------------------------------------------------------------------
# Collinearity index (PCA eigenvalue ratio on the ENU ground track)
# ---------------------------------------------------------------------------

def collinearity_index(enu_points: list[tuple[float, float, float]]) -> float:
    """Ratio of the 2nd/1st eigenvalue of the horizontal (E,N) covariance.

    ~0 => perfectly straight line (single-pass flight: cannot constrain
    roll around the flight axis from GPS alone).
    ~1 => isotropic spread (e.g. a grid/orbit survey).
    """
    if len(enu_points) < 3:
        return 0.0
    import numpy as np

    pts = np.array([(e, n) for e, n, _ in enu_points])
    pts = pts - pts.mean(axis=0)
    cov = np.cov(pts.T)
    eigvals = np.linalg.eigvalsh(cov)
    eigvals = np.sort(eigvals)[::-1]
    if eigvals[0] <= 1e-9:
        return 0.0
    return float(eigvals[1] / eigvals[0])


# ---------------------------------------------------------------------------
# Time-sync telemetry samples to frame timestamps
# ---------------------------------------------------------------------------

def sync_to_frame_times(track: TelemetryTrack, frame_times_s: list[float]) -> list[TelemetrySample | None]:
    """Interpolate telemetry to each frame timestamp (linear on lat/lon/alt/attitude).

    If the track has no real timestamps, `frame_times_s` values are ignored
    and interpolation is proportional over track duration/frame count
    (caller must have already logged this via track.notes).
    """
    if not track:
        return [None for _ in frame_times_s]

    ts = [s.t for s in track.samples]
    if not track.has_timestamps:
        n = len(track.samples)
        span = max(len(frame_times_s) - 1, 1)
        proportional = [i / span for i in range(len(frame_times_s))]
        idx_float = [p * (n - 1) for p in proportional]
        out = []
        for f in idx_float:
            lo = int(math.floor(f))
            hi = min(lo + 1, n - 1)
            frac = f - lo
            out.append(_lerp_sample(track.samples[lo], track.samples[hi], frac))
        return out

    out = []
    for t in frame_times_s:
        if t <= ts[0]:
            out.append(track.samples[0])
            continue
        if t >= ts[-1]:
            out.append(track.samples[-1])
            continue
        lo = 0
        hi = len(ts) - 1
        while hi - lo > 1:
            mid = (lo + hi) // 2
            if ts[mid] <= t:
                lo = mid
            else:
                hi = mid
        span = ts[hi] - ts[lo]
        frac = (t - ts[lo]) / span if span > 1e-9 else 0.0
        out.append(_lerp_sample(track.samples[lo], track.samples[hi], frac))
    return out


def _lerp_sample(a: TelemetrySample, b: TelemetrySample, frac: float) -> TelemetrySample:
    def lerp(x, y):
        if x is None or y is None:
            return x if x is not None else y
        return x + (y - x) * frac

    return TelemetrySample(
        t=lerp(a.t, b.t), lat=lerp(a.lat, b.lat), lon=lerp(a.lon, b.lon), alt=lerp(a.alt, b.alt),
        alt_is_relative=a.alt_is_relative,
        yaw=lerp(a.yaw, b.yaw), pitch=lerp(a.pitch, b.pitch), roll=lerp(a.roll, b.roll),
        gimbal_yaw=lerp(a.gimbal_yaw, b.gimbal_yaw), gimbal_pitch=lerp(a.gimbal_pitch, b.gimbal_pitch),
        gimbal_roll=lerp(a.gimbal_roll, b.gimbal_roll),
    )


In [ ]:
%%writefile sih3d/vendor/__init__.py



In [ ]:
%%writefile sih3d/vendor/dinov2/__init__.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

__version__ = "0.0.1"


In [ ]:
%%writefile sih3d/vendor/dinov2/hub/__init__.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.


In [ ]:
%%writefile sih3d/vendor/dinov2/hub/backbones.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

from enum import Enum
from typing import Union

import torch

from sih3d.vendor.dinov2.hub.utils import (
    _DINOV2_BASE_URL,
    _make_dinov2_model_name,
)


class Weights(Enum):
    LVD142M = "LVD142M"


def _make_dinov2_model(
    *,
    arch_name: str = "vit_large",
    img_size: int = 518,
    patch_size: int = 14,
    init_values: float = 1.0,
    ffn_layer: str = "mlp",
    block_chunks: int = 0,
    num_register_tokens: int = 0,
    interpolate_antialias: bool = False,
    interpolate_offset: float = 0.1,
    pretrained: bool = True,
    weights: Union[Weights, str] = Weights.LVD142M,
    **kwargs,
):
    from ..models import vision_transformer as vits

    if isinstance(weights, str):
        try:
            weights = Weights[weights]
        except KeyError:
            raise AssertionError(f"Unsupported weights: {weights}")

    model_base_name = _make_dinov2_model_name(arch_name, patch_size)
    vit_kwargs = dict(
        img_size=img_size,
        patch_size=patch_size,
        init_values=init_values,
        ffn_layer=ffn_layer,
        block_chunks=block_chunks,
        num_register_tokens=num_register_tokens,
        interpolate_antialias=interpolate_antialias,
        interpolate_offset=interpolate_offset,
    )
    vit_kwargs.update(**kwargs)
    model = vits.__dict__[arch_name](**vit_kwargs)

    if pretrained:
        model_full_name = _make_dinov2_model_name(
            arch_name, patch_size, num_register_tokens
        )
        url = _DINOV2_BASE_URL + f"/{model_base_name}/{model_full_name}_pretrain.pth"
        state_dict = torch.hub.load_state_dict_from_url(url, map_location="cpu")
        model.load_state_dict(state_dict, strict=True)

    return model


def dinov2_vits14(
    *, pretrained: bool = True, weights: Union[Weights, str] = Weights.LVD142M, **kwargs
):
    """
    DINOv2 ViT-S/14 model (optionally) pretrained on the LVD-142M dataset.
    """
    return _make_dinov2_model(
        arch_name="vit_small", pretrained=pretrained, weights=weights, **kwargs
    )


def dinov2_vitb14(
    *, pretrained: bool = True, weights: Union[Weights, str] = Weights.LVD142M, **kwargs
):
    """
    DINOv2 ViT-B/14 model (optionally) pretrained on the LVD-142M dataset.
    """
    return _make_dinov2_model(
        arch_name="vit_base", pretrained=pretrained, weights=weights, **kwargs
    )


def dinov2_vitl14(
    *, pretrained: bool = True, weights: Union[Weights, str] = Weights.LVD142M, **kwargs
):
    """
    DINOv2 ViT-L/14 model (optionally) pretrained on the LVD-142M dataset.
    """
    return _make_dinov2_model(
        arch_name="vit_large", pretrained=pretrained, weights=weights, **kwargs
    )


def dinov2_vitg14(
    *, pretrained: bool = True, weights: Union[Weights, str] = Weights.LVD142M, **kwargs
):
    """
    DINOv2 ViT-g/14 model (optionally) pretrained on the LVD-142M dataset.
    """
    return _make_dinov2_model(
        arch_name="vit_giant2",
        ffn_layer="swiglufused",
        weights=weights,
        pretrained=pretrained,
        **kwargs,
    )


def dinov2_vits14_reg(
    *, pretrained: bool = True, weights: Union[Weights, str] = Weights.LVD142M, **kwargs
):
    """
    DINOv2 ViT-S/14 model with registers (optionally) pretrained on the LVD-142M dataset.
    """
    return _make_dinov2_model(
        arch_name="vit_small",
        pretrained=pretrained,
        weights=weights,
        num_register_tokens=4,
        interpolate_antialias=True,
        interpolate_offset=0.0,
        **kwargs,
    )


def dinov2_vitb14_reg(
    *, pretrained: bool = True, weights: Union[Weights, str] = Weights.LVD142M, **kwargs
):
    """
    DINOv2 ViT-B/14 model with registers (optionally) pretrained on the LVD-142M dataset.
    """
    return _make_dinov2_model(
        arch_name="vit_base",
        pretrained=pretrained,
        weights=weights,
        num_register_tokens=4,
        interpolate_antialias=True,
        interpolate_offset=0.0,
        **kwargs,
    )


def dinov2_vitl14_reg(
    *, pretrained: bool = True, weights: Union[Weights, str] = Weights.LVD142M, **kwargs
):
    """
    DINOv2 ViT-L/14 model with registers (optionally) pretrained on the LVD-142M dataset.
    """
    return _make_dinov2_model(
        arch_name="vit_large",
        pretrained=pretrained,
        weights=weights,
        num_register_tokens=4,
        interpolate_antialias=True,
        interpolate_offset=0.0,
        **kwargs,
    )


def dinov2_vitg14_reg(
    *, pretrained: bool = True, weights: Union[Weights, str] = Weights.LVD142M, **kwargs
):
    """
    DINOv2 ViT-g/14 model with registers (optionally) pretrained on the LVD-142M dataset.
    """
    return _make_dinov2_model(
        arch_name="vit_giant2",
        ffn_layer="swiglufused",
        weights=weights,
        pretrained=pretrained,
        num_register_tokens=4,
        interpolate_antialias=True,
        interpolate_offset=0.0,
        **kwargs,
    )


In [ ]:
%%writefile sih3d/vendor/dinov2/hub/utils.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

import itertools
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

_DINOV2_BASE_URL = "https://dl.fbaipublicfiles.com/dinov2"


def _make_dinov2_model_name(
    arch_name: str, patch_size: int, num_register_tokens: int = 0
) -> str:
    compact_arch_name = arch_name.replace("_", "")[:4]
    registers_suffix = f"_reg{num_register_tokens}" if num_register_tokens else ""
    return f"dinov2_{compact_arch_name}{patch_size}{registers_suffix}"


class CenterPadding(nn.Module):
    def __init__(self, multiple):
        super().__init__()
        self.multiple = multiple

    def _get_pad(self, size):
        new_size = math.ceil(size / self.multiple) * self.multiple
        pad_size = new_size - size
        pad_size_left = pad_size // 2
        pad_size_right = pad_size - pad_size_left
        return pad_size_left, pad_size_right

    @torch.inference_mode()
    def forward(self, x):
        pads = list(
            itertools.chain.from_iterable(self._get_pad(m) for m in x.shape[:1:-1])
        )
        output = F.pad(x, pads)
        return output


In [ ]:
%%writefile sih3d/vendor/dinov2/layers/__init__.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

from sih3d.vendor.dinov2.layers.dino_head import DINOHead  # noqa
from sih3d.vendor.dinov2.layers.mlp import Mlp  # noqa
from sih3d.vendor.dinov2.layers.patch_embed import PatchEmbed  # noqa
from sih3d.vendor.dinov2.layers.swiglu_ffn import (
    SwiGLUFFN,  # noqa
    SwiGLUFFNFused,  # noqa
)
from sih3d.vendor.dinov2.layers.block import NestedTensorBlock  # noqa
from sih3d.vendor.dinov2.layers.attention import MemEffAttention  # noqa


In [ ]:
%%writefile sih3d/vendor/dinov2/layers/attention.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

# References:
#   https://github.com/facebookresearch/dino/blob/master/vision_transformer.py
#   https://github.com/rwightman/pytorch-image-models/tree/master/timm/models/vision_transformer.py

import logging
import os

from torch import nn, Tensor

logger = logging.getLogger("dinov2")


XFORMERS_ENABLED = os.environ.get("XFORMERS_DISABLED") is None
try:
    if XFORMERS_ENABLED:
        from xformers.ops import memory_efficient_attention, unbind

        XFORMERS_AVAILABLE = True
        # warnings.warn("xFormers is available (Attention)")
    else:
        # warnings.warn("xFormers is disabled (Attention)")
        raise ImportError
except ImportError:
    XFORMERS_AVAILABLE = False
    # warnings.warn("xFormers is not available (Attention)")


class Attention(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int = 8,
        qkv_bias: bool = False,
        proj_bias: bool = True,
        attn_drop: float = 0.0,
        proj_drop: float = 0.0,
    ) -> None:
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim**-0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim, bias=proj_bias)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x: Tensor, attn_bias=None) -> Tensor:
        B, N, C = x.shape
        qkv = (
            self.qkv(x)
            .reshape(B, N, 3, self.num_heads, C // self.num_heads)
            .permute(2, 0, 3, 1, 4)
        )

        q, k, v = qkv[0] * self.scale, qkv[1], qkv[2]
        attn = q @ k.transpose(-2, -1)

        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class MemEffAttention(Attention):
    def forward(self, x: Tensor, attn_bias=None) -> Tensor:
        if not XFORMERS_AVAILABLE:
            if attn_bias is not None:
                raise AssertionError("xFormers is required for using nested tensors")
            return super().forward(x)

        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads)

        q, k, v = unbind(qkv, 2)

        x = memory_efficient_attention(q, k, v, attn_bias=attn_bias)
        x = x.reshape([B, N, C])

        x = self.proj(x)
        x = self.proj_drop(x)
        return x


In [ ]:
%%writefile sih3d/vendor/dinov2/layers/block.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

# References:
#   https://github.com/facebookresearch/dino/blob/master/vision_transformer.py
#   https://github.com/rwightman/pytorch-image-models/tree/master/timm/layers/patch_embed.py

import logging
import os
from typing import Any, Callable, Dict, List, Tuple

import torch
from torch import nn, Tensor

from sih3d.vendor.dinov2.layers.attention import (
    Attention,
    MemEffAttention,
)
from sih3d.vendor.dinov2.layers.drop_path import DropPath
from sih3d.vendor.dinov2.layers.layer_scale import LayerScale
from sih3d.vendor.dinov2.layers.mlp import Mlp

logger = logging.getLogger("dinov2")


XFORMERS_ENABLED = os.environ.get("XFORMERS_DISABLED") is None
try:
    if XFORMERS_ENABLED:
        from xformers.ops import fmha, index_select_cat, scaled_index_add

        XFORMERS_AVAILABLE = True
        # warnings.warn("xFormers is available (Block)")
    else:
        # warnings.warn("xFormers is disabled (Block)")
        raise ImportError
except ImportError:
    XFORMERS_AVAILABLE = False
    # warnings.warn("xFormers is not available (Block)")


class Block(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = False,
        proj_bias: bool = True,
        ffn_bias: bool = True,
        drop: float = 0.0,
        attn_drop: float = 0.0,
        init_values=None,
        drop_path: float = 0.0,
        act_layer: Callable[..., nn.Module] = nn.GELU,
        norm_layer: Callable[..., nn.Module] = nn.LayerNorm,
        attn_class: Callable[..., nn.Module] = Attention,
        ffn_layer: Callable[..., nn.Module] = Mlp,
    ) -> None:
        super().__init__()
        # print(f"biases: qkv: {qkv_bias}, proj: {proj_bias}, ffn: {ffn_bias}")
        self.norm1 = norm_layer(dim)
        self.attn = attn_class(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            proj_bias=proj_bias,
            attn_drop=attn_drop,
            proj_drop=drop,
        )
        self.ls1 = (
            LayerScale(dim, init_values=init_values) if init_values else nn.Identity()
        )
        self.drop_path1 = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = ffn_layer(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            act_layer=act_layer,
            drop=drop,
            bias=ffn_bias,
        )
        self.ls2 = (
            LayerScale(dim, init_values=init_values) if init_values else nn.Identity()
        )
        self.drop_path2 = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

        self.sample_drop_ratio = drop_path

    def forward(self, x: Tensor) -> Tensor:
        def attn_residual_func(x: Tensor) -> Tensor:
            return self.ls1(self.attn(self.norm1(x)))

        def ffn_residual_func(x: Tensor) -> Tensor:
            return self.ls2(self.mlp(self.norm2(x)))

        if self.training and self.sample_drop_ratio > 0.1:
            # the overhead is compensated only for a drop path rate larger than 0.1
            x = drop_add_residual_stochastic_depth(
                x,
                residual_func=attn_residual_func,
                sample_drop_ratio=self.sample_drop_ratio,
            )
            x = drop_add_residual_stochastic_depth(
                x,
                residual_func=ffn_residual_func,
                sample_drop_ratio=self.sample_drop_ratio,
            )
        elif self.training and self.sample_drop_ratio > 0.0:
            x = x + self.drop_path1(attn_residual_func(x))
            x = x + self.drop_path1(ffn_residual_func(x))  # FIXME: drop_path2
        else:
            x = x + attn_residual_func(x)
            x = x + ffn_residual_func(x)
        return x


def drop_add_residual_stochastic_depth(
    x: Tensor,
    residual_func: Callable[[Tensor], Tensor],
    sample_drop_ratio: float = 0.0,
) -> Tensor:
    # 1) extract subset using permutation
    b, n, d = x.shape
    sample_subset_size = max(int(b * (1 - sample_drop_ratio)), 1)
    brange = (torch.randperm(b, device=x.device))[:sample_subset_size]
    x_subset = x[brange]

    # 2) apply residual_func to get residual
    residual = residual_func(x_subset)

    x_flat = x.flatten(1)
    residual = residual.flatten(1)

    residual_scale_factor = b / sample_subset_size

    # 3) add the residual
    x_plus_residual = torch.index_add(
        x_flat, 0, brange, residual.to(dtype=x.dtype), alpha=residual_scale_factor
    )
    return x_plus_residual.view_as(x)


def get_branges_scales(x, sample_drop_ratio=0.0):
    b, n, d = x.shape
    sample_subset_size = max(int(b * (1 - sample_drop_ratio)), 1)
    brange = (torch.randperm(b, device=x.device))[:sample_subset_size]
    residual_scale_factor = b / sample_subset_size
    return brange, residual_scale_factor


def add_residual(x, brange, residual, residual_scale_factor, scaling_vector=None):
    if scaling_vector is None:
        x_flat = x.flatten(1)
        residual = residual.flatten(1)
        x_plus_residual = torch.index_add(
            x_flat, 0, brange, residual.to(dtype=x.dtype), alpha=residual_scale_factor
        )
    else:
        x_plus_residual = scaled_index_add(
            x,
            brange,
            residual.to(dtype=x.dtype),
            scaling=scaling_vector,
            alpha=residual_scale_factor,
        )
    return x_plus_residual


attn_bias_cache: Dict[Tuple, Any] = {}


def get_attn_bias_and_cat(x_list, branges=None):
    """
    this will perform the index select, cat the tensors, and provide the attn_bias from cache
    """
    batch_sizes = (
        [b.shape[0] for b in branges]
        if branges is not None
        else [x.shape[0] for x in x_list]
    )
    all_shapes = tuple((b, x.shape[1]) for b, x in zip(batch_sizes, x_list))
    if all_shapes not in attn_bias_cache.keys():
        seqlens = []
        for b, x in zip(batch_sizes, x_list):
            for _ in range(b):
                seqlens.append(x.shape[1])
        attn_bias = fmha.BlockDiagonalMask.from_seqlens(seqlens)
        attn_bias._batch_sizes = batch_sizes
        attn_bias_cache[all_shapes] = attn_bias

    if branges is not None:
        cat_tensors = index_select_cat([x.flatten(1) for x in x_list], branges).view(
            1, -1, x_list[0].shape[-1]
        )
    else:
        tensors_bs1 = tuple(x.reshape([1, -1, *x.shape[2:]]) for x in x_list)
        cat_tensors = torch.cat(tensors_bs1, dim=1)

    return attn_bias_cache[all_shapes], cat_tensors


def drop_add_residual_stochastic_depth_list(
    x_list: List[Tensor],
    residual_func: Callable[[Tensor, Any], Tensor],
    sample_drop_ratio: float = 0.0,
    scaling_vector=None,
) -> Tensor:
    # 1) generate random set of indices for dropping samples in the batch
    branges_scales = [
        get_branges_scales(x, sample_drop_ratio=sample_drop_ratio) for x in x_list
    ]
    branges = [s[0] for s in branges_scales]
    residual_scale_factors = [s[1] for s in branges_scales]

    # 2) get attention bias and index+concat the tensors
    attn_bias, x_cat = get_attn_bias_and_cat(x_list, branges)

    # 3) apply residual_func to get residual, and split the result
    residual_list = attn_bias.split(residual_func(x_cat, attn_bias=attn_bias))  # type: ignore

    outputs = []
    for x, brange, residual, residual_scale_factor in zip(
        x_list, branges, residual_list, residual_scale_factors
    ):
        outputs.append(
            add_residual(
                x, brange, residual, residual_scale_factor, scaling_vector
            ).view_as(x)
        )
    return outputs


class NestedTensorBlock(Block):
    def forward_nested(self, x_list: List[Tensor]) -> List[Tensor]:
        """
        x_list contains a list of tensors to nest together and run
        """
        assert isinstance(self.attn, MemEffAttention)

        if self.training and self.sample_drop_ratio > 0.0:

            def attn_residual_func(x: Tensor, attn_bias=None) -> Tensor:
                return self.attn(self.norm1(x), attn_bias=attn_bias)

            def ffn_residual_func(x: Tensor, attn_bias=None) -> Tensor:
                return self.mlp(self.norm2(x))

            x_list = drop_add_residual_stochastic_depth_list(
                x_list,
                residual_func=attn_residual_func,
                sample_drop_ratio=self.sample_drop_ratio,
                scaling_vector=self.ls1.gamma
                if isinstance(self.ls1, LayerScale)
                else None,
            )
            x_list = drop_add_residual_stochastic_depth_list(
                x_list,
                residual_func=ffn_residual_func,
                sample_drop_ratio=self.sample_drop_ratio,
                scaling_vector=self.ls2.gamma
                if isinstance(self.ls1, LayerScale)
                else None,
            )
            return x_list
        else:

            def attn_residual_func(x: Tensor, attn_bias=None) -> Tensor:
                return self.ls1(self.attn(self.norm1(x), attn_bias=attn_bias))

            def ffn_residual_func(x: Tensor, attn_bias=None) -> Tensor:
                return self.ls2(self.mlp(self.norm2(x)))

            attn_bias, x = get_attn_bias_and_cat(x_list)
            x = x + attn_residual_func(x, attn_bias=attn_bias)
            x = x + ffn_residual_func(x)
            return attn_bias.split(x)

    def forward(self, x_or_x_list):
        if isinstance(x_or_x_list, Tensor):
            return super().forward(x_or_x_list)
        elif isinstance(x_or_x_list, list):
            if not XFORMERS_AVAILABLE:
                raise AssertionError("xFormers is required for using nested tensors")
            return self.forward_nested(x_or_x_list)
        else:
            raise AssertionError


In [ ]:
%%writefile sih3d/vendor/dinov2/layers/dino_head.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

import torch
import torch.nn as nn
from torch.nn.init import trunc_normal_
from torch.nn.utils import weight_norm


class DINOHead(nn.Module):
    def __init__(
        self,
        in_dim,
        out_dim,
        use_bn=False,
        nlayers=3,
        hidden_dim=2048,
        bottleneck_dim=256,
        mlp_bias=True,
    ):
        super().__init__()
        nlayers = max(nlayers, 1)
        self.mlp = _build_mlp(
            nlayers,
            in_dim,
            bottleneck_dim,
            hidden_dim=hidden_dim,
            use_bn=use_bn,
            bias=mlp_bias,
        )
        self.apply(self._init_weights)
        self.last_layer = weight_norm(nn.Linear(bottleneck_dim, out_dim, bias=False))
        self.last_layer.weight_g.data.fill_(1)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.mlp(x)
        eps = 1e-6 if x.dtype == torch.float16 else 1e-12
        x = nn.functional.normalize(x, dim=-1, p=2, eps=eps)
        x = self.last_layer(x)
        return x


def _build_mlp(
    nlayers, in_dim, bottleneck_dim, hidden_dim=None, use_bn=False, bias=True
):
    if nlayers == 1:
        return nn.Linear(in_dim, bottleneck_dim, bias=bias)
    else:
        layers = [nn.Linear(in_dim, hidden_dim, bias=bias)]
        if use_bn:
            layers.append(nn.BatchNorm1d(hidden_dim))
        layers.append(nn.GELU())
        for _ in range(nlayers - 2):
            layers.append(nn.Linear(hidden_dim, hidden_dim, bias=bias))
            if use_bn:
                layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.GELU())
        layers.append(nn.Linear(hidden_dim, bottleneck_dim, bias=bias))
        return nn.Sequential(*layers)


In [ ]:
%%writefile sih3d/vendor/dinov2/layers/drop_path.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

# References:
#   https://github.com/facebookresearch/dino/blob/master/vision_transformer.py
#   https://github.com/rwightman/pytorch-image-models/tree/master/timm/layers/drop.py


from torch import nn


def drop_path(x, drop_prob: float = 0.0, training: bool = False):
    if drop_prob == 0.0 or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (
        x.ndim - 1
    )  # work with diff dim tensors, not just 2D ConvNets
    random_tensor = x.new_empty(shape).bernoulli_(keep_prob)
    if keep_prob > 0.0:
        random_tensor.div_(keep_prob)
    output = x * random_tensor
    return output


class DropPath(nn.Module):
    """Drop paths (Stochastic Depth) per sample (when applied in main path of residual blocks)."""

    def __init__(self, drop_prob=None):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)


In [ ]:
%%writefile sih3d/vendor/dinov2/layers/layer_scale.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

# Modified from: https://github.com/huggingface/pytorch-image-models/blob/main/timm/models/vision_transformer.py#L103-L110

from typing import Union

import torch
from torch import nn, Tensor


class LayerScale(nn.Module):
    def __init__(
        self,
        dim: int,
        init_values: Union[float, Tensor] = 1e-5,
        inplace: bool = False,
    ) -> None:
        super().__init__()
        self.inplace = inplace
        self.gamma = nn.Parameter(init_values * torch.ones(dim))

    def forward(self, x: Tensor) -> Tensor:
        return x.mul_(self.gamma) if self.inplace else x * self.gamma


In [ ]:
%%writefile sih3d/vendor/dinov2/layers/mlp.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

# References:
#   https://github.com/facebookresearch/dino/blob/master/vision_transformer.py
#   https://github.com/rwightman/pytorch-image-models/tree/master/timm/layers/mlp.py


from typing import Callable, Optional

from torch import nn, Tensor


class Mlp(nn.Module):
    def __init__(
        self,
        in_features: int,
        hidden_features: Optional[int] = None,
        out_features: Optional[int] = None,
        act_layer: Callable[..., nn.Module] = nn.GELU,
        drop: float = 0.0,
        bias: bool = True,
    ) -> None:
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features, bias=bias)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features, bias=bias)
        self.drop = nn.Dropout(drop)

    def forward(self, x: Tensor) -> Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


In [ ]:
%%writefile sih3d/vendor/dinov2/layers/patch_embed.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

# References:
#   https://github.com/facebookresearch/dino/blob/master/vision_transformer.py
#   https://github.com/rwightman/pytorch-image-models/tree/master/timm/layers/patch_embed.py

from typing import Callable, Optional, Tuple, Union

import torch.nn as nn
from torch import Tensor


def make_2tuple(x):
    if isinstance(x, tuple):
        assert len(x) == 2
        return x

    assert isinstance(x, int)
    return (x, x)


class PatchEmbed(nn.Module):
    """
    2D image to patch embedding: (B,C,H,W) -> (B,N,D)

    Args:
        img_size: Image size.
        patch_size: Patch token size.
        in_chans: Number of input image channels.
        embed_dim: Number of linear projection output channels.
        norm_layer: Normalization layer.
    """

    def __init__(
        self,
        img_size: Union[int, Tuple[int, int]] = 224,
        patch_size: Union[int, Tuple[int, int]] = 16,
        in_chans: int = 3,
        embed_dim: int = 768,
        norm_layer: Optional[Callable] = None,
        flatten_embedding: bool = True,
    ) -> None:
        super().__init__()

        image_HW = make_2tuple(img_size)
        patch_HW = make_2tuple(patch_size)
        patch_grid_size = (
            image_HW[0] // patch_HW[0],
            image_HW[1] // patch_HW[1],
        )

        self.img_size = image_HW
        self.patch_size = patch_HW
        self.patches_resolution = patch_grid_size
        self.num_patches = patch_grid_size[0] * patch_grid_size[1]

        self.in_chans = in_chans
        self.embed_dim = embed_dim

        self.flatten_embedding = flatten_embedding

        self.proj = nn.Conv2d(
            in_chans, embed_dim, kernel_size=patch_HW, stride=patch_HW
        )
        self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()

    def forward(self, x: Tensor) -> Tensor:
        _, _, H, W = x.shape
        patch_H, patch_W = self.patch_size

        assert H % patch_H == 0, (
            f"Input image height {H} is not a multiple of patch height {patch_H}"
        )
        assert W % patch_W == 0, (
            f"Input image width {W} is not a multiple of patch width: {patch_W}"
        )

        x = self.proj(x)  # B C H W
        H, W = x.size(2), x.size(3)
        x = x.flatten(2).transpose(1, 2)  # B HW C
        x = self.norm(x)
        if not self.flatten_embedding:
            x = x.reshape(-1, H, W, self.embed_dim)  # B H W C
        return x

    def flops(self) -> float:
        Ho, Wo = self.patches_resolution
        flops = (
            Ho
            * Wo
            * self.embed_dim
            * self.in_chans
            * (self.patch_size[0] * self.patch_size[1])
        )
        if self.norm is not None:
            flops += Ho * Wo * self.embed_dim
        return flops


In [ ]:
%%writefile sih3d/vendor/dinov2/layers/swiglu_ffn.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

import os
from typing import Callable, Optional

import torch.nn.functional as F
from torch import nn, Tensor


class SwiGLUFFN(nn.Module):
    def __init__(
        self,
        in_features: int,
        hidden_features: Optional[int] = None,
        out_features: Optional[int] = None,
        act_layer: Callable[..., nn.Module] = None,
        drop: float = 0.0,
        bias: bool = True,
    ) -> None:
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.w12 = nn.Linear(in_features, 2 * hidden_features, bias=bias)
        self.w3 = nn.Linear(hidden_features, out_features, bias=bias)

    def forward(self, x: Tensor) -> Tensor:
        x12 = self.w12(x)
        x1, x2 = x12.chunk(2, dim=-1)
        hidden = F.silu(x1) * x2
        return self.w3(hidden)


XFORMERS_ENABLED = os.environ.get("XFORMERS_DISABLED") is None
try:
    if XFORMERS_ENABLED:
        from xformers.ops import SwiGLU

        XFORMERS_AVAILABLE = True
        # warnings.warn("xFormers is available (SwiGLU)")
    else:
        # warnings.warn("xFormers is disabled (SwiGLU)")
        raise ImportError
except ImportError:
    SwiGLU = SwiGLUFFN
    XFORMERS_AVAILABLE = False

    # warnings.warn("xFormers is not available (SwiGLU)")


class SwiGLUFFNFused(SwiGLU):
    def __init__(
        self,
        in_features: int,
        hidden_features: Optional[int] = None,
        out_features: Optional[int] = None,
        act_layer: Callable[..., nn.Module] = None,
        drop: float = 0.0,
        bias: bool = True,
    ) -> None:
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        hidden_features = (int(hidden_features * 2 / 3) + 7) // 8 * 8
        super().__init__(
            in_features=in_features,
            hidden_features=hidden_features,
            out_features=out_features,
            bias=bias,
        )


In [ ]:
%%writefile sih3d/vendor/dinov2/models/__init__.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

import logging

import sih3d.vendor.dinov2.models.vision_transformer as vits

logger = logging.getLogger("dinov2")


def build_model(args, only_teacher=False, img_size=224):
    args.arch = args.arch.removesuffix("_memeff")
    if "vit" in args.arch:
        vit_kwargs = dict(
            img_size=img_size,
            patch_size=args.patch_size,
            init_values=args.layerscale,
            ffn_layer=args.ffn_layer,
            block_chunks=args.block_chunks,
            qkv_bias=args.qkv_bias,
            proj_bias=args.proj_bias,
            ffn_bias=args.ffn_bias,
            num_register_tokens=args.num_register_tokens,
            interpolate_offset=args.interpolate_offset,
            interpolate_antialias=args.interpolate_antialias,
        )
        teacher = vits.__dict__[args.arch](**vit_kwargs)
        if only_teacher:
            return teacher, teacher.embed_dim
        student = vits.__dict__[args.arch](
            **vit_kwargs,
            drop_path_rate=args.drop_path_rate,
            drop_path_uniform=args.drop_path_uniform,
        )
        embed_dim = student.embed_dim
    return student, teacher, embed_dim


def build_model_from_cfg(cfg, only_teacher=False):
    return build_model(
        cfg.student, only_teacher=only_teacher, img_size=cfg.crops.global_crops_size
    )


In [ ]:
%%writefile sih3d/vendor/dinov2/models/vision_transformer.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

# References:
#   https://github.com/facebookresearch/dino/blob/main/vision_transformer.py
#   https://github.com/rwightman/pytorch-image-models/tree/master/timm/models/vision_transformer.py

import math
from functools import partial
from typing import Callable, Sequence, Tuple, Union

import torch
import torch.nn as nn
from torch.nn.init import trunc_normal_
from torch.utils.checkpoint import checkpoint

from sih3d.vendor.dinov2.layers import (
    MemEffAttention,
    Mlp,
    NestedTensorBlock as Block,
    PatchEmbed,
    SwiGLUFFNFused,
)
from sih3d.vendor.pi3.layers.attention import FlashAttention

# logger = logging.getLogger("dinov2")


def named_apply(
    fn: Callable, module: nn.Module, name="", depth_first=True, include_root=False
) -> nn.Module:
    if not depth_first and include_root:
        fn(module=module, name=name)
    for child_name, child_module in module.named_children():
        child_name = ".".join((name, child_name)) if name else child_name
        named_apply(
            fn=fn,
            module=child_module,
            name=child_name,
            depth_first=depth_first,
            include_root=True,
        )
    if depth_first and include_root:
        fn(module=module, name=name)
    return module


class BlockChunk(nn.ModuleList):
    def forward(self, x):
        for b in self:
            x = b(x)
        return x


class DinoVisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_chans=3,
        embed_dim=768,
        depth=12,
        num_heads=12,
        mlp_ratio=4.0,
        qkv_bias=True,
        ffn_bias=True,
        proj_bias=True,
        drop_path_rate=0.0,
        drop_path_uniform=False,
        init_values=None,  # for layerscale: None or 0 => no layerscale
        embed_layer=PatchEmbed,
        act_layer=nn.GELU,
        block_fn=Block,
        ffn_layer="mlp",
        block_chunks=1,
        num_register_tokens=0,
        interpolate_antialias=False,
        interpolate_offset=0.1,
    ):
        """
        Args:
            img_size (int, tuple): input image size
            patch_size (int, tuple): patch size
            in_chans (int): number of input channels
            embed_dim (int): embedding dimension
            depth (int): depth of transformer
            num_heads (int): number of attention heads
            mlp_ratio (int): ratio of mlp hidden dim to embedding dim
            qkv_bias (bool): enable bias for qkv if True
            proj_bias (bool): enable bias for proj in attn if True
            ffn_bias (bool): enable bias for ffn if True
            drop_path_rate (float): stochastic depth rate
            drop_path_uniform (bool): apply uniform drop rate across blocks
            weight_init (str): weight init scheme
            init_values (float): layer-scale init values
            embed_layer (nn.Module): patch embedding layer
            act_layer (nn.Module): MLP activation layer
            block_fn (nn.Module): transformer block class
            ffn_layer (str): "mlp", "swiglu", "swiglufused" or "identity"
            block_chunks: (int) split block sequence into block_chunks units for FSDP wrap
            num_register_tokens: (int) number of extra cls tokens (so-called "registers")
            interpolate_antialias: (str) flag to apply anti-aliasing when interpolating positional embeddings
            interpolate_offset: (float) work-around offset to apply when interpolating positional embeddings
        """
        super().__init__()
        norm_layer = partial(nn.LayerNorm, eps=1e-6)

        self.num_features = self.embed_dim = (
            embed_dim  # num_features for consistency with other models
        )
        self.num_tokens = 1
        self.n_blocks = depth
        self.num_heads = num_heads
        self.patch_size = patch_size
        self.num_register_tokens = num_register_tokens
        self.interpolate_antialias = interpolate_antialias
        self.interpolate_offset = interpolate_offset

        self.patch_embed = embed_layer(
            img_size=img_size,
            patch_size=patch_size,
            in_chans=in_chans,
            embed_dim=embed_dim,
        )
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(
            torch.zeros(1, num_patches + self.num_tokens, embed_dim)
        )
        assert num_register_tokens >= 0
        self.register_tokens = (
            nn.Parameter(torch.zeros(1, num_register_tokens, embed_dim))
            if num_register_tokens
            else None
        )

        if drop_path_uniform is True:
            dpr = [drop_path_rate] * depth
        else:
            dpr = [
                x.item() for x in torch.linspace(0, drop_path_rate, depth)
            ]  # stochastic depth decay rule

        if ffn_layer == "mlp":
            # logger.info("using MLP layer as FFN")
            ffn_layer = Mlp
        elif ffn_layer == "swiglufused" or ffn_layer == "swiglu":
            # logger.info("using SwiGLU layer as FFN")
            ffn_layer = SwiGLUFFNFused
        elif ffn_layer == "identity":
            # logger.info("using Identity layer as FFN")

            def f(*args, **kwargs):
                return nn.Identity()

            ffn_layer = f
        else:
            raise NotImplementedError

        blocks_list = [
            block_fn(
                dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                proj_bias=proj_bias,
                ffn_bias=ffn_bias,
                drop_path=dpr[i],
                norm_layer=norm_layer,
                act_layer=act_layer,
                ffn_layer=ffn_layer,
                init_values=init_values,
                attn_class=FlashAttention,
            )
            for i in range(depth)
        ]
        if block_chunks > 0:
            self.chunked_blocks = True
            chunked_blocks = []
            chunksize = depth // block_chunks
            for i in range(0, depth, chunksize):
                # this is to keep the block index consistent if we chunk the block list
                chunked_blocks.append(
                    [nn.Identity()] * i + blocks_list[i : i + chunksize]
                )
            self.blocks = nn.ModuleList([BlockChunk(p) for p in chunked_blocks])
        else:
            self.chunked_blocks = False
            self.blocks = nn.ModuleList(blocks_list)

        self.norm = norm_layer(embed_dim)
        self.head = nn.Identity()

        self.mask_token = nn.Parameter(torch.zeros(1, embed_dim))

        self.init_weights()

    def init_weights(self):
        trunc_normal_(self.pos_embed, std=0.02)
        nn.init.normal_(self.cls_token, std=1e-6)
        if self.register_tokens is not None:
            nn.init.normal_(self.register_tokens, std=1e-6)
        named_apply(init_weights_vit_timm, self)

    def interpolate_pos_encoding(self, x, w, h):
        previous_dtype = x.dtype
        npatch = x.shape[1] - 1
        N = self.pos_embed.shape[1] - 1
        if npatch == N and w == h:
            return self.pos_embed
        pos_embed = self.pos_embed.float()
        class_pos_embed = pos_embed[:, 0]
        patch_pos_embed = pos_embed[:, 1:]
        dim = x.shape[-1]
        w0 = w // self.patch_size
        h0 = h // self.patch_size
        M = int(math.sqrt(N))  # Recover the number of patches in each dimension
        assert N == M * M
        kwargs = {}
        if self.interpolate_offset:
            # Historical kludge: add a small number to avoid floating point error in the interpolation, see https://github.com/facebookresearch/dino/issues/8
            # Note: still needed for backward-compatibility, the underlying operators are using both output size and scale factors
            sx = float(w0 + self.interpolate_offset) / M
            sy = float(h0 + self.interpolate_offset) / M
            kwargs["scale_factor"] = (sx, sy)
        else:
            # Simply specify an output size instead of a scale factor
            kwargs["size"] = (w0, h0)
        patch_pos_embed = nn.functional.interpolate(
            patch_pos_embed.reshape(1, M, M, dim).permute(0, 3, 1, 2),
            mode="bicubic",
            antialias=self.interpolate_antialias,
            **kwargs,
        )
        assert (w0, h0) == patch_pos_embed.shape[-2:]
        patch_pos_embed = patch_pos_embed.permute(0, 2, 3, 1).view(1, -1, dim)
        return torch.cat((class_pos_embed.unsqueeze(0), patch_pos_embed), dim=1).to(
            previous_dtype
        )

    def prepare_tokens_with_masks(self, x, masks=None):
        B, nc, w, h = x.shape
        x = self.patch_embed(x)
        if masks is not None:
            x = torch.where(
                masks.unsqueeze(-1), self.mask_token.to(x.dtype).unsqueeze(0), x
            )

        x = torch.cat((self.cls_token.expand(x.shape[0], -1, -1), x), dim=1)
        x = x + self.interpolate_pos_encoding(x, w, h)

        if self.register_tokens is not None:
            x = torch.cat(
                (
                    x[:, :1],
                    self.register_tokens.expand(x.shape[0], -1, -1),
                    x[:, 1:],
                ),
                dim=1,
            )

        return x

    def forward_features_list(self, x_list, masks_list):
        x = [
            self.prepare_tokens_with_masks(x, masks)
            for x, masks in zip(x_list, masks_list)
        ]
        for blk in self.blocks:
            if self.training:
                x = checkpoint(blk, x, use_reentrant=False)
            else:
                x = blk(x)

        all_x = x
        output = []
        for x, masks in zip(all_x, masks_list):
            x_norm = self.norm(x)
            output.append(
                {
                    "x_norm_clstoken": x_norm[:, 0],
                    "x_norm_regtokens": x_norm[:, 1 : self.num_register_tokens + 1],
                    "x_norm_patchtokens": x_norm[:, self.num_register_tokens + 1 :],
                    "x_prenorm": x,
                    "masks": masks,
                }
            )
        return output

    def forward_features(self, x, masks=None):
        if isinstance(x, list):
            return self.forward_features_list(x, masks)

        x = self.prepare_tokens_with_masks(x, masks)

        for blk in self.blocks:
            if self.training:
                x = checkpoint(blk, x, use_reentrant=False)
            else:
                x = blk(x)

        x_norm = self.norm(x)
        return {
            "x_norm_clstoken": x_norm[:, 0],
            "x_norm_regtokens": x_norm[:, 1 : self.num_register_tokens + 1],
            "x_norm_patchtokens": x_norm[:, self.num_register_tokens + 1 :],
            "x_prenorm": x,
            "masks": masks,
        }

    def _get_intermediate_layers_not_chunked(self, x, n=1):
        x = self.prepare_tokens_with_masks(x)
        # If n is an int, take the n last blocks. If it's a list, take them
        output, total_block_len = [], len(self.blocks)
        blocks_to_take = (
            range(total_block_len - n, total_block_len) if isinstance(n, int) else n
        )
        for i, blk in enumerate(self.blocks):
            x = blk(x)
            if i in blocks_to_take:
                output.append(x)
        assert len(output) == len(blocks_to_take), (
            f"only {len(output)} / {len(blocks_to_take)} blocks found"
        )
        return output

    def _get_intermediate_layers_chunked(self, x, n=1):
        x = self.prepare_tokens_with_masks(x)
        output, i, total_block_len = [], 0, len(self.blocks[-1])
        # If n is an int, take the n last blocks. If it's a list, take them
        blocks_to_take = (
            range(total_block_len - n, total_block_len) if isinstance(n, int) else n
        )
        for block_chunk in self.blocks:
            for blk in block_chunk[i:]:  # Passing the nn.Identity()
                x = blk(x)
                if i in blocks_to_take:
                    output.append(x)
                i += 1
        assert len(output) == len(blocks_to_take), (
            f"only {len(output)} / {len(blocks_to_take)} blocks found"
        )
        return output

    def get_intermediate_layers(
        self,
        x: torch.Tensor,
        n: Union[int, Sequence] = 1,  # Layers or n last layers to take
        reshape: bool = False,
        return_class_token: bool = False,
        norm=True,
    ) -> Tuple[Union[torch.Tensor, Tuple[torch.Tensor]]]:
        if self.chunked_blocks:
            outputs = self._get_intermediate_layers_chunked(x, n)
        else:
            outputs = self._get_intermediate_layers_not_chunked(x, n)
        if norm:
            outputs = [self.norm(out) for out in outputs]
        class_tokens = [out[:, 0] for out in outputs]
        outputs = [out[:, 1 + self.num_register_tokens :] for out in outputs]
        if reshape:
            B, _, w, h = x.shape
            outputs = [
                out.reshape(B, w // self.patch_size, h // self.patch_size, -1)
                .permute(0, 3, 1, 2)
                .contiguous()
                for out in outputs
            ]
        if return_class_token:
            return tuple(zip(outputs, class_tokens))
        return tuple(outputs)

    def forward(self, *args, is_training=False, **kwargs):
        ret = self.forward_features(*args, **kwargs)
        if is_training:
            return ret
        else:
            return self.head(ret["x_norm_clstoken"])


def init_weights_vit_timm(module: nn.Module, name: str = ""):
    """ViT weight initialization, original timm impl (for reproducibility)"""
    if isinstance(module, nn.Linear):
        trunc_normal_(module.weight, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)


def vit_small(patch_size=16, num_register_tokens=0, **kwargs):
    model = DinoVisionTransformer(
        patch_size=patch_size,
        embed_dim=384,
        depth=12,
        num_heads=6,
        mlp_ratio=4,
        block_fn=partial(Block, attn_class=MemEffAttention),
        num_register_tokens=num_register_tokens,
        **kwargs,
    )
    return model


def vit_base(patch_size=16, num_register_tokens=0, **kwargs):
    model = DinoVisionTransformer(
        patch_size=patch_size,
        embed_dim=768,
        depth=12,
        num_heads=12,
        mlp_ratio=4,
        block_fn=partial(Block, attn_class=MemEffAttention),
        num_register_tokens=num_register_tokens,
        **kwargs,
    )
    return model


def vit_large(patch_size=16, num_register_tokens=0, **kwargs):
    model = DinoVisionTransformer(
        patch_size=patch_size,
        embed_dim=1024,
        depth=24,
        num_heads=16,
        mlp_ratio=4,
        block_fn=partial(Block, attn_class=MemEffAttention),
        num_register_tokens=num_register_tokens,
        **kwargs,
    )
    return model


def vit_giant2(patch_size=16, num_register_tokens=0, **kwargs):
    """
    Close to ViT-giant, with embed-dim 1536 and 24 heads => embed-dim per head 64
    """
    model = DinoVisionTransformer(
        patch_size=patch_size,
        embed_dim=1536,
        depth=40,
        num_heads=24,
        mlp_ratio=4,
        block_fn=partial(Block, attn_class=MemEffAttention),
        num_register_tokens=num_register_tokens,
        **kwargs,
    )
    return model


In [ ]:
%%writefile sih3d/vendor/dinov2/utils/__init__.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.


In [ ]:
%%writefile sih3d/vendor/dinov2/utils/dtype.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.


from typing import Dict, Union

import numpy as np
import torch

TypeSpec = Union[str, np.dtype, torch.dtype]


_NUMPY_TO_TORCH_DTYPE: Dict[np.dtype, torch.dtype] = {
    np.dtype("bool"): torch.bool,
    np.dtype("uint8"): torch.uint8,
    np.dtype("int8"): torch.int8,
    np.dtype("int16"): torch.int16,
    np.dtype("int32"): torch.int32,
    np.dtype("int64"): torch.int64,
    np.dtype("float16"): torch.float16,
    np.dtype("float32"): torch.float32,
    np.dtype("float64"): torch.float64,
    np.dtype("complex64"): torch.complex64,
    np.dtype("complex128"): torch.complex128,
}


def as_torch_dtype(dtype: TypeSpec) -> torch.dtype:
    if isinstance(dtype, torch.dtype):
        return dtype
    if isinstance(dtype, str):
        dtype = np.dtype(dtype)
    assert isinstance(dtype, np.dtype), (
        f"Expected an instance of nunpy dtype, got {type(dtype)}"
    )
    return _NUMPY_TO_TORCH_DTYPE[dtype]


In [ ]:
%%writefile sih3d/vendor/dinov2/utils/utils.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

import os
import random
import subprocess
from urllib.parse import urlparse

import numpy as np
import torch
from torch import nn

# logger = logging.getLogger("dinov2")


def load_pretrained_weights(model, pretrained_weights, checkpoint_key):
    if urlparse(pretrained_weights).scheme:  # If it looks like an URL
        state_dict = torch.hub.load_state_dict_from_url(
            pretrained_weights, map_location="cpu"
        )
    else:
        state_dict = torch.load(pretrained_weights, map_location="cpu")
    if checkpoint_key is not None and checkpoint_key in state_dict:
        # logger.info(f"Take key {checkpoint_key} in provided checkpoint dict")
        state_dict = state_dict[checkpoint_key]
    # remove `module.` prefix
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    # remove `backbone.` prefix induced by multicrop wrapper
    state_dict = {k.replace("backbone.", ""): v for k, v in state_dict.items()}
    _ = model.load_state_dict(state_dict, strict=False)
    # logger.info("Pretrained weights found at {} and loaded with msg: {}".format(pretrained_weights, msg))


def fix_random_seeds(seed=31):
    """
    Fix random seeds.
    """
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)


def get_sha():
    cwd = os.path.dirname(os.path.abspath(__file__))

    def _run(command):
        return subprocess.check_output(command, cwd=cwd).decode("ascii").strip()

    sha = "N/A"
    diff = "clean"
    branch = "N/A"
    try:
        sha = _run(["git", "rev-parse", "HEAD"])
        subprocess.check_output(["git", "diff"], cwd=cwd)
        diff = _run(["git", "diff-index", "HEAD"])
        diff = "has uncommitted changes" if diff else "clean"
        branch = _run(["git", "rev-parse", "--abbrev-ref", "HEAD"])
    except Exception:
        pass
    message = f"sha: {sha}, status: {diff}, branch: {branch}"
    return message


class CosineScheduler(object):
    def __init__(
        self,
        base_value,
        final_value,
        total_iters,
        warmup_iters=0,
        start_warmup_value=0,
        freeze_iters=0,
    ):
        super().__init__()
        self.final_value = final_value
        self.total_iters = total_iters

        freeze_schedule = np.zeros((freeze_iters))

        warmup_schedule = np.linspace(start_warmup_value, base_value, warmup_iters)

        iters = np.arange(total_iters - warmup_iters - freeze_iters)
        schedule = final_value + 0.5 * (base_value - final_value) * (
            1 + np.cos(np.pi * iters / len(iters))
        )
        self.schedule = np.concatenate((freeze_schedule, warmup_schedule, schedule))

        assert len(self.schedule) == self.total_iters

    def __getitem__(self, it):
        if it >= self.total_iters:
            return self.final_value
        else:
            return self.schedule[it]


def has_batchnorms(model):
    bn_types = (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d, nn.SyncBatchNorm)
    for name, module in model.named_modules():
        if isinstance(module, bn_types):
            return True
    return False


In [ ]:
%%writefile sih3d/vendor/pi3/__init__.py



In [ ]:
%%writefile sih3d/vendor/pi3/layers/__init__.py



In [ ]:
%%writefile sih3d/vendor/pi3/layers/attention.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

# References:
#   https://github.com/facebookresearch/dino/blob/master/vision_transformer.py
#   https://github.com/rwightman/pytorch-image-models/tree/master/timm/models/vision_transformer.py

import logging
import os
import warnings

from torch import Tensor
from torch import nn
import torch

from torch.nn.functional import scaled_dot_product_attention
from torch.nn.attention import SDPBackend

from sih3d.vendor.pi3.layers.prope import _prepare_apply_fns, _prepare_apply_fns_query

XFORMERS_ENABLED = os.environ.get("XFORMERS_DISABLED") is None
try:
    if XFORMERS_ENABLED:
        from xformers.ops import memory_efficient_attention, unbind

        XFORMERS_AVAILABLE = True
        # warnings.warn("xFormers is available (Attention)")
    else:
        # warnings.warn("xFormers is disabled (Attention)")
        raise ImportError
except ImportError:
    XFORMERS_AVAILABLE = False
    # warnings.warn("xFormers is not available (Attention)")


class Attention(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int = 8,
        qkv_bias: bool = False,
        proj_bias: bool = True,
        attn_drop: float = 0.0,
        proj_drop: float = 0.0,
    ) -> None:
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim**-0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim, bias=proj_bias)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x: Tensor, attn_bias=None) -> Tensor:
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        
        q, k, v = qkv[0] * self.scale, qkv[1], qkv[2]
        attn = q @ k.transpose(-2, -1)

        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class MemEffAttention(Attention):
    def forward(self, x: Tensor, attn_bias=None) -> Tensor:
        if not XFORMERS_AVAILABLE:
            if attn_bias is not None:
                raise AssertionError("xFormers is required for using nested tensors")
            return super().forward(x)

        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads)

        # q, k, v = unbind(qkv, 2)
        q, k, v = [qkv[:,:,i] for i in range(3)]

        x = memory_efficient_attention(q, k, v, attn_bias=attn_bias)
        x = x.reshape([B, N, C])

        x = self.proj(x)
        x = self.proj_drop(x)
        return x


    
class FlashAttention(Attention):
    def forward(self, x: Tensor, attn_bias=None) -> Tensor:
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).transpose(1, 3)

        # q, k, v = unbind(qkv, 2)
        q, k, v = [qkv[:,:,i] for i in range(3)]

        if q.dtype == torch.bfloat16:
            with nn.attention.sdpa_kernel(SDPBackend.FLASH_ATTENTION):
                x = scaled_dot_product_attention(q, k, v)
        else:
            with nn.attention.sdpa_kernel([SDPBackend.MATH, SDPBackend.EFFICIENT_ATTENTION]):
                x = scaled_dot_product_attention(q, k, v)

        x = x.transpose(1, 2).reshape([B, N, C])

        x = self.proj(x)
        x = self.proj_drop(x)
        return x


"""
Following is written by GPT-4o
"""
class CrossAttentionRope(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int = 8,
        qkv_bias: bool = False,
        proj_bias: bool = True,
        attn_drop: float = 0.0,
        proj_drop: float = 0.0,
        qk_norm: bool = False,
        norm_layer: nn.Module = nn.LayerNorm,
        rope=None,
    ) -> None:
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim**-0.5

        # Separate projection layers for query, key, and value
        self.q_proj = nn.Linear(dim, dim, bias=qkv_bias)
        self.k_proj = nn.Linear(dim, dim, bias=qkv_bias)
        self.v_proj = nn.Linear(dim, dim, bias=qkv_bias)

        self.q_norm = norm_layer(head_dim) if qk_norm else nn.Identity()
        self.k_norm = norm_layer(head_dim) if qk_norm else nn.Identity()

        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim, bias=proj_bias)
        self.proj_drop = nn.Dropout(proj_drop)

        self.rope = rope

    def forward(self, query: Tensor, key: Tensor, value: Tensor, attn_bias=None, qpos=None, kpos=None) -> Tensor:
        """
        Args:
            query: Tensor of shape (B, N, C), input query
            key: Tensor of shape (B, M, C), input key
            value: Tensor of shape (B, M, C), input value
            attn_bias: Optional tensor for attention bias
        Returns:
            Tensor of shape (B, N, C), output of cross-attention
        """
        B, N, C = query.shape
        _, M, _ = key.shape

        # Project query, key, and value
        q = self.q_proj(query).reshape(B, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        k = self.k_proj(key).reshape(B, M, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        v = self.v_proj(value).reshape(B, M, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        q, k = self.q_norm(q).to(v.dtype), self.k_norm(k).to(v.dtype)

        if self.rope is not None:
            q = self.rope(q, qpos)
            k = self.rope(k, kpos)

        # Scale query
        q = q * self.scale

        # Compute attention scores
        attn = q @ k.transpose(-2, -1)  # (B, num_heads, N, M)
        if attn_bias is not None:
            attn = attn + attn_bias

        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        # Compute attention output
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)  # (B, N, C)

        # Final projection
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class MemEffCrossAttentionRope(CrossAttentionRope):
    def forward(self, query: Tensor, key: Tensor, value: Tensor, attn_bias=None, qpos=None, kpos=None) -> Tensor:
        """
        Args:
            query: Tensor of shape (B, N, C), input query
            key: Tensor of shape (B, M, C), input key
            value: Tensor of shape (B, M, C), input value
            attn_bias: Optional tensor for attention bias
        Returns:
            Tensor of shape (B, N, C), output of cross-attention
        """
        if not XFORMERS_AVAILABLE:
            if attn_bias is not None:
                raise AssertionError("xFormers is required for using nested tensors")
            return super().forward(query, key, value, attn_bias)

        B, N, C = query.shape
        _, M, _ = key.shape

        # Project query, key, and value
        q = self.q_proj(query).reshape(B, N, self.num_heads, C // self.num_heads)
        k = self.k_proj(key).reshape(B, M, self.num_heads, C // self.num_heads)
        v = self.v_proj(value).reshape(B, M, self.num_heads, C // self.num_heads)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        q, k = self.q_norm(q).to(v.dtype), self.k_norm(k).to(v.dtype)

        if self.rope is not None:
            q = self.rope(q, qpos)
            k = self.rope(k, kpos)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)

        # Compute memory-efficient attention
        x = memory_efficient_attention(q, k, v, attn_bias=attn_bias)
        x = x.reshape(B, N, C)

        # Final projection
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class AttentionRope(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int = 8,
        qkv_bias: bool = False,
        proj_bias: bool = True,
        attn_drop: float = 0.0,
        proj_drop: float = 0.0,
        qk_norm: bool = False,
        norm_layer: nn.Module = nn.LayerNorm,
        rope=None
    ) -> None:
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim**-0.5
        self.head_dim = head_dim

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim, bias=proj_bias)
        self.proj_drop = nn.Dropout(proj_drop)

        self.q_norm = norm_layer(head_dim) if qk_norm else nn.Identity()
        self.k_norm = norm_layer(head_dim) if qk_norm else nn.Identity()

        self.rope = rope

    def forward(self, x: Tensor, attn_bias=None, xpos=None) -> Tensor:
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q, k = self.q_norm(q).to(v.dtype), self.k_norm(k).to(v.dtype)

        if self.rope is not None:
            q = self.rope(q, xpos)
            k = self.rope(k, xpos)
        
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)

        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class MemEffAttentionRope(AttentionRope):
    def forward(self, x: Tensor, attn_bias=None, xpos=None) -> Tensor:
        if not XFORMERS_AVAILABLE:
            if attn_bias is not None:
                raise AssertionError("xFormers is required for using nested tensors")
            return super().forward(x)

        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads)
        
        qkv = qkv.transpose(1, 3)
        # q, k, v = unbind(qkv, 2)
        q, k, v = [qkv[:,:,i] for i in range(3)]
        q, k = self.q_norm(q).to(v.dtype), self.k_norm(k).to(v.dtype)

        if self.rope is not None:
            q = self.rope(q, xpos)
            k = self.rope(k, xpos)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        x = memory_efficient_attention(q, k, v, attn_bias=attn_bias)
        x = x.reshape([B, N, C])

        # score_matrix = (q.permute(0, 2, 1, 3) * self.scale @ k.permute(0, 2, 1, 3).transpose(-2, -1)).sum(dim=1).reshape(frame_num, 261, frame_num, 261).mean(dim=[1, 3]).sum(1)         # for frame attention matrix
        # global_valid_id = torch.where(score_matrix > 0)
        # score_matrix = (q.permute(0, 2, 1, 3) * self.scale @ k.permute(0, 2, 1, 3).transpose(-2, -1)).sum(dim=1)

        x = self.proj(x)
        x = self.proj_drop(x)
        return x

    
class FlashAttentionRope(AttentionRope):
    def forward(self, x: Tensor, attn_bias=None, xpos=None) -> Tensor:
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).transpose(1, 3)

        # q, k, v = unbind(qkv, 2)
        q, k, v = [qkv[:,:,i] for i in range(3)]
        q, k = self.q_norm(q).to(v.dtype), self.k_norm(k).to(v.dtype)

        if self.rope is not None:
            q = self.rope(q, xpos)
            k = self.rope(k, xpos)

        if q.dtype == torch.bfloat16:
            with nn.attention.sdpa_kernel(SDPBackend.FLASH_ATTENTION):
                x = scaled_dot_product_attention(q, k, v)
        else:
            with nn.attention.sdpa_kernel([SDPBackend.MATH, SDPBackend.EFFICIENT_ATTENTION]):
                x = scaled_dot_product_attention(q, k, v)

        x = x.transpose(1, 2).reshape([B, N, C])

        x = self.proj(x)
        x = self.proj_drop(x)
        return x

def get_attn_score(blk_class, x, frame_num, token_length, xpos=None):
    x = blk_class.norm1(x)
    
    B, N, C = x.shape
    qkv = blk_class.attn.qkv(x).reshape(B, N, 3, blk_class.attn.num_heads, C // blk_class.attn.num_heads)
    
    qkv = qkv.transpose(1, 3)
    # q, k, v = unbind(qkv, 2)
    q, k, v = [qkv[:,:,i] for i in range(3)]
    q, k = blk_class.attn.q_norm(q).to(v.dtype), blk_class.attn.k_norm(k).to(v.dtype)

    if blk_class.attn.rope is not None:
        q = blk_class.attn.rope(q, xpos)
        k = blk_class.attn.rope(k, xpos)

    q = q.transpose(1, 2)
    k = k.transpose(1, 2)

    score = (q.permute(0, 2, 1, 3) * blk_class.attn.scale @ k.permute(0, 2, 1, 3).transpose(-2, -1)).sum(dim=1).reshape(B, frame_num, token_length, frame_num, token_length).mean(dim=[2, 4]).sum(-1)

    return score


class PRopeFlashAttention(AttentionRope):
    def forward(self, x: Tensor, extrinsics, H, W, patch_h, patch_w, K=None, attn_mask=None) -> Tensor:
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).transpose(1, 3)

        # q, k, v = unbind(qkv, 2)
        q, k, v = [qkv[:,:,i] for i in range(3)]
        q, k = self.q_norm(q).to(v.dtype), self.k_norm(k).to(v.dtype)

        apply_fn_q, apply_fn_kv, apply_fn_o = _prepare_apply_fns(
            head_dim=self.head_dim,
            viewmats=extrinsics,
            Ks=K,
            patches_x=patch_w,
            patches_y=patch_h,
            image_width=W,
            image_height=H,
        )
        q = apply_fn_q(q)
        k = apply_fn_kv(k)
        v = apply_fn_kv(v)

        if q.dtype == torch.bfloat16 and attn_mask is None:
            with nn.attention.sdpa_kernel(SDPBackend.FLASH_ATTENTION):
                x = scaled_dot_product_attention(q, k, v)
        else:
            with nn.attention.sdpa_kernel([SDPBackend.MATH, SDPBackend.EFFICIENT_ATTENTION]):
                x = scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        
        x = apply_fn_o(x)

        x = x.transpose(1, 2).reshape([B, N, C])

        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class FlashCrossAttentionRope(CrossAttentionRope):
    def forward(self, query: Tensor, key: Tensor, value: Tensor, attn_bias=None, qpos=None, kpos=None) -> Tensor:
        """
        Args:
            query: Tensor of shape (B, N, C)
            key: Tensor of shape (B, M, C)
            value: Tensor of shape (B, M, C),
        Returns:
            Tensor of shape (B, N, C),
        """
        B, N, C = query.shape
        _, M, _ = key.shape

        q = self.q_proj(query).reshape(B, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        k = self.k_proj(key).reshape(B, M, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        v = self.v_proj(value).reshape(B, M, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)

        q, k = self.q_norm(q).to(v.dtype), self.k_norm(k).to(v.dtype)
        if self.rope is not None:
            q = self.rope(q, qpos)
            k = self.rope(k, kpos)
        
        dropout_p = self.attn_drop.p if self.training else 0.0
        
        if q.dtype == torch.bfloat16:
            with nn.attention.sdpa_kernel(SDPBackend.FLASH_ATTENTION):
                x = scaled_dot_product_attention(
                    q, k, v, attn_mask=attn_bias, dropout_p=dropout_p
                )
        else:
            with nn.attention.sdpa_kernel([SDPBackend.MATH, SDPBackend.EFFICIENT_ATTENTION]):
                x = scaled_dot_product_attention(
                    q, k, v, attn_mask=attn_bias, dropout_p=dropout_p
                )

        x = x.transpose(1, 2).reshape(B, N, C)

        x = self.proj(x)
        x = self.proj_drop(x)
        return x
    

In [ ]:
%%writefile sih3d/vendor/pi3/layers/block.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the Apache License, Version 2.0
# found in the LICENSE file in the root directory of this source tree.

# References:
#   https://github.com/facebookresearch/dino/blob/master/vision_transformer.py
#   https://github.com/rwightman/pytorch-image-models/tree/master/timm/layers/patch_embed.py

import os
from typing import Any, Callable, Dict, List, Tuple

import torch
from torch import nn, Tensor
import numpy as np
from sih3d.vendor.dinov2.layers.drop_path import DropPath
from sih3d.vendor.dinov2.layers.layer_scale import LayerScale
from sih3d.vendor.dinov2.layers.mlp import Mlp
from sih3d.vendor.pi3.layers.attention import (
    Attention,
    CrossAttentionRope,
    MemEffAttention,
    PRopeFlashAttention
)

def se3_inverse(T):
    """
    Computes the inverse of a batch of SE(3) matrices.
    """

    if torch.is_tensor(T):
        R = T[..., :3, :3]
        t = T[..., :3, 3].unsqueeze(-1)
        R_inv = R.transpose(-2, -1)
        t_inv = -torch.matmul(R_inv, t)
        T_inv = torch.cat([
            torch.cat([R_inv, t_inv], dim=-1),
            torch.tensor([0, 0, 0, 1], device=T.device, dtype=T.dtype).repeat(*T.shape[:-2], 1, 1)
        ], dim=-2)
    else:
        R = T[..., :3, :3]
        t = T[..., :3, 3, np.newaxis]

        R_inv = np.swapaxes(R, -2, -1)
        t_inv = -R_inv @ t

        bottom_row = np.zeros((*T.shape[:-2], 1, 4), dtype=T.dtype)
        bottom_row[..., :, 3] = 1

        top_part = np.concatenate([R_inv, t_inv], axis=-1)
        T_inv = np.concatenate([top_part, bottom_row], axis=-2)
    return T_inv


XFORMERS_ENABLED = os.environ.get("XFORMERS_DISABLED") is None
try:
    if XFORMERS_ENABLED:
        from xformers.ops import fmha, scaled_index_add, index_select_cat

        XFORMERS_AVAILABLE = True
        # warnings.warn("xFormers is available (Block)")
    else:
        # warnings.warn("xFormers is disabled (Block)")
        raise ImportError
except ImportError:
    XFORMERS_AVAILABLE = False
    # warnings.warn("xFormers is not available (Block)")


class Block(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = False,
        proj_bias: bool = True,
        ffn_bias: bool = True,
        drop: float = 0.0,
        attn_drop: float = 0.0,
        init_values=None,
        drop_path: float = 0.0,
        act_layer: Callable[..., nn.Module] = nn.GELU,
        norm_layer: Callable[..., nn.Module] = nn.LayerNorm,
        attn_class: Callable[..., nn.Module] = Attention,
        ffn_layer: Callable[..., nn.Module] = Mlp,
    ) -> None:
        super().__init__()
        # print(f"biases: qkv: {qkv_bias}, proj: {proj_bias}, ffn: {ffn_bias}")
        self.norm1 = norm_layer(dim)
        self.attn = attn_class(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            proj_bias=proj_bias,
            attn_drop=attn_drop,
            proj_drop=drop,
        )

        self.ls1 = LayerScale(dim, init_values=init_values) if init_values else nn.Identity()
        self.drop_path1 = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = ffn_layer(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            act_layer=act_layer,
            drop=drop,
            bias=ffn_bias,
        )
        self.ls2 = LayerScale(dim, init_values=init_values) if init_values else nn.Identity()
        self.drop_path2 = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

        self.sample_drop_ratio = drop_path

    def forward(self, x: Tensor) -> Tensor:
        def attn_residual_func(x: Tensor) -> Tensor:
            return self.ls1(self.attn(self.norm1(x)))

        def ffn_residual_func(x: Tensor) -> Tensor:
            return self.ls2(self.mlp(self.norm2(x)))

        if self.training and self.sample_drop_ratio > 0.1:
            # the overhead is compensated only for a drop path rate larger than 0.1
            x = drop_add_residual_stochastic_depth(
                x,
                residual_func=attn_residual_func,
                sample_drop_ratio=self.sample_drop_ratio,
            )
            x = drop_add_residual_stochastic_depth(
                x,
                residual_func=ffn_residual_func,
                sample_drop_ratio=self.sample_drop_ratio,
            )
        elif self.training and self.sample_drop_ratio > 0.0:
            x = x + self.drop_path1(attn_residual_func(x))
            x = x + self.drop_path1(ffn_residual_func(x))  # FIXME: drop_path2
        else:
            x = x + attn_residual_func(x)
            x = x + ffn_residual_func(x)
        return x


def drop_add_residual_stochastic_depth(
    x: Tensor,
    residual_func: Callable[[Tensor], Tensor],
    sample_drop_ratio: float = 0.0,
) -> Tensor:
    # 1) extract subset using permutation
    b, n, d = x.shape
    sample_subset_size = max(int(b * (1 - sample_drop_ratio)), 1)
    brange = (torch.randperm(b, device=x.device))[:sample_subset_size]
    x_subset = x[brange]

    # 2) apply residual_func to get residual
    residual = residual_func(x_subset)

    x_flat = x.flatten(1)
    residual = residual.flatten(1)

    residual_scale_factor = b / sample_subset_size

    # 3) add the residual
    x_plus_residual = torch.index_add(x_flat, 0, brange, residual.to(dtype=x.dtype), alpha=residual_scale_factor)
    return x_plus_residual.view_as(x)


def get_branges_scales(x, sample_drop_ratio=0.0):
    b, n, d = x.shape
    sample_subset_size = max(int(b * (1 - sample_drop_ratio)), 1)
    brange = (torch.randperm(b, device=x.device))[:sample_subset_size]
    residual_scale_factor = b / sample_subset_size
    return brange, residual_scale_factor


def add_residual(x, brange, residual, residual_scale_factor, scaling_vector=None):
    if scaling_vector is None:
        x_flat = x.flatten(1)
        residual = residual.flatten(1)
        x_plus_residual = torch.index_add(x_flat, 0, brange, residual.to(dtype=x.dtype), alpha=residual_scale_factor)
    else:
        x_plus_residual = scaled_index_add(
            x, brange, residual.to(dtype=x.dtype), scaling=scaling_vector, alpha=residual_scale_factor
        )
    return x_plus_residual


attn_bias_cache: Dict[Tuple, Any] = {}


def get_attn_bias_and_cat(x_list, branges=None):
    """
    this will perform the index select, cat the tensors, and provide the attn_bias from cache
    """
    batch_sizes = [b.shape[0] for b in branges] if branges is not None else [x.shape[0] for x in x_list]
    all_shapes = tuple((b, x.shape[1]) for b, x in zip(batch_sizes, x_list))
    if all_shapes not in attn_bias_cache.keys():
        seqlens = []
        for b, x in zip(batch_sizes, x_list):
            for _ in range(b):
                seqlens.append(x.shape[1])
        attn_bias = fmha.BlockDiagonalMask.from_seqlens(seqlens)
        attn_bias._batch_sizes = batch_sizes
        attn_bias_cache[all_shapes] = attn_bias

    if branges is not None:
        cat_tensors = index_select_cat([x.flatten(1) for x in x_list], branges).view(1, -1, x_list[0].shape[-1])
    else:
        tensors_bs1 = tuple(x.reshape([1, -1, *x.shape[2:]]) for x in x_list)
        cat_tensors = torch.cat(tensors_bs1, dim=1)

    return attn_bias_cache[all_shapes], cat_tensors


def drop_add_residual_stochastic_depth_list(
    x_list: List[Tensor],
    residual_func: Callable[[Tensor, Any], Tensor],
    sample_drop_ratio: float = 0.0,
    scaling_vector=None,
) -> Tensor:
    # 1) generate random set of indices for dropping samples in the batch
    branges_scales = [get_branges_scales(x, sample_drop_ratio=sample_drop_ratio) for x in x_list]
    branges = [s[0] for s in branges_scales]
    residual_scale_factors = [s[1] for s in branges_scales]

    # 2) get attention bias and index+concat the tensors
    attn_bias, x_cat = get_attn_bias_and_cat(x_list, branges)

    # 3) apply residual_func to get residual, and split the result
    residual_list = attn_bias.split(residual_func(x_cat, attn_bias=attn_bias))  # type: ignore

    outputs = []
    for x, brange, residual, residual_scale_factor in zip(x_list, branges, residual_list, residual_scale_factors):
        outputs.append(add_residual(x, brange, residual, residual_scale_factor, scaling_vector).view_as(x))
    return outputs


class NestedTensorBlock(Block):
    def forward_nested(self, x_list: List[Tensor]) -> List[Tensor]:
        """
        x_list contains a list of tensors to nest together and run
        """
        assert isinstance(self.attn, MemEffAttention)

        if self.training and self.sample_drop_ratio > 0.0:

            def attn_residual_func(x: Tensor, attn_bias=None) -> Tensor:
                return self.attn(self.norm1(x), attn_bias=attn_bias)

            def ffn_residual_func(x: Tensor, attn_bias=None) -> Tensor:
                return self.mlp(self.norm2(x))

            x_list = drop_add_residual_stochastic_depth_list(
                x_list,
                residual_func=attn_residual_func,
                sample_drop_ratio=self.sample_drop_ratio,
                scaling_vector=self.ls1.gamma if isinstance(self.ls1, LayerScale) else None,
            )
            x_list = drop_add_residual_stochastic_depth_list(
                x_list,
                residual_func=ffn_residual_func,
                sample_drop_ratio=self.sample_drop_ratio,
                scaling_vector=self.ls2.gamma if isinstance(self.ls1, LayerScale) else None,
            )
            return x_list
        else:

            def attn_residual_func(x: Tensor, attn_bias=None) -> Tensor:
                return self.ls1(self.attn(self.norm1(x), attn_bias=attn_bias))

            def ffn_residual_func(x: Tensor, attn_bias=None) -> Tensor:
                return self.ls2(self.mlp(self.norm2(x)))

            attn_bias, x = get_attn_bias_and_cat(x_list)
            x = x + attn_residual_func(x, attn_bias=attn_bias)
            x = x + ffn_residual_func(x)
            return attn_bias.split(x)

    def forward(self, x_or_x_list):
        if isinstance(x_or_x_list, Tensor):
            return super().forward(x_or_x_list)
        elif isinstance(x_or_x_list, list):
            if not XFORMERS_AVAILABLE:
                raise AssertionError("xFormers is required for using nested tensors")
            return self.forward_nested(x_or_x_list)
        else:
            raise AssertionError

class BlockRope(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = False,
        proj_bias: bool = True,
        ffn_bias: bool = True,
        drop: float = 0.0,
        attn_drop: float = 0.0,
        init_values=None,
        drop_path: float = 0.0,
        act_layer: Callable[..., nn.Module] = nn.GELU,
        norm_layer: Callable[..., nn.Module] = nn.LayerNorm,
        attn_class: Callable[..., nn.Module] = Attention,
        ffn_layer: Callable[..., nn.Module] = Mlp,
        qk_norm: bool=False,
        rope=None
    ) -> None:
        super().__init__()
        # print(f"biases: qkv: {qkv_bias}, proj: {proj_bias}, ffn: {ffn_bias}")
        self.norm1 = norm_layer(dim)
        self.attn = attn_class(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            proj_bias=proj_bias,
            attn_drop=attn_drop,
            proj_drop=drop,
            qk_norm=qk_norm,
            rope=rope
        )

        self.ls1 = LayerScale(dim, init_values=init_values) if init_values else nn.Identity()
        self.drop_path1 = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = ffn_layer(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            act_layer=act_layer,
            drop=drop,
            bias=ffn_bias,
        )
        self.ls2 = LayerScale(dim, init_values=init_values) if init_values else nn.Identity()
        self.drop_path2 = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

        self.sample_drop_ratio = drop_path

    def forward(self, x: Tensor, xpos=None) -> Tensor:
        def attn_residual_func(x: Tensor) -> Tensor:
            return self.ls1(self.attn(self.norm1(x), xpos=xpos))

        def ffn_residual_func(x: Tensor) -> Tensor:
            return self.ls2(self.mlp(self.norm2(x)))

        if self.training and self.sample_drop_ratio > 0.1:
            # the overhead is compensated only for a drop path rate larger than 0.1
            x = drop_add_residual_stochastic_depth(
                x,
                residual_func=attn_residual_func,
                sample_drop_ratio=self.sample_drop_ratio,
            )
            x = drop_add_residual_stochastic_depth(
                x,
                residual_func=ffn_residual_func,
                sample_drop_ratio=self.sample_drop_ratio,
            )
        elif self.training and self.sample_drop_ratio > 0.0:
            x = x + self.drop_path1(attn_residual_func(x))
            x = x + self.drop_path1(ffn_residual_func(x))  # FIXME: drop_path2
        else:
            x = x + attn_residual_func(x)
            x = x + ffn_residual_func(x)
        return x


class CrossBlockRope(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = False,
        proj_bias: bool = True,
        ffn_bias: bool = True,
        act_layer: Callable[..., nn.Module] = nn.GELU,
        norm_layer: Callable[..., nn.Module] = nn.LayerNorm,
        attn_class: Callable[..., nn.Module] = Attention,
        cross_attn_class: Callable[..., nn.Module] = CrossAttentionRope,
        ffn_layer: Callable[..., nn.Module] = Mlp,
        init_values=None,
        qk_norm: bool=False,
        rope=None
    ) -> None:
        super().__init__()
        # print(f"biases: qkv: {qkv_bias}, proj: {proj_bias}, ffn: {ffn_bias}")
        self.ls1 = LayerScale(dim, init_values=init_values) if init_values else nn.Identity()
        self.norm1 = norm_layer(dim)
        self.attn = attn_class(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            proj_bias=proj_bias,
            rope=rope,
            qk_norm=qk_norm
        )

        self.ls2 = LayerScale(dim, init_values=init_values) if init_values else nn.Identity()
        self.ls_y = LayerScale(dim, init_values=init_values) if init_values else nn.Identity()
        self.norm2 = norm_layer(dim)
        self.norm_y = norm_layer(dim)
        self.cross_attn = cross_attn_class(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            proj_bias=proj_bias,
            rope=rope,
            qk_norm=qk_norm
        )

        self.norm3 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = ffn_layer(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            act_layer=act_layer,
            bias=ffn_bias,
        )

    def forward(self, x: Tensor, y: Tensor, xpos=None, ypos=None) -> Tensor:
        def attn_residual_func(x: Tensor) -> Tensor:
            return self.ls1(self.attn(self.norm1(x), xpos=xpos))

        def cross_attn_residual_func(x: Tensor, y: Tensor) -> Tensor:
            return self.ls_y(self.cross_attn(self.norm2(x), y, y, qpos=xpos, kpos=ypos))

        def ffn_residual_func(x: Tensor) -> Tensor:
            return self.ls2(self.mlp(self.norm3(x)))

        x = x + attn_residual_func(x)
        y_ = self.norm_y(y)
        x = x + cross_attn_residual_func(x, y_)
        x = x + ffn_residual_func(x)

        return x
    

class PoseInjectBlock(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = False,
        proj_bias: bool = True,
        ffn_bias: bool = True,
        drop: float = 0.0,
        attn_drop: float = 0.0,
        init_values=None,
        drop_path: float = 0.0,
        act_layer: Callable[..., nn.Module] = nn.GELU,
        norm_layer: Callable[..., nn.Module] = nn.LayerNorm,
        attn_class: Callable[..., nn.Module] = PRopeFlashAttention,
        ffn_layer: Callable[..., nn.Module] = Mlp,
        qk_norm: bool=False,
        rope=None
    ) -> None:
        super().__init__()
        # print(f"biases: qkv: {qkv_bias}, proj: {proj_bias}, ffn: {ffn_bias}")
        self.norm1 = norm_layer(dim)
        self.attn = attn_class(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            proj_bias=proj_bias,
            attn_drop=attn_drop,
            proj_drop=drop,
            qk_norm=qk_norm,
            rope=rope
        )

        self.ls1 = LayerScale(dim, init_values=init_values) if init_values else nn.Identity()
        self.drop_path1 = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = ffn_layer(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            act_layer=act_layer,
            drop=drop,
            bias=ffn_bias,
        )
        self.ls2 = LayerScale(dim, init_values=init_values) if init_values else nn.Identity()
        self.drop_path2 = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

        self.sample_drop_ratio = drop_path

    def forward(self, x: Tensor, poses, H, W, patch_h, patch_w, K=None, connect=False, attn_mask=None) -> Tensor:
        extrinsics = se3_inverse(poses)
        def attn_residual_func(x: Tensor) -> Tensor:
            return self.ls1(self.attn(self.norm1(x), extrinsics, H, W, patch_h, patch_w, K=K, attn_mask=attn_mask))

        def ffn_residual_func(x: Tensor) -> Tensor:
            return self.ls2(self.mlp(self.norm2(x)))

        if connect:
            return x + attn_residual_func(x) + ffn_residual_func(x)
        return attn_residual_func(x) + ffn_residual_func(x)

class CrossOnlyBlockRope(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = False,
        proj_bias: bool = True,
        ffn_bias: bool = True,
        act_layer: Callable[..., nn.Module] = nn.GELU,
        norm_layer: Callable[..., nn.Module] = nn.LayerNorm,
        # attn_class 已被移除，因为它不再被使用
        cross_attn_class: Callable[..., nn.Module] = CrossAttentionRope,
        ffn_layer: Callable[..., nn.Module] = Mlp,
        init_values=None,
        qk_norm: bool=False,
        rope=None
    ) -> None:
        super().__init__()
        # print(f"biases: qkv: {qkv_bias}, proj: {proj_bias}, ffn: {ffn_bias}")
        
        # ---------------------------------------------------

        self.ls2 = LayerScale(dim, init_values=init_values) if init_values else nn.Identity()
        self.ls_y = LayerScale(dim, init_values=init_values) if init_values else nn.Identity()
        self.norm2 = norm_layer(dim)
        self.norm_y = norm_layer(dim)
        self.cross_attn = cross_attn_class(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            proj_bias=proj_bias,
            rope=rope,
            qk_norm=qk_norm
        )

        self.norm3 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = ffn_layer(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            act_layer=act_layer,
            bias=ffn_bias,
        )

    def forward(self, x: Tensor, y: Tensor, xpos=None, ypos=None) -> Tensor:
        
        # ---------------------------

        def cross_attn_residual_func(x: Tensor, y: Tensor) -> Tensor:
            # 注意：self.norm2(x) 是 x 经过 pre-normalization
            return self.ls_y(self.cross_attn(self.norm2(x), y, y, qpos=xpos, kpos=ypos))

        def ffn_residual_func(x: Tensor) -> Tensor:
            return self.ls2(self.mlp(self.norm3(x)))

        # x = x + attn_residual_func(x) 
        
        y_ = self.norm_y(y) 
        x = x + cross_attn_residual_func(x, y_)
        x = x + ffn_residual_func(x)

        return x
    

In [ ]:
%%writefile sih3d/vendor/pi3/layers/camera_head.py
import torch
import torch.nn as nn
from copy import deepcopy
import torch.nn.functional as F

# code adapted from 'https://github.com/nianticlabs/marepo/blob/9a45e2bb07e5bb8cb997620088d352b439b13e0e/transformer/transformer.py#L172'
class ResConvBlock(nn.Module):
    """
    1x1 convolution residual block
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.head_skip = nn.Identity() if self.in_channels == self.out_channels else nn.Conv2d(self.in_channels, self.out_channels, 1, 1, 0)
        # self.res_conv1 = nn.Conv2d(self.in_channels, self.out_channels, 1, 1, 0)
        # self.res_conv2 = nn.Conv2d(self.out_channels, self.out_channels, 1, 1, 0)
        # self.res_conv3 = nn.Conv2d(self.out_channels, self.out_channels, 1, 1, 0)

        # change 1x1 convolution to linear
        self.res_conv1 = nn.Linear(self.in_channels, self.out_channels)
        self.res_conv2 = nn.Linear(self.out_channels, self.out_channels)
        self.res_conv3 = nn.Linear(self.out_channels, self.out_channels)

    def forward(self, res):
        x = F.relu(self.res_conv1(res))
        x = F.relu(self.res_conv2(x))
        x = F.relu(self.res_conv3(x))
        res = self.head_skip(res) + x
        return res

class CameraHead(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        output_dim = dim
        self.res_conv = nn.ModuleList([deepcopy(ResConvBlock(output_dim, output_dim)) 
                for _ in range(2)])
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.more_mlps = nn.Sequential(
            nn.Linear(output_dim,output_dim),
            nn.ReLU(),
            nn.Linear(output_dim,output_dim),
            nn.ReLU()
            )
        self.fc_t = nn.Linear(output_dim, 3)
        self.fc_rot = nn.Linear(output_dim, 9)

    def forward(self, feat, patch_h, patch_w):
        BN, hw, c = feat.shape

        for i in range(2):
            feat = self.res_conv[i](feat)

        # feat = self.avgpool(feat)
        feat = self.avgpool(feat.permute(0, 2, 1).reshape(BN, -1, patch_h, patch_w).contiguous())              ##########
        feat = feat.view(feat.size(0), -1)

        feat = self.more_mlps(feat)  # [B, D_]
        with torch.amp.autocast(device_type='cuda', enabled=False):
            out_t = self.fc_t(feat.float())  # [B,3]
            out_r = self.fc_rot(feat.float())  # [B,9]
            pose = self.convert_pose_to_4x4(BN, out_r, out_t, feat.device)

        return pose

    def convert_pose_to_4x4(self, B, out_r, out_t, device):
        out_r = self.svd_orthogonalize(out_r)  # [N,3,3]
        pose = torch.zeros((B, 4, 4), device=device)
        pose[:, :3, :3] = out_r
        pose[:, :3, 3] = out_t
        pose[:, 3, 3] = 1.
        return pose

    def svd_orthogonalize(self, m):
        """Convert 9D representation to SO(3) using SVD orthogonalization.

        Args:
          m: [BATCH, 3, 3] 3x3 matrices.

        Returns:
          [BATCH, 3, 3] SO(3) rotation matrices.
        """
        if m.dim() < 3:
            m = m.reshape((-1, 3, 3))
        m_transpose = torch.transpose(torch.nn.functional.normalize(m, p=2, dim=-1), dim0=-1, dim1=-2)
        u, s, v = torch.svd(m_transpose)
        det = torch.det(torch.matmul(v, u.transpose(-2, -1)))
        # Check orientation reflection.
        r = torch.matmul(
            torch.cat([v[:, :, :-1], v[:, :, -1:] * det.view(-1, 1, 1)], dim=2),
            u.transpose(-2, -1)
        )
        return r
    

In [ ]:
%%writefile sih3d/vendor/pi3/layers/conv_head.py
"""
Conv head is from MoGe (https://github.com/microsoft/moge)
"""

import torch
import torch.nn as nn
from typing import *
import torch.nn.functional as F


def normalized_view_plane_uv(width: int, height: int, aspect_ratio: float = None, dtype: torch.dtype = None, device: torch.device = None) -> torch.Tensor:
    "UV with left-top corner as (-width / diagonal, -height / diagonal) and right-bottom corner as (width / diagonal, height / diagonal)"
    if aspect_ratio is None:
        aspect_ratio = width / height
    
    span_x = aspect_ratio / (1 + aspect_ratio ** 2) ** 0.5
    span_y = 1 / (1 + aspect_ratio ** 2) ** 0.5

    u = torch.linspace(-span_x * (width - 1) / width, span_x * (width - 1) / width, width, dtype=dtype, device=device)
    v = torch.linspace(-span_y * (height - 1) / height, span_y * (height - 1) / height, height, dtype=dtype, device=device)
    u, v = torch.meshgrid(u, v, indexing='xy')
    uv = torch.stack([u, v], dim=-1)
    return uv

class ResidualConvBlock(nn.Module):  
    def __init__(self, in_channels: int, out_channels: int = None, hidden_channels: int = None, padding_mode: str = 'replicate', activation: Literal['relu', 'leaky_relu', 'silu', 'elu'] = 'relu', norm: Literal['group_norm', 'layer_norm'] = 'group_norm'):  
        super(ResidualConvBlock, self).__init__()  
        if out_channels is None:  
            out_channels = in_channels
        if hidden_channels is None:
            hidden_channels = in_channels

        if activation =='relu':
            activation_cls = lambda: nn.ReLU(inplace=True)
        elif activation == 'leaky_relu':
            activation_cls = lambda: nn.LeakyReLU(negative_slope=0.2, inplace=True)
        elif activation =='silu':
            activation_cls = lambda: nn.SiLU(inplace=True)
        elif activation == 'elu':
            activation_cls = lambda: nn.ELU(inplace=True)
        else:
            raise ValueError(f'Unsupported activation function: {activation}')

        self.layers = nn.Sequential(
            nn.GroupNorm(1, in_channels),
            activation_cls(),
            nn.Conv2d(in_channels, hidden_channels, kernel_size=3, padding=1, padding_mode=padding_mode),
            nn.GroupNorm(hidden_channels // 32 if norm == 'group_norm' else 1, hidden_channels),
            activation_cls(),
            nn.Conv2d(hidden_channels, out_channels, kernel_size=3, padding=1, padding_mode=padding_mode)
        )
        
        self.skip_connection = nn.Conv2d(in_channels, out_channels, kernel_size=1, padding=0) if in_channels != out_channels else nn.Identity()  
  
    def forward(self, x):  
        skip = self.skip_connection(x)  
        x = self.layers(x)
        x = x + skip
        return x  


class ConvHead(nn.Module):
    def __init__(
        self, 
        num_features: int,
        dim_in: int, 
        dim_out: List[int], 
        dim_proj: int = 512,
        dim_upsample: List[int] = [256, 128, 128],
        dim_times_res_block_hidden: int = 1,
        num_res_blocks: int = 1,
        res_block_norm: Literal['group_norm', 'layer_norm'] = 'group_norm',
        last_res_blocks: int = 0,
        last_conv_channels: int = 32,
        last_conv_size: int = 1,
        projects: nn.Module = None,
        using_uv: bool = True
    ):
        super().__init__()
        
        self.using_uv = using_uv
        if projects is not None:
            self.projects = projects

        self.upsample_blocks = nn.ModuleList([
            nn.Sequential(
                self._make_upsampler(in_ch + 2 if using_uv else in_ch, out_ch),
                *(ResidualConvBlock(out_ch, out_ch, dim_times_res_block_hidden * out_ch, activation="relu", norm=res_block_norm) for _ in range(num_res_blocks))
            ) for in_ch, out_ch in zip([dim_proj] + dim_upsample[:-1], dim_upsample)
        ])

        self.output_block = nn.ModuleList([
            self._make_output_block(
                dim_upsample[-1] + 2 if using_uv else dim_upsample[-1], dim_out_, dim_times_res_block_hidden, last_res_blocks, last_conv_channels, last_conv_size, res_block_norm,
            ) for dim_out_ in dim_out
        ])
    
    def _make_upsampler(self, in_channels: int, out_channels: int):
        upsampler = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, padding_mode='replicate')
        )
        upsampler[0].weight.data[:] = upsampler[0].weight.data[:, :, :1, :1]
        return upsampler

    def _make_output_block(self, dim_in: int, dim_out: int, dim_times_res_block_hidden: int, last_res_blocks: int, last_conv_channels: int, last_conv_size: int, res_block_norm: Literal['group_norm', 'layer_norm']):
        return nn.Sequential(
            nn.Conv2d(dim_in, last_conv_channels, kernel_size=3, stride=1, padding=1, padding_mode='replicate'),
            *(ResidualConvBlock(last_conv_channels, last_conv_channels, dim_times_res_block_hidden * last_conv_channels, activation='relu', norm=res_block_norm) for _ in range(last_res_blocks)),
            nn.ReLU(inplace=True),
            nn.Conv2d(last_conv_channels, dim_out, kernel_size=last_conv_size, stride=1, padding=last_conv_size // 2, padding_mode='replicate'),
        )
            
    def forward(self, hidden_states: torch.Tensor, image: torch.Tensor = None, patch_h: int = None, patch_w: int = None):
        if image is not None:
            img_h, img_w = image.shape[-2:]
            patch_h, patch_w = img_h // 14, img_w // 14
        else:
            assert patch_h is not None and patch_w is not None
            img_h = patch_h * 14
            img_w = patch_w * 14

        # Process the hidden states
        if self.projects is not None:
            x = self.projects(hidden_states).permute(0, 2, 1).unflatten(2, (patch_h, patch_w)).contiguous()
        else:
            x = hidden_states
        
        # Upsample stage
        # (patch_h, patch_w) -> (patch_h * 2, patch_w * 2) -> (patch_h * 4, patch_w * 4) -> (patch_h * 8, patch_w * 8)
        for i, block in enumerate(self.upsample_blocks):
            # UV coordinates is for awareness of image aspect ratio
            if self.using_uv:
                uv = normalized_view_plane_uv(width=x.shape[-1], height=x.shape[-2], aspect_ratio=img_w / img_h, dtype=x.dtype, device=x.device)
                uv = uv.permute(2, 0, 1).unsqueeze(0).expand(x.shape[0], -1, -1, -1)
                x = torch.cat([x, uv], dim=1)
            for layer in block:
                x = torch.utils.checkpoint.checkpoint(layer, x, use_reentrant=False)
        
        # (patch_h * 8, patch_w * 8) -> (img_h, img_w)
        x = F.interpolate(x, (img_h, img_w), mode="bilinear", align_corners=False)
        if self.using_uv:
            uv = normalized_view_plane_uv(width=x.shape[-1], height=x.shape[-2], aspect_ratio=img_w / img_h, dtype=x.dtype, device=x.device)
            uv = uv.permute(2, 0, 1).unsqueeze(0).expand(x.shape[0], -1, -1, -1)
            x = torch.cat([x, uv], dim=1)

        if isinstance(self.output_block, nn.ModuleList):
            output = [torch.utils.checkpoint.checkpoint(block, x, use_reentrant=False) for block in self.output_block]
        else:
            output = torch.utils.checkpoint.checkpoint(self.output_block, x, use_reentrant=False)
        
        return output
    

In [ ]:
%%writefile sih3d/vendor/pi3/layers/pos_embed.py
# Copyright (C) 2022-present Naver Corporation. All rights reserved.
# Licensed under CC BY-NC-SA 4.0 (non-commercial use only).


# --------------------------------------------------------
# Position embedding utils
# --------------------------------------------------------



import numpy as np

import torch

# --------------------------------------------------------
# 2D sine-cosine position embedding
# References:
# MAE: https://github.com/facebookresearch/mae/blob/main/util/pos_embed.py
# Transformer: https://github.com/tensorflow/models/blob/master/official/nlp/transformer/model_utils.py
# MoCo v3: https://github.com/facebookresearch/moco-v3
# --------------------------------------------------------
def get_2d_sincos_pos_embed(embed_dim, grid_size, n_cls_token=0):
    """
    grid_size: int of the grid height and width
    return:
    pos_embed: [grid_size*grid_size, embed_dim] or [n_cls_token+grid_size*grid_size, embed_dim] (w/ or w/o cls_token)
    """
    grid_h = np.arange(grid_size, dtype=np.float32)
    grid_w = np.arange(grid_size, dtype=np.float32)
    grid = np.meshgrid(grid_w, grid_h)  # here w goes first
    grid = np.stack(grid, axis=0)

    grid = grid.reshape([2, 1, grid_size, grid_size])
    pos_embed = get_2d_sincos_pos_embed_from_grid(embed_dim, grid)
    if n_cls_token>0:
        pos_embed = np.concatenate([np.zeros([n_cls_token, embed_dim]), pos_embed], axis=0)
    return pos_embed


def get_2d_sincos_pos_embed_from_grid(embed_dim, grid):
    assert embed_dim % 2 == 0

    # use half of dimensions to encode grid_h
    emb_h = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[0])  # (H*W, D/2)
    emb_w = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[1])  # (H*W, D/2)

    emb = np.concatenate([emb_h, emb_w], axis=1) # (H*W, D)
    return emb


def get_1d_sincos_pos_embed_from_grid(embed_dim, pos):
    """
    embed_dim: output dimension for each position
    pos: a list of positions to be encoded: size (M,)
    out: (M, D)
    """
    assert embed_dim % 2 == 0
    omega = np.arange(embed_dim // 2, dtype=float)
    omega /= embed_dim / 2.
    omega = 1. / 10000**omega  # (D/2,)

    pos = pos.reshape(-1)  # (M,)
    out = np.einsum('m,d->md', pos, omega)  # (M, D/2), outer product

    emb_sin = np.sin(out) # (M, D/2)
    emb_cos = np.cos(out) # (M, D/2)

    emb = np.concatenate([emb_sin, emb_cos], axis=1)  # (M, D)
    return emb


# --------------------------------------------------------
# Interpolate position embeddings for high-resolution
# References:
# MAE: https://github.com/facebookresearch/mae/blob/main/util/pos_embed.py
# DeiT: https://github.com/facebookresearch/deit
# --------------------------------------------------------
def interpolate_pos_embed(model, checkpoint_model):
    if 'pos_embed' in checkpoint_model:
        pos_embed_checkpoint = checkpoint_model['pos_embed']
        embedding_size = pos_embed_checkpoint.shape[-1]
        num_patches = model.patch_embed.num_patches
        num_extra_tokens = model.pos_embed.shape[-2] - num_patches
        # height (== width) for the checkpoint position embedding
        orig_size = int((pos_embed_checkpoint.shape[-2] - num_extra_tokens) ** 0.5)
        # height (== width) for the new position embedding
        new_size = int(num_patches ** 0.5)
        # class_token and dist_token are kept unchanged
        if orig_size != new_size:
            print("Position interpolate from %dx%d to %dx%d" % (orig_size, orig_size, new_size, new_size))
            extra_tokens = pos_embed_checkpoint[:, :num_extra_tokens]
            # only the position tokens are interpolated
            pos_tokens = pos_embed_checkpoint[:, num_extra_tokens:]
            pos_tokens = pos_tokens.reshape(-1, orig_size, orig_size, embedding_size).permute(0, 3, 1, 2)
            pos_tokens = torch.nn.functional.interpolate(
                pos_tokens, size=(new_size, new_size), mode='bicubic', align_corners=False)
            pos_tokens = pos_tokens.permute(0, 2, 3, 1).flatten(1, 2)
            new_pos_embed = torch.cat((extra_tokens, pos_tokens), dim=1)
            checkpoint_model['pos_embed'] = new_pos_embed


#----------------------------------------------------------
# RoPE2D: RoPE implementation in 2D
#----------------------------------------------------------

# try:
#     from models.curope import cuRoPE2D
#     RoPE2D = cuRoPE2D
# except ImportError:
print('Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead')
class RoPE2D(torch.nn.Module):
        
        def __init__(self, freq=100.0, F0=1.0):
            super().__init__()
            self.base = freq 
            self.F0 = F0
            self.cache = {}

        def get_cos_sin(self, D, seq_len, device, dtype):
            seq_len = int(seq_len)
            D = int(D)

            if D <= 0 or D % 2 != 0:
                raise RuntimeError(f"Invalid RoPE D={D}, seq_len={seq_len}, device={device}, dtype={dtype}")

            if seq_len <= 0 or seq_len > 4097:
                raise RuntimeError(f"Invalid RoPE seq_len={seq_len}, D={D}, device={device}, dtype={dtype}")
            
            key = (D, seq_len, str(device), str(dtype))
            if key not in self.cache:
                inv_freq = 1.0 / (
                    self.base ** (torch.arange(0, D, 2, device=device).float() / D)
                )
                t = torch.arange(seq_len, device=device, dtype=inv_freq.dtype)
                freqs = torch.einsum("i,j->ij", t, inv_freq).to(dtype)
                freqs = torch.cat((freqs, freqs), dim=-1)
                cos = freqs.cos()
                sin = freqs.sin()
                self.cache[key] = (cos, sin)

            return self.cache[key]

            # if (D,seq_len,device,dtype) not in self.cache:
            #     inv_freq = 1.0 / (self.base ** (torch.arange(0, D, 2).float().to(device) / D))
            #     t = torch.arange(seq_len, device=device, dtype=inv_freq.dtype)
            #     freqs = torch.einsum("i,j->ij", t, inv_freq).to(dtype)
            #     freqs = torch.cat((freqs, freqs), dim=-1)
            #     cos = freqs.cos() # (Seq, Dim)
            #     sin = freqs.sin()
            #     self.cache[D,seq_len,device,dtype] = (cos,sin)
            # return self.cache[D,seq_len,device,dtype]
            
        @staticmethod
        def rotate_half(x):
            x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
            return torch.cat((-x2, x1), dim=-1)
            
        def apply_rope1d(self, tokens, pos1d, cos, sin):
            assert pos1d.ndim==2
            cos = torch.nn.functional.embedding(pos1d, cos)[:, None, :, :]
            sin = torch.nn.functional.embedding(pos1d, sin)[:, None, :, :]
            return (tokens * cos) + (self.rotate_half(tokens) * sin)
            
        def forward(self, tokens, positions):
            """
            input:
                * tokens: batch_size x nheads x ntokens x dim
                * positions: batch_size x ntokens x 2 (y and x position of each token)
            output:
                * tokens after applying RoPE2D
            """
            if tokens.size(3) % 2 != 0:
                raise RuntimeError(
                    f"Invalid RoPE token dim: tokens_shape={tuple(tokens.shape)}"
                )

            D = tokens.size(3) // 2

            if positions.ndim != 3 or positions.shape[-1] != 2:
                raise RuntimeError(
                    f"Invalid RoPE positions shape before cast: "
                    f"positions_shape={tuple(positions.shape)}, "
                    f"tokens_shape={tuple(tokens.shape)}"
                )

            # 只在 positions 是浮点时检查 NaN/Inf
            if torch.is_floating_point(positions):
                if not bool(torch.isfinite(positions).all().item()):
                    raise RuntimeError(
                        f"Non-finite RoPE positions before cast: "
                        f"positions_shape={tuple(positions.shape)}, "
                        f"tokens_shape={tuple(tokens.shape)}, "
                        f"positions_dtype={positions.dtype}"
                    )

            positions = positions.to(device=tokens.device, dtype=torch.long).contiguous()

            if positions.shape[0] != tokens.shape[0] or positions.shape[1] != tokens.shape[2]:
                raise RuntimeError(
                    f"RoPE shape mismatch: "
                    f"positions_shape={tuple(positions.shape)}, "
                    f"tokens_shape={tuple(tokens.shape)}"
                )

            if positions.numel() == 0:
                raise RuntimeError(
                    f"Empty RoPE positions: "
                    f"positions_shape={tuple(positions.shape)}, "
                    f"tokens_shape={tuple(tokens.shape)}"
                )

            min_pos = int(positions.amin().item())
            max_pos = int(positions.amax().item())
            seq_len = max_pos + 1

            if min_pos < 0 or max_pos > 4096 or seq_len <= 0 or seq_len > 4097:
                raise RuntimeError(
                    f"Invalid RoPE positions: min={min_pos}, max={max_pos}, seq_len={seq_len}, "
                    f"positions_shape={tuple(positions.shape)}, dtype={positions.dtype}, "
                    f"tokens_shape={tuple(tokens.shape)}, tokens_dtype={tokens.dtype}, "
                    f"device={tokens.device}"
                )

            cos, sin = self.get_cos_sin(D, seq_len, tokens.device, tokens.dtype)

            y, x = tokens.chunk(2, dim=-1)
            y = self.apply_rope1d(y, positions[:, :, 0], cos, sin)
            x = self.apply_rope1d(x, positions[:, :, 1], cos, sin)
            tokens = torch.cat((y, x), dim=-1)
            return tokens


# # patch embedding
# class PositionGetter(object):
#     """ return positions of patches """

#     def __init__(self):
#         self.cache_positions = {}
        
#     def __call__(self, b, h, w, device):
#         if not (h,w) in self.cache_positions:
#             x = torch.arange(w, device=device)
#             y = torch.arange(h, device=device)
#             self.cache_positions[h,w] = torch.cartesian_prod(y, x) # (h, w, 2)
#         pos = self.cache_positions[h,w].view(1, h*w, 2).expand(b, -1, 2).clone()
#         return pos

class PositionGetter(object):
    """return positions of patches"""

    def __init__(self):
        self.cache_positions = {}

    def __call__(self, b, h, w, device):
        b = int(b)
        h = int(h)
        w = int(w)
        key = (h, w, str(device))

        if h <= 0 or w <= 0:
            raise RuntimeError(f"Invalid PositionGetter h/w: b={b}, h={h}, w={w}, device={device}")

        if key not in self.cache_positions:
            x = torch.arange(w, device=device, dtype=torch.long)
            y = torch.arange(h, device=device, dtype=torch.long)
            self.cache_positions[key] = torch.cartesian_prod(y, x).contiguous()

        pos = self.cache_positions[key].view(1, h * w, 2).expand(b, -1, 2).clone()

        if pos.shape != (b, h * w, 2):
            raise RuntimeError(
                f"Invalid PositionGetter output: "
                f"pos_shape={tuple(pos.shape)}, expected={(b, h*w, 2)}, "
                f"h={h}, w={w}, device={device}"
            )

        return pos

In [ ]:
%%writefile sih3d/vendor/pi3/layers/prope.py
# MIT License
#
# Copyright (c) Authors of
# "PRoPE: Projective Positional Encoding for Multiview Transformers"
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

# How to use PRoPE attention for self-attention:
# 
# 1. Easiest way (fast):
#    attn = PropeDotProductAttention(...)
#    o = attn(q, k, v, viewmats, Ks)
#
# 2. More flexible way (fast):
#    attn = PropeDotProductAttention(...)
#    attn._precompute_and_cache_apply_fns(viewmats, Ks)
#    q = attn._apply_to_q(q)
#    k = attn._apply_to_kv(k)
#    v = attn._apply_to_kv(v)
#    o = F.scaled_dot_product_attention(q, k, v, **kwargs)
#    o = attn._apply_to_o(o)
# 
# 3. The most flexible way (but slower because repeated computation of RoPE coefficients):
#    o = prope_dot_product_attention(q, k, v, ...)
# 
# How to use PRoPE attention for cross-attention:
# 
#    attn_src = PropeDotProductAttention(...)
#    attn_tgt = PropeDotProductAttention(...)
#    attn_src._precompute_and_cache_apply_fns(viewmats_src, Ks_src)
#    attn_tgt._precompute_and_cache_apply_fns(viewmats_tgt, Ks_tgt)
#    q_src = attn_src._apply_to_q(q_src)
#    k_tgt = attn_tgt._apply_to_kv(k_tgt)
#    v_tgt = attn_tgt._apply_to_kv(v_tgt)
#    o_src = F.scaled_dot_product_attention(q_src, k_tgt, v_tgt, **kwargs)
#    o_src = attn_src._apply_to_o(o_src)

from functools import partial
from typing import Callable, Optional, Tuple, List

import torch
import torch.nn.functional as F


class PropeDotProductAttention(torch.nn.Module):
    """PRoPE attention with precomputed RoPE coefficients."""

    coeffs_x_0: torch.Tensor
    coeffs_x_1: torch.Tensor
    coeffs_y_0: torch.Tensor
    coeffs_y_1: torch.Tensor

    def __init__(
        self,
        head_dim: int,
        patches_x: int,
        patches_y: int,
        image_width: int,
        image_height: int,
        freq_base: float = 100.0,
        freq_scale: float = 1.0,
    ):
        super().__init__()
        self.head_dim = head_dim
        self.patches_x = patches_x
        self.patches_y = patches_y
        self.image_width = image_width
        self.image_height = image_height

        coeffs_x: Tuple[torch.Tensor, torch.Tensor] = _rope_precompute_coeffs(
            torch.tile(torch.arange(patches_x), (patches_y,)),
            freq_base=freq_base,
            freq_scale=freq_scale,
            feat_dim=head_dim // 4,
        )
        coeffs_y: Tuple[torch.Tensor, torch.Tensor] = _rope_precompute_coeffs(
            torch.repeat_interleave(torch.arange(patches_y), patches_x),
            freq_base=freq_base,
            freq_scale=freq_scale,
            feat_dim=head_dim // 4,
        )
        # Do not save coeffs to checkpoint as `cameras` might change during testing.
        self.register_buffer("coeffs_x_0", coeffs_x[0], persistent=False)
        self.register_buffer("coeffs_x_1", coeffs_x[1], persistent=False)
        self.register_buffer("coeffs_y_0", coeffs_y[0], persistent=False)
        self.register_buffer("coeffs_y_1", coeffs_y[1], persistent=False)

    # override load_state_dict to not load coeffs if they exist (for backward compatibility)
    def load_state_dict(self, state_dict, strict=True):
        # remove coeffs from state_dict
        state_dict.pop("coeffs_x_0", None)
        state_dict.pop("coeffs_x_1", None)
        state_dict.pop("coeffs_y_0", None)
        state_dict.pop("coeffs_y_1", None)
        super().load_state_dict(state_dict, strict)

    def forward(
        self,
        q: torch.Tensor,  # (batch, num_heads, seqlen, head_dim)
        k: torch.Tensor,  # (batch, num_heads, seqlen, head_dim)
        v: torch.Tensor,  # (batch, num_heads, seqlen, head_dim)
        viewmats: torch.Tensor,  # (batch, cameras, 4, 4)
        Ks: Optional[torch.Tensor],  # (batch, cameras, 3, 3)
        **kwargs,
    ) -> torch.Tensor:
        return prope_dot_product_attention(
            q,
            k,
            v,
            viewmats=viewmats,
            Ks=Ks,
            patches_x=self.patches_x,
            patches_y=self.patches_y,
            image_width=self.image_width,
            image_height=self.image_height,
            coeffs_x=(self.coeffs_x_0, self.coeffs_x_1),
            coeffs_y=(self.coeffs_y_0, self.coeffs_y_1),
            **kwargs,
        )

    def _precompute_and_cache_apply_fns(
        self, viewmats: torch.Tensor, Ks: Optional[torch.Tensor]
    ):
        (batch, cameras, _, _) = viewmats.shape
        assert viewmats.shape == (batch, cameras, 4, 4)
        assert Ks is None or Ks.shape == (batch, cameras, 3, 3)
        self.cameras = cameras

        self.apply_fn_q, self.apply_fn_kv, self.apply_fn_o = _prepare_apply_fns(
            head_dim=self.head_dim,
            viewmats=viewmats,
            Ks=Ks,
            patches_x=self.patches_x,
            patches_y=self.patches_y,
            image_width=self.image_width,
            image_height=self.image_height,
            coeffs_x=(self.coeffs_x_0, self.coeffs_x_1),
            coeffs_y=(self.coeffs_y_0, self.coeffs_y_1),
        )

    def _apply_to_q(self, q: torch.Tensor) -> torch.Tensor:
        (batch, num_heads, seqlen, head_dim) = q.shape
        assert seqlen == self.cameras * self.patches_x * self.patches_y
        assert head_dim == self.head_dim
        assert q.shape == (batch, num_heads, seqlen, head_dim)
        assert self.apply_fn_q is not None
        return self.apply_fn_q(q)

    def _apply_to_kv(self, kv: torch.Tensor) -> torch.Tensor:
        (batch, num_heads, seqlen, head_dim) = kv.shape
        assert seqlen == self.cameras * self.patches_x * self.patches_y
        assert head_dim == self.head_dim
        assert kv.shape == (batch, num_heads, seqlen, head_dim)
        assert self.apply_fn_kv is not None
        return self.apply_fn_kv(kv)

    def _apply_to_o(self, o: torch.Tensor) -> torch.Tensor:
        (batch, num_heads, seqlen, head_dim) = o.shape
        assert seqlen == self.cameras * self.patches_x * self.patches_y
        assert head_dim == self.head_dim
        assert o.shape == (batch, num_heads, seqlen, head_dim)
        assert self.apply_fn_o is not None
        return self.apply_fn_o(o)


def prope_dot_product_attention(
    q: torch.Tensor,  # (batch, num_heads, seqlen, head_dim)
    k: torch.Tensor,  # (batch, num_heads, seqlen, head_dim)
    v: torch.Tensor,  # (batch, num_heads, seqlen, head_dim)
    *,
    viewmats: torch.Tensor,  # (batch, cameras, 4, 4)
    Ks: Optional[torch.Tensor],  # (batch, cameras, 3, 3)
    patches_x: int,  # How many patches wide is each image?
    patches_y: int,  # How many patches tall is each image?
    image_width: int,  # Width of the image. Used to normalize intrinsics.
    image_height: int,  # Height of the image. Used to normalize intrinsics.
    coeffs_x: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    coeffs_y: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    **kwargs,
) -> torch.Tensor:
    """Similar to torch.nn.functional.scaled_dot_product_attention, but applies PRoPE-style
    positional encoding.

    Currently, we assume that the sequence length is equal to:

        cameras * patches_x * patches_y

    And token ordering allows the `(seqlen,)` axis to be reshaped into
    `(cameras, patches_x, patches_y)`.
    """
    # We're going to assume self-attention: all inputs are the same shape.
    (batch, num_heads, seqlen, head_dim) = q.shape
    cameras = viewmats.shape[1]
    assert q.shape == k.shape == v.shape
    assert viewmats.shape == (batch, cameras, 4, 4)
    assert Ks is None or Ks.shape == (batch, cameras, 3, 3)
    assert seqlen == cameras * patches_x * patches_y

    apply_fn_q, apply_fn_kv, apply_fn_o = _prepare_apply_fns(
        head_dim=head_dim,
        viewmats=viewmats,
        Ks=Ks,
        patches_x=patches_x,
        patches_y=patches_y,
        image_width=image_width,
        image_height=image_height,
        coeffs_x=coeffs_x,
        coeffs_y=coeffs_y,
    )

    out = F.scaled_dot_product_attention(
        query=apply_fn_q(q),
        key=apply_fn_kv(k),
        value=apply_fn_kv(v),
        **kwargs,
    )
    out = apply_fn_o(out)
    assert out.shape == (batch, num_heads, seqlen, head_dim)
    return out


def _prepare_apply_fns(
    head_dim: int,  # Q/K/V will have this last dimension
    viewmats: torch.Tensor,  # (batch, cameras, 4, 4)
    Ks: Optional[torch.Tensor],  # (batch, cameras, 3, 3)
    patches_x: int,  # How many patches wide is each image?
    patches_y: int,  # How many patches tall is each image?
    image_width: int,  # Width of the image. Used to normalize intrinsics.
    image_height: int,  # Height of the image. Used to normalize intrinsics.
    coeffs_x: Optional[torch.Tensor] = None,
    coeffs_y: Optional[torch.Tensor] = None,
) -> Tuple[
    Callable[[torch.Tensor], torch.Tensor],
    Callable[[torch.Tensor], torch.Tensor],
    Callable[[torch.Tensor], torch.Tensor],
]:
    """Prepare transforms for PRoPE-style positional encoding."""
    device = viewmats.device
    (batch, cameras, _, _) = viewmats.shape

    # Normalize camera intrinsics.
    if Ks is not None:
        Ks_norm = torch.zeros_like(Ks)
        Ks_norm[..., 0, 0] = Ks[..., 0, 0] / image_width
        Ks_norm[..., 1, 1] = Ks[..., 1, 1] / image_height
        Ks_norm[..., 0, 2] = Ks[..., 0, 2] / image_width - 0.5
        Ks_norm[..., 1, 2] = Ks[..., 1, 2] / image_height - 0.5
        Ks_norm[..., 2, 2] = 1.0
        del Ks

        # Compute the camera projection matrices we use in PRoPE.
        # - K is an `image<-camera` transform.
        # - viewmats is a `camera<-world` transform.
        # - P = lift(K) @ viewmats is an `image<-world` transform.
        P = torch.einsum("...ij,...jk->...ik", _lift_K(Ks_norm), viewmats)
        P_T = P.transpose(-1, -2)
        P_inv = torch.einsum(
            "...ij,...jk->...ik",
            _invert_SE3(viewmats),
            _lift_K(_invert_K(Ks_norm)),
        )

    else:
        # GTA formula. P is `camera<-world` transform.
        P = viewmats
        P_T = P.transpose(-1, -2)
        P_inv = _invert_SE3(viewmats)

    assert P.shape == P_inv.shape == (batch, cameras, 4, 4)

    # Precompute cos/sin terms for RoPE. We use tiles/repeats for 'row-major'
    # broadcasting.
    if coeffs_x is None:
        coeffs_x = _rope_precompute_coeffs(
            torch.tile(torch.arange(patches_x, device=device), (patches_y * cameras,)),
            freq_base=100.0,
            freq_scale=1.0,
            feat_dim=head_dim // 4,
        )
    if coeffs_y is None:
        coeffs_y = _rope_precompute_coeffs(
            torch.tile(
                torch.repeat_interleave(
                    torch.arange(patches_y, device=device), patches_x
                ),
                (cameras,),
            ),
            freq_base=100.0,
            freq_scale=1.0,
            feat_dim=head_dim // 4,
        )

    # Block-diagonal transforms to the inputs and outputs of the attention operator.
    assert head_dim % 4 == 0
    transforms_q = [
        (partial(_apply_tiled_projmat, matrix=P_T), head_dim // 2),
        (partial(_rope_apply_coeffs, coeffs=coeffs_x), head_dim // 4),
        (partial(_rope_apply_coeffs, coeffs=coeffs_y), head_dim // 4),
    ]
    transforms_kv = [
        (partial(_apply_tiled_projmat, matrix=P_inv), head_dim // 2),
        (partial(_rope_apply_coeffs, coeffs=coeffs_x), head_dim // 4),
        (partial(_rope_apply_coeffs, coeffs=coeffs_y), head_dim // 4),
    ]
    transforms_o = [
        (partial(_apply_tiled_projmat, matrix=P), head_dim // 2),
        (partial(_rope_apply_coeffs, coeffs=coeffs_x, inverse=True), head_dim // 4),
        (partial(_rope_apply_coeffs, coeffs=coeffs_y, inverse=True), head_dim // 4),
    ]

    apply_fn_q = partial(_apply_block_diagonal, func_size_pairs=transforms_q)
    apply_fn_kv = partial(_apply_block_diagonal, func_size_pairs=transforms_kv)
    apply_fn_o = partial(_apply_block_diagonal, func_size_pairs=transforms_o)
    return apply_fn_q, apply_fn_kv, apply_fn_o


def _apply_tiled_projmat(
    feats: torch.Tensor,  # (batch, num_heads, seqlen, feat_dim)
    matrix: torch.Tensor,  # (batch, cameras, D, D)
) -> torch.Tensor:
    """Apply projection matrix to features."""
    # - seqlen => (cameras, patches_x * patches_y)
    # - feat_dim => (feat_dim // 4, 4)
    (batch, num_heads, seqlen, feat_dim) = feats.shape
    cameras = matrix.shape[1]
    assert seqlen > cameras and seqlen % cameras == 0
    D = matrix.shape[-1]
    assert matrix.shape == (batch, cameras, D, D)
    assert feat_dim % D == 0
    return torch.einsum(
        "bcij,bncpkj->bncpki",
        matrix,
        feats.reshape((batch, num_heads, cameras, -1, feat_dim // D, D)),
    ).reshape(feats.shape)


def _rope_precompute_coeffs(
    positions: torch.Tensor,  # (seqlen,)
    freq_base: float,
    freq_scale: float,
    feat_dim: int,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Precompute RoPE coefficients."""
    assert len(positions.shape) == 1
    assert feat_dim % 2 == 0
    num_freqs = feat_dim // 2
    freqs = freq_scale * (
        freq_base
        ** (
            -torch.arange(num_freqs, device=positions.device)[None, None, None, :]
            / num_freqs
        )
    )
    angles = positions[None, None, :, None] * freqs
    # Shape should be: `(batch, num_heads, seqlen, num_freqs)`; we're
    # broadcasting across `batch` and `num_heads`.
    assert angles.shape == (1, 1, positions.shape[0], num_freqs)
    return torch.cos(angles), torch.sin(angles)


def _rope_apply_coeffs(
    feats: torch.Tensor,  # (batch, num_heads, seqlen, feat_dim)
    coeffs: Tuple[torch.Tensor, torch.Tensor],
    inverse: bool = False,
) -> torch.Tensor:
    """Apply RoPE coefficients to features. We adopt a 'split' ordering
    convention. (in contrast to 'interleaved')"""
    cos, sin = coeffs
    # We allow (cos, sin) to be either with shape (1, 1, seqlen, feat_dim // 2),
    # or (1, 1, seqlen_per_image, feat_dim // 2) and we repeat it to
    # match the shape of feats.
    if cos.shape[2] != feats.shape[2]:
        n_repeats = feats.shape[2] // cos.shape[2]
        cos = cos.repeat(1, 1, n_repeats, 1)
        sin = sin.repeat(1, 1, n_repeats, 1)
    assert len(feats.shape) == len(cos.shape) == len(sin.shape) == 4
    assert cos.shape[-1] == sin.shape[-1] == feats.shape[-1] // 2
    x_in = feats[..., : feats.shape[-1] // 2]
    y_in = feats[..., feats.shape[-1] // 2 :]
    return torch.cat(
        (
            [cos * x_in + sin * y_in, -sin * x_in + cos * y_in]
            if not inverse
            else [cos * x_in - sin * y_in, sin * x_in + cos * y_in]
        ),
        dim=-1,
    )


def _apply_block_diagonal(
    feats: torch.Tensor,  # (..., dim)
    func_size_pairs: List[Tuple[Callable[[torch.Tensor], torch.Tensor], int]],
) -> torch.Tensor:
    """Apply a block-diagonal function to an input array.

    Each function is specified as a tuple with form:

        ((Tensor) -> Tensor, int)

    Where the integer is the size of the input to the function.
    """
    funcs, block_sizes = zip(*func_size_pairs)
    assert feats.shape[-1] == sum(block_sizes)
    x_blocks = torch.split(feats, block_sizes, dim=-1)
    out = torch.cat(
        [f(x_block) for f, x_block in zip(funcs, x_blocks)],
        dim=-1,
    )
    assert out.shape == feats.shape, "Input/output shapes should match."
    return out


def _invert_SE3(transforms: torch.Tensor) -> torch.Tensor:
    """Invert a 4x4 SE(3) matrix."""
    assert transforms.shape[-2:] == (4, 4)
    Rinv = transforms[..., :3, :3].transpose(-1, -2)
    out = torch.zeros_like(transforms)
    out[..., :3, :3] = Rinv
    out[..., :3, 3] = -torch.einsum("...ij,...j->...i", Rinv, transforms[..., :3, 3])
    out[..., 3, 3] = 1.0
    return out


def _lift_K(Ks: torch.Tensor) -> torch.Tensor:
    """Lift 3x3 matrices to homogeneous 4x4 matrices."""
    assert Ks.shape[-2:] == (3, 3)
    out = torch.zeros(Ks.shape[:-2] + (4, 4), device=Ks.device)
    out[..., :3, :3] = Ks
    out[..., 3, 3] = 1.0
    return out


def _invert_K(Ks: torch.Tensor) -> torch.Tensor:
    """Invert 3x3 intrinsics matrices. Assumes no skew."""
    assert Ks.shape[-2:] == (3, 3)
    out = torch.zeros_like(Ks)
    out[..., 0, 0] = 1.0 / Ks[..., 0, 0]
    out[..., 1, 1] = 1.0 / Ks[..., 1, 1]
    out[..., 0, 2] = -Ks[..., 0, 2] / Ks[..., 0, 0]
    out[..., 1, 2] = -Ks[..., 1, 2] / Ks[..., 1, 1]
    out[..., 2, 2] = 1.0
    return out

def _prepare_apply_fns_query(
    head_dim: int,  # Q/K/V will have this last dimension
    viewmats_src: torch.Tensor,  # (batch, cameras, 4, 4)
    viewmats_query: torch.Tensor,  # (batch, cameras, 4, 4)
    Ks_src: Optional[torch.Tensor],  # (batch, cameras, 3, 3)
    Ks_query: Optional[torch.Tensor],  # (batch, cameras, 3, 3)
    patches_x: int,  # How many patches wide is each image?
    patches_y: int,  # How many patches tall is each image?
    image_width: int,  # Width of the image. Used to normalize intrinsics.
    image_height: int,  # Height of the image. Used to normalize intrinsics.
    coeffs_x: Optional[torch.Tensor] = None,
    coeffs_y: Optional[torch.Tensor] = None,
) -> Tuple[
    Callable[[torch.Tensor], torch.Tensor],
    Callable[[torch.Tensor], torch.Tensor],
    Callable[[torch.Tensor], torch.Tensor],
]:
    """Prepare transforms for PRoPE-style positional encoding."""
    device = viewmats_src.device
    (batch, cameras_src, _, _) = viewmats_src.shape
    (batch, cameras_query, _, _) = viewmats_query.shape

    # Normalize camera intrinsics.
    if Ks_src is not None:
        Ks_src_norm = torch.zeros_like(Ks_src)
        Ks_src_norm[..., 0, 0] = Ks_src[..., 0, 0] / image_width
        Ks_src_norm[..., 1, 1] = Ks_src[..., 1, 1] / image_height
        Ks_src_norm[..., 0, 2] = Ks_src[..., 0, 2] / image_width - 0.5
        Ks_src_norm[..., 1, 2] = Ks_src[..., 1, 2] / image_height - 0.5
        Ks_src_norm[..., 2, 2] = 1.0
        del Ks_src

        Ks_query_norm = torch.zeros_like(Ks_query)
        Ks_query_norm[..., 0, 0] = Ks_query[..., 0, 0] / image_width
        Ks_query_norm[..., 1, 1] = Ks_query[..., 1, 1] / image_height
        Ks_query_norm[..., 0, 2] = Ks_query[..., 0, 2] / image_width - 0.5
        Ks_query_norm[..., 1, 2] = Ks_query[..., 1, 2] / image_height - 0.5
        Ks_query_norm[..., 2, 2] = 1.0
        del Ks_query

        # Compute the camera projection matrices we use in PRoPE.
        # - K is an `image<-camera` transform.
        # - viewmats is a `camera<-world` transform.
        # - P = lift(K) @ viewmats is an `image<-world` transform.
        P_src = torch.einsum("...ij,...jk->...ik", _lift_K(Ks_src_norm), viewmats_src)
        # P_src_T = P_src.transpose(-1, -2)
        P_src_inv = torch.einsum(
            "...ij,...jk->...ik",
            _invert_SE3(viewmats_src),
            _lift_K(_invert_K(Ks_src_norm)),
        )

        P_query = torch.einsum("...ij,...jk->...ik", _lift_K(Ks_query_norm), viewmats_query)
        P_query_T = P_query.transpose(-1, -2)
        # P_query_inv = torch.einsum(
        #     "...ij,...jk->...ik",
        #     _invert_SE3(viewmats_query),
        #     _lift_K(_invert_K(Ks_query_norm)),
        # )

    else:
        # GTA formula. P is `camera<-world` transform.
        P_src = viewmats_src
        # P_src_T = P_src.transpose(-1, -2)
        P_src_inv = _invert_SE3(viewmats_src)

        P_query = viewmats_query
        P_query_T = P_query.transpose(-1, -2)
        # P_query_inv = _invert_SE3(viewmats_query)

    # Precompute cos/sin terms for RoPE. We use tiles/repeats for 'row-major'
    # broadcasting.
    # 1. 为 Query (Q) 和 Output (O) 创建 2D RoPE
    if coeffs_x is None: # (假设 coeffs_x 和 coeffs_y 是 Q 的)
        coeffs_x_q = _rope_precompute_coeffs(
            torch.tile(torch.arange(patches_x, device=device), (patches_y * cameras_query,)),
            freq_base=100.0,
            freq_scale=1.0,
            feat_dim=head_dim // 4,
        )
    else:
        coeffs_x_q = coeffs_x

    if coeffs_y is None: # (假设 coeffs_x 和 coeffs_y 是 Q 的)
        coeffs_y_q = _rope_precompute_coeffs(
            torch.tile(
                torch.repeat_interleave(
                    torch.arange(patches_y, device=device), patches_x
                ),
                (cameras_query,),
            ),
            freq_base=100.0,
            freq_scale=1.0,
            feat_dim=head_dim // 4,
        )
    else:
        coeffs_y_q = coeffs_y

    # 2. 为 Key (K) 和 Value (V) 创建 2D RoPE
    # (这里我们假设 K/V 的 RoPE 总是需要新计算，或者您需要
    # 额外传入 coeffs_x_s, coeffs_y_s)
    coeffs_x_s = _rope_precompute_coeffs(
        torch.tile(torch.arange(patches_x, device=device), (patches_y * cameras_src,)),
        freq_base=100.0,
        freq_scale=1.0,
        feat_dim=head_dim // 4,
    )
    coeffs_y_s = _rope_precompute_coeffs(
        torch.tile(
            torch.repeat_interleave(
                torch.arange(patches_y, device=device), patches_x
            ),
            (cameras_src,),
        ),
        freq_base=100.0,
        freq_scale=1.0,
        feat_dim=head_dim // 4,
    )

    # Block-diagonal transforms to the inputs and outputs of the attention operator.
    assert head_dim % 4 == 0
    transforms_q = [
        (partial(_apply_tiled_projmat, matrix=P_query_T), head_dim // 2),
        (partial(_rope_apply_coeffs, coeffs=coeffs_x_q), head_dim // 4),
        (partial(_rope_apply_coeffs, coeffs=coeffs_y_q), head_dim // 4),
    ]
    transforms_kv = [
        (partial(_apply_tiled_projmat, matrix=P_src_inv), head_dim // 2),
        (partial(_rope_apply_coeffs, coeffs=coeffs_x_s), head_dim // 4),
        (partial(_rope_apply_coeffs, coeffs=coeffs_y_s), head_dim // 4),
    ]
    transforms_o = [
        (partial(_apply_tiled_projmat, matrix=P_query), head_dim // 2),
        (partial(_rope_apply_coeffs, coeffs=coeffs_x_q, inverse=True), head_dim // 4),
        (partial(_rope_apply_coeffs, coeffs=coeffs_y_q, inverse=True), head_dim // 4),
    ]

    apply_fn_q = partial(_apply_block_diagonal, func_size_pairs=transforms_q)
    apply_fn_kv = partial(_apply_block_diagonal, func_size_pairs=transforms_kv)
    apply_fn_o = partial(_apply_block_diagonal, func_size_pairs=transforms_o)
    return apply_fn_q, apply_fn_kv, apply_fn_o

In [ ]:
%%writefile sih3d/vendor/pi3/layers/transformer_head.py
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

from sih3d.vendor.dinov2.layers import Mlp
from sih3d.vendor.pi3.layers.attention import FlashAttentionRope, FlashCrossAttentionRope
from sih3d.vendor.pi3.layers.block import BlockRope, CrossOnlyBlockRope
   
class TransformerDecoder(nn.Module):
    def __init__(
        self,
        in_dim,
        out_dim,
        dec_embed_dim=512,
        depth=5,
        dec_num_heads=8,
        mlp_ratio=4,
        rope=None,
        need_project=True,
        use_checkpoint=False,
    ):
        super().__init__()

        self.projects = nn.Linear(in_dim, dec_embed_dim) if need_project else nn.Identity()
        self.use_checkpoint = use_checkpoint

        self.blocks = nn.ModuleList([
            BlockRope(
                dim=dec_embed_dim,
                num_heads=dec_num_heads,
                mlp_ratio=mlp_ratio,
                qkv_bias=True,
                proj_bias=True,
                ffn_bias=True,
                drop_path=0.0,
                norm_layer=partial(nn.LayerNorm, eps=1e-6),
                act_layer=nn.GELU,
                ffn_layer=Mlp,
                init_values=None,
                qk_norm=False,
                # attn_class=MemEffAttentionRope,
                attn_class=FlashAttentionRope,
                rope=rope
            ) for _ in range(depth)])

        self.linear_out = nn.Linear(dec_embed_dim, out_dim)

    def forward(self, hidden, xpos=None):
        hidden = self.projects(hidden)
        # for i, blk in enumerate(self.blocks):
        #     if self.use_checkpoint and self.training:
        #         hidden = checkpoint(blk, hidden, xpos=xpos, use_reentrant=False)
        #     else:
        #         hidden = blk(hidden, xpos=xpos)

        if xpos is not None:
            xpos = xpos.to(device=hidden.device, dtype=torch.long).contiguous().detach().clone()

        for i, blk in enumerate(self.blocks):
            if self.use_checkpoint and self.training:
                def run_blk(x, xpos_i, blk=blk):
                    return blk(x, xpos=xpos_i)
                hidden = checkpoint(run_blk, hidden, xpos, use_reentrant=False)
            else:
                hidden = blk(hidden, xpos=xpos)


        out = self.linear_out(hidden)
        return out

class LinearPts3d (nn.Module):
    """ 
    Linear head for dust3r
    Each token outputs: - 16x16 3D points (+ confidence)
    """

    def __init__(self, patch_size, dec_embed_dim, output_dim=3,):
        super().__init__()
        self.patch_size = patch_size

        self.proj = nn.Linear(dec_embed_dim, (output_dim)*self.patch_size**2)

    def forward(self, decout, img_shape):
        H, W = img_shape
        tokens = decout[-1]
        B, S, D = tokens.shape

        # extract 3D points
        feat = self.proj(tokens)  # B,S,D
        feat = feat.transpose(-1, -2).view(B, -1, H//self.patch_size, W//self.patch_size)
        feat = F.pixel_shuffle(feat, self.patch_size)  # B,3,H,W

        # permute + norm depth
        return feat.permute(0, 2, 3, 1)
    


class ContextOnlyTransformerDecoder(nn.Module):
    def __init__(
        self,
        in_dim,
        out_dim,
        dec_embed_dim=512,
        depth=5,
        dec_num_heads=8,
        mlp_ratio=4,
        rope=None,
        prenorm=False,
        use_checkpoint=True,
    ):
        super().__init__()

        if prenorm:
            self.pre_norm = nn.LayerNorm(in_dim)
        else:
            self.pre_norm = None

        self.projects_x = nn.Linear(in_dim, dec_embed_dim)
        self.projects_y = nn.Linear(in_dim, dec_embed_dim)
        self.use_checkpoint = use_checkpoint

        self.blocks = nn.ModuleList([
            CrossOnlyBlockRope(
                dim=dec_embed_dim,
                num_heads=dec_num_heads,
                mlp_ratio=mlp_ratio,
                qkv_bias=True,
                proj_bias=True,
                ffn_bias=True,
                norm_layer=partial(nn.LayerNorm, eps=1e-6),
                act_layer=nn.GELU,
                ffn_layer=Mlp,
                init_values=None,
                qk_norm=False,
                cross_attn_class=FlashCrossAttentionRope,
                rope=rope
            ) for _ in range(depth)])

        self.linear_out = nn.Linear(dec_embed_dim, out_dim)

    def forward(self, hidden, context, xpos=None, ypos=None):
        if self.pre_norm is not None:
            hidden = self.pre_norm(hidden)
            context = self.pre_norm(context)

        hidden = self.projects_x(hidden)
        context = self.projects_y(context)

        # for i, blk in enumerate(self.blocks):
        #     if self.use_checkpoint and self.training:
        #         hidden = checkpoint(blk, hidden, context, xpos=xpos, ypos=ypos, use_reentrant=False)
        #     else:
        #         hidden = blk(hidden, context, xpos=xpos, ypos=ypos)
        
        if xpos is not None:
            xpos = xpos.to(device=hidden.device, dtype=torch.long).contiguous().detach().clone()
        if ypos is not None:
            ypos = ypos.to(device=hidden.device, dtype=torch.long).contiguous().detach().clone()

        for i, blk in enumerate(self.blocks):
            if self.use_checkpoint and self.training:
                def run_blk(x, ctx, xpos_i, ypos_i, blk=blk):
                    return blk(x, ctx, xpos=xpos_i, ypos=ypos_i)

                hidden = checkpoint(
                    run_blk,
                    hidden,
                    context,
                    xpos,
                    ypos,
                    use_reentrant=False,
                )
            else:
                hidden = blk(hidden, context, xpos=xpos, ypos=ypos)

        out = self.linear_out(hidden)
        return out
    

In [ ]:
%%writefile sih3d/vendor/pi3/models/__init__.py



In [ ]:
%%writefile sih3d/vendor/pi3/models/pi3.py
from copy import deepcopy
from functools import partial

import torch
import torch.nn as nn
from huggingface_hub import PyTorchModelHubMixin
from torch.utils.checkpoint import checkpoint

from sih3d.vendor.dinov2.hub.backbones import dinov2_vitl14_reg
from sih3d.vendor.dinov2.layers import Mlp
from sih3d.vendor.pi3.layers.attention import FlashAttentionRope
from sih3d.vendor.pi3.layers.block import BlockRope
from sih3d.vendor.pi3.layers.camera_head import CameraHead
from sih3d.vendor.pi3.layers.pos_embed import PositionGetter, RoPE2D
from sih3d.vendor.pi3.layers.transformer_head import (
    LinearPts3d,
    TransformerDecoder,
)


def homogenize_points(
    points,
):
    """Convert batched points (xyz) to (xyz1)."""
    return torch.cat([points, torch.ones_like(points[..., :1])], dim=-1)


class Pi3(nn.Module, PyTorchModelHubMixin):
    def __init__(
        self,
        pos_type="rope100",
        decoder_size="large",
        gradient_checkpointing=False,
        num_dec_blk_not_to_checkpoint=4,
    ):
        super().__init__()

        self.gradient_checkpointing = gradient_checkpointing
        self.num_dec_blk_not_to_checkpoint = num_dec_blk_not_to_checkpoint

        # ----------------------
        #        Encoder
        # ----------------------
        self.encoder = dinov2_vitl14_reg(pretrained=False)
        self.patch_size = 14
        del self.encoder.mask_token

        # Wrap encoder blocks with checkpointing
        if self.gradient_checkpointing:
            for i in range(len(self.encoder.blocks)):
                self.encoder.blocks[i] = self.wrap_module_with_gradient_checkpointing(
                    self.encoder.blocks[i]
                )

        # ----------------------
        #  Positonal Encoding
        # ----------------------
        self.pos_type = pos_type if pos_type is not None else "none"
        self.rope = None
        if self.pos_type.startswith("rope"):  # eg rope100
            if RoPE2D is None:
                raise ImportError(
                    "Cannot find cuRoPE2D, please install it following the README instructions"
                )
            freq = float(self.pos_type[len("rope") :])
            self.rope = RoPE2D(freq=freq)
            self.position_getter = PositionGetter()
        else:
            raise NotImplementedError

        # ----------------------
        #        Decoder
        # ----------------------
        if decoder_size == "small":
            dec_embed_dim = 384
            dec_num_heads = 6
            mlp_ratio = 4
            dec_depth = 24
        elif decoder_size == "base":
            dec_embed_dim = 768
            dec_num_heads = 12
            mlp_ratio = 4
            dec_depth = 24
        elif decoder_size == "large":
            dec_embed_dim = 1024
            dec_num_heads = 16
            mlp_ratio = 4
            dec_depth = 36
        else:
            raise NotImplementedError
        self.decoder = nn.ModuleList(
            [
                BlockRope(
                    dim=dec_embed_dim,
                    num_heads=dec_num_heads,
                    mlp_ratio=mlp_ratio,
                    qkv_bias=True,
                    proj_bias=True,
                    ffn_bias=True,
                    drop_path=0.0,
                    norm_layer=partial(nn.LayerNorm, eps=1e-6),
                    act_layer=nn.GELU,
                    ffn_layer=Mlp,
                    init_values=0.01,
                    qk_norm=True,
                    attn_class=FlashAttentionRope,
                    rope=self.rope,
                )
                for _ in range(dec_depth)
            ]
        )
        self.dec_embed_dim = dec_embed_dim

        # ----------------------
        #     Register_token
        # ----------------------
        num_register_tokens = 5
        self.patch_start_idx = num_register_tokens
        self.register_token = nn.Parameter(
            torch.randn(1, 1, num_register_tokens, self.dec_embed_dim)
        )
        nn.init.normal_(self.register_token, std=1e-6)

        # ----------------------
        #  Local Points Decoder
        # ----------------------
        self.point_decoder = TransformerDecoder(
            in_dim=2 * self.dec_embed_dim,
            dec_embed_dim=1024,
            dec_num_heads=16,
            out_dim=1024,
            rope=self.rope,
            use_checkpoint=gradient_checkpointing,
            # use_checkpoint=False,
        )
        self.point_head = LinearPts3d(patch_size=14, dec_embed_dim=1024, output_dim=3)

        # ----------------------
        #     Conf Decoder
        # ----------------------
        self.conf_decoder = deepcopy(self.point_decoder)
        self.conf_head = LinearPts3d(patch_size=14, dec_embed_dim=1024, output_dim=1)

        # ----------------------
        #  Camera Pose Decoder
        # ----------------------
        self.camera_decoder = TransformerDecoder(
            in_dim=2 * self.dec_embed_dim,
            dec_embed_dim=1024,
            dec_num_heads=16,  # 8
            out_dim=512,
            rope=self.rope,
            use_checkpoint=gradient_checkpointing,
            # use_checkpoint=False,
        )
        self.camera_head = CameraHead(dim=512)

        # For ImageNet Normalize
        image_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        image_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

        self.register_buffer("image_mean", image_mean)
        self.register_buffer("image_std", image_std)
    
    def wrap_module_with_gradient_checkpointing(self, module: nn.Module):
        class _CheckpointingWrapper(module.__class__):
            _restore_cls = module.__class__
            def forward(self, *args, **kwargs):
                return checkpoint(super().forward, *args, use_reentrant=False, **kwargs)
        module.__class__ = _CheckpointingWrapper
        return module

    def decode(self, hidden, N, H, W):
        BN, hw, _ = hidden.shape
        B = BN // N

        final_output = []

        hidden = hidden.reshape(B * N, hw, -1)

        register_token = self.register_token.repeat(B, N, 1, 1).reshape(
            B * N, *self.register_token.shape[-2:]
        )

        # Concatenate special tokens with patch tokens
        hidden = torch.cat([register_token, hidden], dim=1)
        hw = hidden.shape[1]

        if self.pos_type.startswith("rope"):
            pos = self.position_getter(
                B * N, H // self.patch_size, W // self.patch_size, hidden.device
            )

        if self.patch_start_idx > 0:
            # do not use position embedding for special tokens (camera and register tokens)
            # so set pos to 0 for the special tokens
            pos = pos + 1
            pos_special = (
                torch.zeros(B * N, self.patch_start_idx, 2)
                .to(hidden.device)
                .to(pos.dtype)
            )
            pos = torch.cat([pos_special, pos], dim=1)

        for i in range(len(self.decoder)):
            blk = self.decoder[i]

            if i % 2 == 0:
                pos = pos.reshape(B * N, hw, -1)
                hidden = hidden.reshape(B * N, hw, -1)
            else:
                pos = pos.reshape(B, N * hw, -1)
                hidden = hidden.reshape(B, N * hw, -1)

            if i >= self.num_dec_blk_not_to_checkpoint and self.training and self.gradient_checkpointing:
                hidden = checkpoint(blk, hidden, xpos=pos, use_reentrant=False)
            else:
                hidden = blk(hidden, xpos=pos)
            
            if i + 1 in [len(self.decoder) - 1, len(self.decoder)]:
                final_output.append(hidden.reshape(B * N, hw, -1))

        return torch.cat([final_output[0], final_output[1]], dim=-1), pos.reshape(
            B * N, hw, -1
        )

    def forward(self, imgs):
        imgs = (imgs - self.image_mean) / self.image_std

        B, N, _, H, W = imgs.shape
        patch_h, patch_w = H // 14, W // 14

        # encode by dinov2
        imgs = imgs.reshape(B * N, _, H, W)
        hidden = self.encoder(imgs, is_training=True)

        if isinstance(hidden, dict):
            hidden = hidden["x_norm_patchtokens"]

        hidden, pos = self.decode(hidden, N, H, W)

        point_hidden = self.point_decoder(hidden, xpos=pos)
        conf_hidden = self.conf_decoder(hidden, xpos=pos)
        camera_hidden = self.camera_decoder(hidden, xpos=pos)

        with torch.amp.autocast(device_type="cuda", enabled=False):
            # local points
            point_hidden = point_hidden.float()
            ret = self.point_head(
                [point_hidden[:, self.patch_start_idx :]], (H, W)
            ).reshape(B, N, H, W, -1)
            xy, z = ret.split([2, 1], dim=-1)
            z = torch.exp(z)
            # z = torch.exp(z.clamp(max=15.0))
            local_points = torch.cat([xy * z, z], dim=-1)

            # confidence
            conf_hidden = conf_hidden.float()
            conf = self.conf_head(
                [conf_hidden[:, self.patch_start_idx :]], (H, W)
            ).reshape(B, N, H, W, -1)

            # camera
            camera_hidden = camera_hidden.float()
            camera_poses = self.camera_head(
                camera_hidden[:, self.patch_start_idx :], patch_h, patch_w
            ).reshape(B, N, 4, 4)

            # unproject local points using camera poses
            points = torch.einsum(
                "bnij, bnhwj -> bnhwi", camera_poses, homogenize_points(local_points)
            )[..., :3]

        return dict(
            points=points,
            local_points=local_points,
            conf=conf,
            camera_poses=camera_poses,
        )


In [ ]:
%%writefile sih3d/vendor/pi3/models/pi3x.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial
from copy import deepcopy
from huggingface_hub import PyTorchModelHubMixin
import numpy as np
from torch.utils.checkpoint import checkpoint

from sih3d.vendor.dinov2.hub.backbones import dinov2_vitl14_reg
from sih3d.vendor.pi3.layers.pos_embed import PositionGetter, RoPE2D
from sih3d.vendor.pi3.layers.block import BlockRope, PoseInjectBlock
from sih3d.vendor.pi3.layers.attention import FlashAttentionRope
from sih3d.vendor.dinov2.layers import Mlp, PatchEmbed
from sih3d.vendor.pi3.layers.camera_head import CameraHead
from sih3d.vendor.pi3.layers.conv_head import ConvHead
from sih3d.vendor.pi3.layers.transformer_head import TransformerDecoder, ContextOnlyTransformerDecoder

def se3_inverse(T):
    """
    Computes the inverse of a batch of SE(3) matrices.
    """

    if torch.is_tensor(T):
        R = T[..., :3, :3]
        t = T[..., :3, 3].unsqueeze(-1)
        R_inv = R.transpose(-2, -1)
        t_inv = -torch.matmul(R_inv, t)
        T_inv = torch.cat([
            torch.cat([R_inv, t_inv], dim=-1),
            torch.tensor([0, 0, 0, 1], device=T.device, dtype=T.dtype).repeat(*T.shape[:-2], 1, 1)
        ], dim=-2)
    else:
        R = T[..., :3, :3]
        t = T[..., :3, 3, np.newaxis]

        R_inv = np.swapaxes(R, -2, -1)
        t_inv = -R_inv @ t

        bottom_row = np.zeros((*T.shape[:-2], 1, 4), dtype=T.dtype)
        bottom_row[..., :, 3] = 1

        top_part = np.concatenate([R_inv, t_inv], axis=-1)
        T_inv = np.concatenate([top_part, bottom_row], axis=-2)
    return T_inv


def get_pixel(H, W):
    # get 2D pixels (u, v) for image_a in cam_a pixel space
    u_a, v_a = np.meshgrid(np.arange(W), np.arange(H))
    # u_a = np.flip(u_a, axis=1)
    # v_a = np.flip(v_a, axis=0)
    pixels_a = np.stack([
        u_a.flatten() + 0.5, 
        v_a.flatten() + 0.5, 
        np.ones_like(u_a.flatten())
    ], axis=0)
    return pixels_a


def homogenize_points(
    points,
):
    """Convert batched points (xyz) to (xyz1)."""
    return torch.cat([points, torch.ones_like(points[..., :1])], dim=-1)



class Pi3X(nn.Module, PyTorchModelHubMixin):
    def __init__(
            self,
            ckpt=None,    
            use_multimodal=True,
            gradient_checkpointing=False,
            checkpoint_strategy="all",  # "all" or "global_only"
        ):
        super().__init__()

        self.use_multimodal = use_multimodal
        self.gradient_checkpointing = gradient_checkpointing
        self.checkpoint_strategy = checkpoint_strategy

        # ----------------------
        #        Encoder
        # ----------------------
        self.encoder = dinov2_vitl14_reg(pretrained=False)
        self.patch_size = 14
        del self.encoder.mask_token

        # Wrap encoder blocks with checkpointing
        if self.gradient_checkpointing:
            for i in range(len(self.encoder.blocks)):
                self.encoder.blocks[i] = self.wrap_module_with_gradient_checkpointing(
                    self.encoder.blocks[i]
                )

        # ----------------------
        #  Positonal Encoding
        # ----------------------
        freq = 100
        self.rope = RoPE2D(freq=freq)
        self.position_getter = PositionGetter()

        # ----------------------
        #        Decoder
        # ----------------------
        dec_embed_dim = 1024
        dec_num_heads = 16
        mlp_ratio = 4
        dec_depth = 36      
        self.decoder = nn.ModuleList([
            BlockRope(
                dim=dec_embed_dim,
                num_heads=dec_num_heads,
                mlp_ratio=mlp_ratio,
                qkv_bias=True,
                proj_bias=True,
                ffn_bias=True,
                drop_path=0.0,
                norm_layer=partial(nn.LayerNorm, eps=1e-6),
                act_layer=nn.GELU,
                ffn_layer=Mlp,
                init_values=0.01,
                qk_norm=True,
                attn_class=FlashAttentionRope,
                rope=self.rope
        ) for _ in range(dec_depth)])
        self.dec_embed_dim = dec_embed_dim

        num_register_tokens = 5
        self.patch_start_idx = num_register_tokens
        self.register_token = nn.Parameter(torch.randn(1, 1, num_register_tokens, self.dec_embed_dim))
        nn.init.normal_(self.register_token, std=1e-6)

        # -----------------------
        #       multi-modal
        # -----------------------
        if use_multimodal:
            ## Depth encoder
            self.depth_encoder = deepcopy(self.encoder)
            del self.depth_encoder.patch_embed
            self.depth_encoder.patch_embed = PatchEmbed(img_size=224, patch_size=14, in_chans=2, embed_dim=1024)
            self.depth_emb = nn.Parameter(torch.zeros(1, 1, 1024))

            ## Ray embedding
            self.ray_embed = PatchEmbed(img_size=224, patch_size=14, in_chans=2, embed_dim=1024)
            nn.init.constant_(self.ray_embed.proj.weight, 0)
            nn.init.constant_(self.ray_embed.proj.bias, 0)

            ## Pose inject blocks
            self.pose_inject_blk = nn.ModuleList([PoseInjectBlock(
                dim=1024,
                num_heads=16,
                mlp_ratio=4,
                qkv_bias=True,
                proj_bias=True,
                ffn_bias=True,
                drop_path=0.0,
                norm_layer=partial(nn.LayerNorm, eps=1e-6),
                act_layer=nn.GELU,
                ffn_layer=Mlp,
                init_values=0.01,
                qk_norm=True,
            ) for _ in range(5)])


        # ------------------------------
        #           Head
        # ------------------------------
        ## --------------- Point ---------------
        self.point_decoder = TransformerDecoder(
            in_dim=2*self.dec_embed_dim, 
            dec_embed_dim=1024,
            dec_num_heads=16,                # 8
            out_dim=1024,
            rope=self.rope,
            use_checkpoint=self.gradient_checkpointing
            # use_checkpoint=False,
        )
        # self.point_head = LinearPts3d(patch_size=14, dec_embed_dim=1024, output_dim=3)
        self.point_head = ConvHead(
                num_features=4, 
                dim_in=dec_embed_dim,
                # projects=nn.Linear(1024, 1024),
                projects=nn.Identity(),
                dim_out=[2, 1], 
                dim_proj=1024,
                dim_upsample=[256, 128, 64],
                dim_times_res_block_hidden=2,
                num_res_blocks=2,
                res_block_norm='group_norm',
                last_res_blocks=0,
                last_conv_channels=32,
                last_conv_size=1,
                using_uv=True
            )

        ## --------------- Camera ---------------
        self.camera_decoder = TransformerDecoder(
            in_dim=2*self.dec_embed_dim, 
            dec_embed_dim=1024,
            dec_num_heads=16,                # 8
            out_dim=512,
            rope=self.rope,
            # use_checkpoint=self.gradient_checkpointing
            use_checkpoint=False,
        )
        self.camera_head = CameraHead(dim=512)

        ## --------------- Metric ---------------
        self.metric_token = nn.Parameter(torch.randn(1, 1, 2*self.dec_embed_dim))
        self.metric_decoder = ContextOnlyTransformerDecoder(
            in_dim=2*self.dec_embed_dim, 
            dec_embed_dim=512,
            dec_num_heads=8,                # 8
            out_dim=512,
            rope=self.rope,
            # use_checkpoint=self.gradient_checkpointing
            use_checkpoint=False
        )
        self.metric_head = nn.Linear(512, 1)
        nn.init.normal_(self.metric_token, std=1e-6)


        ## -------------- Conf ------------------
        self.conf_decoder = TransformerDecoder(
            in_dim=2*self.dec_embed_dim, 
            dec_embed_dim=1024,
            dec_num_heads=16,                # 8
            out_dim=1024,
            rope=self.rope,
            use_checkpoint=self.gradient_checkpointing
            # use_checkpoint=False
        )
        self.conf_head = ConvHead(
            num_features=4, 
            dim_in=dec_embed_dim,
            # projects=nn.Linear(1024, 1024),
            projects=nn.Identity(),
            dim_out=[1], 
            dim_proj=1024,
            dim_upsample=[256, 128, 64],
            dim_times_res_block_hidden=2,
            num_res_blocks=2,
            res_block_norm='group_norm',
            last_res_blocks=0,
            last_conv_channels=32,
            last_conv_size=1,
            using_uv=True
        )

        # For ImageNet Normalize
        image_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        image_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

        self.register_buffer("image_mean", image_mean)
        self.register_buffer("image_std", image_std)


    def wrap_module_with_gradient_checkpointing(self, module: nn.Module):
        class _CheckpointingWrapper(module.__class__):
            _restore_cls = module.__class__
            def forward(self, *args, **kwargs):
                return checkpoint(super().forward, *args, use_reentrant=False, **kwargs)
        module.__class__ = _CheckpointingWrapper
        return module
    

    def disable_multimodal(self, free_cuda_cache: bool = True):
        """
        Disables multimodal branches and releases their modules/parameters.
        Use this when no multimodal conditions are provided.
        """
        self.use_multimodal = False
        for attr in ("depth_encoder", "depth_emb", "ray_embed", "pose_inject_blk"):
            if hasattr(self, attr):
                delattr(self, attr)

        if free_cuda_cache and torch.cuda.is_available():
            torch.cuda.empty_cache()


    def forward(
        self,
        imgs,
        depths=None,
        intrinsics=None,
        rays=None,
        poses=None,
        pose_mask=None,
        with_prior=None,
        overall_prob=1.0,
        ray_dirs_prob=0.0,
        depth_prob=0.0,
        cam_prob=0.0,
    ):
        """
        Forward pass with optional multimodal conditions.

        Args:
            imgs (torch.Tensor): Input RGB images valued in [0, 1].
                Shape: (B, N, 3, H, W).
            intrinsics (torch.Tensor, optional): Camera intrinsic matrices.
                Shape: (B, N, 3, 3).
                Values are in pixel coordinates (not normalized).
            rays (torch.Tensor, optional): Pre-computed ray directions (unit vectors).
                Shape: (B, N, H, W, 3).
                Can replace `intrinsics` as a geometric condition.
            poses (torch.Tensor, optional): Camera-to-World matrices.
                Shape: (B, N, 4, 4).
                Coordinate system: OpenCV convention (Right-Down-Forward).
            depths (torch.Tensor, optional): Ground truth or prior depth maps.
                Shape: (B, N, H, W).
                Invalid values (e.g., sky or missing data) should be set to 0.
            mask_add_depth (torch.Tensor, optional): Mask for depth condition.
                Shape: (B, N, N).
            mask_add_ray (torch.Tensor, optional): Mask for ray/intrinsic condition.
                Shape: (B, N, N).
            mask_add_pose (torch.Tensor, optional): Mask for pose condition.
                Shape: (B, N, N).
                Note: Requires at least two frames to be True to establish a meaningful
                coordinate system (absolute pose for a single frame provides no relative constraint).

        Returns:
            dict: Model outputs containing 'points', 'conf', etc.
        """
        imgs = (imgs - self.image_mean) / self.image_std

        B, N, _, H, W = imgs.shape
        patch_h, patch_w = H // 14, W // 14

        # encode
        hidden, poses_, use_depth_mask, use_pose_mask, norm_factor = self.encode(
            imgs,
            with_prior=with_prior,
            depths=depths,
            intrinsics=intrinsics,
            poses=poses,
            pose_mask=pose_mask,
            rays=rays,
            overall_prob=overall_prob,
            ray_dirs_prob=ray_dirs_prob,
            depth_prob=depth_prob,
            cam_prob=cam_prob,
        )
        hidden = hidden.reshape(B, N, -1, self.dec_embed_dim)

        # decode
        hidden, pos = self.decode(hidden, N, H, W, poses_, use_pose_mask)

        # # head
        outputs = self.forward_head(hidden, pos, B, N, H, W, patch_h, patch_w)

        return outputs
    
    def encode(
        self,
        imgs,
        with_prior=None,
        depths=None,
        rays=None,
        intrinsics=None,
        poses=None,
        pose_mask=None,
        overall_prob=1.0,
        ray_dirs_prob=0.0,
        depth_prob=0.0,
        cam_prob=0.0,
    ):
        B, N, _, H, W = imgs.shape
        device = imgs.device

        # encode by dinov2
        imgs = imgs.reshape(B*N, _, H, W)
        hidden = self.encoder(imgs, is_training=True)["x_norm_patchtokens"]

        if self.use_multimodal:
            with torch.amp.autocast(device_type='cuda', enabled=False):

                if with_prior is False:
                    p_depth = 0.0
                    p_ray = 0.0
                    p_pose = 0.0
                else:
                    # with_prior=None: training mode, sample according to task probabilities.
                    # with_prior=True: explicit prior mode, also use provided task probabilities.
                    # This is different from the original public inference code, but matches
                    # the training need where each task has its own prior probabilities.
                    if torch.rand(1, device=device) < float(overall_prob):
                        p_depth = float(depth_prob)
                        p_ray = float(ray_dirs_prob)
                        p_pose = float(cam_prob)
                    else:
                        p_depth = 0.0
                        p_ray = 0.0
                        p_pose = 0.0

                if depths is None:
                    p_depth = 0.0
                    depths = torch.zeros((B, N, H, W), device=imgs.device)

                if rays is not None:
                    rays = rays[..., :2] / (rays[..., 2:3] + 1e-6)
                else:
                    if intrinsics is None:
                        p_ray = 0.0
                        rays = torch.zeros((B, N, H, W, 2), device=imgs.device)
                    else:
                        pix = torch.from_numpy(get_pixel(H, W).T.reshape(H, W, 3)).to(device).float()[None].repeat(B, 1, 1, 1)
                        rays = torch.einsum('bnij, bhwj -> bnhwi', torch.inverse(intrinsics), pix)[..., :2]
                        # rays = F.normalize(rays, dim=-1).reshape(B, N, H, W, 3)                   # don't normalize, so the pred['xy'] is the same as input rays

                if poses is None:
                    p_pose = 0.0
                    poses = torch.eye(4, device=device)[None, None].repeat(B, N, 1, 1)
                else:
                    assert rays is not None                     # rays should be along with poses
                    
                mask_add_depth = torch.rand((B, N), device=device) <= p_depth
                mask_add_ray = torch.rand((B, N), device=device) <= p_ray
                if pose_mask is not None:
                    if pose_mask.shape != (B, N):
                        raise ValueError("pose_mask must have shape [B, N]")
                    mask_add_pose = pose_mask.to(device=device, dtype=torch.bool)
                else:
                    mask_add_pose = torch.rand((B, N), device=device) <= p_pose

                # pose is injected relatively. so at least two frame should be true.
                num_valid_pose = mask_add_pose.sum(dim=1)
                bad_indices = (num_valid_pose == 1)
                mask_add_pose[bad_indices] = False

                # normalize depth and pose
                normalized_depths, dep_median = self.normalize_depth(depths, method='mean')
                scale_aug = 0.8 + torch.rand((B,), device=device) * 0.4
                normalized_depths /= scale_aug.view(B, 1, 1, 1)
                dep_median *= scale_aug

                depths_masks = (normalized_depths > 0).float()
                depths_masks = depths_masks.reshape(B*N, 1, H, W)

                if pose_mask is not None:
                    # Choose the first actually available pose as reference;
                    # masked/missing poses must not define the coordinate frame.
                    first_valid = mask_add_pose.float().argmax(dim=1)
                    ref_poses = poses[torch.arange(B, device=device), first_valid]
                else:
                    ref_poses = poses[:, 0]
                poses_ = torch.einsum('bij, bnjk -> bnik', se3_inverse(ref_poses), poses)
                poses_[..., :3, 3] /= dep_median.view(B, 1, 1)

                # noramlize for the batch not using depth
                use_depth_batch_mask = mask_add_depth.sum(dim=1) > 0
                if (~use_depth_batch_mask).sum() > 0 and N > 1:
                    if pose_mask is not None:
                        pose_dist = poses_[..., :3, 3].norm(dim=-1)
                        pose_valid = mask_add_pose.to(pose_dist.dtype)
                        pose_count = pose_valid.sum(dim=1)
                        pose_scale = (pose_dist * pose_valid).sum(dim=1) / pose_count.clamp_min(1.0)
                        static_threshold = 2e-2
                        masked_max = (pose_dist * pose_valid).max(dim=1)[0]
                        is_static_mask = masked_max < static_threshold
                        has_pose_baseline = pose_count >= 2
                    else:
                        pose_scale_per_view = poses_[..., 1:, :3, 3].norm(dim=-1)
                        static_threshold = 2e-2
                        is_static_mask = pose_scale_per_view.max(dim=1)[0] < static_threshold
                        pose_scale = pose_scale_per_view.mean(dim=1)
                        has_pose_baseline = torch.ones(B, dtype=torch.bool, device=device)
                    scale_aug = 0.8 + torch.rand((B,), device=device) * 0.4
                    pose_scale *= scale_aug

                    final_moving_mask = (
                        (~use_depth_batch_mask) & (~is_static_mask) & has_pose_baseline
                    )
                    poses_[final_moving_mask, ..., :3, 3] /= (pose_scale.view(B, 1, 1)[final_moving_mask] + 1e-8)
                    normalized_depths[final_moving_mask] /= (pose_scale.view(B, 1, 1, 1)[final_moving_mask] + 1e-8)

                    dep_median[final_moving_mask] *= pose_scale[final_moving_mask]
                
                # if with_prior is None:
                #     add_noise_batch = torch.rand(B) > 0.5
                # else:
                #     add_noise_batch = torch.rand(B) > 1   

                # if N > 1:
                #     poses_[add_noise_batch, 1:] = add_randomized_smooth_pose_noise_torch(poses_[add_noise_batch, 1:])

                normalized_depths = normalized_depths.reshape(B*N, 1, H, W)

            if mask_add_depth.sum() > 0:
                depth_emb = self.depth_encoder(
                    torch.cat([normalized_depths, depths_masks], dim=1),
                    is_training=True,
                )["x_norm_patchtokens"] + self.depth_emb
            else:
                depth_emb = torch.zeros_like(hidden)

            if mask_add_ray.sum() > 0:
                ray_emb = self.ray_embed(
                    rays.reshape(B * N, H, W, 2).permute(0, 3, 1, 2)
                )
            else:
                ray_emb = torch.zeros_like(hidden)
            
            
            use_depth_mask = mask_add_depth
            use_pose_mask = mask_add_pose

            # hidden = hidden + ray_emb * mask_add_ray.reshape(B*N, 1, 1)
            # hidden = hidden + depth_emb * mask_add_depth.reshape(B*N, 1, 1)

            depth_mask = mask_add_depth.reshape(B * N, 1, 1)
            ray_mask = mask_add_ray.reshape(B * N, 1, 1)
            
            depth_emb = torch.where(depth_mask, depth_emb, torch.zeros_like(depth_emb))
            ray_emb = torch.where(ray_mask, ray_emb, torch.zeros_like(ray_emb))

            hidden = hidden + ray_emb
            hidden = hidden + depth_emb

            return hidden, poses_, use_depth_mask, use_pose_mask, dep_median
        
        return hidden, None, None, None, None
    
    def _chunked_conv_head(self, head, feat, patch_h, patch_w, chunk_size=64):
        BN = feat.shape[0]
        if BN <= chunk_size:
            return head(feat, patch_h=patch_h, patch_w=patch_w)
        outputs = [[] for _ in range(len(head.output_block))] if isinstance(head.output_block, nn.ModuleList) else []
        for i in range(0, BN, chunk_size):
            chunk_out = head(feat[i:i+chunk_size], patch_h=patch_h, patch_w=patch_w)
            if isinstance(chunk_out, list):
                for j, o in enumerate(chunk_out):
                    outputs[j].append(o)
            else:
                outputs.append(chunk_out)
        if isinstance(outputs[0], list):
            return [torch.cat(parts, dim=0) for parts in outputs]
        return torch.cat(outputs, dim=0)

    def forward_head(self, hidden, pos, B, N, H, W, patch_h, patch_w):
        device = hidden.device
        hw = patch_h*patch_w+self.patch_start_idx

        # decode point
        ret_point = self.point_decoder(hidden, xpos=pos)

        # decode camera
        ret_camera = self.camera_decoder(hidden, xpos=pos)

        # decode metric
        pos_hw = pos.reshape(B, N*hw, -1)
        ret_metric = self.metric_decoder(self.metric_token.repeat(B, 1, 1), hidden.reshape(B, N*hw, -1), xpos=pos_hw[:, 0:1], ypos=pos_hw)

        # decode conf
        ret_conf = self.conf_decoder(hidden, xpos=pos)

        with torch.amp.autocast(device_type='cuda', enabled=False):
            point_feat = ret_point[:, self.patch_start_idx:].float()
            xy, z = self._chunked_conv_head(self.point_head, point_feat, patch_h, patch_w)
            del point_feat

            # xy = xy.permute(0, 2, 3, 1).reshape(B, N, H, W, -1)
            # z = z.permute(0, 2, 3, 1).reshape(B, N, H, W, -1)

            # # z = torch.exp(z.clamp(max=15.0))
            # z = torch.nan_to_num(z.float(), nan=0.0, posinf=15.0, neginf=-15.0)
            # z = torch.exp(z.clamp(min=-15.0, max=15.0))

            # local_points = torch.cat([xy * z, z], dim=-1)
            # rays = F.normalize(torch.cat([xy, torch.ones_like(z)], dim=-1), dim=-1)


            xy = xy.permute(0, 2, 3, 1).reshape(B, N, H, W, -1)
            z = z.permute(0, 2, 3, 1).reshape(B, N, H, W, -1)

            xy = torch.nan_to_num(xy.float(), nan=0.0, posinf=1e4, neginf=-1e4,)
            xy = xy.clamp(min=-1e4, max=1e4)

            z = torch.nan_to_num(z.float(), nan=0.0, posinf=15.0, neginf=-15.0,)
            z = torch.exp(z.clamp(min=-15.0, max=15.0))

            local_points = torch.cat([xy * z, z], dim=-1)
            local_points = torch.nan_to_num(local_points,nan=0.0, posinf=1e6, neginf=-1e6,)

            ray_input = torch.cat([xy, torch.ones_like(z)], dim=-1)
            ray_input = torch.nan_to_num(ray_input, nan=0.0, posinf=1e4, neginf=-1e4)
            rays = F.normalize(ray_input, dim=-1, eps=1e-6)
            rays = torch.nan_to_num(rays, nan=0.0, posinf=1.0, neginf=-1.0)

            camera_poses = self.camera_head(ret_camera[:, self.patch_start_idx:].float(), patch_h, patch_w).reshape(B, N, 4, 4)

            # metric = self.metric_head(ret_metric.float()).reshape(B).exp()
            metric_log = self.metric_head(ret_metric.float()).reshape(B)
            metric_log = torch.nan_to_num(metric_log, nan=0.0, posinf=10.0, neginf=-10.0)
            metric = metric_log.clamp(min=-10.0, max=10.0).exp()

            # conf
            conf_feat = ret_conf[:, self.patch_start_idx:].float()
            conf = self._chunked_conv_head(self.conf_head, conf_feat, patch_h, patch_w)[0]
            del conf_feat
            conf = conf.permute(0, 2, 3, 1).reshape(B, N, H, W, -1)

            # # points
            # points = torch.einsum('bnij, bnhwj -> bnhwi', camera_poses, homogenize_points(local_points))[..., :3] * metric.view(B, 1, 1, 1, 1)

            # # convert camera poses to metric
            # camera_poses[..., :3, 3] = camera_poses[..., :3, 3] * metric.view(B, 1, 1)

            # # convert local_points to metric
            # local_points = local_points * metric.view(B, 1, 1, 1, 1)

            # convert local_points to metric
            metric_points = metric.view(B, 1, 1, 1, 1)
            metric_pose_t = metric.view(B, 1, 1, 1)

            local_points_metric = local_points * metric_points
            local_points_metric = torch.nan_to_num(local_points_metric, nan=0.0, posinf=1e6, neginf=-1e6,)

            # build metric camera poses without inplace modification
            camera_R = camera_poses[..., :3, :3]
            camera_t = camera_poses[..., :3, 3:4] * metric_pose_t
            camera_bottom = camera_poses[..., 3:4, :]

            camera_poses_metric = torch.cat(
                [
                    torch.cat([camera_R, camera_t], dim=-1),
                    camera_bottom,
                ],
                dim=-2,
            )
            camera_poses_metric = torch.nan_to_num(camera_poses_metric,nan=0.0, posinf=1e6, neginf=-1e6,)

            # points in metric world frame
            points = torch.einsum(
                'bnij, bnhwj -> bnhwi',
                camera_poses_metric,
                homogenize_points(local_points_metric),
            )[..., :3]

            points = torch.nan_to_num(points, nan=0.0, posinf=1e6, neginf=-1e6,)

            camera_poses = camera_poses_metric
            local_points = local_points_metric

        return dict(
            points=points,
            local_points=local_points,
            rays=rays,
            conf=conf,
            camera_poses=camera_poses,  
            metric=metric,
        )


    def decode(self, hidden, N, H, W, poses, use_pose_mask):
        device = hidden.device

        if len(hidden.shape) == 4:
            B, N, hw, _ = hidden.shape
        else:
            BN, hw, _ = hidden.shape
            B = BN // N

        hidden = hidden.reshape(B*N, hw, -1)

        register_token = self.register_token.repeat(B, N, 1, 1).reshape(B*N, *self.register_token.shape[-2:])
        hidden = torch.cat([register_token, hidden], dim=1)
        hw = hidden.shape[1]
        pose_inject_blk_idx = 0

        pos = self.position_getter(B*N, H//self.patch_size, W//self.patch_size, hidden.device)
        if self.patch_start_idx > 0:
            # do not use position embedding for special tokens (camera and register tokens)
            # so set pos to 0 for the special tokens
            pos_patch = pos + 1
            pos_special = torch.zeros(B * N, self.patch_start_idx, 2).to(hidden.device).to(pos.dtype)
            pos = torch.cat([pos_special, pos_patch], dim=1)

        if self.use_multimodal:
            if use_pose_mask.sum() == B * N:
                pose_inject_mask = None
            else:
                view_interaction_mask = use_pose_mask.unsqueeze(2) & use_pose_mask.unsqueeze(1)
                token_interaction_mask = view_interaction_mask.repeat_interleave(hw - self.patch_start_idx, dim=1)
                token_interaction_mask = token_interaction_mask.repeat_interleave(hw - self.patch_start_idx, dim=2)
                pose_inject_mask = token_interaction_mask[:, None]

        for i in range(len(self.decoder)):
            blk = self.decoder[i]

            if i % 2 == 0:
                pos = pos.reshape(B*N, hw, -1)
                hidden = hidden.reshape(B*N, hw, -1)
            else:
                pos = pos.reshape(B, N*hw, -1)
                hidden = hidden.reshape(B, N*hw, -1)

            # hidden = blk(hidden, xpos=pos)

            do_checkpoint = False
            if self.gradient_checkpointing:
                if self.checkpoint_strategy == 'all':
                    do_checkpoint = True
                elif self.checkpoint_strategy == 'global_only':
                    if i % 2 != 0:
                        do_checkpoint = True

            # if self.training and do_checkpoint:
            #     hidden = checkpoint(blk, hidden, xpos=pos, attn_mask=None, use_reentrant=False)
            # else:
            #     hidden = blk(hidden, xpos=pos)

            pos = pos.to(device=hidden.device, dtype=torch.long).contiguous().detach().clone()
            if self.training and do_checkpoint:
                def run_blk(x, xpos, blk=blk):
                    return blk(x, xpos=xpos)
                hidden = checkpoint(run_blk, hidden, pos, use_reentrant=False)
            else:
                hidden = blk(hidden, xpos=pos)

            if self.use_multimodal:
                if i in [1, 9, 17, 25, 33] and use_pose_mask.sum() > 0:
                    hidden = hidden.reshape(B, N, -1, 1024)
                    poses_feat = self.pose_inject_blk[pose_inject_blk_idx](hidden[..., self.patch_start_idx:, :].reshape(B, N*(hw-self.patch_start_idx), -1), poses, H, W, H//14, W//14, attn_mask=pose_inject_mask).reshape(B, N, -1, 1024)
                    # hidden[..., self.patch_start_idx:, :] += poses_feat * use_pose_mask.view(B, N, 1, 1)

                    patch_hidden = hidden[..., self.patch_start_idx:, :]
                    patch_hidden = patch_hidden + poses_feat * use_pose_mask.view(B, N, 1, 1)
                    hidden = torch.cat([hidden[..., :self.patch_start_idx, :], patch_hidden], dim=2)

                    hidden = hidden.reshape(B, N*hw, -1)
                    pose_inject_blk_idx += 1

            if i == len(self.decoder) - 2:
                temp_features = hidden.clone().reshape(B*N, hw, -1)

        concatenated = torch.cat((temp_features, hidden.reshape(B*N, hw, -1)), dim=-1)

        return concatenated, pos.reshape(B*N, hw, -1)
    
    
    def normalize_depth(self, depths: torch.Tensor, method: str = 'median') -> tuple[torch.Tensor, torch.Tensor]:
        """
        Normalizes a batch of depth maps using either median or mean normalization.

        Args:
            depths (torch.Tensor): A batch of depth maps with shape [B, N, H, W].
                                Non-positive values are treated as invalid depth data.
            method (str, optional): The normalization method to use.
                                    Can be 'median' or 'mean'. Defaults to 'median'.

        Returns:
            tuple[torch.Tensor, torch.Tensor]: A tuple containing:
                - The normalized depth maps.
                - The normalization factors (medians or means) used for each batch element.

        Raises:
            ValueError: If the method is not 'median' or 'mean'.
        """
        # 确保输入是 torch.Tensor
        if not isinstance(depths, torch.Tensor):
            depths = torch.tensor(depths, dtype=torch.float32)

        if method not in ['median', 'mean']:
            raise ValueError(f"Invalid normalization method: '{method}'. Choose 'median' or 'mean'.")

        B, N, H, W = depths.shape
        epsilon = 1e-8

        # Create a mask for valid depth values (positive values)
        valid_depths = torch.where(depths > 0, depths, float('nan'))
        valid_depths_reshaped = valid_depths.view(B, -1)

        if method == 'median':
            # Calculate the median for each depth map in the batch
            factors, _ = torch.nanmedian(valid_depths_reshaped, dim=1)
        elif method == 'mean':
            # Calculate the mean for each depth map in the batch
            factors = torch.nanmean(valid_depths_reshaped, dim=1)
        
        # Handle cases where all values might be NaN (e.g., all depths are 0 or negative)
        # In such cases, use 1.0 as the normalization factor to prevent division by zero.
        factors = torch.nan_to_num(factors, nan=1.0)
        
        # Reshape factors for broadcasting during division
        factors_for_division = factors.view(B, 1, 1, 1)

        # Perform normalization, adding a small epsilon to prevent division by zero
        normalized_depths = depths / (factors_for_division + epsilon)

        return normalized_depths, factors.reshape(-1)


In [ ]:
%%writefile sih3d/viewer.py
"""Writes a single self-contained viewer.html: three.js (loaded from a CDN,
the only external dependency — everything else is embedded inline as
base64 so the file works standalone regardless of how it's downloaded off
Kaggle, without relying on relative fetch() of sibling files, which file://
URLs frequently block).

Embeds: the point cloud (positions/colors/confidence as base64-encoded
typed arrays — far more compact than a JSON number array), the textured/
vertex-colored mesh as a base64 GLB blob (parsed via GLTFLoader.parse from
an ArrayBuffer, no network fetch needed), and both trajectory tracks as
small local-ENU-meter coordinate arrays (not the GeoJSON/KML lon-lat
versions, which are for GIS tools, not directly usable in the same 3D
scene as the point cloud/mesh without reprojecting).
"""

from __future__ import annotations

import base64
import json
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np

from .events import EventBus
from .fusion import FusedPointCloud


@dataclass
class ExportStatus:
    name: str
    path: Path | None
    ok: bool
    skipped_reason: str | None = None
    size_bytes: int = 0
    timing_s: float = 0.0


def _b64(arr: np.ndarray) -> str:
    return base64.b64encode(np.ascontiguousarray(arr).tobytes()).decode("ascii")


def write_viewer_html(
    out_path: Path,
    bus: EventBus,
    *,
    cloud: FusedPointCloud | None,
    mesh_glb_path: Path | None,
    gps_track_enu: list[tuple[float, float, float]] | None,
    camera_track_enu: list[tuple[float, float, float]] | None,
    georeferenced: bool,
    max_points: int = 200_000,
) -> ExportStatus:
    t0 = time.time()

    points_b64 = colors_b64 = conf_b64 = ""
    n_points = 0
    if cloud is not None and len(cloud.points):
        pts, cols, conf = cloud.points, cloud.colors, cloud.confidence
        if len(pts) > max_points:
            idx = np.random.default_rng(0).choice(len(pts), size=max_points, replace=False)
            pts, cols, conf = pts[idx], cols[idx], conf[idx]
        n_points = len(pts)
        points_b64 = _b64(pts.astype(np.float32))
        colors_b64 = _b64(np.clip(cols, 0, 255).astype(np.uint8))
        conf_max = float(conf.max()) if len(conf) else 1.0
        conf_norm = (conf / conf_max) if conf_max > 0 else conf
        conf_b64 = _b64(conf_norm.astype(np.float32))

    mesh_b64 = ""
    if mesh_glb_path is not None and Path(mesh_glb_path).exists():
        mesh_b64 = base64.b64encode(Path(mesh_glb_path).read_bytes()).decode("ascii")

    gps_json = json.dumps([list(p) for p in (gps_track_enu or [])])
    cam_json = json.dumps([list(p) for p in (camera_track_enu or [])])

    html = _TEMPLATE.format(
        n_points=n_points, points_b64=points_b64, colors_b64=colors_b64, conf_b64=conf_b64,
        mesh_b64=mesh_b64, gps_json=gps_json, cam_json=cam_json,
        georeferenced_label="Georeferenced (metric, CRS-aligned)" if georeferenced else "APPROXIMATE SCALE — NOT GEOREFERENCED",
        georeferenced_color="#2ea043" if georeferenced else "#d29922",
    )
    out_path.write_text(html)
    size = out_path.stat().st_size
    bus.log(f"Wrote {out_path.name}: {n_points} points embedded, mesh={'yes' if mesh_b64 else 'no'} ({size / 1e6:.1f} MB)")
    return ExportStatus(name="viewer.html", path=out_path, ok=True, size_bytes=size, timing_s=time.time() - t0)


_TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>SIH26158 3D Viewer</title>
<style>
  html, body {{ margin:0; height:100%; background:#0b0f14; color:#e6edf3; font-family: -apple-system, sans-serif; overflow:hidden; }}
  #canvas-wrap {{ position:absolute; inset:0; }}
  #hud {{ position:absolute; top:10px; left:10px; z-index:10; background:rgba(13,17,23,0.85);
          border:1px solid #30363d; border-radius:8px; padding:10px 14px; max-width:340px; }}
  #hud h1 {{ font-size:14px; margin:0 0 8px 0; }}
  #hud label {{ display:block; font-size:12px; margin:6px 0 2px; }}
  #hud select, #hud button {{ font-size:12px; padding:3px 6px; margin-right:4px; }}
  .badge {{ display:inline-block; padding:2px 8px; border-radius:4px; background:{georeferenced_color}; font-size:11px; margin-bottom:6px; }}
  #measure-readout {{ font-size:12px; margin-top:6px; min-height:16px; color:#79c0ff; }}
</style>
</head>
<body>
<div id="canvas-wrap"></div>
<div id="hud">
  <h1>SIH26158 — 3D Model Viewer</h1>
  <div class="badge">{georeferenced_label}</div>
  <div>Points embedded: {n_points}</div>
  <label>Point color</label>
  <select id="color-mode">
    <option value="rgb">RGB</option>
    <option value="confidence">Confidence</option>
  </select>
  <label>Overlays</label>
  <button id="toggle-points">Toggle points</button>
  <button id="toggle-mesh">Toggle mesh</button>
  <button id="toggle-trajectory">Toggle trajectory</button>
  <label>Measure</label>
  <select id="measure-mode">
    <option value="none">Off</option>
    <option value="distance">Distance</option>
    <option value="height">Height</option>
    <option value="area">Area (ground-projected)</option>
  </select>
  <button id="measure-reset">Reset</button>
  <div id="measure-readout"></div>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r158/three.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/three@0.158.0/examples/js/controls/OrbitControls.js"></script>
<script src="https://cdn.jsdelivr.net/npm/three@0.158.0/examples/js/loaders/GLTFLoader.js"></script>
<script>
const POINTS_B64 = "{points_b64}";
const COLORS_B64 = "{colors_b64}";
const CONF_B64 = "{conf_b64}";
const MESH_B64 = "{mesh_b64}";
const GPS_TRACK = {gps_json};
const CAM_TRACK = {cam_json};

function b64ToArrayBuffer(b64) {{
  if (!b64) return new ArrayBuffer(0);
  const bin = atob(b64);
  const buf = new Uint8Array(bin.length);
  for (let i = 0; i < bin.length; i++) buf[i] = bin.charCodeAt(i);
  return buf.buffer;
}}

const scene = new THREE.Scene();
scene.background = new THREE.Color(0x0b0f14);
const camera = new THREE.PerspectiveCamera(60, window.innerWidth / window.innerHeight, 0.01, 100000);
camera.position.set(20, 20, 20);
const renderer = new THREE.WebGLRenderer({{ antialias: true }});
renderer.setSize(window.innerWidth, window.innerHeight);
document.getElementById("canvas-wrap").appendChild(renderer.domElement);
scene.add(new THREE.AmbientLight(0xffffff, 0.8));
const dirLight = new THREE.DirectionalLight(0xffffff, 0.6);
dirLight.position.set(1, 2, 1);
scene.add(dirLight);
scene.add(new THREE.AxesHelper(5));

const controls = new THREE.OrbitControls(camera, renderer.domElement);
controls.enableDamping = true;

// -- point cloud --------------------------------------------------------
let pointCloud = null;
let rgbColors = null, confColors = null;
{{
  const posBuf = new Float32Array(b64ToArrayBuffer(POINTS_B64));
  if (posBuf.length > 0) {{
    const colorBytes = new Uint8Array(b64ToArrayBuffer(COLORS_B64));
    const confFloats = new Float32Array(b64ToArrayBuffer(CONF_B64));
    rgbColors = new Float32Array(colorBytes.length);
    for (let i = 0; i < colorBytes.length; i++) rgbColors[i] = colorBytes[i] / 255.0;
    confColors = new Float32Array(confFloats.length * 3);
    for (let i = 0; i < confFloats.length; i++) {{
      const v = confFloats[i];
      // simple blue->yellow->red ramp
      confColors[i*3]   = Math.min(1, v * 2);
      confColors[i*3+1] = Math.min(1, 2 - v * 2);
      confColors[i*3+2] = Math.max(0, 1 - v * 2);
    }}
    const geo = new THREE.BufferGeometry();
    geo.setAttribute("position", new THREE.BufferAttribute(posBuf, 3));
    geo.setAttribute("color", new THREE.BufferAttribute(rgbColors.slice(), 3));
    const mat = new THREE.PointsMaterial({{ size: 0.03, vertexColors: true }});
    pointCloud = new THREE.Points(geo, mat);
    scene.add(pointCloud);

    const box = new THREE.Box3().setFromObject(pointCloud);
    const center = box.getCenter(new THREE.Vector3());
    const size = box.getSize(new THREE.Vector3()).length();
    camera.position.copy(center).add(new THREE.Vector3(size * 0.6, size * 0.6, size * 0.6));
    controls.target.copy(center);
    controls.update();
  }}
}}

document.getElementById("color-mode").addEventListener("change", (e) => {{
  if (!pointCloud) return;
  const arr = e.target.value === "confidence" ? confColors : rgbColors;
  pointCloud.geometry.setAttribute("color", new THREE.BufferAttribute(arr.slice(), 3));
}});

// -- mesh ------------------------------------------------------------------
let meshObject = null;
if (MESH_B64) {{
  const loader = new THREE.GLTFLoader();
  loader.parse(b64ToArrayBuffer(MESH_B64), "", (gltf) => {{
    meshObject = gltf.scene;
    scene.add(meshObject);
  }}, (err) => console.error("GLB parse failed", err));
}}

// -- trajectory overlay ------------------------------------------------
let trajectoryGroup = new THREE.Group();
function addTrack(points, color) {{
  if (!points || points.length < 2) return;
  const pts = points.map(p => new THREE.Vector3(p[0], p[1], p[2]));
  const geo = new THREE.BufferGeometry().setFromPoints(pts);
  const mat = new THREE.LineBasicMaterial({{ color, linewidth: 2 }});
  trajectoryGroup.add(new THREE.Line(geo, mat));
}}
addTrack(GPS_TRACK, 0x8b949e);
addTrack(CAM_TRACK, 0x2ea043);
scene.add(trajectoryGroup);

// -- toggles -------------------------------------------------------------
document.getElementById("toggle-points").onclick = () => {{ if (pointCloud) pointCloud.visible = !pointCloud.visible; }};
document.getElementById("toggle-mesh").onclick = () => {{ if (meshObject) meshObject.visible = !meshObject.visible; }};
document.getElementById("toggle-trajectory").onclick = () => {{ trajectoryGroup.visible = !trajectoryGroup.visible; }};

// -- measurement tool ---------------------------------------------------
const raycaster = new THREE.Raycaster();
raycaster.params.Points.threshold = 0.1;
const mouse = new THREE.Vector2();
let measureMode = "none";
let measurePoints = [];
let measureMarkers = new THREE.Group();
scene.add(measureMarkers);

document.getElementById("measure-mode").addEventListener("change", (e) => {{
  measureMode = e.target.value;
  resetMeasure();
}});
document.getElementById("measure-reset").onclick = resetMeasure;

function resetMeasure() {{
  measurePoints = [];
  measureMarkers.clear();
  document.getElementById("measure-readout").textContent = "";
}}

function pickPoint(event) {{
  const rect = renderer.domElement.getBoundingClientRect();
  mouse.x = ((event.clientX - rect.left) / rect.width) * 2 - 1;
  mouse.y = -((event.clientY - rect.top) / rect.height) * 2 + 1;
  raycaster.setFromCamera(mouse, camera);
  const targets = [pointCloud, meshObject].filter(Boolean);
  const hits = raycaster.intersectObjects(targets, true);
  return hits.length ? hits[0].point.clone() : null;
}}

function polygonAreaGroundProjected(points) {{
  // Shoelace formula on the XY (ground) projection — the standard
  // convention for aerial/survey area measurement (ground footprint,
  // not slanted 3D surface area).
  let area = 0;
  for (let i = 0; i < points.length; i++) {{
    const a = points[i], b = points[(i + 1) % points.length];
    area += a.x * b.y - b.x * a.y;
  }}
  return Math.abs(area) / 2;
}}

function addMarker(p) {{
  const geo = new THREE.SphereGeometry(0.05, 8, 8);
  const mat = new THREE.MeshBasicMaterial({{ color: 0xffa657 }});
  const marker = new THREE.Mesh(geo, mat);
  marker.position.copy(p);
  measureMarkers.add(marker);
}}

renderer.domElement.addEventListener("dblclick", (event) => {{
  if (measureMode === "none") return;
  const p = pickPoint(event);
  if (!p) return;
  measurePoints.push(p);
  addMarker(p);

  const readout = document.getElementById("measure-readout");
  if (measureMode === "distance" && measurePoints.length >= 2) {{
    const d = measurePoints[measurePoints.length - 2].distanceTo(p);
    readout.textContent = `Distance: ${{d.toFixed(3)}} m`;
  }} else if (measureMode === "height" && measurePoints.length >= 2) {{
    const dz = Math.abs(measurePoints[measurePoints.length - 2].z - p.z);
    readout.textContent = `Height: ${{dz.toFixed(3)}} m`;
  }} else if (measureMode === "area" && measurePoints.length >= 3) {{
    const area = polygonAreaGroundProjected(measurePoints);
    readout.textContent = `Area (${{measurePoints.length}} pts, ground-projected): ${{area.toFixed(2)}} m²`;
  }}
}});

// -- render loop -----------------------------------------------------------
window.addEventListener("resize", () => {{
  camera.aspect = window.innerWidth / window.innerHeight;
  camera.updateProjectionMatrix();
  renderer.setSize(window.innerWidth, window.innerHeight);
}});

function animate() {{
  requestAnimationFrame(animate);
  controls.update();
  renderer.render(scene, camera);
}}
animate();
</script>
</body>
</html>
"""


In [ ]:
# ============================== SETUP: env check, install, detect inputs ====
# One cell, top to bottom: environment checks -> install only what's missing
# -> detect the video/telemetry/checkpoints already attached under
# /kaggle/input -> ready for the Launch cell below. Kaggle's image already
# has torch preinstalled — nothing here reinstalls/pins it (GeoFF3D's own
# pyproject pins torch==2.5.0, exactly the kind of forced-reinstall the task
# spec says to avoid; see PHASE0_NOTES.md section 4).
import importlib
import subprocess
import sys
import time
from pathlib import Path

_t0 = time.time()

# -- 1. environment checks ---------------------------------------------------

def _check_internet(timeout_s: float = 5.0) -> bool:
    import urllib.request
    try:
        urllib.request.urlopen("https://huggingface.co", timeout=timeout_s)
        return True
    except Exception:
        return False

def _check_gpu() -> tuple[bool, str]:
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                              capture_output=True, text=True, timeout=15)
        if out.returncode == 0 and out.stdout.strip():
            return True, out.stdout.strip()
    except Exception as e:
        return False, str(e)
    return False, "nvidia-smi returned no GPUs"

_internet_ok = _check_internet()
_gpu_ok, _gpu_info = _check_gpu()
print(f"Internet: {'ON' if _internet_ok else 'OFF'}")
print(f"GPU(s):\n{_gpu_info if _gpu_ok else '(none detected)'}")

if not _internet_ok:
    raise RuntimeError(
        "Internet is OFF for this notebook session. Enable it under "
        "Notebook Settings -> Internet -> On, then Run All again. "
        "Model weights (MapAnything) and pip installs need it."
    )
if not _gpu_ok:
    print("WARNING: no GPU detected. The pipeline will still run on CPU as a "
          "last-resort fallback, but this will be VERY slow. Enable a GPU "
          "under Notebook Settings -> Accelerator.")

sys.path.insert(0, str(Path.cwd()))

# -- 2. install ---------------------------------------------------------------

from sih3d.io_detect import find_cache_dir  # noqa: E402

_cache_ds = find_cache_dir(Path(INPUT_ROOT))
_pip_extra = ["--find-links", str(_cache_ds)] if _cache_ds is not None else []
if _cache_ds is not None:
    print(f"\nFound a cache dataset at {_cache_ds} — pip will prefer any wheels there and skip re-downloading.")

def _try_import(module_name: str) -> bool:
    try:
        importlib.import_module(module_name)
        return True
    except Exception:
        return False

def _pip_install(spec: str, extra_args: list[str] | None = None, label: str | None = None) -> None:
    label = label or spec
    t0 = time.time()
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + _pip_extra + (extra_args or []) + [spec]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
        elapsed = time.time() - t0
        if result.returncode != 0:
            print(f"  [FAILED, {elapsed:.0f}s] {label}: {result.stderr[-500:]}")
        else:
            print(f"  [OK, {elapsed:.0f}s] {label}")
    except Exception as e:
        print(f"  [FAILED, {time.time()-t0:.0f}s] {label}: {e}")

print("\nChecking/installing dependencies (only what's missing)...")

# Core numeric/vision/geo stack (no torch dependency in any of these).
# Deliberately no anywidget/plotly here — see dashboard.py's module
# docstring: a first real Kaggle run hit "No version of module anywidget
# is registered" because any custom ipywidgets extension needs its JS
# registered with the front end at kernel start, not installable mid-run;
# the dashboard only uses ipywidgets' own built-in widgets now.
_simple_deps = [
    ("scipy", "scipy"), ("xatlas", "xatlas"), ("PIL", "pillow"),
    ("laspy", "laspy"), ("rasterio", "rasterio"), ("pyproj", "pyproj"),
    ("trimesh", "trimesh"), ("open3d", "open3d"), ("pymavlink", "pymavlink"),
    ("ipywidgets", "ipywidgets"),
    ("huggingface_hub", "huggingface_hub"), ("safetensors", "safetensors"),
]
for _mod, _pkg in _simple_deps:
    if not _try_import(_mod):
        _pip_install(_pkg)
    else:
        print(f"  [already present] {_pkg}")

# GPU video decode: deliberately NOT pip-installing torchcodec. A first
# real Kaggle run stalled 20+ minutes in frame extraction (CPU 100%, GPU
# idle) because the generic PyPI torchcodec wheel silently decoded on CPU
# despite claiming a CUDA device — it imports fine either way, so pip
# installing "some torchcodec build" isn't actually useful without also
# pinning an exact CUDA-matching wheel from the PyTorch index, which is
# fragile across Kaggle image updates. decode.py now verifies real GPU
# throughput before trusting torchcodec (using it only if it's already
# importable AND passes that check) and otherwise uses ffmpeg -hwaccel
# cuda directly — no extra install needed, and it's what's actually
# driving GPU decode on this run either way.
if _try_import("torchcodec"):
    print("  [already present] torchcodec (will only be trusted if decode.py's real GPU-throughput check passes)")
else:
    print("  [skipped] torchcodec (relying on ffmpeg -hwaccel cuda instead — see decode.py)")

# Dynamic-object masking (best-effort; masks.py degrades to all-static).
if not _try_import("ultralytics"):
    _pip_install("ultralytics", label="ultralytics (YOLO-seg masking)")

# MapAnything (Backbone C, the default) — a PLAIN pip install, no --no-deps
# and no hand-picked extra-deps list: PHASE0_NOTES.md confirmed its own
# pyproject has no torch/CUDA pin, so letting pip resolve its declared deps
# itself (rather than guessing which ones it needs) is both simpler and
# more correct than maintaining a hand-picked list that can drift from the
# upstream pyproject.
if not _try_import("mapanything"):
    _pip_install("git+https://github.com/facebookresearch/map-anything.git", label="mapanything (Backbone C)")

# Verify MapAnything actually works — import AND construct the real model —
# before the pipeline starts, so a broken install fails here with a clear
# message instead of deep inside the first chunk's geometric_reconstruction
# stage. This also front-loads the pretrained-weights download the pipeline
# needs anyway, so it isn't wasted work.
print("\nVerifying MapAnything (import + model init)...")
try:
    from mapanything.models import MapAnything as _MapAnythingCheck
    _mapanything_model_check = _MapAnythingCheck.from_pretrained("facebook/map-anything-apache")
    del _mapanything_model_check
    print("  [OK] mapanything imports and facebook/map-anything-apache loads")
except Exception as e:
    print(f"  [FAILED] MapAnything verification: {type(e).__name__}: {e}")
    print("  The pipeline will hit this same error in geometric_reconstruction; fix the install above and Run All again.")

print(f"\nInstalls done in {time.time()-_t0:.1f}s. If anything above FAILED, the "
      f"corresponding pipeline stage will log a fallback rather than crash — "
      f"check report.json after the run.")

# -- 3. detect inputs ---------------------------------------------------------

import sih3d.io_detect as io_detect
import sih3d.telemetry as telemetry_mod
importlib.reload(io_detect)
importlib.reload(telemetry_mod)

detected = io_detect.detect_all(Path(INPUT_ROOT))
if detected.video is None:
    raise RuntimeError(
        f"No video file found under {INPUT_ROOT}. Attach a dataset containing "
        f"a drone video (.mp4/.mov/.mkv/.avi/.m4v/.ts) — see RUN_ON_KAGGLE.md."
    )

print(f"\nVideo: {detected.video.path.name} "
      f"({detected.video.width}x{detected.video.height}, {detected.video.fps:.1f}fps, "
      f"{detected.video.duration_s:.0f}s)")
print(f"Telemetry: {detected.telemetry_path or '(none found)'} (kind={detected.telemetry_kind})")
print(f"Intrinsics: {detected.intrinsics_path or '(none — will estimate)'}")
print(f"Fine-tuned checkpoints detected: {detected.checkpoints or '(none)'}")
for w in detected.warnings:
    print(f"  note: {w}")

telemetry = None
if detected.telemetry_path is not None:
    telemetry = telemetry_mod.parse_telemetry(detected.telemetry_path, detected.telemetry_kind)
    print(f"Parsed telemetry: {len(telemetry)} samples ({telemetry.kind}); notes: {telemetry.notes}")
elif detected.telemetry_kind == "embedded":
    telemetry = telemetry_mod.extract_embedded_telemetry(detected.video.path)
    print(f"Parsed embedded telemetry: {len(telemetry)} samples; notes: {telemetry.notes}")

if telemetry is None or len(telemetry) == 0:
    print("\nNO GPS TELEMETRY FOUND. Continuing RGB-only: outputs will be "
          "APPROXIMATE SCALE — NOT GEOREFERENCED. See report.json/viewer.html "
          "for this run's labeling.")

print(f"\nSetup done in {time.time()-_t0:.1f}s total.")


### Publishing a cache dataset (speeds up future runs)

After the first successful run, downloaded wheels and model checkpoints are
copied into `/kaggle/working/cache/`. To skip re-downloading on future runs:

1. In the Kaggle notebook viewer, open the **Data** pane → **Output**.
2. Click **New Dataset** from the `cache/` folder, name it (e.g.
   `sih3d-cache`), and publish it.
3. Add that dataset as an input to this notebook (**Add Input** → search
   your username → select it).
4. On the next run, `find_cache_dir()` will detect it automatically under
   `/kaggle/input/` and skip downloads it already has.


In [ ]:
# ============================== LAUNCH =======================================
import importlib
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import sih3d.events as events_mod
import sih3d.gpu_monitor as gpu_monitor_mod
import sih3d.report as report_mod
import sih3d.dashboard as dashboard_mod
import sih3d.pipeline as pipeline_mod
for _m in (events_mod, gpu_monitor_mod, report_mod, dashboard_mod, pipeline_mod):
    importlib.reload(_m)

from sih3d.events import EventBus
from sih3d.gpu_monitor import GpuMonitor
from sih3d.report import ReportBuilder
from sih3d.dashboard import Dashboard
from sih3d.pipeline import Pipeline, PipelineConfig

bus = EventBus()
gpu_monitor = GpuMonitor(bus, interval_s=0.5)
report = ReportBuilder()

cfg = PipelineConfig(
    mode=MODE, backbone_choice=BACKBONE, prior_mode=PRIOR_MODE,
    use_finetuned=USE_FINETUNED_CHECKPOINT, output_dir=Path(OUTPUT_DIR),
    device0="cuda:0" if gpu_monitor.device_count >= 1 else "cpu",
    device1="cuda:1" if gpu_monitor.device_count >= 2 else ("cuda:0" if gpu_monitor.device_count >= 1 else "cpu"),
)

header_info = {
    "video_name": detected.video.path.name,
    "resolution": f"{detected.video.width}x{detected.video.height}",
    "fps": round(detected.video.fps, 1),
    "duration_s": detected.video.duration_s,
    "telemetry_type": detected.telemetry_kind or "none",
    "telemetry_sample_count": len(telemetry) if telemetry else 0,
    "sync_method": "timestamp" if (telemetry and telemetry.has_timestamps) else "proportional interpolation",
    "mode": MODE, "gpu_count": gpu_monitor.device_count, "gpu_names": gpu_monitor.device_names,
    "backbone": BACKBONE, "prior_mode": PRIOR_MODE,
}

dashboard = Dashboard(header_info, refresh_hz=2.0)
pipeline = Pipeline(cfg, detected, telemetry, bus, gpu_monitor, report)

dashboard.display()
pipeline.start()

# Drain the bus in the main thread, fanning each event out to both the
# dashboard and the report (report.py's own docstring calls this out as
# its intended usage — a queue.Queue can only have one destructive
# consumer, so this loop is that single consumer). render_if_due() throttles
# actual redraws to <=2Hz internally; this loop itself can poll faster.
while pipeline.is_alive():
    for evt in bus.drain():
        dashboard.on_event(evt)
        report.on_event(evt)
    dashboard.render_if_due()
    time.sleep(0.1)

# Final drain after the pipeline thread exits, so nothing published right
# at the end is lost, then one final forced render + report write.
for evt in bus.drain():
    dashboard.on_event(evt)
    report.on_event(evt)
dashboard.render_if_due(force=True)

report.write_json(Path(OUTPUT_DIR) / "report.json")
report.write_html(Path(OUTPUT_DIR) / "report.html", dashboard_snapshot_html=dashboard.snapshot_html())

print(f"\nPipeline finished: success={pipeline.result.success if pipeline.result else 'UNKNOWN'}")


In [ ]:
# ============================== RESULTS ======================================
# Everything here renders INLINE — nothing needs to be downloaded to check
# the run. All the underlying files are still saved under /kaggle/working/
# outputs/ (table + download links at the end of this cell) for later use.
%matplotlib inline
import importlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import sih3d.stage_views as stage_views_mod
importlib.reload(stage_views_mod)
from sih3d.stage_views import (
    render_frame_extraction, render_poses, render_geometric_reconstruction,
    render_large_scale_alignment, render_dense_point_cloud,
    render_mesh_textured_model, render_final_summary,
)
from IPython.display import display, FileLink, HTML

artifacts = pipeline.artifacts

print("=" * 80); print("KEYFRAMES"); print("=" * 80)
render_frame_extraction(artifacts)

print("=" * 80); print("CAMERA TRAJECTORY"); print("=" * 80)
render_poses(artifacts)

print("=" * 80); print("SAMPLE DEPTH / CONFIDENCE MAPS"); print("=" * 80)
render_geometric_reconstruction(artifacts)

print("=" * 80); print("LARGE-SCALE ALIGNMENT"); print("=" * 80)
render_large_scale_alignment(artifacts)

print("=" * 80); print("DENSE POINT CLOUD"); print("=" * 80)
render_dense_point_cloud(artifacts)

print("=" * 80); print("TEXTURED MESH + DSM/ORTHOMOSAIC"); print("=" * 80)
render_mesh_textured_model(artifacts)

print("=" * 80); print("TIMINGS"); print("=" * 80)
render_final_summary(report)

print("\n" + "=" * 80); print("OUTPUT FILES (saved under", OUTPUT_DIR, ")"); print("=" * 80)
out_dir = Path(OUTPUT_DIR)
rows = []
for status in getattr(pipeline, "_export_statuses", []):
    size_mb = (status.size_bytes or 0) / 1e6
    state = "OK" if status.ok else f"SKIPPED ({status.skipped_reason})"
    rows.append(f"<tr><td>{status.name}</td><td>{state}</td><td>{size_mb:.2f} MB</td></tr>")
display(HTML("<table><tr><th>File</th><th>Status</th><th>Size</th></tr>" + "".join(rows) + "</table>"))
for status in getattr(pipeline, "_export_statuses", []):
    if status.ok and status.path is not None:
        display(FileLink(str(status.path)))


In [ ]:
# ============================== DEBUG (run this if anything above failed) ===
# One block to paste back for help: traceback, last 100 log lines, GPU/env
# info, and everything that was auto-detected about the inputs.
import subprocess
import sys

print("=" * 80)
print("PIPELINE RESULT")
print("=" * 80)
if "pipeline" in dir() and pipeline.result is not None:
    print(f"success: {pipeline.result.success}")
    if not pipeline.result.success:
        print(f"error: {pipeline.result.error}")
        print("\ntraceback:")
        print(pipeline.result.traceback)
else:
    print("Pipeline object not found or never finished — check the Launch cell output above.")

print("\n" + "=" * 80)
print(f"LAST {min(len(bus.recent_logs()) if 'bus' in dir() else 0, 100)} LOG LINES")
print("=" * 80)
if "bus" in dir():
    for line in bus.recent_logs()[-100:]:
        print(line)

print("\n" + "=" * 80)
print("GPU / ENVIRONMENT")
print("=" * 80)
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=15).stdout)
except Exception as e:
    print(f"nvidia-smi failed: {e}")
try:
    import torch
    print(f"torch: {torch.__version__}, cuda available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"  GPU{i}: {torch.cuda.get_device_name(i)}, capability {torch.cuda.get_device_capability(i)}")
except Exception as e:
    print(f"torch import failed: {e}")
print(f"python: {sys.version}")

print("\n" + "=" * 80)
print("DETECTED INPUTS")
print("=" * 80)
if "detected" in dir():
    print(f"video: {detected.video}")
    print(f"telemetry_path: {detected.telemetry_path}, kind: {detected.telemetry_kind}")
    print(f"intrinsics_path: {detected.intrinsics_path}")
    print(f"cache_dir: {detected.cache_dir}")
    print(f"checkpoints: {detected.checkpoints}")
    print(f"warnings: {detected.warnings}")
else:
    print("`detected` not found — the input-detection cell may not have run.")

print("\nCopy everything above this line when asking for help.")
